# Module


In [ ]:
# =============================================================================
# Path Configuration
# Set the following directories according to your environment before running
# =============================================================================
import os

DATA_DIR = '/path/to/data'       # Root directory for input data
FIG_DIR  = '/path/to/figures'    # Directory to save output figures
CODE_DIR = '/path/to/code'       # Root directory for code
HOME_DIR = os.path.expanduser('~')


In [ ]:
import numpy as np
import xarray as xr
import xskillscore as xs
import pandas as pd

from netCDF4 import Dataset, MFDataset
import netCDF4

import matplotlib.ticker as mticker
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
from scipy import stats
import shutil
from scipy.spatial import distance
# import metpy

import cartopy.feature as cfeature
import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
from cartopy.util import add_cyclic_point
from cartopy.feature import NaturalEarthFeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter

from dask.diagnostics import ProgressBar
from glob import *
import sys, os, time, warnings
import pytz

warnings.filterwarnings(action='ignore')
warnings.simplefilter(action='ignore')


In [ ]:
import os
def makedirs(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [ ]:
def grid_transfer(data):
    data=data.sortby(data['latitude'], ascending=True) 
    data['longitude'] = xr.where(data['longitude'] > 180, data['longitude'] - 360, data['longitude']) 
    data = data.sortby(data['longitude']) 
    
    return data

In [ ]:
grid_frame = None
import xesmf as xe
def regrid(change_data, target_data):
    global grid_frame
    if grid_frame is None:
        grid_frame = xe.Regridder(change_data, target_data,"bilinear",periodic = False)
    regrid_data = grid_frame(change_data)
    return regrid_data

In [ ]:
ref_grid = grid_transfer(xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/daily_total.nc').rename(lon = 'longitude',lat = 'latitude'))
ref_grid = xr.Dataset(
    coords={
        'longitude': ref_grid.longitude,
        'latitude': ref_grid.latitude
    }
)

# fig


## 1. Observation Station Map


In [ ]:
obs_list = sorted(glob('${DATA_DIR}/ground_based_obs/*/daily/*.nc'))
results_df = []
for nc_file in  obs_list:
    nc = xr.open_dataset(nc_file)
    lon = nc.longitude.item()
    lat = nc.latitude.item()
    use_day = len(nc.time)
    source = nc_file.split('/')[-3]

    results_df.append({
        'longitude': lon,
        'latitude': lat,
        'use_days': use_day,
        'source': source
    })
result_df = pd.DataFrame(results_df)
result_df['source'].value_counts()

In [ ]:
from scipy.stats import gaussian_kde
import numpy as np

def create_source_plot(ax, result_df, extent):
    projection = ccrs.PlateCarree()
    crs = ccrs.PlateCarree()
    
    gl = ax.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=.2, linestyle='-.')
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.FixedLocator(np.arange(-120, 120 + 1, 60))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 61, 20))
    gl.xpadding = 10
    gl.ypadding = 10
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    gl.top_labels = None
    gl.right_labels = None
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"))
    ax.add_feature(cfeature.BORDERS.with_scale("110m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    source_styles = {
        'GSDR':  {'color': '#1f77b4', 'marker': 'o', 'cmap': 'Blues'},
        'HadISD':{'color': '#ff7f0e', 'marker': 's', 'cmap': 'Oranges'},
        'DWD':   {'color': '#2ca02c', 'marker': '^', 'cmap': 'Greens'},
        'MSC':   {'color': '#d62728', 'marker': 'D', 'cmap': 'Reds'},
        'KMA':   {'color': '#9467bd', 'marker': 'v', 'cmap': 'Purples'},
        'CMFD':  {'color': '#8c564b', 'marker': '<', 'cmap': 'copper'},
    }
    
    source_counts = result_df['source'].value_counts()
    
    # density threshold:   KDE ,   
    DENSITY_THRESHOLD = 100  # threshold based on number of stations per source
    
    legend_handles = []
    
    for source in ['GSDR', 'HadISD', 'DWD', 'CMFD', 'MSC', 'KMA']:
        mask = result_df['source'] == source
        count = source_counts.get(source, 0)
        style = source_styles[source]
        label = 'ECCC' if source == 'MSC' else source
        
        lons = result_df.loc[mask, 'longitude'].values
        lats = result_df.loc[mask, 'latitude'].values
        
        if count >= DENSITY_THRESHOLD:
            # KDE   
            xy = np.vstack([lons, lats])
            kde = gaussian_kde(xy, bw_method=0.15)
            
            # generate grid
            lon_grid = np.linspace(extent[0], extent[1], 300)
            lat_grid = np.linspace(extent[2], extent[3], 150)
            XX, YY = np.meshgrid(lon_grid, lat_grid)
            ZZ = kde(np.vstack([XX.ravel(), YY.ravel()])).reshape(XX.shape)
            
            # mask low-density regions (remove noise)
            threshold = ZZ.max() * 0.05
            ZZ_masked = np.where(ZZ > threshold, ZZ, np.nan)
            
            # filled contour (filled)
            levels = np.linspace(ZZ.max() * 0.1, ZZ.max(), 8)
            cf = ax.contourf(XX, YY, ZZ_masked,
                            levels=levels,
                            cmap=style['cmap'],
                            alpha=0.6,
                            transform=crs)
            
            # contour lines
            ax.contour(XX, YY, ZZ_masked,
                      levels=levels,
                      colors=[style['color']],
                      linewidths=0.5,
                      alpha=0.8,
                      transform=crs)
            
            # legend patch
            import matplotlib.patches as mpatches
            handle = mpatches.Patch(color=style['color'], alpha=0.7,
                                   label=f'{label} [{count}]')
        else:
            # sparse regions shown as scatter points
            sc = ax.scatter(lons, lats,
                           s=20,
                           c=style['color'],
                           marker=style['marker'],
                           edgecolor='black',
                           linewidth=0.4,
                           alpha=0.8,
                           transform=crs,
                           zorder=5)
            import matplotlib.lines as mlines
            handle = mlines.Line2D([], [],
                                  color=style['color'],
                                  marker=style['marker'],
                                  linestyle='None',
                                  markersize=7,
                                  markeredgecolor='black',
                                  markeredgewidth=0.4,
                                  label=f'{label} [{count}]')
        
        legend_handles.append(handle)
    
    legend = ax.legend(handles=legend_handles,
                      bbox_to_anchor=(0.45, 0.8),
                      fontsize=12,
                      frameon=True,
                      edgecolor='black',
                      fancybox=False,
                      framealpha=1)
    legend.get_frame().set_linewidth(1.0)
    
    return legend_handles[0]


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-150, 150, -.001, 60.001]
    fig = plt.figure(figsize=(15, 23))
    
    # Source Plot
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    scatter1 = create_source_plot(ax1, result_df, extent)
    plt.show()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/Obs_info.png',format='png', dpi=300,bbox_inches='tight')

if __name__ == "__main__":
    main()

In [ ]:
"""
Station location visualization - 1°×1° grid tile partitioning method
────────────────────────────────────────────────
Rules:
  - 1 source in grid: fill with that color
  - 2+ sources in grid: split grid vertically into n parts, assign color for each source
  - alpha: COUNT_TICKS=[1,4,7,10] stepped by interval, count>=10clipped
  - sparse sources(KMA, MSC/ECCC): shown separately as scatter points
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LongitudeFormatter, LatitudeFormatter
import matplotlib.ticker as mticker
from collections import defaultdict

# ──  ──────────────────────────────────────────────────────────────────────
GRID_SIZE    = 1.0  # grid size ()
ALPHA_MIN    = 0.2
ALPHA_MAX    = 1.0
COUNT_MAX    = 10  #   ALPHA_MAX 
COUNT_TICKS  = [1, 4, 7, 10]  #  tick  alpha  

SOURCE_ORDER = ['GSDR', 'HadISD', 'DWD', 'CMFD', 'MSC', 'KMA']

SOURCE_STYLES = {
    'GSDR':  {'color': '#1f77b4', 'marker': 'o', 'label': 'GSDR'},
    'HadISD':{'color': '#ff7f0e', 'marker': 's', 'label': 'HadISD'},
    'DWD':   {'color': '#2ca02c', 'marker': '^', 'label': 'DWD'},
    'MSC':   {'color': '#d62728', 'marker': 'D', 'label': 'ECCC'},
    'KMA':   {'color': '#9467bd', 'marker': 'v', 'label': 'KMA'},
    'CMFD':  {'color': '#8c564b', 'marker': 's', 'label': 'CMFD'},
}

#    scatter  sparse sources
SCATTER_ONLY = {'KMA', 'MSC'}


# ──   ────────────────────────────────────────────────────────────────

def count_to_alpha(count,
                   alpha_min=ALPHA_MIN, alpha_max=ALPHA_MAX,
                   count_max=COUNT_MAX, count_ticks=COUNT_TICKS):
    """
    COUNT_TICKS stepped by interval alpha Returns.
    clip to ALPHA_MAX when count >= COUNT_MAX.
    e.g. COUNT_TICKS=[1,4,7,10]:
        count in [1,4)  → step 0 → alpha 0.20
        count in [4,7)  → step 1 → alpha 0.47
        count in [7,10) → step 2 → alpha 0.73
        count >= 10     → step 3 → alpha 1.00
    """
    count = min(count, count_max)
    n     = len(count_ticks)

    for step_idx in range(n - 1, -1, -1):
        if count >= count_ticks[step_idx]:
            t = step_idx / (n - 1)
            return alpha_min + t * (alpha_max - alpha_min)

    return alpha_min


def aggregate_to_grid(result_df, grid_size=GRID_SIZE):
    """
    Aggregate to grid_size degree grid excluding SCATTER_ONLY sources.
    Returns: {(grid_center_lon, grid_center_lat): {source: count, ...}, ...}
    """
    df = result_df[~result_df['source'].isin(SCATTER_ONLY)].copy()
    df['grid_lon'] = (np.floor(df['longitude'] / grid_size) * grid_size
                      + grid_size / 2)
    df['grid_lat'] = (np.floor(df['latitude']  / grid_size) * grid_size
                      + grid_size / 2)

    grid_data = defaultdict(lambda: defaultdict(int))
    for _, row in df.iterrows():
        grid_data[(row['grid_lon'], row['grid_lat'])][row['source']] += 1
    return grid_data


def draw_cell(ax, grid_lon, grid_lat, source_count_dict, grid_size=GRID_SIZE):
    """
    Draw single grid cell split vertically by source count.
    Sort by SOURCE_ORDER for consistency.
    """
    sources = [s for s in SOURCE_ORDER
               if s in source_count_dict and s not in SCATTER_ONLY]
    n = len(sources)
    if n == 0:
        return

    lon_left   = grid_lon - grid_size / 2
    lat_bottom = grid_lat - grid_size / 2
    tile_w     = grid_size / n

    for i, src in enumerate(sources):
        count = source_count_dict[src]
        alpha = count_to_alpha(count)
        rect  = mpatches.Rectangle(
            (lon_left + i * tile_w, lat_bottom),
            tile_w, grid_size,
            facecolor=SOURCE_STYLES[src]['color'],
            edgecolor='none',
            alpha=alpha,
            transform=ccrs.PlateCarree(),
            zorder=4,
        )
        ax.add_patch(rect)


def add_alpha_colorbar(ax, grid_data, source_styles, scatter_only, source_order,
                       alpha_min=ALPHA_MIN, alpha_max=ALPHA_MAX,
                       count_ticks=COUNT_TICKS):
    """
    Insert stepped alpha colorbar per source at bottom-left of map.
      - vertical bar for each source color (n_steps intervals)
      -  count  : 1, 4, 7, ≥10
      -  = alpha  /  = alpha 
    """
    active_sources = [s for s in source_order
                      if s not in scatter_only
                      and any(s in cell for cell in grid_data.values())]
    n_src   = len(active_sources)
    n_steps = len(count_ticks)

    if n_src == 0:
        return

    # ── inset axes  ────────────────────────────────────────────────────────
    # total_w:   (axes fraction )
    # : tick label  +  bar
    bar_h   = 0.26  # inset  (axes fraction)
    bar_w   = 0.014  #  bar   (axes fraction)
    bar_gap = 0.003  # bar   (axes fraction)
    pad_l   = 0.040  #  tick label  (axes fraction)
    pad_r   = 0.005

    total_w = pad_l + n_src * bar_w + (n_src - 1) * bar_gap + pad_r

    inset = ax.inset_axes(
        [0.01, 0.04, total_w, bar_h],
        transform=ax.transAxes,
    )

    # inset  : x=[0,1], y=[0, n_steps]
    # x=0 ~ pad_x: tick label 
    # x=pad_x ~  : bar 
    pad_x = 0.25  # inset x  tick   (data , 0~1 )

    inset.set_xlim(0, 1)
    inset.set_ylim(0, n_steps)
    inset.axis('off')

    # bar     x (inset data )
    bar_data_w   = (1.0 - pad_x - 0.02) / n_src  # bar 
    bar_data_gap = bar_data_w * 0.12  # bar 

    # ──    bar ──────────────────────────────────────────────────
    for i, src in enumerate(active_sources):
        color = source_styles[src]['color']
        label = source_styles[src]['label']
        x0    = pad_x + i * (bar_data_w)

        for step in range(n_steps):
            t     = step / (n_steps - 1)
            alpha = alpha_min + t * (alpha_max - alpha_min)

            rect = mpatches.Rectangle(
                (x0, step),
                bar_data_w - bar_data_gap, 1.0,
                facecolor=color,
                edgecolor='gray',
                linewidth=0.4,
                alpha=alpha,
                transform=inset.transData,
                clip_on=False,
            )
            inset.add_patch(rect)

        #   (bar )
        # inset.text(
        #     x0 + (bar_data_w - bar_data_gap) / 2, -0.08,
        #     label,
        #     ha='center', va='top', fontsize=7,
        #     transform=inset.transData,
        # )

    # ── tick  () ──────────────────────────────────────────────────────
    for step, cnt in enumerate(count_ticks):
        y_pos = step

        # tick  (label   → bar )
        inset.plot([pad_x - 0.02, pad_x], [y_pos, y_pos],
                   color='gray', lw=0.7, transform=inset.transData)

        tick_label = f'≥{cnt}' if step == n_steps - 1 else str(cnt)
        inset.text(
            pad_x - 0.03, y_pos, tick_label,
            ha='right', va='center', fontsize=7.5,
            transform=inset.transData,
        )

    x_end = pad_x + n_src * bar_data_w - bar_data_gap
    inset.plot([pad_x, x_end], [n_steps, n_steps],
               color='gray', lw=0.7, transform=inset.transData)

    inset.text(
        (pad_x + x_end) / 2, n_steps + 0.12,
        'Count per grid',
        ha='center', va='bottom', fontsize=8,
        transform=inset.transData,
    )

    # inset 
    inset.set_facecolor('white')
    inset.patch.set_alpha(0.85)
    inset.patch.set_visible(True)


def create_source_plot(ax, result_df, extent):
    """
      .
    ax        : cartopy GeoAxes (PlateCarree projection)
    result_df : 'source', 'longitude', 'latitude'   DataFrame
    extent    : [lon_min, lon_max, lat_min, lat_max]
    """
    crs = ccrs.PlateCarree()

    # ── Gridlines ──────────────────────────────────────────────────────────────
    gl = ax.gridlines(crs=crs, draw_labels=True, linewidth=1,
                      color='gray', alpha=0.2, linestyle='-.')
    gl.xlabel_style = {'size': 10}
    gl.ylabel_style = {'size': 10}
    gl.xlocator = mticker.FixedLocator(np.arange(-120, 121, 60))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 61, 20))
    gl.xpadding = 10
    gl.ypadding = 10
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    gl.top_labels   = None
    gl.right_labels = None

    # ── Map features ──────────────────────────────────────────────────────────
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), lw=0.8,  zorder=6)
    ax.add_feature(cfeature.OCEAN.with_scale('50m'),               zorder=2)
    ax.add_feature(cfeature.BORDERS.with_scale('110m'),  lw=0.3,  zorder=6)
    ax.set_extent(extent, crs=crs)

    # ──   ──────────────────────────────────────────────────────────────
    grid_data = aggregate_to_grid(result_df)

    # ──   ────────────────────────────────────────────────────────────
    for (glon, glat), src_cnt in grid_data.items():
        if not (extent[0] - GRID_SIZE < glon < extent[1] + GRID_SIZE and
                extent[2] - GRID_SIZE < glat < extent[3] + GRID_SIZE):
            continue
        draw_cell(ax, glon, glat, src_cnt)

    # ── Scatter-only  ──────────────────────────────────────────────────────
    source_counts = result_df['source'].value_counts()

    for src in SCATTER_ONLY:
        if src not in source_counts:
            continue
        mask  = result_df['source'] == src
        style = SOURCE_STYLES[src]
        ax.scatter(
            result_df.loc[mask, 'longitude'],
            result_df.loc[mask, 'latitude'],
            s=30, c=style['color'], marker=style['marker'],
            edgecolor='black', linewidth=0.8,
            alpha=0.6, transform=crs, zorder=9,
        )

    # ──  ──────────────────────────────────────────────────────────────────
    legend_handles = []
    for src in SOURCE_ORDER:
        if src not in source_counts:
            continue
        count = source_counts[src]
        style = SOURCE_STYLES[src]

        if src in SCATTER_ONLY:
            h = mlines.Line2D(
                [], [], color=style['color'], marker=style['marker'],
                linestyle='None', markersize=9,
                markeredgecolor='black', markeredgewidth=0.8,
                label=f"{style['label']} [{count}]",
            )
        else:
            h = mpatches.Patch(
                facecolor=style['color'], edgecolor='black',
                linewidth=0.5, alpha=0.80,
                label=f"{style['label']} [{count}]",
            )
        legend_handles.append(h)

    legend = ax.legend(
        handles=legend_handles,
        bbox_to_anchor=(0.45, 0.8),
        fontsize=12, frameon=True,
        edgecolor='black', fancybox=False, framealpha=1,
        markerscale=1.2,
    )
    legend.get_frame().set_linewidth(1.0)

    # ── Alpha  ───────────────────────────────────────────────────────────
    add_alpha_colorbar(
        ax, grid_data,
        source_styles=SOURCE_STYLES,
        scatter_only=SCATTER_ONLY,
        source_order=SOURCE_ORDER,
        alpha_min=ALPHA_MIN,
        alpha_max=ALPHA_MAX,
        count_ticks=COUNT_TICKS,
    )

    return legend_handles


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    extent = [-150, 150, -0.001, 60.001]

    fig = plt.figure(figsize=(15, 7))
    ax  = fig.add_subplot(1, 1, 1, projection=ccrs.PlateCarree())

    create_source_plot(ax, result_df, extent)
    # ax.set_title('Location of Ground-based observations', fontsize=16)
    # ax.text(-150, 61, '(a)', fontsize=16, ha='left', va='bottom')

    plt.tight_layout()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/Obs_info.png',
    #             format='png', dpi=300, bbox_inches='tight')
    plt.show()


if __name__ == '__main__':
    main()

In [ ]:
def create_source_plot(ax, result_df, extent):
    projection = ccrs.PlateCarree()
    crs = ccrs.PlateCarree()
    
    gl = ax.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=.2, linestyle='-.')
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.FixedLocator(np.arange(-120, 120 + 1, 60))
    # gl.ylocator = mticker.FixedLocator(np.arange(extent[2], extent[3] + 1, 20))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 61, 20))     # 0, 20, 40, 60

    gl.xpadding = 10
    gl.ypadding = 10
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    gl.top_labels = None
    gl.right_labels = None
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"))
    ax.add_feature(cfeature.BORDERS.with_scale("110m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    source_styles = {
        'GSDR': {'color': '#1f77b4', 'marker': 'o'},
        'HadISD': {'color': '#ff7f0e', 'marker': 's'},
        'DWD': {'color': '#2ca02c', 'marker': '^'},
        'MSC': {'color': '#d62728', 'marker': 'D'},
        'KMA': {'color': '#9467bd', 'marker': 'v'},
        'CMFD': {'color': '#8c564b', 'marker': '<'}}
    
    source_counts = result_df['source'].value_counts()
    
    scatter_plots = []
    for source in ['GSDR', 'HadISD', 'DWD','CMFD', 'MSC', 'KMA']:
        mask = result_df['source'] == source
        count = source_counts[source]
        style = source_styles[source]
        if source == 'MSC':
            scatter = ax.scatter(result_df.loc[mask, 'longitude'], 
                                   result_df.loc[mask, 'latitude'],
                                   s=15,
                                   c=style['color'],
                                   marker=style['marker'],
                                   label=f'ECCC [{count}]',
                                   edgecolor='black',
                                   linewidth=0.5,
                                   alpha=0.7,  # alpha  
                                   transform=ccrs.PlateCarree())
        else:
            
            scatter = ax.scatter(result_df.loc[mask, 'longitude'], 
                               result_df.loc[mask, 'latitude'],
                               s=15,
                               c=style['color'],
                               marker=style['marker'],
                               label=f'{source} [{count}]',
                               edgecolor='black',
                               linewidth=0.5,
                               alpha=0.7,  # alpha  
                               transform=ccrs.PlateCarree())
        scatter_plots.append(scatter)
            
        legend = ax.legend(bbox_to_anchor=(0.45, 0.8),  #  -30,  30 
                          fontsize=12, 
                          frameon=True,
                          edgecolor='black',
                          fancybox=False,
                          framealpha=1,markerscale=1.4)
        legend.get_frame().set_linewidth(1.0)
    
    return scatter_plots[0]  #   scatter  Returns

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-150, 150, -.001, 60.001]
    fig = plt.figure(figsize=(15, 23))
    
    # Source Plot
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    scatter1 = create_source_plot(ax1, result_df, extent)
    # ax1.set_title('Location of Ground-based observations', fontsize=18)
    # ax1.text(-150,61,'(a)' ,fontsize=18,ha='left', va='bottom')

    plt.show()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/Obs_info.png',format='png', dpi=300,bbox_inches='tight')

if __name__ == "__main__":
    main()

In [ ]:
def create_source_plot(ax, result_df, extent):  # for extended thesis
    projection = ccrs.PlateCarree()
    crs = ccrs.PlateCarree()
    
    gl = ax.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=.2, linestyle='-.')
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.FixedLocator(np.arange(-120, 120 + 1, 60))
    # gl.ylocator = mticker.FixedLocator(np.arange(extent[2], extent[3] + 1, 20))
    gl.ylocator = mticker.FixedLocator(np.arange(0, 61, 20))     # 0, 20, 40, 60

    gl.xpadding = 10
    gl.ypadding = 10
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    gl.top_labels = None
    gl.right_labels = None
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"))
    ax.add_feature(cfeature.BORDERS.with_scale("110m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    source_styles = {
        'GSDR': {'color': '#1f77b4', 'marker': 'o'},
        'HadISD': {'color': '#ff7f0e', 'marker': 's'},
        'DWD': {'color': '#2ca02c', 'marker': '^'},
        'MSC': {'color': '#d62728', 'marker': 'D'},
        'KMA': {'color': '#9467bd', 'marker': 'v'},
        'CMFD': {'color': '#8c564b', 'marker': '<'}}
    
    source_counts = result_df['source'].value_counts()
    
    scatter_plots = []
    for source in ['GSDR', 'HadISD', 'DWD','CMFD', 'MSC', 'KMA']:
        mask = result_df['source'] == source
        count = source_counts[source]
        style = source_styles[source]
        scatter = ax.scatter(result_df.loc[mask, 'longitude'], 
                           result_df.loc[mask, 'latitude'],
                           s=15,
                           c=style['color'],
                           marker=style['marker'],
                           label=f'{source}', #[{count}]',
                           edgecolor='black',
                           linewidth=0.5,
                           alpha=0.7,  # alpha  
                           transform=ccrs.PlateCarree())
        scatter_plots.append(scatter)
            
        legend = ax.legend(bbox_to_anchor=(0.45, 0.7),  #  -30,  30 
                          fontsize=12, 
                          frameon=True,
                          edgecolor='black',
                          fancybox=False,
                          framealpha=1,markerscale=1.4)
        legend.get_frame().set_linewidth(1.0)
    
    return scatter_plots[0]  #   scatter  Returns

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-150, 150, -.001, 60.001]
    fig = plt.figure(figsize=(15, 23))
    
    # Source Plot
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    scatter1 = create_source_plot(ax1, result_df, extent)
    # ax1.set_title('Location of Ground-based observations', fontsize=18)
    # ax1.text(-150,61,'(a)' ,fontsize=18,ha='left', va='bottom')

    # plt.show()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/Obs_info.png',format='png', dpi=400,bbox_inches='tight')

if __name__ == "__main__":
    main()

## 1. High-Frequency Ratio (HFR)
'
'
'


In [ ]:
###################################
#          Movind Window          #
###################################
# #obs
# obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr.nc')

# #satellite
# IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp')
# TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp')
# CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp')
# GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp')
# MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp')

# #reanalysis
# ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr.nc')
# JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr.nc')
# NARRa_hfr= xr.open_dataset('${DATA_DIR}/NARR/NARRa_hfr.nc').rename(__xarray_dataarray_variable__='hfr_tp') #   analysis  firstguess     ..
# NARRf_hfr = xr.open_dataset('${DATA_DIR}/NARR/acum_after/NARRf_hfr.nc')
# MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr.nc')


obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr_28h.nc')

#satellite
IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_28h.nc').rename(__xarray_dataarray_variable__='hfr_tp')
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_28h.nc').rename(__xarray_dataarray_variable__='hfr_tp')
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_28h.nc').rename(__xarray_dataarray_variable__='hfr_tp')
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_28h.nc').rename(__xarray_dataarray_variable__='hfr_tp')
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_28h.nc').rename(__xarray_dataarray_variable__='hfr_tp')

#reanalysis
ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr_28.nc')
JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr_28.nc')
MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr_28.nc')

In [ ]:
obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr_26h_30.nc')#.rename({'hfr':'hfr_tp'})

IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

In [ ]:
sat_mean = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr],'data').mean('data')
rean_mean = xr.concat([ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data')

In [ ]:
total_mask = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr,ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data',skipna = False)['hfr_tp'].isnull()

In [ ]:
IMERG_hfr = IMERG_hfr.where(~total_mask)
TRMM_hfr = TRMM_hfr.where(~total_mask)
CMORPH_hfr = CMORPH_hfr.where(~total_mask)
GSMaP_hfr = GSMaP_hfr.where(~total_mask)
MSWEP_hfr = MSWEP_hfr.where(~total_mask)

#reanalysis
ERA5_hfr = ERA5_hfr.where(~total_mask)
JRA3Q_hfr = JRA3Q_hfr.where(~total_mask)
# NARRa_hfr = NARRa_hfr.where(~total_mask)
# NARRf_hfr = NARRf_hfr.where(~total_mask)
MERRA2_hfr = MERRA2_hfr.where(~total_mask)

In [ ]:
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
sat_mean

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    bounds = np.linspace(70, 100, levels + 1)  # +1  
    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 60)
    
    #  True  1,  NaN 
    hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
               hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    if ref_data is not None:
        mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
        obs = ref_data.where(mask)['hfr']
        model = ds.where(mask)[var]
        
        r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
        mae = np.mean(np.abs(model - obs)).item()
        
        stats_text = f"r = {r:.2f}\n" \
                     f"MAE = {mae:.2f}%"
        
        ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
                fontsize=12, bbox=dict(facecolor='white', alpha=1),
                transform=ccrs.PlateCarree())
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / HFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('HFR [%]', fontsize=10)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=9, frameon=False, facecolor='none')


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 30  # 30  
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, sat_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax1.set_title('(a) Multi-Satellite(MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.1, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, sat_mean, 'hfr_tp', lsm, extent,True)
    ax1_hist.set_xlim(45, 95)  #  x  
    ax1_hist.set_xticks(np.arange(60, 81,10))
    ax1_hist.set_xlabel('HFR [%]', fontsize=12)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, rean_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax2.set_title('(b) Multi-Model(MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.1,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, rean_mean, 'hfr_tp', lsm, extent)
    ax2_hist.set_xlim(45, 95)  #  x  
    ax2_hist.set_xticks(np.arange(60, 81,10))
    ax2_hist.set_xlabel('HFR [%]', fontsize=12)
    ax2_hist.tick_params(labelsize=9)
    
    cbar_ticks = np.linspace(70, 100, 7)  # 70, 75, 80, 85, 90, 95, 100
    
    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])

    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='min', ticks=cbar_ticks)
    cbar.set_label('HFR[%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/HFR_map.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(30, 80, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
               hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    if ref_data is not None:
        mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
        obs = ref_data.where(mask)['hfr']
        model = ds.where(mask)[var]
        
        r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
        mae = np.mean(np.abs(model - obs)).item()
        
        stats_text = f"r = {r:.2f}\n" \
                     f"MAE = {mae:.2f}%"
        
        ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
                fontsize=12, bbox=dict(facecolor='white', alpha=1),
                transform=ccrs.PlateCarree())
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('SFR [%]', fontsize=10)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=9, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, sat_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax1.set_title('(a) Multi-Satellite(MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, sat_mean, 'hfr_tp', lsm, extent,True)
    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    ax1_hist.set_xlabel('SFR [%]', fontsize=12)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, rean_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax2.set_title('(b) Multi-Model(MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, rean_mean, 'hfr_tp', lsm, extent)
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    ax2_hist.set_xlabel('SFR [%]', fontsize=12)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(30, 80, 6)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])

    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label('SFR [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(30, 80, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
               hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    if ref_data is not None:
        mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
        obs = ref_data.where(mask)['hfr']
        model = ds.where(mask)[var]
        
        r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
        mae = np.mean(np.abs(model - obs)).item()
        
        stats_text = f"r = {r:.2f}"
        # stats_text = f"r = {r:.2f}\n" \
        #              f"MAE = {mae:.2f}%"
        ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
                # fontsize=13, 
                # transform=ccrs.PlateCarree())
                fontsize=13, bbox=dict(facecolor='white', alpha=.8),
                transform=ccrs.PlateCarree())

    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('$rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')
        # ax.legend(
        #     loc='lower left',
        # bbox_to_anchor=(.0, .96),      #   =  
        #     fontsize=12,
        #     frameon=True,
        #     facecolor='white',
        #     edgecolor='gray',
        #     framealpha=0.9        )


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, sat_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, sat_mean, 'hfr_tp', lsm, extent,True)
    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    # ax1_hist.set_xlabel('$rat_{subD}$ [%]', fontsize=12)
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, rean_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, rean_mean, 'hfr_tp', lsm, extent)
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    # ax2_hist.set_xlabel('$rat_{subD}$ [%]', fontsize=12)
    ax2_hist.set_xlabel(None)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(30, 80, 6)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])

    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label('$rat_{subD}$ [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(30, 80, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
               hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    if ref_data is not None:
        mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
        obs = ref_data.where(mask)['hfr']
        model = ds.where(mask)[var]
        
        r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
        mae = np.mean(np.abs(model - obs)).item()
        
        stats_text = f"r = {r:.2f}\n" \
                     f"MAE = {mae:.2f}%"
        
        ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
                fontsize=12, bbox=dict(facecolor='white', alpha=1),
                transform=ccrs.PlateCarree())
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('$FR_{high}$ [%]', fontsize=10)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=9, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    cmap = cmaps.WhiteYellowOrangeRed
    levels = 40  # 30  
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, sat_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax1.set_title('(c) Multi-Satellite(MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, sat_mean, 'hfr_tp', lsm, extent,True)
    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    ax1_hist.set_xlabel('$FR_{high}$ [%]', fontsize=12)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, rean_mean, extent, cmap, levels, 'hfr_tp', obs_hfr)
    ax2.set_title('(d) Multi-Model(MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, rean_mean, 'hfr_tp', lsm, extent)
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    ax2_hist.set_xlabel('$FR_{high}$ [%]', fontsize=12)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(30, 80, 6)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])

    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label('$FR_{high}$ [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/HFR_map_26h_30.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, bounds,hatchs,ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(70, 100, levels + 1)  # +1  
    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)
    if hatchs == True :
        
        hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 60)
        
        #  True  1,  NaN 
        hatch_data = xr.where(hatch_mask, 1, np.nan)
        
        # contourf    (  )
        ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
                   hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    if ref_data is not None:
        mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
        obs = ref_data.where(mask)['hfr']
        model = ds.where(mask)[var]
        
        r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
        mae = np.mean(np.abs(model - obs)).item()
        
        stats_text = f"r = {r:.2f}\n" \
                     f"MAE = {mae:.2f}%"
        
        ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
                fontsize=12, bbox=dict(facecolor='white', alpha=1),
                transform=ccrs.PlateCarree())
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / HFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('HFR [%]', fontsize=10)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=9, frameon=False,bbox_to_anchor=(0.1, 1.), facecolor='none')


def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11/1.2, 11))
    cmap = cmaps.WhiteBlueGreenYellowRed
    cmap1 = cmaps.balance

    levels = 30  # 30  
    bounds = np.linspace(70, 100, levels + 1)  # +1  
    bounds1 = np.linspace(-20, 20, 40 + 1)  # +1  

    #   : Multi satellite
    ax1 = fig.add_subplot(3, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, sat_mean, extent, cmap, levels, 'hfr_tp', bounds,True,obs_hfr)
    ax1.set_title('(a) Multi-Satellite(MS)', fontsize=18)
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.1, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, sat_mean, 'hfr_tp', lsm, extent,True)
    ax1_hist.set_xlim(45, 95)  #  x  
    ax1_hist.set_xticks(np.arange(60, 81,10))
    ax1_hist.set_xlabel('HFR [%]', fontsize=12)
    ax1_hist.tick_params(labelsize=9)
    #   : Multi reanalysis
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, rean_mean, extent, cmap, levels, 'hfr_tp', bounds,True,obs_hfr)
    ax2.set_title('(b) Multi-Model(MM)', fontsize=18)
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.1,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, rean_mean, 'hfr_tp', lsm, extent)
    ax2_hist.set_xlim(45, 95)  #  x  
    ax2_hist.set_xticks(np.arange(60, 81,10))
    ax2_hist.set_xlabel('HFR [%]', fontsize=12)
    ax2_hist.tick_params(labelsize=9)

    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())  # 3 
    scatter3 = create_local_solar_time_plot(ax3, (rean_mean-sat_mean), extent, cmap1, levels, 'hfr_tp',bounds1,False)
    ax3.set_title('(c) MM-MS', fontsize=18)

    cbar_ticks = np.linspace(70, 100, 7)  # 70, 75, 80, 85, 90, 95, 10
    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    cax = fig.add_axes([ax2_hist.get_position().x1 + 0.04,  (pos1.y1-pos2.y0), 0.02, (pos1.y1-pos2.y0)*(2/3)])
    cbar = fig.colorbar(scatter2, cax=cax, orientation='vertical', 
                       extend='min', ticks=cbar_ticks)
    cbar.set_label('HFR[%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    cbar_ticks3 = np.linspace(-20,20, 5)  # 70, 75, 80, 85, 90, 95, 10
    pos3 = ax3.get_position()
    cax1 = fig.add_axes([pos3.x1 + 0.04,  pos3.y0, 0.02, (pos3.y1-pos3.y0)])
    cbar1 = fig.colorbar(scatter3, cax=cax1, orientation='vertical', 
                       extend='both', ticks=cbar_ticks3)
    cbar1.set_label('HFR[%]', fontsize=12)
    cbar1.ax.tick_params(labelsize=11)

    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/HFR_map_minus.png', dpi=400, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
TRMM_hfr = TRMM_hfr.clip(0,100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def create_grouped_hfr_boxplots():
    """// HFR boxplot """
    
    satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    
    reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
    reanalysis_names = ['ERA5', 'JRA3Q', 'MERRA2']
    
    all_datasets = satellite_datasets + reanalysis_datasets
    all_names = satellite_names + reanalysis_names
    
    plot_data = []
    
    for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
        global_data = ds['hfr_tp'].values.flatten()
        global_data = global_data[~np.isnan(global_data)]
        
        land_data = ds.where(lsm)['hfr_tp'].values.flatten()
        land_data = land_data[~np.isnan(land_data)]
        
        ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        data_type = 'Satellite' if i < 5 else 'Reanalysis'
        
        for val in global_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Global', 
                            'Type': data_type, 'Position': i})
        
        for val in land_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Land', 
                            'Type': data_type, 'Position': i})
        
        for val in ocean_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Ocean', 
                            'Type': data_type, 'Position': i})
    
    df = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    
    surface_colors = {
        'Land': '#D9665B',
        'Ocean': '#03738C',
        'Global': '#54728C'
    }
    
    sns.boxplot(data=df, x='Dataset', y='HFR', hue='Surface',
                order=all_names, hue_order=['Global', 'Land', 'Ocean'],
                palette=surface_colors, ax=ax, showfliers=False,
                showmeans=True, meanprops={"marker":"o","markerfacecolor":"white", 
                                          "markeredgecolor":"black","markersize":5})
    
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1)
    
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', labelsize=11, rotation=45)
    ax.tick_params(axis='y', labelsize=10)
    
    ax.set_title('High Frequency Ratio Distribution by Surface Type', 
                fontsize=14, pad=15)
    
    ax.legend( fontsize=10, 
             loc='lower left', frameon=False)
    
    # ax.text(2, -12, 'Satellite Products', ha='center', va='top', 
    #        fontsize=12, fontweight='bold', transform=ax.transData)
    # ax.text(6, -12, 'Reanalysis Products', ha='center', va='top', 
    #        fontsize=12, fontweight='bold', transform=ax.transData)
    
    sns.despine()
    
    plt.tight_layout()
    plt.show()

def create_grouped_hfr_boxplots_matplotlib():
    """matplotlib     """
    
    satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    
    reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
    reanalysis_names = ['ERA5', 'JRA3Q', 'MERRA2']
    
    all_datasets = satellite_datasets + reanalysis_datasets
    all_names = satellite_names + reanalysis_names
    
    land_data_all = []
    ocean_data_all = []
    global_data_all = []
    
    for ds in all_datasets:
        global_data = ds['hfr_tp'].values.flatten()
        global_data = global_data[~np.isnan(global_data)]
        global_data_all.append(global_data)
        
        land_data = ds.where(lsm)['hfr_tp'].values.flatten()
        land_data = land_data[~np.isnan(land_data)]
        land_data_all.append(land_data)
        
        ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        ocean_data_all.append(ocean_data)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    
    land_color = '#D9665B'
    ocean_color = '#03738C'
    global_color = '#54728C'
    
    width = 0.25
    positions = np.arange(len(all_names))
    
    for i, pos in enumerate(positions):
        bp1 = ax.boxplot([global_data_all[i]], positions=[pos - width], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=global_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
        
        bp2 = ax.boxplot([land_data_all[i]], positions=[pos], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=land_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
        
        bp3 = ax.boxplot([ocean_data_all[i]], positions=[pos + width], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=ocean_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
    
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=11)
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_ylim(0, 100)
    
    ax.set_title('High Frequency Ratio Distribution by Surface Type', 
                fontsize=14, pad=15)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=global_color, alpha=0.8, label='Global'),
                      Patch(facecolor=land_color, alpha=0.8, label='Land'),
                      Patch(facecolor=ocean_color, alpha=0.8, label='Ocean')]
    ax.legend(handles=legend_elements, fontsize=10, loc='lower left', frameon=False)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(-.5,7.5)
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/HFR_boxplot.png', dpi=400, bbox_inches='tight')
    # plt.show()
# print("Seaborn :")
# create_grouped_hfr_boxplots()

# print("\nMatplotlib :")
create_grouped_hfr_boxplots_matplotlib()

In [ ]:
obs_hfr = obs_hfr.rename({'hfr':'hfr_tp'})

In [ ]:
satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']

reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']

all_datasets = satellite_datasets + reanalysis_datasets
all_names = satellite_names + reanalysis_names

plot_data = []
for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
    global_data = ds['hfr_tp'].values.flatten()
    global_data = global_data[~np.isnan(global_data)]
    
    land_data = ds.where(lsm)['hfr_tp'].values.flatten()
    land_data = land_data[~np.isnan(land_data)]
    
    ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
    ocean_data = ocean_data[~np.isnan(ocean_data)]
    
    data_type = 'Satellite' if i < 5 else 'Reanalysis'
    
    for val in global_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Global', 
                        'Type': data_type, 'Position': i})
    
    for val in land_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Land', 
                        'Type': data_type, 'Position': i})
    
    for val in ocean_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Ocean', 
                        'Type': data_type, 'Position': i})
df = pd.DataFrame(plot_data)


In [ ]:
df.to_csv('${CODE_DIR}/Diurnal_cycle/HFR/hfr_csv.csv',index=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

def create_split_violin_hfr():
    """   HFR  
    Left: Global, Right: Land & Ocean (overlaid with alpha)
    """
    
    #   (   )
    satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    
    reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    
    all_datasets = satellite_datasets + reanalysis_datasets
    all_names = satellite_names + reanalysis_names
    
    plot_data = []
    for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
        global_data = ds['hfr_tp'].values.flatten()
        global_data = global_data[~np.isnan(global_data)]
        
        land_data = ds.where(lsm)['hfr_tp'].values.flatten()
        land_data = land_data[~np.isnan(land_data)]
        
        ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        data_type = 'Satellite' if i < 5 else 'Reanalysis'
        
        for val in global_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Global', 
                            'Type': data_type, 'Position': i})
        
        for val in land_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Land', 
                            'Type': data_type, 'Position': i})
        
        for val in ocean_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Ocean', 
                            'Type': data_type, 'Position': i})
    df = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(7, 5))
    
    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global median  ( )
            global_median = np.median(global_vals)
            ax.plot([i-0.4, i], [global_median, global_median], 
                   color=global_color, linewidth=1.5, alpha=0.9)
            
            # Global mean 
            global_mean = np.mean(global_vals)
            ax.scatter([i-0.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
        
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land median  ( )
            land_median = np.median(land_vals)
            ax.plot([i, i+0.4], [land_median, land_median], 
                   color=land_color, linewidth=1.5, alpha=0.9)
            
            # Land mean 
            land_mean = np.mean(land_vals)
            ax.scatter([i+0.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
        
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)  #    Land 
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean median  ( ,  )
            ocean_median = np.median(ocean_vals)
            ax.plot([i, i+0.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=1.5, alpha=0.8, linestyle='-')
            
            # Ocean mean  (  )
            ocean_mean = np.mean(ocean_vals)
            ax.scatter([i+0.2], [ocean_mean], color='white', s=40, 
                      marker='s', edgecolor=ocean_color, linewidth=1.5, zorder=10)
    
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1.5)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=11)
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_ylim(10, 100)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    ax.set_title('(C) High Frequency Ratio Distribution', 
                fontsize=14, pad=15)
    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=9, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def calc_kdepss(ref, mod):
    """ Perkins Skill Score  """
    if np.std(ref) != 0 and np.std(mod) != 0:
        kde_ref = stats.gaussian_kde(ref)
        kde_mod = stats.gaussian_kde(mod)
        vmin = min([ref.min(), mod.min()])
        vmax = max([ref.max(), mod.max()])
        
        KDEnbins = 1000
        dx = ((vmax-vmin)*2/KDEnbins)
        ls = np.linspace(vmin-(vmax-vmin)/2, vmax+(vmax-vmin)/2, KDEnbins)
        
        arr = np.empty([2, KDEnbins])
        arr[:] = 0.
        arr[0,:] = kde_ref(ls)
        arr[1,:] = kde_mod(ls)
        del kde_ref
        del kde_mod
        kde_pss = np.sum(np.min(arr,axis=0))*dx
    else:
        kde_pss = np.nan
    
    return kde_pss

def create_single_violin_hfr():
    """   HFR   (  )"""
    
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    obs_names = ['Obs']
    all_names = satellite_names + reanalysis_names + obs_names
    
    #   (obs_hfr )
    all_datasets = satellite_datasets + reanalysis_datasets + [obs_hfr]
    
    fig, ax = plt.subplots(figsize=((7/8)*9, 5))
    
    satellite_color = '#2E8B57'
    reanalysis_color = '#4169E1'
    obs_color = '#DC143C'
    
    positions = np.arange(len(all_names))
    
    #      (land only)
    obs_valid_mask = ~np.isnan(obs_hfr['hfr_tp'].values) & lsm.values
    obs_data = obs_hfr.where(obs_valid_mask)['hfr_tp'].values.flatten()
    obs_data = obs_data[~np.isnan(obs_data)]
    
    pss_scores = []  # PSS  
    
    plot_data = []
    
    for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
        if name == 'Obs':
            #  : land   
            valid_data = ds.where(obs_valid_mask)['hfr_tp'].values.flatten()
            valid_data = valid_data[~np.isnan(valid_data)]
            color = obs_color
            data_type = 'Observation'
            pss_score = 1.0  #   PSS 1.0
        else:
            #  :     
            valid_data = ds.where(obs_valid_mask)['hfr_tp'].values.flatten()
            valid_data = valid_data[~np.isnan(valid_data)]
            
            # PSS  (  )
            pss_score = calc_kdepss(obs_data, valid_data)
            
            if i < 5:
                color = satellite_color
                data_type = 'Satellite'
            else:
                color = reanalysis_color  
                data_type = 'Reanalysis'
        
        pss_scores.append(pss_score)
        
        for val in valid_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Type': data_type, 'Position': i})
        
        if len(valid_data) > 0:
            parts = ax.violinplot([valid_data], positions=[i], widths=0.6, 
                                showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.6)
                pc.set_edgecolor(color)
                pc.set_linewidth(1.2)
            
            q1 = np.percentile(valid_data, 25)
            median = np.median(valid_data)
            q3 = np.percentile(valid_data, 75)
            mean = np.mean(valid_data)
            
            # Q1 (25%)
            ax.plot([i-0.25, i+0.25], [q1, q1], 
                   color=color, linewidth=1.5, alpha=0.8, linestyle=':')
            # Median (50%)
            ax.plot([i-0.3, i+0.3], [median, median], 
                   color=color, linewidth=3, alpha=0.9)
            # Q3 (75%)
            ax.plot([i-0.25, i+0.25], [q3, q3], 
                   color=color, linewidth=1.5, alpha=0.8, linestyle=':')
            
            # Mean 
            ax.scatter([i], [mean], color='white', s=60, 
                      edgecolor=color, linewidth=2, zorder=10)
            
            # PSS     
            y_max = np.max(valid_data)
            y_text = y_max + 2
            if i != len(all_names)-1:
                
                ax.text(i, 99,  f'{pss_score:.2f}',#f'PSS:{pss_score:.3f}', 
                       ha='center', va='top', fontsize=11)
    
    for i in range(len(all_names) - 1):
        if i == 4:
            ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
        elif i == 7:
            ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
        else:
            ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=11)
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_ylim(40, 100)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    ax.set_title('(D)', 
                fontsize=14, pad=15)
    
    legend_elements = [
        Patch(facecolor=satellite_color, alpha=0.6, label='Satellite'),
        Patch(facecolor=reanalysis_color, alpha=0.6, label='Reanalysis'),
        Patch(facecolor=obs_color, alpha=0.6, label='Observation'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=10, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # PSS
    plt.tight_layout()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/HFR_violin_PSS.png', 
    #             dpi=400, bbox_inches='tight')
    plt.show()

#   (   )
satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]

create_single_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def calc_kdepss(ref, mod):
    """ Perkins Skill Score  """
    if np.std(ref) != 0 and np.std(mod) != 0:
        kde_ref = stats.gaussian_kde(ref)
        kde_mod = stats.gaussian_kde(mod)
        vmin = min([ref.min(), mod.min()])
        vmax = max([ref.max(), mod.max()])
        
        KDEnbins = 1000
        dx = ((vmax-vmin)*2/KDEnbins)
        ls = np.linspace(vmin-(vmax-vmin)/2, vmax+(vmax-vmin)/2, KDEnbins)
        
        arr = np.empty([2, KDEnbins])
        arr[:] = 0.
        arr[0,:] = kde_ref(ls)
        arr[1,:] = kde_mod(ls)
        del kde_ref
        del kde_mod
        kde_pss = np.sum(np.min(arr,axis=0))*dx
    else:
        kde_pss = np.nan
    
    return kde_pss

def create_single_violin_hfr():
    """   HFR   (  )"""
    
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    obs_names = ['Obs']
    all_names = satellite_names + reanalysis_names + obs_names
    
    #   (obs_hfr )
    all_datasets = satellite_datasets + reanalysis_datasets + [obs_hfr]
    
    fig, ax = plt.subplots(figsize=((7/8)*9, 5))
    
    satellite_color = '#2E8B57'
    reanalysis_color = '#4169E1'
    obs_color = '#DC143C'
    ymax = 80
    ymin = 20
    positions = np.arange(len(all_names))
    
    #      (land only)
    obs_valid_mask = ~np.isnan(obs_hfr['hfr_tp'].values) & lsm.values
    obs_data = obs_hfr.where(obs_valid_mask)['hfr_tp'].values.flatten()
    obs_data = obs_data[~np.isnan(obs_data)]
    
    pss_scores = []  # PSS  
    
    plot_data = []
    
    for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
        if name == 'Obs':
            #  : land   
            valid_data = ds.where(obs_valid_mask)['hfr_tp'].values.flatten()
            valid_data = valid_data[~np.isnan(valid_data)]
            color = obs_color
            data_type = 'Observation'
            pss_score = 1.0  #   PSS 1.0
        else:
            #  :     
            valid_data = ds.where(obs_valid_mask)['hfr_tp'].values.flatten()
            valid_data = valid_data[~np.isnan(valid_data)]
            
            # PSS  (  )
            pss_score = calc_kdepss(obs_data, valid_data)
            
            if i < 5:
                color = satellite_color
                data_type = 'Satellite'
            else:
                color = reanalysis_color  
                data_type = 'Reanalysis'
        
        pss_scores.append(pss_score)
        
        for val in valid_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Type': data_type, 'Position': i})
        
        if len(valid_data) > 0:
            parts = ax.violinplot([valid_data], positions=[i], widths=0.6, 
                                showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts['bodies']:
                pc.set_facecolor(color)
                pc.set_alpha(0.6)
                pc.set_edgecolor(color)
                pc.set_linewidth(1.2)
            
            q1 = np.percentile(valid_data, 25)
            median = np.median(valid_data)
            q3 = np.percentile(valid_data, 75)
            mean = np.mean(valid_data)
            
            # Q1 (25%)
            ax.plot([i-0.25, i+0.25], [q1, q1], 
                   color=color, linewidth=1.5, alpha=0.8, linestyle=':')
            # Median (50%)
            ax.plot([i-0.3, i+0.3], [median, median], 
                   color=color, linewidth=3, alpha=0.9)
            # Q3 (75%)
            ax.plot([i-0.25, i+0.25], [q3, q3], 
                   color=color, linewidth=1.5, alpha=0.8, linestyle=':')
            
            # Mean 
            ax.scatter([i], [mean], color='white', s=60, 
                      edgecolor=color, linewidth=2, zorder=10)
            
            # PSS     
            y_max = np.max(valid_data)
            y_text = y_max + 2
            if i != len(all_names)-1:
                
                ax.text(i, ymax-1,  f'{pss_score:.2f}',#f'PSS:{pss_score:.3f}', 
                       ha='center', va='top', fontsize=12)
    
    for i in range(len(all_names) - 1):
        if i == 4:
            ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
        elif i == 7:
            ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
        else:
            ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('SFR [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.text('(D)',0.,1. ,fontsize=20)
    ax.text(-0.5, ymax,  '(D)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    # ax.set_title('(D) High Frequency Ratio Distribution with Observation', 
    #             fontsize=14, pad=15)
    legend_elements = [
        Patch(facecolor=satellite_color, alpha=0.6, label='Satellite'),
        Patch(facecolor=reanalysis_color, alpha=0.6, label='Reanalysis'),
        Patch(facecolor=obs_color, alpha=0.6, label='Observation'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    # PSS
    plt.tight_layout()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violin_PSS_28h.png', 
    #             dpi=400, bbox_inches='tight')
    plt.show()

#   (   )
satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]

create_single_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    fig, ax = plt.subplots(figsize=(7, 5))
    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('SFR [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(C)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30.png', dpi=400, bbox_inches='tight')
    # plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    # fig, ax = plt.subplots(figsize=(7, 5))
    fig, ax = plt.subplots(figsize=(7, 4.25))

    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('$rat_{subD}$ [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(b)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    # plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    # fig, ax = plt.subplots(figsize=(7, 5))
    fig, ax = plt.subplots(figsize=(7, 4.25))

    ymax = 80
    ymin = 0

    global_color = '#6B8E8F'
    land_color = '#C17C58'
    ocean_color = '#0A5F8C'
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor(ocean_color)
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor(land_color)
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('$FR_{high}$ [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(b)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/HFR_violinplot_26h_30.png', dpi=400, bbox_inches='tight')
    # plt.show()

create_split_violin_hfr()

## 2. Relative Contributions of 1st, 2nd, and Higher Harmonic Amplitudes in Diurnal Cycle[]
'
'


In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc')  # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
sat_mean = xr.concat([imerg ,trmm,cmorph,gsmap,mswep],'data').mean('data')#,skipna = False)

In [ ]:
rean_mean = xr.concat([jra55,era5,merra2],'data').mean('data',skipna = False)

In [ ]:
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D
import mpltern
from matplotlib.patches import ArrowStyle, FancyArrowPatch
from mpltern.datasets import get_dirichlet_pdfs
from matplotlib.colors import to_rgb

from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D

def rgb_colormap(t, l, r, tmin, lmin, rmin,tmax, lmax, rmax, colors):
    """
    Custom colormap function that interpolates between three HEX-defined colors.
    """    
    # Create masks for out-of-bounds regions
    t_out = t > tmax
    l_out = l > lmax
    r_out = r < rmin
    
    # Normalize the coordinates
    total = t + l + r
    # t_norm = t / tmax
    # l_norm = l / lmax 
    t_norm = (t - tmin) / (tmax - tmin)
    l_norm = (l - lmin) / (lmax - lmin)
    r_norm = (r - rmin) / (rmax - rmin)  # r 30-100  
    # r_norm = r /rmax 
    t_norm_total = t / total
    l_norm_total = l / total
    r_norm_total = r / total
    
    # Interpolate between the three corner colors
    color_top = np.array(to_rgb(colors['top']))     # HEX to RGB #2nd
    color_left = np.array(to_rgb(colors['left']))   # HEX to RGB
    color_right = np.array(to_rgb(colors['right'])) # HEX to RGB
    
    # Weighted average of the corner colors
    red = t_norm * color_top[0] + l_norm * color_left[0] + r_norm * color_right[0]
    green = t_norm * color_top[1] + l_norm * color_left[1] + r_norm * color_right[1]
    blue = t_norm * color_top[2] + l_norm * color_left[2] + r_norm * color_right[2]
    
    colors_rgba = np.column_stack([red, green, blue, np.ones_like(red)])
    colors_rgba[t_out] = [*color_top, 1]
    colors_rgba[l_out] = [*color_left, 1]
    colors_rgba[r_out] = [*color_right, 1]
    
    # Clip values
    colors_rgba = np.clip(colors_rgba, 0, 1)
    
    # Calculate alpha based on distance from center
    center_dist = np.sqrt((t_norm_total - 1/3)**2 + (l_norm_total - 1/3)**2 + (r_norm_total - 1/3)**2)
    max_dist = np.sqrt(1/2)
    normalized_dist = center_dist / max_dist
    alpha = normalized_dist * 0.9 + 0.0
    alpha = np.clip(alpha, 0, 1)
    
    # Apply alpha
    colors_rgba[:, 3] = alpha
    
    return colors_rgba

def rgb_array(first, second, tmin, lmin, rmin, tmax, lmax, rmax, colors_hex):
    """
    Create an RGB DataArray using rgb_colormap for xarray DataArrays.
    """
    # Create an empty DataArray
    tfs_afs_rgb = xr.DataArray(
        data=np.empty([len(first.latitude), len(first.longitude), 4]) * np.nan,
        dims=["lat", "lon", "RGBs"],
        coords=dict(
            lat=(["lat"], first.latitude.values),
            lon=(["lon"], first.longitude.values),
            RGBs=(["RGBs"], [x for x in range(4)]),
        ),
    )
    
    # Flatten the DataArrays
    r_flat = first.values.flatten()  # 1st (, Right Vertex)
    t_flat = second.values.flatten()  # 2nd (, Top Vertex)
    l_flat = (100 - t_flat - r_flat)  # higher order (, Left Vertex)
    
    # Filter out NaN values
    valid_mask = ~np.isnan(t_flat) & ~np.isnan(l_flat) & ~np.isnan(r_flat)
    t_valid = t_flat[valid_mask]
    l_valid = l_flat[valid_mask]
    r_valid = r_flat[valid_mask]

    t_valid = np.clip(t_valid, tmin, tmax)
    l_valid = np.clip(l_valid, lmin, lmax)
    r_valid = np.clip(r_valid, rmin, rmax)
    
    # Compute colors using rgb_colormap
    rgba_colors = rgb_colormap(t_valid, l_valid, r_valid,tmin, lmin, rmin, tmax, lmax, rmax, colors_hex)
    
    color_top = np.array([*to_rgb(colors_hex['top']), 1])
    color_left = np.array([*to_rgb(colors_hex['left']), 1])
    color_right = np.array([*to_rgb(colors_hex['right']), 1])
    
    t_out = t_valid > tmax
    l_out = l_valid > lmax
    r_out = r_valid < rmin
    
    rgba_colors[t_out] = color_top
    rgba_colors[l_out] = color_left
    rgba_colors[r_out] = color_right
    
    # Assign computed colors back to the DataArray
    tfs_afs_rgb.values.reshape(-1, 4)[valid_mask] = rgba_colors
    
    return tfs_afs_rgb

In [ ]:
result_ds_list = [sat_mean,rean_mean]

In [ ]:
titles = ['(a) Multi-Satellite (MS)','(b) Multi-Model (MM)']

In [ ]:
rgb_list = []
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65

for i in range(len(result_ds_list)):
    rgb_list.append(rgb_array(result_ds_list[i]['1st'], result_ds_list[i]['2nd'], tmin, lmin, rmin, tmax, lmax, rmax, colors_hex))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def creat_ampratio(ax,ddd,raw_ds,extent):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)

    data = ddd
    data_T = data[::-1,:,:]
    cs=ax.imshow(data_T, extent=[data.lon.values.min(), data.lon.values.max(), data.lat.values.min(), data.lat.values.max()])
    # ax.coastlines()
    ax.contourf(
        ddd.lon, ddd.lat, ddd[:,:,0].isnull(),
        transform=ccrs.PlateCarree(),
        colors='gray',
        levels=[0.5,1.5]
    ) 
    return cs

def main():
    # extent = [-150, 150, -50, 50]
    extent=[-180, 180, -60, 60]
    fig = plt.figure(figsize=(11, 8.5))
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = creat_ampratio(ax1,rgb_list[0],result_ds_list[0],extent)
    ax1.set_title(titles[0], fontsize=18)
    # ax1.text(-150, 51, '(a)', fontsize=14, ha='left', va='bottom')
    
    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = creat_ampratio(ax2,rgb_list[1],result_ds_list[1],extent)
    ax2.set_title(titles[1], fontsize=18)
    # ax2.text(-150, 51, '(b)', fontsize=14, ha='left', va='bottom')
    
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/amp_ratio_finish.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np  # incorrect version
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def creat_ampratio(ax,ddd,raw_ds,extent):
    crs = ccrs.PlateCarree()    
    gl = ax.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=.2, linestyle='-.')
    gl.xlabel_style = {"size": 10}
    gl.ylabel_style = {"size": 10}
    gl.xlocator = mticker.FixedLocator(np.arange(extent[0], extent[1] + 1, 50))
    gl.ylocator = mticker.FixedLocator([-30,0,30])
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    gl.top_labels = None
    gl.right_labels = None
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    data = ddd
    data_T = data[::-1,:,:]
    cs=ax.imshow(data_T, extent=[data.lon.values.min(), data.lon.values.max(), data.lat.values.min(), data.lat.values.max()])
    # ax.coastlines()
    ax.contourf(
        ddd.lon, ddd.lat, ddd[:,:,0].isnull(),
        transform=ccrs.PlateCarree(),
        colors='gray',
        levels=[0.5,1.5]
    ) 
    return cs

def main():
    extent = [-150, 150, -50, 50]
    fig = plt.figure(figsize=(11, 6))
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = creat_ampratio(ax1,rgb_list[0],result_ds_list[0],extent)
    ax1.set_title('Multi satellite', fontsize=14)
    ax1.text(-150, 51, '(a)', fontsize=14, ha='left', va='bottom')
    
    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = creat_ampratio(ax2,rgb_list[1],result_ds_list[1],extent)
    ax2.set_title('Multi reanalysis', fontsize=14)
    ax2.text(-150, 51, '(b)', fontsize=14, ha='left', va='bottom')
    
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/HFR_map.png', dpi=400, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
data_title = ['Multi-Satellite (MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
               'Multi-Model (MM)','JRA-3Q','ERA5','MERRA-2']

markers = ['$S$','^' ,'s', 'H','X', 'D',
          '$R$','<', 'p', '*']

In [ ]:
sat_right_mask = cmorph['right_mask']&imerg ['right_mask']&trmm['right_mask']&gsmap['right_mask']&mswep['right_mask']

In [ ]:
rea_right_mask = jra55['right_mask']&era5['right_mask']&merra2['right_mask']

In [ ]:
sat_mean['right_mask'] = sat_right_mask
rean_mean['right_mask'] = rea_right_mask

In [ ]:
result_ds_list = [sat_mean,cmorph,imerg ,trmm,gsmap,mswep,
                  rean_mean,jra55,era5,merra2]

In [ ]:
def return_mean(data,lsm):
    # lat/lon restricted to 50 deg, no mask
    # land_1st = data.where((lsm==1)&(~(cmorph['right_mask']|imerg['right_mask']|trmm['right_mask']|gsmap['right_mask']|mswep['right_mask']|jra55['right_mask']|era5['right_mask']|merra2['right_mask']))).sel(latitude = slice(-60,60))['1st'].mean().item()
    # land_2nd = data.where((lsm==1)&(~(cmorph['right_mask']|imerg['right_mask']|trmm['right_mask']|gsmap['right_mask']|mswep['right_mask']|jra55['right_mask']|era5['right_mask']|merra2['right_mask']))).sel(latitude = slice(-60,60))['2nd'].mean().item()
    land_1st = data.where(lsm==1).sel(latitude = slice(-50,50))['1st'].mean().item()
    land_2nd = data.where(lsm==1).sel(latitude = slice(-50,50))['2nd'].mean().item()
    land_high =100-land_1st-land_2nd

    o_1st = data.where(lsm==0).sel(latitude = slice(-50,50))['1st'].mean().item()
    o_2nd = data.where(lsm==0).sel(latitude = slice(-50,50))['2nd'].mean().item()
    o_high =100-o_1st-o_2nd
    
    return((land_1st,land_2nd,land_high),(o_1st,o_2nd,o_high))

In [ ]:
mean_list = []  # original version
for i in range(len(result_ds_list)):
    print(i)
    mean_list.append(return_mean(result_ds_list[i],lsm))

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=200, alpha=(10, 10, 10))

# Define HEX colors for the vertices

tmax = 50
lmax = 40
rmax = 90
tmin = 10
lmin = 0 
rmin = 50
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=50,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 10  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
for i in range(len(mean_list)):
    #land
    ax.scatter(mean_list[i][0][1],mean_list[i][0][2],mean_list[i][0][0], marker=markers[i],edgecolor='black', color='#F24405', s=200,zorder = 2,lw = .7,alpha = .7)
    #Ocean     label=data_title[i]+" Land",
    ax.scatter(mean_list[i][1][1],mean_list[i][1][2],mean_list[i][1][0], marker=markers[i],edgecolor='black', color='#C2E5F2', s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color='white', s=150, marker=markers[i],zorder = 2,lw = 1,alpha = 1,label = data_title[i],edgecolor='black'))

ll = ax.scatter([],[],[], color='#F24405', s=150,zorder = 2,lw = .7,edgecolor='black',alpha = 1.,label = 'Land')#,edgecolor='black')
oo = ax.scatter([],[],[], color='#C2E5F2', s=150,zorder = 2,lw = .7,edgecolor='black',alpha = 1.,label = 'Ocean')#,edgecolor='black')
legend1 = ax.legend(handles=[ll, oo], loc='upper right', bbox_to_anchor=(1.1, 1.1), fontsize=15,frameon=False)
ax.add_artist(legend1)

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_non_obs.png',format='png', dpi=300,bbox_inches='tight')


In [ ]:
new_markers = ['o']+markers
new_data_title = ['Obs']+data_title

In [ ]:
print(len(new_markers))
print(len(new_data_title))

In [ ]:
obs_result = xr.open_dataset('${DATA_DIR}/obs/obs_gridded.nc')

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D
def rgb_colormap(t, l, r, tmin, lmin, rmin,tmax, lmax, rmax, colors):
    """
    Custom colormap function that interpolates between three HEX-defined colors.
    """
    # Define rmin
    # rmin = 65
    
    # Create masks for out-of-bounds regions
    t_out = t > tmax
    l_out = l > lmax
    r_out = r < rmin
    
    # Normalize the coordinates
    total = t + l + r
    # t_norm = t / tmax
    # l_norm = l / lmax 
    t_norm = (t - tmin) / (tmax - tmin)
    l_norm = (l - lmin) / (lmax - lmin)
    r_norm = (r - rmin) / (rmax - rmin)  # r 30-100  
    # r_norm = r /rmax 
    t_norm_total = t / total
    l_norm_total = l / total
    r_norm_total = r / total
    
    # Interpolate between the three corner colors
    color_top = np.array(to_rgb(colors['top']))     # HEX to RGB #2nd
    color_left = np.array(to_rgb(colors['left']))   # HEX to RGB
    color_right = np.array(to_rgb(colors['right'])) # HEX to RGB
    
    # Weighted average of the corner colors
    red = t_norm * color_top[0] + l_norm * color_left[0] + r_norm * color_right[0]
    green = t_norm * color_top[1] + l_norm * color_left[1] + r_norm * color_right[1]
    blue = t_norm * color_top[2] + l_norm * color_left[2] + r_norm * color_right[2]
    
    colors_rgba = np.column_stack([red, green, blue, np.ones_like(red)])
    colors_rgba[t_out] = [*color_top, 1]
    colors_rgba[l_out] = [*color_left, 1]
    colors_rgba[r_out] = [*color_right, 1]
    
    # Clip values
    colors_rgba = np.clip(colors_rgba, 0, 1)
    
    # Calculate alpha based on distance from center
    center_dist = np.sqrt((t_norm_total - 1/3)**2 + (l_norm_total - 1/3)**2 + (r_norm_total - 1/3)**2)
    max_dist = np.sqrt(1/2)
    normalized_dist = center_dist / max_dist
    alpha = normalized_dist * 0.9 + 0.0
    alpha = np.clip(alpha, 0, 1)
    
    # Apply alpha
    colors_rgba[:, 3] = alpha
    
    return colors_rgba


In [ ]:
# no filtering applied
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']
for i in range(len(new_data_title)):
    if i == 0 :
        ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        lll = 100-ttt-rrr
    else:
        ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        lll = 100-ttt-rrr   

    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.show()

In [ ]:
# no filtering applied
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']
for i in range(len(new_data_title)):
    if i == 0 :
        ttt = obs_result.where(condition)['2nd'].mean().item()
        rrr = obs_result.where(condition)['1st'].mean().item()
        lll = 100-ttt-rrr
    else:
        ttt = result_ds_list[i-1].where(condition)['2nd'].mean().item()
        rrr = result_ds_list[i-1].where(condition)['1st'].mean().item()
        lll = 100-ttt-rrr   

    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# no filtering applied
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']
for i in range(len(new_data_title)):
    if i == 0 :
        # ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr

        ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1)&(obs_result['1st']>=50))['2nd'].mean().item()
        rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1)&(obs_result['1st']>=50))['1st'].mean().item()
        lll = 100-ttt-rrr
    else:
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr    #land
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['1st'].mean().item()

        ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(obs_result['1st']>=50))['2nd'].mean().item()
        rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(obs_result['1st']>=50))['1st'].mean().item()
        lll = 100-ttt-rrr   

    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# filtered by right mask
# obs unfiltered other datasets filtered by prior studies
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 30
lmax = 20
rmax = 90
tmin = 10
lmin = 0 
rmin = 70
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']
for i in range(len(new_data_title)):
    if i == 0 :
        # ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr

        ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1)&(obs_result['1st']>=50))['2nd'].mean().item()
        rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1)&(obs_result['1st']>=50))['1st'].mean().item()
        lll = 100-ttt-rrr
    else:
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr    #land
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['1st'].mean().item()

        ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(obs_result['1st']>=50))['2nd'].mean().item()
        rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(obs_result['1st']>=50))['1st'].mean().item()
        lll = 100-ttt-rrr   

    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# filtered by obs left mask original method
# exclude regions where obs 1st amplitude < 70
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 30
lmax = 20
rmax = 90
tmin = 10
lmin = 0 
rmin = 70
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']
for i in range(len(new_data_title)):
    if i == 0 :
        # ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr

        ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask']))['2nd'].mean().item()
        rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask']))['1st'].mean().item()
        lll = 100-ttt-rrr
    else:
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1))['1st'].mean().item()
        # lll = 100-ttt-rrr    #land
        # ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['2nd'].mean().item()
        # rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~result_ds_list[i-1]['right_mask']))['1st'].mean().item()

        ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask']))['2nd'].mean().item()
        rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask']))['1st'].mean().item()
        lll = 100-ttt-rrr   

    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_filter_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
new_data_title

In [ ]:
fig = plt.figure(figsize=(11, 11))
regions = ['NH', 'CONUS','EU', 'EA']
titles = ['(a) NH', '(b) CONUS', '(c) Europe', '(d) East Asia']
sliced = [[slice(None), slice(None)],
          [slice(-130, -60), slice(25, 50)],
          [slice(-20, 30), slice(30, 60)],
          [slice(70, 146), slice(20, 46)]]
def rgb_colormap(t, l, r, tmax, lmax, rmax, colors):
    """
    Custom colormap function that interpolates between three HEX-defined colors.
    """
    # Define rmin
    rmin = 65
    
    # Create masks for out-of-bounds regions
    t_out = t > tmax
    l_out = l > lmax
    r_out = r < rmin
    
    # Normalize the coordinates
    total = t + l + r
    t_norm = t / tmax
    l_norm = l / lmax 
    r_norm = (r - rmin) / (rmax - rmin)  # r 30-100  
    
    t_norm_total = t / total
    l_norm_total = l / total
    r_norm_total = r / total
    
    # Interpolate between the three corner colors
    color_top = np.array(to_rgb(colors['top']))     # HEX to RGB #2nd
    color_left = np.array(to_rgb(colors['left']))   # HEX to RGB
    color_right = np.array(to_rgb(colors['right'])) # HEX to RGB
    
    # Weighted average of the corner colors
    red = t_norm * color_top[0] + l_norm * color_left[0] + r_norm * color_right[0]
    green = t_norm * color_top[1] + l_norm * color_left[1] + r_norm * color_right[1]
    blue = t_norm * color_top[2] + l_norm * color_left[2] + r_norm * color_right[2]
    
    colors_rgba = np.column_stack([red, green, blue, np.ones_like(red)])
    colors_rgba[t_out] = [*color_top, 1]
    colors_rgba[l_out] = [*color_left, 1]
    colors_rgba[r_out] = [*color_right, 1]
    
    # Clip values
    colors_rgba = np.clip(colors_rgba, 0, 1)
    
    # Calculate alpha based on distance from center
    center_dist = np.sqrt((t_norm_total - 1/3)**2 + (l_norm_total - 1/3)**2 + (r_norm_total - 1/3)**2)
    max_dist = np.sqrt(1/2)
    normalized_dist = center_dist / max_dist
    alpha = normalized_dist * 0.9 + 0.0
    alpha = np.clip(alpha, 0, 1)
    
    # Apply alpha
    colors_rgba[:, 3] = alpha
    
    return colors_rgba
    
t, l, r, v = get_dirichlet_pdfs(n=200, alpha=(10, 10, 10))
tmax = 35
lmax = 35
rmax = 100
tmin = 0
lmin = 0 
rmin = 65

colors = rgb_colormap(t*100, l*100, r*100,tmax,lmax,rmax, colors_hex)

for idx, (region, title, slice_coords) in enumerate(zip(regions, titles, sliced), 1):
    ax = fig.add_subplot(2, 2, idx, projection='ternary', ternary_sum=100.0)
    
    ax.scatter(t, l, r, c=colors, s=50, marker='D')
    
    #   ...
    arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
    kwargs_arrow = {
        'transform': ax.transAxes,
        'arrowstyle': arrowstyle,
        'linewidth': 3,
        'clip_on': False,
        'zorder': -10,
    }
    
    ta = np.array([ 0.0, -13,  113])
    la = np.array([ 113,  0.0, -13])
    ra = np.array([-13,  113,  0.0])
    tb = np.array([ 100, -13,  13])
    lb = np.array([ 13,  100, -13])
    rb = np.array([-13,  13,  100])
    
    f = ax.transAxesProjection.transform
    
    tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
    larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
    rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
    ax.add_patch(tarrow)
    ax.add_patch(larrow)
    ax.add_patch(rarrow)
    
    kwargs_label = {'fontsize':13,
        'transform': ax.transTernaryAxes,
        'backgroundcolor': 'w',
        'ha': 'center',
        'va': 'center',
        'rotation_mode': 'anchor',
        'zorder': -9,
    }
    
    tpos = (ta + tb) * 0.5
    lpos = (la + lb) * 0.5
    rpos = (ra + rb) * 0.5
    
    ax.text(*tpos, '2nd harmonic amplitude [%]', color='black', rotation=-60, **kwargs_label)
    ax.text(*lpos, 'Higher harmonic amplitude [%]', color='black', rotation=60, **kwargs_label)
    ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=0, **kwargs_label)
    
    sec_legend = []
    for i in range(len(new_data_title)):
        ####
        ####
        if region == 'CONUS':
            if i == 0:
                ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask'])&conus_mask)['2nd'].mean().item()
                rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask'])&conus_mask)['1st'].mean().item()
            else:
                ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask'])&conus_mask)['2nd'].mean().item()
                rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask'])&conus_mask)['1st'].mean().item()
        else:
            
            if i == 0:
                ttt = obs_result.where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask'])).sel(longitude=slice_coords[0], latitude=slice_coords[1])['2nd'].mean().item()
                rrr = obs_result.where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask'])).sel(longitude=slice_coords[0], latitude=slice_coords[1])['1st'].mean().item()
            else:
                ttt = result_ds_list[i-1].where((~obs_result['2nd'].isnull())&(lsm==1)&(~obs_result['left_mask'])).sel(longitude=slice_coords[0], latitude=slice_coords[1])['2nd'].mean().item()
                rrr = result_ds_list[i-1].where((~obs_result['1st'].isnull())&(lsm==1)&(~obs_result['left_mask'])).sel(longitude=slice_coords[0], latitude=slice_coords[1])['1st'].mean().item()
        
        lll = 100-ttt-rrr
        
        ax.scatter(ttt, lll, rrr, marker=new_markers[i], edgecolor='black', 
                  color=data_colors[i], s=200, zorder=2, lw=.7, alpha=.8)
        
        if idx == 1:
            sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, 
                                       marker=new_markers[i], zorder=2, lw=.7, 
                                       alpha=1, label=new_data_title[i], 
                                       edgecolor='black'))
    
    ax.set_ternary_lim(0, 35, 0, 35, 65, 100)
    ax.taxis.set_label_position("tick1")
    ax.laxis.set_label_position("tick1")
    ax.raxis.set_label_position("tick1")
    
    ax.taxis.set_ticks(np.arange(0, 35, 10), labels=np.arange(0, 35, 10), fontsize=12)
    ax.laxis.set_ticks(np.arange(0, 35, 10), labels=np.arange(0, 35, 10), fontsize=12)
    ax.raxis.set_ticks(np.arange(70, 101, 10), labels=np.arange(70, 101, 10), fontsize=12)
    
    ax.set_title(title, fontsize=17, pad=50)
    ax.grid()
    
    if idx == 1:
        legend2 = ax.legend(handles=sec_legend, loc='upper left', 
                          bbox_to_anchor=(.9, -.1), fontsize=15, frameon=False)

# plt.tight_layout()
plt.subplots_adjust(wspace = .5,hspace = .5)
# plt.show()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_region.png',format='png', dpi=300,bbox_inches='tight')

## 3. Harmonic disagreement[]


In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc')  # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
sat_mean = xr.concat([imerg,trmm,cmorph, gsmap,mswep],'data').mean('data')#,skipna=False)
rean_mean = xr.concat([era5,jra55,merra2],'data').mean('data')#,skipna=False)

In [ ]:
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
xr.corr( sat_mean['SAD']<4,sat_mean['1st']>70,['longitude','latitude'])

In [ ]:
xr.corr( rean_mean['SAD']<4,rean_mean['1st']>70,['longitude','latitude'])

In [ ]:
sat_mean['1st'].plot()

In [ ]:
rean_mean['1st'].plot()

In [ ]:
def HD_figure(dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extend,alb):

    with ProgressBar():  #  ?

        #utc
        nan_mask = np.isnan(dataset[var]) 
        hatch_mask = dataset['1st']<50    ##['SAD']>=4  
        tick_spacing = 30

        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        # fig = plt.figure(figsize=(20,10))

        # varMin, varMax, varInt = 0, 24, 0.5
        levels = np.arange(varMin, varMax+varInt, varInt)
        nlevs  = levels.size
        # tick_interval = 2
        # cmap = plt.get_cmap(cmap, nlevs)
        # cmap = plt.get_cm'Accent', nlevs)

        extent=[-180, 180, -60, 60]
        # crs = ccrs.PlateCarree()
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=crs)
        
        #  gridlines labels 
        x_ticks = [-180, -120, -60, 0, 60, 120, 180]
        y_ticks = [-60, -30, 0, 30, 60]
        
        axis.set_xticks(x_ticks, crs=crs)
        axis.set_yticks(y_ticks, crs=crs)
        axis.xaxis.set_major_formatter(LongitudeFormatter())
        axis.yaxis.set_major_formatter(LatitudeFormatter())
        axis.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)

        lons = dataset['longitude']
        lats = dataset['latitude']
        if var == 'lst':

            values = dataset[var]
            cbarlabel = 'Local Solar Time of Max[hour]'

        elif var == 'new_harm_amp' :
            values = dataset[var]
            cbarlabel = '[mm/3h]'
        else:
            values = dataset[var]
            cbarlabel = '[-]'
        lons, lats = np.meshgrid(dataset['longitude'], dataset['latitude'])

        cnplot = axis.contourf(lons, lats, values,cmap=cmap,levels=levels,zorder=0,transform=ccrs.PlateCarree(),extend =extend)

        # cbar = plt.colorbar(cnplot,ticks=np.arange(varMin, varMax+tick_interval, tick_interval), 
        #                     orientation='vertical', pad=0.01, shrink=.658) 

        axis.contourf(
            lons, lats, (nan_mask==True),
            transform=ccrs.PlateCarree(),
            colors='gray',
            levels=[0.5, 1.5],
        )  # #    


        axis.contourf(
            lons, lats, hatch_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['\\\\\\'],
        ) 
        axis.set_title(title ,fontsize=21)
        # axis.text(-150,51,alb ,fontsize=19,ha='left', va='bottom')

        return(cnplot)
        # plt.show()
        


In [ ]:
# result_ds_list = [cmorph,imerg ,sat_mean,
#                   trmm,gsmap,mswep,
#                   jra55,era5,merra2]

# phase_title = ['CMORPH [1998-2022]','IMERG [2000-2023]','Multi-Satellite',
#                'TRMM [2000-2019]','GSMaP [1998-2022]','MSWEP [2000-2023]',
#                'JRA-3Q [2000-2023]','ERA5 [2000-2023]','MERRA2 [2000-2023]']

result_ds_list = [sat_mean,rean_mean]
phase_title = ['(a) Multi-Satellite (MS)','(b) Multi-Model (MM)']

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(11 ,8.5))
axs = []
nrows, ncols = 2, 1
# extent= eu
var = '1st'
varMin, varMax, varInt =20, 100., 1.
tick_interval = 10
extend = 'min'

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
# cmap = 'RdBu_r'
cmap = cmaps.torch
# cmap = cmaps.lipari
# cmap = cmaps.roma_r
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # (dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extent)
    # cnplot = diurnal_cycle_figure(result_ds_list[i],phase_title[i],var,'YlOrBr',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])

''' '''
pos1 = axs[0].get_position()
pos2 = axs[1].get_position()

cax = fig.add_axes([pos1.x1+.1 , (pos1.y1-pos2.y0)/3, 0.03, (pos1.y1-pos2.y0)*(2/3)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label('$rat_1 $ [-]', fontsize=12)
cbar1.ax.tick_params(labelsize=11)

# plt.subplots_adjust(wspace = .01,hspace = -.4)
# axs[2].set_visible(False) 
# cbar_ax.set_visible(False)
plt.tight_layout()

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/rat_1_fig.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
def HD_figure(dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extend,alb):

    with ProgressBar():  #  ?

        #utc
        nan_mask = np.isnan(dataset[var]) 
        hatch_mask = dataset['SAD']>=4  # dataset['left_mask'] #ERA5 20%  
        tick_spacing = 30

        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        # fig = plt.figure(figsize=(20,10))

        # varMin, varMax, varInt = 0, 24, 0.5
        levels = np.arange(varMin, varMax+varInt, varInt)
        nlevs  = levels.size
        # tick_interval = 2
        # cmap = plt.get_cmap(cmap, nlevs)
        # cmap = plt.get_cm'Accent', nlevs)

        extent=[-180, 180, -60, 60]
        # crs = ccrs.PlateCarree()
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=crs)
        
        #  gridlines labels 
        x_ticks = [-180, -120, -60, 0, 60, 120, 180]
        y_ticks = [-60, -30, 0, 30, 60]
        
        axis.set_xticks(x_ticks, crs=crs)
        axis.set_yticks(y_ticks, crs=crs)
        axis.xaxis.set_major_formatter(LongitudeFormatter())
        axis.yaxis.set_major_formatter(LatitudeFormatter())
        axis.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)

        lons = dataset['longitude']
        lats = dataset['latitude']
        if var == 'lst':

            values = dataset[var]
            cbarlabel = 'Local Solar Time of Max[hour]'

        elif var == 'new_harm_amp' :
            values = dataset[var]
            cbarlabel = '[mm/3h]'
        else:
            values = dataset[var]
            cbarlabel = '[-]'
        lons, lats = np.meshgrid(dataset['longitude'], dataset['latitude'])

        cnplot = axis.contourf(lons, lats, values,cmap=cmap,levels=levels,zorder=0,transform=ccrs.PlateCarree(),extend =extend)

        # cbar = plt.colorbar(cnplot,ticks=np.arange(varMin, varMax+tick_interval, tick_interval), 
        #                     orientation='vertical', pad=0.01, shrink=.658) 

        axis.contourf(
            lons, lats, (nan_mask==True),
            transform=ccrs.PlateCarree(),
            colors='gray',
            levels=[0.5, 1.5],
        )  # #    


        axis.contourf(
            lons, lats, hatch_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['\\\\\\'],
        ) 
        axis.set_title(title ,fontsize=21)
        # axis.text(-150,51,alb ,fontsize=19,ha='left', va='bottom')

        return(cnplot)
        # plt.show()
        


In [ ]:
# result_ds_list = [cmorph,imerg ,sat_mean,
#                   trmm,gsmap,mswep,
#                   jra55,era5,merra2]

# phase_title = ['CMORPH [1998-2022]','IMERG [2000-2023]','Multi-Satellite',
#                'TRMM [2000-2019]','GSMaP [1998-2022]','MSWEP [2000-2023]',
#                'JRA-3Q [2000-2023]','ERA5 [2000-2023]','MERRA2 [2000-2023]']

result_ds_list = [sat_mean,rean_mean]
phase_title = ['(a) Multi-Satellite(MS)','(b) Multi-Model(MM)']

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(11 ,8.5))
axs = []
nrows, ncols = 2, 1
# extent= eu
var = 'SAD'
varMin, varMax, varInt =1, 7., .25
tick_interval = 1
extend = 'both'

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
# cmap = 'RdBu_r'
cmap = cmaps.BlueWhiteOrangeRed
# cmap = cmaps.roma_r
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # (dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extent)
    # cnplot = diurnal_cycle_figure(result_ds_list[i],phase_title[i],var,'YlOrBr',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])

''' '''
pos1 = axs[0].get_position()
pos2 = axs[1].get_position()

cax = fig.add_axes([pos1.x1+.1 , (pos1.y1-pos2.y0)/3, 0.03, (pos1.y1-pos2.y0)*(2/3)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label('$HD_1 $ [-]', fontsize=12)
cbar1.ax.tick_params(labelsize=11)

# plt.subplots_adjust(wspace = .01,hspace = -.4)
# axs[2].set_visible(False) 
# cbar_ax.set_visible(False)
plt.tight_layout()

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/HD_figure_mean_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(11 ,8.5))
axs = []
nrows, ncols = 2, 1
# extent= eu
var = 'SAD'
varMin, varMax, varInt =1, 7., .25
tick_interval = 1
extend = 'both'

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
cmap = 'RdBu_r'
# cmap = cmaps.BlueWhiteOrangeRed
# cmap = cmaps.lipari_r
# cmap = cmaps.roma_r
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # (dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extent)
    # cnplot = diurnal_cycle_figure(result_ds_list[i],phase_title[i],var,'YlOrBr',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])

''' '''
pos1 = axs[0].get_position()
pos2 = axs[1].get_position()

cax = fig.add_axes([pos1.x1+.1 , (pos1.y1-pos2.y0)/3, 0.03, (pos1.y1-pos2.y0)*(2/3)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label('$HD_1 $ [-]', fontsize=12)
cbar1.ax.tick_params(labelsize=11)

# plt.subplots_adjust(wspace = .01,hspace = -.4)
# axs[2].set_visible(False) 
# cbar_ax.set_visible(False)
plt.tight_layout()

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/HD_figure_mean_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(11 ,6))
axs = []
nrows, ncols = 2, 1
# extent= eu
var = 'SAD'
varMin, varMax, varInt =1, 7., .25
tick_interval = 1
extend = 'both'

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
# cmap = 'RdBu_r'
cmap = cmaps.BlueWhiteOrangeRed
# cmap = cmaps.roma_r
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # (dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extent)
    # cnplot = diurnal_cycle_figure(result_ds_list[i],phase_title[i],var,'YlOrBr',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])

''' '''
pos1 = axs[0].get_position()
pos2 = axs[1].get_position()

cax = fig.add_axes([pos1.x1+.06 , (pos1.y1-pos2.y0)/3, 0.03, (pos1.y1-pos2.y0)*(2/3)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label('$HD_1 $ [-]', fontsize=15)
cbar1.ax.tick_params(labelsize=15)

# plt.subplots_adjust(wspace = .01,hspace = -.4)
# axs[2].set_visible(False)
# cbar_ax.set_visible(False)
plt.tight_layout()

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/HD_figure_mean_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
# extent= eu
var = 'SAD'
varMin, varMax, varInt =0, 8., .25
tick_interval = 2
extend = 'neither'

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
# cmap = 'RdBu_r'
cmap = cmaps.BlueWhiteOrangeRed
# cmap = cmaps.roma_r
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # (dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extent)
    # cnplot = diurnal_cycle_figure(result_ds_list[i],phase_title[i],var,'YlOrBr',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])


cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar.set_label('[-]', fontsize=15)
cbar.ax.tick_params(labelsize=15)

''' '''
pos1 = axs[1].get_position()
cax = fig.add_axes([pos1.x1+.016 , pos2.y0-.01, 0.014, (pos1.y1-pos1.y0+.01)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label('$HD_1 $ [-]', fontsize=15)
cbar1.ax.tick_params(labelsize=15)

plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
cbar_ax.set_visible(False)

# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/HD_figure_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

### Updated Version


In [ ]:
#rat_subD
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc')  # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
sat_mean_rat1 = xr.concat([imerg,trmm,cmorph, gsmap,mswep],'data').mean('data')#,skipna=False)
rean_mean_rat1 = xr.concat([era5,jra55,merra2],'data').mean('data')#,skipna=False)

In [ ]:
sat_mean_rat1 = sat_mean_rat1.clip(0,100)
rean_mean_rat1 = rean_mean_rat1.clip(0,100)

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
sat_mean['total_var'].plot()

In [ ]:
(((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100))*100).plot()

In [ ]:
(((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100))*100).plot()

In [ ]:
def HD_figure(dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extend,alb):

    with ProgressBar():  #  ?

        #utc
        nan_mask = np.isnan(dataset) 
        # hatch_mask = dataset['1st']<50    ##['SAD']>=4  
        tick_spacing = 30

        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        # fig = plt.figure(figsize=(20,10))

        # varMin, varMax, varInt = 0, 24, 0.5
        levels = np.arange(varMin, varMax+varInt, varInt)
        nlevs  = levels.size
        # tick_interval = 2
        # cmap = plt.get_cmap(cmap, nlevs)
        # cmap = plt.get_cm'Accent', nlevs)

        extent=[-180, 180, -60, 60]
        # crs = ccrs.PlateCarree()
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=crs)
        
        #  gridlines labels 
        x_ticks = [-180, -120, -60, 0, 60, 120, 180]
        y_ticks = [-60, -30, 0, 30, 60]
        
        axis.set_xticks(x_ticks, crs=crs)
        axis.set_yticks(y_ticks, crs=crs)
        axis.xaxis.set_major_formatter(LongitudeFormatter())
        axis.yaxis.set_major_formatter(LatitudeFormatter())
        axis.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)

        lons = dataset['longitude']
        lats = dataset['latitude']
        if var == 'lst':

            values = dataset
            cbarlabel = 'Local Solar Time of Max[hour]'

        elif var == 'new_harm_amp' :
            values = dataset
            # cbarlabel = '[mm/3h]'
        else:
            values = dataset
            # cbarlabel = '[%]'
        lons, lats = np.meshgrid(dataset['longitude'], dataset['latitude'])

        cnplot = axis.contourf(lons, lats, values,cmap=cmap,levels=levels,zorder=0,transform=ccrs.PlateCarree(),extend =extend)

        axis.contourf(
            lons, lats, (nan_mask==True),
            transform=ccrs.PlateCarree(),
            colors='gray',
            levels=[0.5, 1.5],
        )  # #    


        # axis.contourf(
        #     lons, lats, hatch_mask,
        #     transform=ccrs.PlateCarree(),
        #     colors='none',
        #     levels=[0.5,1.5],
        #     hatches=['\\\\\\'],
        # ) 
        axis.set_title(title ,fontsize=21)
        # axis.text(-150,51,alb ,fontsize=19,ha='left', va='bottom')

        return(cnplot)
        # plt.show()

In [ ]:
Colormap.

In [ ]:
(((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100))*100).plot(vmin = 20, vmax =  70, cmap = Colormap('colorcet:CET_L17').to_mpl())

In [ ]:
import cmocean
import colormaps as cmaps

fig = plt.figure(figsize=(11 ,8.5))
axs = []
nrows, ncols = 2, 1
var = '1st'
varMin, varMax, varInt =20, 60., 2.5
tick_interval = 10
extend = 'both'
result_ds_list = [(((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100))*100),
                  (((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100))*100)]
alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
           '(f)','(g)','(h)']
# cmap = 'RdBu_r'
# cmap = cmaps.torch
# cmap = cmaps.lipari
cmap = Colormap('colorcet:CET_L17').to_mpl()
# cmap  = cmaps.vik
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot = HD_figure(result_ds_list[i],phase_title[i],var,cmap,axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])

''' '''
pos1 = axs[0].get_position()
pos2 = axs[1].get_position()

cax = fig.add_axes([pos1.x1+.1 , (pos1.y1-pos2.y0)/3, 0.03, (pos1.y1-pos2.y0)*(2/3)])
cbar1 = plt.colorbar(cnplot, cax=cax, orientation='vertical',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)
cbar1.set_label(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
cbar1.ax.tick_params(labelsize=11)

plt.tight_layout()

# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/rat_1_fig.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(20, 60, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    # hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    # hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    # ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
    #            hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    # if ref_data is not None:
    #     mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
    #     obs = ref_data.where(mask)['hfr']
    #     model = ds.where(mask)[var]
        
    #     r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
    #     mae = np.mean(np.abs(model - obs)).item()
        
    #     stats_text = f"r = {r:.2f}"
    #     # stats_text = f"r = {r:.2f}\n" \
    #     #              f"MAE = {mae:.2f}%"
    # #      
    #     ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
    #             # fontsize=13, 
    #             # transform=ccrs.PlateCarree())
    #             fontsize=13, bbox=dict(facecolor='white', alpha=.8),
    #             transform=ccrs.PlateCarree())

    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,30))
    ax.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')

result_ds_list = [(((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100))*100),
                  (((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100))*100)]

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    # cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    cmap = Colormap('colorcet:CET_L17').to_mpl()
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name = 'rat'), 'rat', lsm, extent,True)
    ax1_hist.set_xlim(10, 60)  #  x  
    ax1_hist.set_xticks([20,30,40,50])
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name = 'rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(10, 60)  #  x  
    ax2_hist.set_xticks([20,30,40,50])
    ax2_hist.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=11)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(20, 60, 5)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0)-.06,0.03])
    # cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])  #  
    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(20, 60, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    # hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    # hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    # ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
    #            hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    # if ref_data is not None:
    #     mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
    #     obs = ref_data.where(mask)['hfr']
    #     model = ds.where(mask)[var]
        
    #     r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
    #     mae = np.mean(np.abs(model - obs)).item()
        
    #     stats_text = f"r = {r:.2f}"
    #     # stats_text = f"r = {r:.2f}\n" \
    #     #              f"MAE = {mae:.2f}%"
    # #      
    #     ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
    #             # fontsize=13, 
    #             # transform=ccrs.PlateCarree())
    #             fontsize=13, bbox=dict(facecolor='white', alpha=.8),
    #             transform=ccrs.PlateCarree())

    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,30))
    ax.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')

result_ds_list = [(((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100))*100),
                  (((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100))*100)]

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    # cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    cmap = Colormap('colorcet:CET_L17').to_mpl()
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name = 'rat'), 'rat', lsm, extent,True)
    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name = 'rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    ax2_hist.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=11)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(20, 60, 5)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0)-.06,0.03])
    # cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])  #  
    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
result_ds_list = [(sat_mean['total_var']*((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100)))/9,
                  (rean_mean['total_var']*((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100)))/9]



In [ ]:
result_ds_list[0].plot(vmin = 0, vmax = 2, cmap = Colormap('colorcet:CET_L17').to_mpl())

In [ ]:
result_ds_list[1].plot(vmin = 0, vmax = 2, cmap = Colormap('colorcet:CET_L17').to_mpl())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(0, 2, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    # hatch_mask = (~np.isnan(ds[var])) & (ds[var] < 30)
    
    #  True  1,  NaN 
    # hatch_data = xr.where(hatch_mask, 1, np.nan)
    
    # contourf    (  )
    # ax.contourf(ds.longitude, ds.latitude, hatch_data, levels=[0.5, 1.5], 
    #            hatches=['xxx'], colors='none', transform=ccrs.PlateCarree())

    # if ref_data is not None:
    #     mask = ~(ref_data['hfr'].isnull() | ds[var].isnull())
        
    #     obs = ref_data.where(mask)['hfr']
    #     model = ds.where(mask)[var]
        
    #     r = xr.corr(obs, model, dim=['longitude', 'latitude']).item()
    #     mae = np.mean(np.abs(model - obs)).item()
        
    #     stats_text = f"r = {r:.2f}"
    #     # stats_text = f"r = {r:.2f}\n" \
    #     #              f"MAE = {mae:.2f}%"
    # #      
    #     ax.text(extent[0] + 5, extent[2] + 5, stats_text, 
    #             # fontsize=13, 
    #             # transform=ccrs.PlateCarree())
    #             fontsize=13, bbox=dict(facecolor='white', alpha=.8),
    #             transform=ccrs.PlateCarree())

    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,30))
    ax.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')

result_ds_list = [(sat_mean['total_var']*((sat_mean_rat1['1st']/100)*(sat_mean['hfr_tp']/100)))/9,
                  (rean_mean['total_var']*((rean_mean_rat1['1st']/100)*(rean_mean['hfr_tp']/100)))/9]

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    # cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    cmap = Colormap('colorcet:CET_L17').to_mpl()
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name = 'rat'), 'rat', lsm, extent,True)
    ax1_hist.set_xlim(0, 1)  #  x  
    ax1_hist.set_xticks([0.5])
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name = 'rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 1)  #  x  
    ax2_hist.set_xticks([0.5])
    ax2_hist.set_xlabel(r'$\sigma_{1st}$ [$mm^2/h^2$]', fontsize=11)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(0, 2, 5)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0)-.06,0.03])
    # cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])  #  
    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='max', ticks=cbar_ticks)
    cbar.set_label(r'$\sigma_{1st}$ [$mm^2/h^2$]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()
    # both
if __name__ == "__main__":
    main()

## 4. Harmonic Phase []
'


In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc')  # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
result_ds_list = [cmorph,imerg ,sat_mean,
                  trmm,gsmap,mswep,
                  jra55,era5,merra2]

In [ ]:
phase_title = ['(a) CMORPH [1998-2022]','(b) IMERG [2000-2023]','Multi-Satellite',
               '(c) TRMM [2000-2019]','(d) GSMaP [1998-2022]','(e) MSWEP [2000-2023]',
               '(f) JRA-3Q [2000-2023]','(g) ERA5 [2000-2023]','(h) MERRA-2 [2000-2023]']

### Phase figure


In [ ]:
(cmorph['SAD']>4).plot()

In [ ]:
def diurnal_cycle_figure(dataset,title,utc_or_lst,cmap,axis,alb):
# HD 4  = //
#   = \\
#    = X
    with ProgressBar():  #  ?

        #utc
        nan_mask = np.isnan(dataset['lst']) 
        left_mask = dataset['SAD']>4#dataset['left_mask'] 
        right_mask = dataset['right_mask'] 
        both_mask = left_mask&right_mask

        tick_spacing = 30

        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        # fig = plt.figure(figsize=(20,10))

        varMin, varMax, varInt = 0, 24, 0.5
        levels = np.arange(varMin, varMax+varInt, varInt)
        nlevs  = levels.size
        tick_interval = 2

        extent=[-180, 180, -60, 60]
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=1, linestyle='-.')

        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None

        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())

        lons = dataset['longitude']
        lats = dataset['latitude']
        if utc_or_lst == 'lst':

            values = dataset['lst']
            cbarlabel = 'Local Solar Time of Max[hour]'

        else :
            values = dataset['utc']
            cbarlabel = 'UTC[hour]'
        lons, lats = np.meshgrid(dataset['longitude'], dataset['latitude'])

        cnplot = axis.contourf(lons, lats, values,cmap=cmap,levels=levels,zorder=0,transform=ccrs.PlateCarree())


        axis.contourf(
            lons, lats, (nan_mask==True),
            transform=ccrs.PlateCarree(),
            colors='gray',
            levels=[0.5, 1.5]
        )  # #    


        axis.contourf(
            lons, lats, left_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['///']
        ) 
        axis.contourf(
            lons, lats, right_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=["\\\\\\"]
        ) 
        axis.contourf(
            lons, lats, both_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['xxx']
        ) 
        axis.set_title(title ,fontsize=18)
        axis.text(-180,61,alb ,fontsize=19,ha='left', va='bottom')
        return(cnplot)
        


In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 2,1
extent= [-180,180,-60,70]
var = 'lst'
varMin, varMax= 0, 24

alphabet = ['(a)','(b)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot =diurnal_cycle_figure(result_ds_list[i],phase_title[i],'lst',srtoss,axs[i],alphabet[i])
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
extent= [-180,180,-60,70]
var = 'lst'
varMin, varMax= 0, 24

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
            '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot =diurnal_cycle_figure(result_ds_list[i],phase_title[i],'lst',srtoss,axs[i],alphabet[i])
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
extent= [-180,180,-60,70]
var = 'lst'
varMin, varMax= 0, 24

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
            '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot =diurnal_cycle_figure(result_ds_list[i],phase_title[i],'lst',srtoss,axs[i],alphabet[i])
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
extent= [-180,180,-60,70]
var = 'lst'
varMin, varMax= 0, 24

alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
            '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot =diurnal_cycle_figure(result_ds_list[i],phase_title[i],'lst',diurnal_cmap,axs[i],alphabet[i])
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import PolyCollection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import PolyCollection

def circle_sine_ramp(r_max=1, r_min=0.0, amp=np.pi/5, cycles=50, power=2, nr=50, ntheta=1025):
    r, t = np.mgrid[r_min:r_max:nr*1j, 0:2*np.pi:ntheta*1j]
    r_norm = (r - r_min)/(r_max - r_min)
    vals = amp * r_norm**power * np.sin(cycles*t) + t
    vals = np.mod(vals, 2*np.pi)
    return t, r, vals

from matplotlib import patheffects

def circle_sine_ramp(r_max=1, r_min=0.0, amp=np.pi/5, cycles=50, power=2, nr=50, ntheta=1025):
    r, t = np.mgrid[r_min:r_max:nr*1j, 0:2*np.pi:ntheta*1j]
    r_norm = (r - r_min)/(r_max - r_min)
    vals = amp * r_norm**power * np.sin(cycles*t) + t
    vals = np.mod(vals, 2*np.pi)
    return t, r, vals


def plot_circular_colormap(ax,r_max, r_min=0.0,nr=6, ntheta=96, cmap=None, figsize=(3, 3), show_labels=True, 
                          alpha_center=0.2, alpha_edge=1.0, fontsize=20):
    """
    Function to draw circular colormap
    
    Parameters:
    -----------
    nr : int
        Radial resolution (default: 6)
    ntheta : int
        Angular resolution (default: 96, 24multiples recommended)
    cmap : matplotlib colormap or None
        Colormap to use (Noneuse default sunset colormap)
    figsize : tuple
        Figure size (default: (3, 3))
    show_labels : bool
        Whether to show time labels (default: True)
    alpha_center : float
          (default: 0.2)
    alpha_edge : float
          (default: 1.0)
    fontsize : int
           (default: 15)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    t_small, r_small, vals_small = circle_sine_ramp(r_max=r_max, r_min=r_min,cycles=0, nr=nr, ntheta=ntheta)
    
    # 24  
    segment_size = 2 * np.pi / 24
    vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)
    
    center_distance_small = r_small / np.max(r_small)
    alpha_range = alpha_edge - alpha_center
    alpha_values_small = center_distance_small * alpha_range + alpha_center
    
    # RGB  
    colors_rgb_small = cmap(vals_quantized_normalized_small)
    
    polygons_small = []
    rgba_colors_small = []
    nr_small, ntheta_small = t_small.shape
    
    for i in range(nr_small-1):
        for j in range(ntheta_small-1):
            corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
            corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
            
            #   :   π/2  12  0  
            corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
            
            corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            
            polygon = list(zip(corners_x, corners_y))
            polygons_small.append(polygon)
            
            cell_rgba = [
                colors_rgb_small[i, j, 0],
                colors_rgb_small[i, j, 1], 
                colors_rgb_small[i, j, 2],
                alpha_values_small[i, j]
            ]
            rgba_colors_small.append(cell_rgba)
    
    collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
    ax.add_collection(collection_final)
    
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.axis('off')
    
    if show_labels:
        for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
            #  : 12 (π/2)   
            angle = np.pi/2 - i * 2 * np.pi / 8  # subtract angle for clockwise direction
            x = (r_max+.15) * np.cos(angle)
            y = (r_max+.15) * np.sin(angle)
            ax.text(x, y, hour, ha='center', va='center', fontsize=fontsize)
    # ** :   **
    if show_labels:
        # amplitude  
        r_values = np.linspace(r_min, r_max, nr)  # nr+1 ()
        
        # (00 )   
        for r_val in r_values[1:]:  # r_min=0 
            x = 0
            y = r_val - 0.04
            # ax.text(x, y, f'{r_val:.1f}', ha='center', va='top', 
            #        fontsize=fontsize-6, color='grey')
            ax.text(x, y, f'{r_val:.1f}', ha='center', va='top', 
                   fontsize=fontsize-6, color='grey',
                   path_effects=[patheffects.withStroke(linewidth=3, foreground='white')])
        #     ()
        ax.plot([0, 0], [0, +r_max], ls = '-',c = 'grey', linewidth=.6, alpha=0.6)
        
        # # "Amplitude"  
        # ax.text(0, -r_max - 0.2, 'Amplitude', ha='center', va='top', 
        #        fontsize=fontsize-1, fontweight='bold')
        
    plt.tight_layout()
    return fig, ax

diurnal_cmap = LinearSegmentedColormap.from_list('diurnal', 
                                               [
                                               "#ff0000","#94001e",  "#FF4800", "#fffb00",
                                               "#00ffc3", "#3494de", "#2b00ff", "#ae00b7", 
                                               "#ff0033"
                                               ], 
                                               N=48)

fig, ax = plot_circular_colormap(r_max=.7, r_min=0.0,nr=6, ntheta=24*4, alpha_center=0.2, fontsize=12,alpha_edge=1.2,cmap=diurnal_cmap)
plt.show()

In [ ]:
fig, ax = plot_circular_colormap(r_max=.6, r_min=0.0,nr=4, ntheta=24*4, alpha_center=0.4, fontsize=12,alpha_edge=1.2,cmap=diurnal_cmap)
plt.show()

In [ ]:
def circular_colormap_mapping(phase, amplitude, max_amp=0.7, cmap=None):
    """
    Convert Phase and Amplitude to circular colormap RGBA values
    
    Parameters:
    -----------
    phase : array-like
        Phase value (0-24 )
    amplitude : array-like
        Amplitude value
    max_amp : float
        Maximum amplitude (normalization reference)
    cmap : matplotlib colormap
        Colormap to use
        
    Returns:
    --------
    rgba_array : array
        RGBA color array (shape: [..., 4])
    """    
    # Phase 0-1   (24 -> 1)
    phase_normalized = (phase % 24) / 24.0
    
    # Amplitude 0-1  
    amplitude_normalized = np.clip(amplitude / max_amp, 0, 1)
    
    # Phase   
    colors = cmap(phase_normalized)
    
    #  amplitude  
    # amplitude 0 (alpha=0.1), max_amp  (alpha=1.0)
    alphas = amplitude_normalized * 0.9 + 0.1
    
    # RGBA  
    rgba_array = np.zeros((*phase.shape, 4))
    rgba_array[..., :3] = colors[..., :3]  # RGB
    rgba_array[..., 3] = alphas  # Alpha
    
    return rgba_array
def create_phase_amplitude_rgba_array(dataset, phase_var='lst', amp_var='amplitude', max_amp=0.7,cmap=None):
    """
    Generate Phase-Amplitude RGBA array from Dataset
    
    Parameters:
    -----------
    dataset : xarray Dataset
        
    phase_var : str
         
    amp_var : str
         
    max_amp : float
        Maximum amplitude
        
    Returns:
    --------
    rgba_array : xarray DataArray
        RGBA color array
    """    
    phases = dataset[phase_var].values
    amplitudes = dataset[amp_var].values
    
    # NaN 
    nan_mask = np.isnan(phases) | np.isnan(amplitudes)
    
    rgba_colors = circular_colormap_mapping(phases, amplitudes, max_amp, cmap)
    
    # NaN   (, )
    rgba_colors[nan_mask] = [0.5, 0.5, 0.5, 0.3]
    
    # xarray DataArray 
    rgba_array = xr.DataArray(
        data=rgba_colors,
        dims=["latitude", "longitude", "rgba"],
        coords=dict(
            latitude=(["latitude"], dataset.latitude.values),
            longitude=(["longitude"], dataset.longitude.values),
            rgba=(["rgba"], ["R", "G", "B", "A"]),
        ),
    )
    
    return rgba_array
    
def phase_amplitude_figure(dataset, title, axis,phase_var='lst', amp_var='amplitude', 
                          max_amp=0.7,  alb='',cmap = diurnal_cmap):
    """
    Function to visualize Phase and Amplitude on a single map (using imshow)
    """
        
    with ProgressBar():
        left_mask = dataset['SAD'] > 4
        right_mask = dataset['right_mask'] 
        both_mask = dataset['both_mask'] 
        
        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        extent = [-180, 180, -60, 60]
        
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, 
                           color='gray', alpha=1, linestyle='-.')
        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), 
                        edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())
        
        # RGBA  
        rgba_array = create_phase_amplitude_rgba_array(dataset, phase_var, amp_var, max_amp,cmap)
        
        lons = dataset['longitude'].values
        lats = dataset['latitude'].values
        extent_data = [lons.min(), lons.max(), lats.min(), lats.max()]
        # data_T = rgba_array[::-1,:,:]

        # imshow 
        im = axis.imshow(rgba_array[::-1,:,:].values, 
                        extent=extent_data,
                        transform=ccrs.PlateCarree(),
                        origin='upper',
                        aspect='equal')
        
        #   ()
        lons_mesh, lats_mesh = np.meshgrid(lons, lats)
        
        axis.contourf(lons_mesh, lats_mesh, left_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['///'])
        
        axis.contourf(lons_mesh, lats_mesh, right_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=["\\\\\\"])
        
        axis.contourf(lons_mesh, lats_mesh, both_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['xxx'])
        
        axis.set_title(title, fontsize=18)
        axis.text(-180, 61, alb, fontsize=19, ha='left', va='bottom')
        
        return im

In [ ]:
def plot_circular_colormap(ax,r_max=1, r_min=0.0,nr=6, ntheta=96, cmap=None, figsize=(3, 3), show_labels=True, 
                          alpha_center=0.2, alpha_edge=1.0, fontsize=20):
    """
    Function to draw circular colormap
    
    Parameters:
    -----------
    nr : int
        Radial resolution (default: 6)
    ntheta : int
        Angular resolution (default: 96, 24multiples recommended)
    cmap : matplotlib colormap or None
        Colormap to use (Noneuse default sunset colormap)
    figsize : tuple
        Figure size (default: (3, 3))
    show_labels : bool
        Whether to show time labels (default: True)
    alpha_center : float
          (default: 0.2)
    alpha_edge : float
          (default: 1.0)
    fontsize : int
           (default: 15)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    t_small, r_small, vals_small = circle_sine_ramp(r_max=1, r_min=0.0,cycles=0, nr=nr, ntheta=ntheta)
    
    # 24  
    segment_size = 2 * np.pi / 24
    vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)
    
    center_distance_small = r_small / np.max(r_small)
    alpha_range = alpha_edge - alpha_center
    alpha_values_small = center_distance_small * alpha_range + alpha_center
    
    # RGB  
    colors_rgb_small = cmap(vals_quantized_normalized_small)
    
    polygons_small = []
    rgba_colors_small = []
    nr_small, ntheta_small = t_small.shape
    
    for i in range(nr_small-1):
        for j in range(ntheta_small-1):
            corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
            corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
            
            #   :   π/2  12  0  
            corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
            
            corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            
            polygon = list(zip(corners_x, corners_y))
            polygons_small.append(polygon)
            
            cell_rgba = [
                colors_rgb_small[i, j, 0],
                colors_rgb_small[i, j, 1], 
                colors_rgb_small[i, j, 2],
                alpha_values_small[i, j]
            ]
            rgba_colors_small.append(cell_rgba)
    
    collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
    ax.add_collection(collection_final)
    
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.axis('off')
    
    if show_labels:
        for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
            #  : 12 (π/2)   
            angle = np.pi/2 - i * 2 * np.pi / 8  # subtract angle for clockwise direction
            x = 1.15 * np.cos(angle)
            y = 1.15 * np.sin(angle)
            ax.text(x, y, hour, ha='center', va='center', fontsize=fontsize)
    
    plt.tight_layout()
    # return fig, ax

diurnal_cmap = LinearSegmentedColormap.from_list('diurnal', 
                                               [
                                               "#ff0000","#94001e",  "#FF4800", "#fffb00",
                                               "#00ffc3", "#3494de", "#2b00ff", "#ae00b7", 
                                               "#ff0033"
                                               ], 
                                               N=48)


In [ ]:
def circular_colormap_mapping(phase, amplitude, max_amp=0.6, cmap=None, 
                             alpha_center=0.2, alpha_edge=1.0, nr=3):
    """
    Convert Phase and Amplitude to circular colormap RGBA values
    """    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    # Phase    24  
    phase_rad = (phase % 24) * 2 * np.pi / 24
    segment_size = 2 * np.pi / 24
    vals_quantized = np.floor(phase_rad / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized = vals_quantized / (2 * np.pi)
    colors = cmap(vals_quantized_normalized)
    
    # Amplitude 3  
    amplitude_clipped = np.clip(amplitude, 0, max_amp)
    step_size = max_amp / nr  # nr=3, max_amp=0.6 → step_size=0.2
    
    #    (0, 1, 2)
    bin_indices = np.floor(amplitude_clipped / step_size).astype(int)
    bin_indices = np.clip(bin_indices, 0, nr-1)
    
    #   alpha  
    alpha_levels = np.linspace( alpha_center, alpha_edge,nr)  # np.array([0.2, 0.6, 1.0])   # 3 
    alphas = alpha_levels[bin_indices]
    
    # amplitude max_amp    alpha
    alphas[amplitude > max_amp] = 1.0
    
    alphas = np.clip(alphas, 0, 1)
    
    # RGBA  
    rgba_array = np.zeros((*phase.shape, 4))
    rgba_array[..., :3] = colors[..., :3]  # RGB
    rgba_array[..., 3] = alphas  # Alpha
    
    return rgba_array


def create_phase_amplitude_rgba_array(dataset, phase_var='lst', amp_var='amplitude', max_amp=0.7, cmap=None,
                                     alpha_center=0.2, alpha_edge=1.2,nr=6):
    """
    Generate Phase-Amplitude RGBA array from Dataset
    """    
    phases = dataset[phase_var].values
    amplitudes = dataset[amp_var].values
    # NaN 
    nan_mask = np.isnan(phases) | np.isnan(amplitudes)
    
    rgba_colors = circular_colormap_mapping(phases, amplitudes, max_amp, cmap, 
                                           alpha_center, alpha_edge,nr)
    
    # NaN   (, )
    rgba_colors[nan_mask] = [0.5, 0.5, 0.5, 0.3]
    
    # xarray DataArray 
    rgba_array = xr.DataArray(
        data=rgba_colors,
        dims=["latitude", "longitude", "rgba"],
        coords=dict(
            latitude=(["latitude"], dataset.latitude.values),
            longitude=(["longitude"], dataset.longitude.values),
            rgba=(["rgba"], ["R", "G", "B", "A"]),
        ),
    )
    
    return rgba_array

def phase_amplitude_figure(dataset, title, axis,phase_var='lst', amp_var='amplitude', 
                          max_amp=0.7,  alb='',cmap = diurnal_cmap,alpha_center=0.2, alpha_edge=1.2,nr=6):
    """
    Function to visualize Phase and Amplitude on a single map (using imshow)
    """
        
    with ProgressBar():
        left_mask = dataset['SAD'] > 4
        right_mask = dataset['right_mask'] 
        both_mask = dataset['both_mask'] 
        
        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        extent = [-180, 180, -60, 60]
        
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, 
                           color='gray', alpha=1, linestyle='-.')
        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), 
                        edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())
        
        # RGBA  
        rgba_array = create_phase_amplitude_rgba_array(dataset, phase_var, amp_var, max_amp,cmap,alpha_center, alpha_edge,nr=6)
        lons = dataset['longitude'].values
        lats = dataset['latitude'].values
        extent_data = [lons.min(), lons.max(), lats.min(), lats.max()]

        # imshow 
        im = axis.imshow(rgba_array[::-1,:,:].values, 
                        extent=extent_data,
                        transform=ccrs.PlateCarree(),
                        origin='upper',
                        aspect='equal')
        
        #   ()
        lons_mesh, lats_mesh = np.meshgrid(lons, lats)
        
        axis.contourf(lons_mesh, lats_mesh, left_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['//'])
        
        axis.contourf(lons_mesh, lats_mesh, right_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=["\\\\"])
        
        axis.contourf(lons_mesh, lats_mesh, both_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['xx'])
        
        axis.set_title(title, fontsize=18)
        axis.text(-180, 61, alb, fontsize=19, ha='left', va='bottom')
        
        return im

In [ ]:
from matplotlib import patheffects
# main version.
def circle_sine_ramp(r_max=1, r_min=0.0, amp=np.pi/5, cycles=50, power=2, nr=50, ntheta=1025):
    r, t = np.mgrid[r_min:r_max:nr*1j, 0:2*np.pi:ntheta*1j]
    r_norm = (r - r_min)/(r_max - r_min)
    vals = amp * r_norm**power * np.sin(cycles*t) + t
    vals = np.mod(vals, 2*np.pi)
    return t, r, vals


def plot_circular_colormap(ax,r_max, r_min=0.0,nr=6, ntheta=96, cmap=None, figsize=(3, 3), show_labels=True, 
                          alpha_center=0.2, alpha_edge=1.0, fontsize=20):
    """
    Function to draw circular colormap
    
    Parameters:
    -----------
    nr : int
        Radial resolution (default: 6)
    ntheta : int
        Angular resolution (default: 96, 24multiples recommended)
    cmap : matplotlib colormap or None
        Colormap to use (Noneuse default sunset colormap)
    figsize : tuple
        Figure size (default: (3, 3))
    show_labels : bool
        Whether to show time labels (default: True)
    alpha_center : float
          (default: 0.2)
    alpha_edge : float
          (default: 1.0)
    fontsize : int
           (default: 15)
    
    Returns:
    --------
    fig, ax : matplotlib figure and axes objects
    """
    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    t_small, r_small, vals_small = circle_sine_ramp(r_max=r_max, r_min=r_min,cycles=0, nr=nr, ntheta=ntheta)
    
    # 24  
    segment_size = 2 * np.pi / 24
    vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)
    
    center_distance_small = r_small / np.max(r_small)
    alpha_range = alpha_edge - alpha_center
    alpha_values_small = center_distance_small * alpha_range + alpha_center
    
    # RGB  
    colors_rgb_small = cmap(vals_quantized_normalized_small)
    
    polygons_small = []
    rgba_colors_small = []
    nr_small, ntheta_small = t_small.shape
    
    for i in range(nr_small-1):
        for j in range(ntheta_small-1):
            corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
            corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
            
            #   :   π/2  12  0  
            corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
            
            corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            
            polygon = list(zip(corners_x, corners_y))
            polygons_small.append(polygon)
            
            cell_rgba = [
                colors_rgb_small[i, j, 0],
                colors_rgb_small[i, j, 1], 
                colors_rgb_small[i, j, 2],
                alpha_values_small[i, j]
            ]
            rgba_colors_small.append(cell_rgba)
    
    collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
    ax.add_collection(collection_final)
    
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect('equal')
    ax.axis('off')
    
    if show_labels:
        for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
            #  : 12 (π/2)   
            angle = np.pi/2 - i * 2 * np.pi / 8  # subtract angle for clockwise direction
            x = (r_max+.15) * np.cos(angle)
            y = (r_max+.15) * np.sin(angle)
            ax.text(x, y, hour, ha='center', va='center', fontsize=fontsize)
    # ** :   **
    if show_labels:
        # amplitude  
        r_values = np.linspace(r_min, r_max, nr)  # nr+1 ()
        
        # (00 )   
        for r_val in r_values[1:]:  # r_min=0 
            x = 0
            y = r_val - 0.04
            # ax.text(x, y, f'{r_val:.1f}', ha='center', va='top', 
            #        fontsize=fontsize-6, color='grey')
            ax.text(x, y, f'{r_val:.1f}', ha='center', va='top', 
                   fontsize=fontsize-6, color='grey',
                   path_effects=[patheffects.withStroke(linewidth=3, foreground='white')])
        #     ()
        ax.plot([0, 0], [0, +r_max], ls = '-',c = 'grey', linewidth=.6, alpha=0.6)        
    plt.tight_layout()

In [ ]:
for i in range(len(result_ds_list)):

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
# alphabet = ['(a)','(b)','(c)',
#            '(c)','(d)','(e)',
#             '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # Phase-Amplitude  
    cnplot = phase_amplitude_figure(result_ds_list[i], phase_title[i], axs[i],'lst', 'amplitude', .8, None,diurnal_cmap,alpha_center=0.2, alpha_edge=1.,nr=4)
colorbar_ax = fig.add_axes([0.67, 0.64, 0.27, 0.27])  # [left, bottom, width, height]
colorbar_ax.set_aspect('equal')  # 1:1  
plot_circular_colormap(colorbar_ax,r_max=.8, r_min=0.0,nr=5, ntheta=24*4, alpha_center=0.2, fontsize=18,alpha_edge=1.2,cmap=diurnal_cmap)
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
cbar_ax.set_visible(False)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
# alphabet = ['(a)','(b)','(c)',
#            '(c)','(d)','(e)',
#             '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # Phase-Amplitude  
    cnplot = phase_amplitude_figure(result_ds_list[i], phase_title[i], axs[i],'lst', 'amplitude', .8, None,diurnal_cmap,alpha_center=0.2, alpha_edge=1.,nr=4)
colorbar_ax = fig.add_axes([0.67, 0.64, 0.27, 0.27])  # [left, bottom, width, height]
colorbar_ax.set_aspect('equal')  # 1:1  
plot_circular_colormap(colorbar_ax,r_max=.8, r_min=0.0,nr=5, ntheta=24*4, alpha_center=0.2, fontsize=18,alpha_edge=1.2,cmap=diurnal_cmap)
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
cbar_ax.set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
fig = plt.figure(figsize=(30, 15))  # alternative color version
axs = []
nrows, ncols = 3, 3
# alphabet = ['(a)','(b)','(c)',
#            '(c)','(d)','(e)',
#             '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    # Phase-Amplitude  
    cnplot = phase_amplitude_figure(result_ds_list[i], phase_title[i], axs[i],'lst', 'amplitude', .8, None,diurnal_cmap,alpha_center=0.2, alpha_edge=1.,nr=4)
colorbar_ax = fig.add_axes([0.67, 0.64, 0.27, 0.27])  # [left, bottom, width, height]
colorbar_ax.set_aspect('equal')  # 1:1  
plot_circular_colormap(colorbar_ax,r_max=.8, r_min=0.0,nr=5, ntheta=24*4, alpha_center=0.2, fontsize=18,alpha_edge=1.2,cmap=diurnal_cmap)
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
cbar_ax.set_visible(False)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase_finish.png',format='png', dpi=300,bbox_inches='tight')
# plt.plot()

In [ ]:
diurnal_cmap = LinearSegmentedColormap.from_list('diurnal', 
                                               [
                                               "#8B0000",  #   (-3)
                                               "#FF4500",  #  (3-6)
                                               "#FFD700",  #  (6-9)
                                               "#32CD32",  #  (9-12)
                                               "#30F8FF",  #  (12-15)
                                               "#0037DB",  #  (15-18)
                                               "#8400FF",  #  (18-21)
                                               "#DC143C"  #  (21-24)
                                               ], 
                                               N=8*3)
diurnal_cmap

In [ ]:
diurnal_cmap = LinearSegmentedColormap.from_list('diurnal', 
                                               [
                                               "#ff0000","#94001e",  "#FF4800", "#fffb00",
                                               "#00ffc3", "#3494de", "#2b00ff", "#ae00b7", 
                                               "#ff0033"
                                               ], 
                                               N=24)
diurnal_cmap

In [ ]:
# when radius exceeds 1
def plot_circular_colormap(ax, r_max, r_min=0.0, nr=6, ntheta=96, cmap=None, figsize=(3, 3), show_labels=True,  
                          alpha_center=0.2, alpha_edge=1.0, fontsize=20):
    """
    Function to draw circular colormap
    """
    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    t_small, r_small, vals_small = circle_sine_ramp(r_max=r_max, r_min=r_min, cycles=0, nr=nr, ntheta=ntheta)
    
    # 24  
    segment_size = 2 * np.pi / 24
    vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)
    
    #   -    !
    # r_max 1       
    if r_max <= 1.0:
        #  : r_max 1  
        center_distance_small = r_small / r_max
    else:
        # New : r_max 1  , 1   
        center_distance_small = r_small / r_max
        #  1   :
        # center_distance_small = np.minimum(r_small, 1.0)
    
    alpha_range = alpha_edge - alpha_center
    alpha_values_small = center_distance_small * alpha_range + alpha_center
    
    # RGB  
    colors_rgb_small = cmap(vals_quantized_normalized_small)
    
    polygons_small = []
    rgba_colors_small = []
    nr_small, ntheta_small = t_small.shape
    
    for i in range(nr_small-1):
        for j in range(ntheta_small-1):
            corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
            corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
            
            corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
            
            corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
            
            polygon = list(zip(corners_x, corners_y))
            polygons_small.append(polygon)
            
            cell_rgba = [
                colors_rgb_small[i, j, 0],
                colors_rgb_small[i, j, 1], 
                colors_rgb_small[i, j, 2],
                alpha_values_small[i, j]
            ]
            rgba_colors_small.append(cell_rgba)
    
    collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
    ax.add_collection(collection_final)
    
    #   - r_max  
    plot_limit = r_max * 1.1
    ax.set_xlim(-plot_limit, plot_limit)
    ax.set_ylim(-plot_limit, plot_limit)
    ax.set_aspect('equal')
    ax.axis('off')
    
    if show_labels:
        for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
            angle = np.pi/2 - i * 2 * np.pi / 8
            x = (r_max + 0.15) * np.cos(angle)
            y = (r_max + 0.15) * np.sin(angle)
            ax.text(x, y, hour, ha='center', va='center', fontsize=fontsize)
    
    if show_labels:
        r_values = np.linspace(r_min, r_max, nr)
        
        for r_val in r_values[1:]:  # r_min=0 
            x = 0
            y = r_val - 0.04
            ax.text(x, y, f'{r_val:.0f}', ha='center', va='top', 
                   fontsize=fontsize-6, color='grey',
                   path_effects=[patheffects.withStroke(linewidth=3, foreground='white')])
        
        ax.plot([0, 0], [0, +r_max], ls = '-', c = 'grey', linewidth=.6, alpha=0.6)
        
    plt.tight_layout()
fig = plt.figure(figsize=(30, 15))

colorbar_ax = fig.add_axes([0.67, 0.61, 0.32, 0.32])
colorbar_ax.set_aspect('equal')

#  r_max=4.0   4 
plot_circular_colormap(colorbar_ax, 
                      r_max=4.0,  # 0 4
                      r_min=0.0, 
                      nr=5,  # 0, 1, 2, 3, 4 5 
                      ntheta=24*4, 
                      alpha_center=0.2, 
                      fontsize=18,
                      alpha_edge=1.2,
                      cmap=diurnal_cmap)
plt.plot()
#      :
# Ring 1: 0 ≤ r < 1
# Ring 2: 1 ≤ r < 2  
# Ring 3: 2 ≤ r < 3
# Ring 4: 3 ≤ r ≤ 4

In [ ]:
def circular_colormap_mapping(phase, amplitude, max_amp=4.0, cmap=None, 
                             alpha_center=0.2, alpha_edge=1.2, nr=5):
    """
    Convert Phase and Amplitude to circular colormap RGBA values
    - new settings: max_amp=4.0, nr=5 (0-1, 1-2, 2-3, 3-44 intervals)
    """    
    if cmap is None:
        cmap = LinearSegmentedColormap.from_list('sunset', 
                                                [
                                                "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                                "#FF851B", "#FC3026", "#85144b", "#00021f"
                                                ], 
                                                N=24)
    
    # Phase    24  
    phase_rad = (phase % 24) * 2 * np.pi / 24
    segment_size = 2 * np.pi / 24
    vals_quantized = np.floor(phase_rad / segment_size) * segment_size + segment_size / 2
    vals_quantized_normalized = vals_quantized / (2 * np.pi)
    colors = cmap(vals_quantized_normalized)
    
    # Amplitude nr-1   (nr=5 4 )
    amplitude_clipped = np.clip(amplitude, 0, max_amp)
    step_size = max_amp / (nr - 1)  # nr=5, max_amp=4.0 → step_size=1.0
    
    #    (0, 1, 2, 3)
    bin_indices = np.floor(amplitude_clipped / step_size).astype(int)
    bin_indices = np.clip(bin_indices, 0, nr-2)  #   nr-2 (4  0,1,2,3)
    
    #   alpha   (nr-1  )
    alpha_levels = np.linspace(alpha_center, alpha_edge, nr-1)  # 4 : [0.2, 0.5333, 0.8667, 1.2]
    alphas = alpha_levels[bin_indices]
    
    # amplitude max_amp    alpha (4   )
    alphas[amplitude >= max_amp] = alpha_edge
    
    #   (1.0  )
    alphas = np.clip(alphas, 0, 1.0)
    
    # RGBA  
    rgba_array = np.zeros((*phase.shape, 4))
    rgba_array[..., :3] = colors[..., :3]  # RGB
    rgba_array[..., 3] = alphas  # Alpha
    
    return rgba_array


def create_phase_amplitude_rgba_array(dataset, phase_var='lst', amp_var='amplitude', 
                                     max_amp=4.0, cmap=None,
                                     alpha_center=0.2, alpha_edge=1.2, nr=5):
    """
    Generate Phase-Amplitude RGBA array from Dataset
    - updated parameters: max_amp=4.0, nr=5
    """    
    phases = dataset[phase_var].values
    amplitudes = dataset[amp_var].values
    
    # NaN 
    nan_mask = np.isnan(phases) | np.isnan(amplitudes)
    
    #    (New  )
    rgba_colors = circular_colormap_mapping(phases, amplitudes, max_amp, cmap, 
                                           alpha_center, alpha_edge, nr)
    
    # NaN   (, )
    rgba_colors[nan_mask] = [0.5, 0.5, 0.5, 0.3]
    
    # xarray DataArray 
    rgba_array = xr.DataArray(
        data=rgba_colors,
        dims=["latitude", "longitude", "rgba"],
        coords=dict(
            latitude=(["latitude"], dataset.latitude.values),
            longitude=(["longitude"], dataset.longitude.values),
            rgba=(["rgba"], ["R", "G", "B", "A"]),
        ),
    )
    
    return rgba_array

def phase_amplitude_figure(dataset, title, axis, phase_var='lst', amp_var='amplitude', 
                          max_amp=4.0, alb='', cmap=None, alpha_center=0.2, alpha_edge=1.2, nr=5):
    """
    Function to visualize Phase and Amplitude on a single map
    - updated parameters: max_amp=4.0, nr=5 (matching colorbar)
    """
    if cmap is None:
        cmap = diurnal_cmap
        
    with ProgressBar():
        left_mask = dataset['SAD'] > 4
        right_mask = dataset['right_mask'] 
        both_mask = dataset['both_mask'] 
        
        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        extent = [-180, 180, -60, 60]
        
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, 
                           color='gray', alpha=1, linestyle='-.')
        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), 
                        edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())
        
        # RGBA   (New  )
        rgba_array = create_phase_amplitude_rgba_array(dataset, phase_var, amp_var, 
                                                      max_amp, cmap, alpha_center, alpha_edge, nr)
        
        lons = dataset['longitude'].values
        lats = dataset['latitude'].values
        extent_data = [lons.min(), lons.max(), lats.min(), lats.max()]

        # imshow 
        im = axis.imshow(rgba_array[::-1,:,:].values, 
                        extent=extent_data,
                        transform=ccrs.PlateCarree(),
                        origin='upper',
                        aspect='equal')
        
        #   ()
        lons_mesh, lats_mesh = np.meshgrid(lons, lats)
        
        axis.contourf(lons_mesh, lats_mesh, left_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['///'])
        
        axis.contourf(lons_mesh, lats_mesh, right_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=["\\\\\\"])
        
        axis.contourf(lons_mesh, lats_mesh, both_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['xxx'])
        
        axis.set_title(title, fontsize=18)
        axis.text(-180, 61, alb, fontsize=19, ha='left', va='bottom')
        
        return im

In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
alphabet = ['(a)','(b)','(c)',
           '(c)','(d)','(e)',
            '(f)','(g)','(h)']
tick_interval = 2

for i in range(nrows * ncols):
    position = i + 1
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    
    # Phase-Amplitude  
    # cnplot = phase_amplitude_figure(result_ds_list[i], phase_title[i], axs[i],'lst', 'amplitude', 0.6, alphabet[i],diurnal_cmap,alpha_center=0.2, alpha_edge=1.,nr=3)
colorbar_ax = fig.add_axes([0.67, 0.61, 0.32, 0.32])  # [left, bottom, width, height]
colorbar_ax.set_aspect('equal')  # 1:1  
plot_circular_colormap(colorbar_ax,r_max=.6, r_min=0.0,nr=4, ntheta=24*4, alpha_center=0.2, fontsize=18,alpha_edge=1.2,cmap=diurnal_cmap)

cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval))
cbar.set_label('Local Solar Time of Max[hour]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
cbar_ax.set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_phase_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
def circular_colormap_mapping(phase, amplitude, max_amp=0.7, cmap=None):
    """
    Convert Phase and Amplitude to circular colormap RGBA values
    
    Parameters:
    -----------
    phase : array-like
        Phase value (0-24 )
    amplitude : array-like
        Amplitude value
    max_amp : float
        Maximum amplitude (normalization reference)
    cmap : matplotlib colormap
        Colormap to use
        
    Returns:
    --------
    rgba_array : array
        RGBA color array (shape: [..., 4])
    """    
    # Phase 0-1   (24 -> 1)
    phase_normalized = (phase % 24) / 24.0
    
    # Amplitude 0-1  
    amplitude_normalized = np.clip(amplitude / max_amp, 0, 1)
    
    # Phase   
    colors = cmap(phase_normalized)
    
    #  amplitude  
    # amplitude 0 (alpha=0.1), max_amp  (alpha=1.0)
    alphas = amplitude_normalized * 0.9 + 0.1
    
    # RGBA  
    rgba_array = np.zeros((*phase.shape, 4))
    rgba_array[..., :3] = colors[..., :3]  # RGB
    rgba_array[..., 3] = alphas  # Alpha
    
    return rgba_array
def create_phase_amplitude_rgba_array(dataset, phase_var='lst', amp_var='amplitude', max_amp=0.7,cmap=None):
    """
    Generate Phase-Amplitude RGBA array from Dataset
    
    Parameters:
    -----------
    dataset : xarray Dataset
        
    phase_var : str
         
    amp_var : str
         
    max_amp : float
        Maximum amplitude
        
    Returns:
    --------
    rgba_array : xarray DataArray
        RGBA color array
    """    
    phases = dataset[phase_var].values
    amplitudes = dataset[amp_var].values
    
    # NaN 
    nan_mask = np.isnan(phases) | np.isnan(amplitudes)
    
    rgba_colors = circular_colormap_mapping(phases, amplitudes, max_amp, cmap)
    
    # NaN   (, )
    rgba_colors[nan_mask] = [0.5, 0.5, 0.5, 0.3]
    
    # xarray DataArray 
    rgba_array = xr.DataArray(
        data=rgba_colors,
        dims=["latitude", "longitude", "rgba"],
        coords=dict(
            latitude=(["latitude"], dataset.latitude.values),
            longitude=(["longitude"], dataset.longitude.values),
            rgba=(["rgba"], ["R", "G", "B", "A"]),
        ),
    )
    
    return rgba_array
    
def phase_amplitude_figure(dataset, title, axis,phase_var='lst', amp_var='amplitude', 
                          max_amp=0.7,  alb='',cmap = diurnal_cmap):
    """
    Function to visualize Phase and Amplitude on a single map (using imshow)
    """
        
    with ProgressBar():
        left_mask = dataset['SAD'] > 4
        right_mask = dataset['right_mask'] 
        both_mask = dataset['both_mask'] 
        
        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()
        extent = [-180, 180, -60, 60]
        
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, 
                           color='gray', alpha=1, linestyle='-.')
        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None
        
        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), 
                        edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())
        
        # RGBA  
        rgba_array = create_phase_amplitude_rgba_array(dataset, phase_var, amp_var, max_amp,cmap)
        
        lons = dataset['longitude'].values
        lats = dataset['latitude'].values
        extent_data = [lons.min(), lons.max(), lats.min(), lats.max()]
        # data_T = rgba_array[::-1,:,:]

        # imshow 
        im = axis.imshow(rgba_array[::-1,:,:].values, 
                        extent=extent_data,
                        transform=ccrs.PlateCarree(),
                        origin='upper',
                        aspect='equal')
        
        #   ()
        lons_mesh, lats_mesh = np.meshgrid(lons, lats)
        
        axis.contourf(lons_mesh, lats_mesh, left_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['///'])
        
        axis.contourf(lons_mesh, lats_mesh, right_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=["\\\\\\"])
        
        axis.contourf(lons_mesh, lats_mesh, both_mask.astype(int),
                     transform=ccrs.PlateCarree(),
                     colors='none',
                     levels=[0.5, 1.5],
                     hatches=['xxx'])
        
        axis.set_title(title, fontsize=18)
        axis.text(-180, 61, alb, fontsize=19, ha='left', va='bottom')
        
        return im

### amplitude figure


In [ ]:
def amp_figure(dataset,title,var,cmap,axis,varMin, varMax, varInt,tick_interval,extend,alb):

    with ProgressBar():  #  ?

        #utc
        nan_mask = np.isnan(dataset[var]) 
        left_mask = dataset['SAD']>4#dataset['left_mask'] 
        right_mask = dataset['right_mask'] 
        both_mask = dataset['both_mask'] 

        projection = ccrs.PlateCarree()
        crs = ccrs.PlateCarree()

        levels = np.arange(varMin, varMax+varInt, varInt)
        nlevs  = levels.size

        extent=[-180, 180, -60, 60]
        gl = axis.gridlines(crs=crs, draw_labels=True, linewidth=1, color='gray', alpha=1, linestyle='-.')
        gl.top_labels = None
        gl.right_labels = None
        gl.bottom_labels = None
        gl.left_labels = None

        axis.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.8)
        axis.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
        axis.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
        axis.set_extent(extent, crs=ccrs.PlateCarree())

        lons = dataset['longitude']
        lats = dataset['latitude']
        if var == 'lst':

            values = dataset[var]
            cbarlabel = 'Local Solar Time of Max[hour]'

        elif var == 'amplitude' :
            values = dataset[var]
            cbarlabel = '[mm/3h]'
        else:
            pass
        lons, lats = np.meshgrid(dataset['longitude'], dataset['latitude'])

        cnplot = axis.contourf(lons, lats, values,cmap=cmap,levels=levels,zorder=0,transform=ccrs.PlateCarree(),extend =extend)


        axis.contourf(
            lons, lats, (nan_mask==True),
            transform=ccrs.PlateCarree(),
            colors='gray',
            levels=[0.5, 1.5],
        )  # #    


        axis.contourf(
            lons, lats, left_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['///']
        ) 
        axis.contourf(
            lons, lats, right_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=["\\\\\\"]
        ) 
        axis.contourf(
            lons, lats, both_mask,
            transform=ccrs.PlateCarree(),
            colors='none',
            levels=[0.5,1.5],
            hatches=['xxx']
        )  # #    
        axis.set_title(title ,fontsize=18)
        axis.text(-180,61,alb ,fontsize=19,ha='left', va='bottom')

        return(cnplot)
        


In [ ]:
fig = plt.figure(figsize=(30, 15))
axs = []
nrows, ncols = 3, 3
# extent= eu
var = 'amplitude'
varMin, varMax, varInt =0, .7, 0.025/2
tick_interval = .1
extend = 'max'
for i in range(nrows * ncols):
    position = i + 1
    
    ax = fig.add_subplot(nrows, ncols, position, projection=ccrs.PlateCarree())
    axs.append(ax)
    cnplot = amp_figure(result_ds_list[i],phase_title[i],var,'turbo',axs[i],varMin, varMax, varInt,tick_interval,extend,alphabet[i])
    
cbar_ax = fig.add_axes([0.263, .14, 0.5, 0.03])  # [,,,]
cbar = plt.colorbar(cnplot, cax=cbar_ax, orientation='horizontal',ticks=np.arange(varMin, varMax+tick_interval, tick_interval),extend = extend)

cbar.set_label('[mm/3hr]', fontsize=15)
cbar.ax.tick_params(labelsize=15)
plt.subplots_adjust(wspace = .01,hspace = -.4)
axs[2].set_visible(False)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/total_amp.png',format='png', dpi=300,bbox_inches='tight')
plt.plot()

In [ ]:
diurnal = LinearSegmentedColormap.from_list('diurnal', 
                                           [
                                           "#f81745", "#fffe42", "#fcffaa", "#ffffff",
                                           "#45facf", "#3494de", "#5f40f3", "#ae00b7", 
                                           "#f20249"
                                           ], 
                                           N=24)
diurnal

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import PolyCollection

def circle_sine_ramp(r_max=1, r_min=0.0, amp=np.pi/5, cycles=50, power=2, nr=50, ntheta=1025):
    r, t = np.mgrid[r_min:r_max:nr*1j, 0:2*np.pi:ntheta*1j]
    r_norm = (r - r_min)/(r_max - r_min)
    vals = amp * r_norm**power * np.sin(cycles*t) + t
    vals = np.mod(vals, 2*np.pi)
    return t, r, vals

t, r, vals = circle_sine_ramp(cycles=0, nr=100, ntheta=24*2)

srtoss = LinearSegmentedColormap.from_list('sunset', 
                                            [
                                            "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                            "#FF851B", "#FC3026", "#85144b", "#00021f"
                                            ], 
                                            N=24)

# 24  
segment_size = 2 * np.pi / 24
vals_quantized = np.floor(vals / segment_size) * segment_size + segment_size / 2
vals_quantized_normalized = vals_quantized / (2 * np.pi)

center_distance = r / np.max(r)
alpha_values = center_distance *1  + 0.1

# RGB  
colors_rgb = srtoss(vals_quantized_normalized)

fig, ax = plt.subplots(figsize=(3,3))

t_small, r_small, vals_small = circle_sine_ramp(cycles=0, nr=6, ntheta=24*4)
vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)
center_distance_small = r_small / np.max(r_small)
alpha_values_small = center_distance_small * 1. + 0.2
colors_rgb_small = srtoss(vals_quantized_normalized_small)

polygons_small = []
rgba_colors_small = []
nr_small, ntheta_small = t_small.shape

for i in range(nr_small-1):
    for j in range(ntheta_small-1):
        corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
        corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
        
        #   :   π/2  12  0  
        corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
        
        corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
        corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
        
        polygon = list(zip(corners_x, corners_y))
        polygons_small.append(polygon)
        
        cell_rgba = [
            colors_rgb_small[i, j, 0],
            colors_rgb_small[i, j, 1], 
            colors_rgb_small[i, j, 2],
            alpha_values_small[i, j]
        ]
        rgba_colors_small.append(cell_rgba)

collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
ax.add_collection(collection_final)

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect('equal')

#    -   
for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
    #  : 12 (π/2)   
    angle = np.pi/2 - i * 2 * np.pi / 8  # subtract angle for clockwise direction
    x = 1.15 * np.cos(angle)
    y = 1.15 * np.sin(angle)
    ax.text(x, y, hour, ha='center', va='center', fontsize=15)

ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.collections import PolyCollection

def circle_sine_ramp(r_max=1, r_min=0.0, amp=np.pi/5, cycles=50, power=2, nr=50, ntheta=1025):
    r, t = np.mgrid[r_min:r_max:nr*1j, 0:2*np.pi:ntheta*1j]
    r_norm = (r - r_min)/(r_max - r_min)
    vals = amp * r_norm**power * np.sin(cycles*t) + t
    vals = np.mod(vals, 2*np.pi)
    return t, r, vals

t, r, vals = circle_sine_ramp(cycles=0, nr=100, ntheta=24*2)

srtoss = LinearSegmentedColormap.from_list('sunset', 
                                            [
                                            "#001f3f", "#0074D9", "#40DF20", "#FFDC00",
                                            "#FF851B", "#FC3026", "#85144b", "#00021f"
                                            ], 
                                            N=24)

# 24  
segment_size = 2 * np.pi / 24
vals_quantized = np.floor(vals / segment_size) * segment_size + segment_size / 2
vals_quantized_normalized = vals_quantized / (2 * np.pi)

#     -  1  
center_distance = r / np.max(r)
alpha_values = center_distance * 0.8 + 0.2  #  0.2,  1.0

# RGB  
colors_rgb = srtoss(vals_quantized_normalized)

fig, ax = plt.subplots(figsize=(3,3))

t_small, r_small, vals_small = circle_sine_ramp(cycles=0, nr=6, ntheta=24)
vals_quantized_small = np.floor(vals_small / segment_size) * segment_size + segment_size / 2
vals_quantized_normalized_small = vals_quantized_small / (2 * np.pi)

#  alpha = 1  
center_distance_small = r_small / np.max(r_small)
alpha_values_small = center_distance_small * 0.8 + 0.2  #  0.2,  1.0

colors_rgb_small = srtoss(vals_quantized_normalized_small)

polygons_small = []
rgba_colors_small = []
nr_small, ntheta_small = t_small.shape

for i in range(nr_small-1):
    for j in range(ntheta_small-1):
        corners_t = [t_small[i, j], t_small[i, j+1], t_small[i+1, j+1], t_small[i+1, j]]
        corners_r = [r_small[i, j], r_small[i, j+1], r_small[i+1, j+1], r_small[i+1, j]]
        
        #   :   π/2  12  0  
        corners_t_clock = [-t_val + np.pi/2 for t_val in corners_t]
        
        corners_x = [r_val * np.cos(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
        corners_y = [r_val * np.sin(t_val) for r_val, t_val in zip(corners_r, corners_t_clock)]
        
        polygon = list(zip(corners_x, corners_y))
        polygons_small.append(polygon)
        
        cell_rgba = [
            colors_rgb_small[i, j, 0],
            colors_rgb_small[i, j, 1], 
            colors_rgb_small[i, j, 2],
            alpha_values_small[i, j]
        ]
        rgba_colors_small.append(cell_rgba)

collection_final = PolyCollection(polygons_small, facecolors=rgba_colors_small, edgecolors='none')
ax.add_collection(collection_final)

ax.set_xlim(-1.1, 1.1)
ax.set_ylim(-1.1, 1.1)
ax.set_aspect('equal')

#    -   
for i, hour in enumerate(['24/0', '3', '6', '9', '12', '15', '18', '21']):
    #  : 12 (π/2)   
    angle = np.pi/2 - i * 2 * np.pi / 8  # subtract angle for clockwise direction
    x = 1.15 * np.cos(angle)
    y = 1.15 * np.sin(angle)
    ax.text(x, y, hour, ha='center', va='center', fontsize=15)

ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Global evaluation of the diurnal cycle of precipitation(bias)[]


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#     -1  
def circular_mean_hours(hours):
    angles = np.array(hours) * 2 * np.pi / 24
    mean_sin = np.mean(np.sin(angles))
    mean_cos = np.mean(np.cos(angles))
    mean_angle = np.arctan2(mean_sin, mean_cos)
    if mean_angle < 0:
        mean_angle += 2 * np.pi
    return (mean_angle * 24 / (2 * np.pi)) % 24


def circular_correlation(x, y):
# S. R. Jammalamadaka, A. SenGupta. “Topics in Circular Statistics”. Series on Multivariate Analysis, Vol. 5, 2001.
# C. Agostinelli, U. Lund. “Circular Statistics from ‘Topics in Circular Statistics (2001)’”. 2015.

    x = np.array(x)
    y = np.array(y)
    n = len(x)
    
    #    ( 0 2π )
    x_rad = x * 2 * np.pi / 24
    y_rad = y * 2 * np.pi / 24
    
    x_bar = np.arctan2(np.mean(np.sin(x_rad)), np.mean(np.cos(x_rad)))
    y_bar = np.arctan2(np.mean(np.sin(y_rad)), np.mean(np.cos(y_rad)))
    
    sin_x_dev = np.sin(x_rad - x_bar)
    sin_y_dev = np.sin(y_rad - y_bar)
    
    #   (8.2.5  )
    numerator = np.sum(sin_x_dev * sin_y_dev)
    denominator = np.sqrt(np.sum(sin_x_dev**2) * np.sum(sin_y_dev**2))
    
    r_c_n = numerator / denominator
    
    return r_c_n
def permutation_test(x, y, n_permutations=10000):
    observed_corr = circular_correlation(x, y)
    permuted_corrs = []
    
    for _ in range(n_permutations):
        y_permuted = np.random.permutation(y)
        permuted_corr = circular_correlation(x, y_permuted)
        permuted_corrs.append(permuted_corr)
    
    p_value = np.mean(np.abs(permuted_corrs) >= np.abs(observed_corr))
    return p_value

def fisher_lee_test(x, y):
    n = len(x)
    rho = circular_correlation(x, y)
    z = rho * np.sqrt(n)
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    return p_value


In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc')  # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
obs_result = xr.open_dataset('${DATA_DIR}/obs/obs_gridded.nc')

In [ ]:
result_ds_list = [cmorph,imerg ,trmm,gsmap,mswep,  # MS can be removed
                  narr,jra55,era5,merra2]

In [ ]:
for i in range(len(result_ds_list)):

In [ ]:
(result_ds_list[-1]['amplitude']-obs_result['amplitude']).mean()

In [ ]:
# data_title = ['Multi-Satellite(MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
#                'Multi-Reanalysis(MR)','NARR','JRA-3Q','ERA5','MERRA2']

# markers = ['$S$','^' ,'s', 'H','X', 'D',
#           '$R$','P','<', 'p', '*']
data_title = ['CMORPH','IMERG','TRMM','GSMaP','MSWEP',
               'NARR','JRA-3Q','ERA5','MERRA-2']

markers = ['^' ,'s', 'H','X', 'D',
          'P','<', 'p', '*']

In [ ]:
new_markers = ['o']+markers
new_data_title = ['Obs']+data_title

In [ ]:
new_data_title

In [ ]:
import cartopy.io.shapereader as shpreader  #  must be executed.
from shapely.geometry import Point
from shapely.prepared import prep

def create_us_mask(da):
    """
    CONUS(CONUS) generates a mask matching the shape.
    
    Parameters:
    -----------
    da : xarray.DataArray
        DataArray to apply mask to, must have latitude and longitude coordinates
    
    Returns:
    --------
    xarray.DataArray
        CONUS mask matching the shape
    """
    states_shp = shpreader.natural_earth(resolution='50m',
                                         category='cultural',
                                         name='admin_1_states_provinces')
    
    # CONUS   (, ,   )
    reader = shpreader.Reader(states_shp)
    us_states = []
    
    excluded_states = ['Alaska', 'Hawaii', 'Puerto Rico']
    
    for state in reader.records():
        name = state.attributes['name']
        if state.attributes['admin'] == 'United States of America' and name not in excluded_states:
            us_states.append(state.geometry)
    
    #  CONUS     
    from shapely.ops import unary_union
    us_outline = unary_union(us_states)
    prepared_geom = prep(us_outline)
    
    #  CONUS   
    mask = np.zeros((len(da.latitude), len(da.longitude)), dtype=bool)
    
    for i, lat in enumerate(da.latitude.values):
        for j, lon in enumerate(da.longitude.values):
            point = Point(lon, lat)
            mask[i, j] = prepared_geom.contains(point)
    
    # xarray DataArray 
    mask_da = xr.DataArray(
        mask,
        coords={
            'latitude': da.latitude,
            'longitude': da.longitude
        },
        dims=['latitude', 'longitude']
    )
    
    #  land-sea mask  ( CONUS )
    final_mask = da & mask_da
    
    return final_mask

conus_mask = create_us_mask(lsm)

plt.figure(figsize=(12, 10))

#  land-sea mask
plt.subplot(2, 1, 1)
lsm.plot(cmap='Blues')
plt.title('Original Land-Sea Mask')

# CONUS mask
plt.subplot(2, 1, 2)
conus_mask.plot(cmap='Blues')
plt.title('CONUS Land Mask')

plt.tight_layout()
plt.savefig('precise_conus_mask.png')
plt.show()

mask_ds = xr.Dataset({'conus_mask': conus_mask})

print("Precise CONUS mask created")
print(f"Number of True values before masking: {lsm.sum().values}")
print(f"Number of True values after masking: {conus_mask.sum().values}")

In [ ]:
for_compare = []
for i in range(len(new_data_title)):
    if i ==0:
        for_compare.append(obs_result.where((~obs_result['lst'].isnull())&(lsm==1)&(obs_result['SAD']<4)))
    else:
        for_compare.append(result_ds_list[i-1].where((~obs_result['lst'].isnull())&(lsm==1)&(obs_result['SAD']<4)))
        print(data_title[i-1])

In [ ]:
corr_data = {}
i=0
for title in new_data_title[1:]:
    i=i+1
    if title == 'Obs':
        lst_column = 'lst'
        amp_column = 'amplitude'
    else:
        lst_column = f'lst_{title}'
        amp_column = f'amp_{title}'
    merged_ds = xr.merge([for_compare[0][['lst','amplitude']],
                          for_compare[i].rename(lst = lst_column,amplitude = amp_column)[[lst_column,amp_column]]])
    df_drop = merged_ds.to_dataframe().dropna() 
    # df_drop = final_df[['lst','amplitude',lst_column,amp_column]].dropna() 

    Had_hours = df_drop['lst'].tolist()
    Had_amp = df_drop['amplitude'].tolist()

    hours = df_drop[lst_column].tolist()
    amp = df_drop[amp_column].tolist()

    lst_r = circular_correlation(Had_hours, hours)
    lst_p = fisher_lee_test(Had_hours, hours)

    amp_r = stats.spearmanr(Had_amp, amp)[0]
    amp_p = stats.spearmanr(Had_amp, amp)[1]
    corr_data[title] = {
        'lst_corr': lst_r,
        'lst_p': lst_p,
        'amp_corr': amp_r,
        'amp_p': amp_p}

In [ ]:
cor_df = pd.DataFrame(corr_data)
cor_df 

In [ ]:
conus_ds={}
for i in range(len(new_data_title)):
    
    conus_ds[new_data_title[i]] = for_compare[i].where(conus_mask)

In [ ]:
eu_markers =  ['o','^' ,'s', 'X', 'D',
          '<', 'p', '*']
eu_data_title =['Obs','CMORPH','IMERG','GSMaP','MSWEP',
               'JRA-3Q','ERA5','MERRA-2']

print(len(eu_markers)) 
print(len(eu_data_title))

In [ ]:
eu_ds={}
for i in range(len(new_data_title)):
    
    eu_ds[new_data_title[i]] = for_compare[i].sel(longitude = slice(-20,30),latitude = slice(30,60))

In [ ]:
eu_corr_data = {}
i=0
for title in eu_data_title[1:]:
    i=i+1
    if title == 'Obs':
        lst_column = 'lst'
        amp_column = 'amplitude'
    else:
        lst_column = f'lst_{title}'
        amp_column = f'amp_{title}'
    merged_ds = xr.merge([eu_ds['Obs'][['lst','amplitude']],
                          eu_ds[title].rename(lst = lst_column,amplitude = amp_column)[[lst_column,amp_column]]])
    df_drop = merged_ds.to_dataframe().dropna() 
    # df_drop = final_df[['lst','amplitude',lst_column,amp_column]].dropna() 

    Had_hours = df_drop['lst'].tolist()
    Had_amp = df_drop['amplitude'].tolist()

    hours = df_drop[lst_column].tolist()
    amp = df_drop[amp_column].tolist()

    lst_r = circular_correlation(Had_hours, hours)
    lst_p = fisher_lee_test(Had_hours, hours)

    amp_r = stats.spearmanr(Had_amp, amp)[0]
    amp_p = stats.spearmanr(Had_amp, amp)[1]
    eu_corr_data[title] = {
        'lst_corr': lst_r,
        'lst_p': lst_p,
        'amp_corr': amp_r,
        'amp_p': amp_p}

In [ ]:
eu_cor_df = pd.DataFrame(eu_corr_data)
eu_cor_df

In [ ]:
#EA, NH
# ea_markers = ['o','$S$','^' ,'s', 'H','X', 'D',
#                  '$R$','<', 'p', '*']

# ea_data_title = ['Obs','Multi-Satellite(MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
#               'Multi-Reanalysis(MR)','JRA-3Q','ERA5','MERRA2']
ea_markers = ['o','^' ,'s','H','X', 'D',
                 '<', 'p', '*']

ea_data_title = ['Obs','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
              'JRA-3Q','ERA5','MERRA-2']

print(len(ea_markers)) #EA, NH
print(len(ea_data_title))

In [ ]:
ea_ds={}
for i in range(len(new_data_title)):

    ea_ds[new_data_title[i]] = for_compare[i].sel(longitude = slice(70,146),
                                                     latitude = slice(20,46))

In [ ]:
ea_corr_data = {}
i=0
for title in ea_data_title[1:]:
    i=i+1
    if title == 'Obs':
        lst_column = 'lst'
        amp_column = 'amplitude'
    else:
        lst_column = f'lst_{title}'
        amp_column = f'amp_{title}'
    merged_ds = xr.merge([ea_ds['Obs'][['lst','amplitude']],
                          ea_ds[title].rename(lst = lst_column,amplitude = amp_column)[[lst_column,amp_column]]])
    df_drop = merged_ds.to_dataframe().dropna() 
    # df_drop = final_df[['lst','amplitude',lst_column,amp_column]].dropna() 

    Had_hours = df_drop['lst'].tolist()
    Had_amp = df_drop['amplitude'].tolist()

    hours = df_drop[lst_column].tolist()
    amp = df_drop[amp_column].tolist()

    lst_r = circular_correlation(Had_hours, hours)
    lst_p = fisher_lee_test(Had_hours, hours)

    amp_r = stats.spearmanr(Had_amp, amp)[0]
    amp_p = stats.spearmanr(Had_amp, amp)[1]
    ea_corr_data[title] = {
        'lst_corr': lst_r,
        'lst_p': lst_p,
        'amp_corr': amp_r,
        'amp_p': amp_p,
    }

In [ ]:
ea_cor_df = pd.DataFrame(ea_corr_data)
ea_cor_df

In [ ]:
NH_ds = {}
for i in range(len(new_data_title)):
    NH_ds[new_data_title[i]] = for_compare[i]

In [ ]:
print(len(new_markers))#CONUS
print(len(new_data_title))

In [ ]:
new_data_title

In [ ]:
#EA, NH
# ea_markers = ['o','$S$','^' ,'s', 'H','X', 'D',
#                  '$R$','<', 'p', '*']

# ea_data_title = ['Obs','Multi-Satellite(MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
#               'Multi-Reanalysis(MR)','JRA-3Q','ERA5','MERRA2']
# ea_markers = ['o','$S$','^' ,'s', 'H','X', 'D',
#                  '<', 'p', '*']

# ea_data_title = ['Obs','Multi-Satellite(MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
#               'JRA-3Q','ERA5','MERRA2']

# print(len(ea_markers)) #EA, NH
# print(len(ea_data_title))

In [ ]:
# EU
# markers =  ['o','$S$','^' ,'s', 'X', 'D',
#           '$R$','<', 'p', '*']
# titles =['Obs','Multi-Satellite(MS)','CMORPH','IMERG','GSMaP','MSWEP',
#                'Multi-Reanalysis(MR)','JRA-3Q','ERA5','MERRA2']
markers =  ['o','^' ,'s', 'X', 'D',
          '<', 'p', '*']
titles =['Obs','CMORPH','IMERG','GSMaP','MSWEP',
               'JRA-3Q','ERA5','MERRA-2']

print(len(markers)) 
print(len(titles))

In [ ]:
# regions where obs SAD < 4& when observation is available
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

fig = plt.figure(figsize=(8, 8))

regions = {
    'NH': (NH_ds, '(a) [NH]', 0.4),
    'CONUS': (conus_ds, '(b) [CONUS]', 0.4),
    'EU': (eu_ds, '(c) [EU]', 0.4),
    'EA': (ea_ds, '(d) [EA]', 0.4)}

def get_color(index): # NH, CONUS, EA
    if index == 0:
        return 'black'
    elif 1 <= index <= 6:
        return '#0554F2'
    else:
        return '#F25244'
def get_color1(index):#EU
    if index == 0:
        return 'black'
    elif 1 <= index <= 5:
        return '#0554F2'
    else:
        return '#F25244'

for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 2, idx, projection='polar')
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
        
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EU
        for i, marker in enumerate(markers):
            name = titles[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.1
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    
    plt.title(title, fontsize=16, loc='left', x=0)

# idx==1   scatter  legend 
legend = fig.legend(bbox_to_anchor=(1.,.9), loc='upper left',
                   labelspacing=1.2, markerscale=1, handletextpad=0.5, frameon=False)

for i, text in enumerate(legend.get_texts()):
    if i == 0:
        text.set_color('black')
    elif 1 <= i <= 6:
        text.set_color('#0554F2')
    else:
        text.set_color('#F25244')

plt.tight_layout()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/regional_circles_finish.png', format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# regions where obs SAD < 4& when observation is available
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

fig = plt.figure(figsize=(8, 8))

regions = {
    'NH': (NH_ds, '(a) Global', 0.8),
    'CONUS': (conus_ds, '(b) CONUS', 0.8),
    'EU': (eu_ds, '(c) EU', 0.8),
    'EA': (ea_ds, '(d) EA', 0.8)}

def get_color(index): # NH, CONUS, EA
    if index == 0:
        return 'black'
    elif 1 <= index <= 5:
        return '#0554F2'
    else:
        return '#F25244'
def get_color1(index):#EU
    if index == 0:
        return 'black'
    elif 1 <= index <= 4:
        return '#0554F2'
        
    else:
        return '#F25244'

for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 2, idx, projection='polar')
    # print(idx)
    # print(region)
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
        
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EA
        for i, marker in enumerate(eu_markers):
            name = eu_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    
    plt.title(title, fontsize=16, loc='left', x=0)

# idx==1   scatter  legend 
legend = fig.legend(bbox_to_anchor=(1.,.9), loc='upper left',
                   labelspacing=1.2, markerscale=1, handletextpad=0.5, frameon=False)

for i, text in enumerate(legend.get_texts()):
    if i == 0:
        text.set_color('black')
    elif 1 <= i <= 5:
        text.set_color('#0554F2')
    else:
        text.set_color('#F25244')

plt.tight_layout()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/regional_circles_finish.png', format='png', dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt  # standard version
import numpy as np
import matplotlib.colors as mcolors

regions_data = {
    'NH': {'data': NH_ds, 'title': '(a) Global', 'titles': ea_data_title[1:], 'markers': ea_markers},
    'CONUS': {'data': conus_ds, 'title': '(b) CONUS', 'titles': new_data_title[1:], 'markers': new_markers},
    'EU': {'data': eu_ds, 'title': '(c) EU', 'titles': titles[1:], 'markers': markers},
    'EA': {'data': ea_ds, 'title': '(d) EA', 'titles': ea_data_title[1:], 'markers': ea_markers}
}

all_corr_results = {}

amp_color = '#E74C3C'
lst_color = '#27AE60'
for region_name, region_info in regions_data.items():
    corr_data = {}
    data = region_info['data']
    region_titles = region_info['titles']
    
    i = 0
    for title in region_titles:
        i += 1
        if title == 'Obs':
            lst_column = 'lst'
            amp_column = 'amplitude'
        else:
            lst_column = f'lst_{title}'
            amp_column = f'amp_{title}'
            
        if region_name == 'NH':
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
        else:
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
            
        df_drop = merged_ds.to_dataframe().dropna()
        Had_hours = df_drop['lst'].tolist()
        Had_amp = df_drop['amplitude'].tolist()
        hours = df_drop[lst_column].tolist()
        amp = df_drop[amp_column].tolist()
        
        lst_r = circular_correlation(Had_hours, hours)
        lst_p = fisher_lee_test(Had_hours, hours)
        amp_r = stats.spearmanr(Had_amp, amp)[0]
        amp_p = stats.spearmanr(Had_amp, amp)[1]
        
        corr_data[title] = {
            'lst_corr': lst_r,
            'lst_p': lst_p,
            'amp_corr': amp_r,
            'amp_p': amp_p,
        }
    
    all_corr_results[region_name] = pd.DataFrame(corr_data)

fig = plt.figure(figsize=(10, 10))

for idx, (region_name, region_info) in enumerate(regions_data.items(), 1):
    ax = fig.add_subplot(2, 2, idx, projection='polar')
    
    cor_df = all_corr_results[region_name]
    datasets = cor_df.columns
    lst_corr = cor_df.loc['lst_corr']
    amp_corr = cor_df.loc['amp_corr']
    
    angles = np.linspace(0, 2*np.pi, len(datasets), endpoint=False)
    lst_corr = np.concatenate((lst_corr, [lst_corr[0]]))
    amp_corr = np.concatenate((amp_corr, [amp_corr[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    y_min = min(min(lst_corr), min(amp_corr), 0.0)
    y_max = max(max(lst_corr), max(amp_corr))
    ax.set_ylim(y_min, y_max)
    
    ax.yaxis.grid(True, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill(np.linspace(0, 2*np.pi, 100), np.zeros(100), alpha=0.1, color='blue')
    
    #   - label   subplot 
    if idx == 1:
        ax.plot(angles, lst_corr, '-', linewidth=1.5, label='R(Phase)',c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5, label='R(Amplitude)',c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)

    else:
        ax.plot(angles, lst_corr, '-', linewidth=1.5,c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5,c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.set_yticks(np.arange(0, y_max + 0.2, 0.2))
    ax.set_title(region_info['title'], fontsize=16, loc='left', x=0, y=1.1)
    
    k = 0
    for angle, label in zip(angles[:-1], datasets):
        if angle == 0:
            ha, va = 'left', 'center'
        elif 0 < angle < np.pi/2:
            ha, va = 'left', 'bottom'
        elif angle == np.pi/2:
            ha, va = 'center', 'bottom'
        elif np.pi/2 < angle < np.pi:
            ha, va = 'right', 'bottom'
        elif angle == np.pi:
            ha, va = 'right', 'center'
        elif np.pi < angle < 3*np.pi/2:
            ha, va = 'right', 'top'
        elif angle == 3*np.pi/2:
            ha, va = 'center', 'top'
        else:
            ha, va = 'left', 'top'
            
        k += 1
        radius = 1.03 if region_name in ['NH', 'CONUS', 'EA'] else 0.83
        
        if k == 0:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='black')#, weight='bold')
        elif 1 <= k <= (4 if region_name == 'EU' else 5):  # EU 5,  6
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#0554F2')#, weight='bold')
        else:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#F25244')#, weight='bold')

#  figure    
legend = fig.legend(bbox_to_anchor=(.46, 0.5), loc='center',
                   labelspacing=1.2, markerscale=1, handletextpad=0.5,
                   frameon=False, fontsize=12)
plt.subplots_adjust(wspace = .7,hspace = .1)
# plt.tight_layout()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/regional_correlation_circles_finish.png', format='png', dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

#  figure  (2 4)
fig = plt.figure(figsize=(20, 8))

#   :    
regions = {
    'NH': (NH_ds, '(a) Global', 0.8),
    'CONUS': (conus_ds, '(b) CONUS', 0.8),
    'EU': (eu_ds, '(c) EU', 0.8),
    'EA': (ea_ds, '(d) EA', 0.8)
}

def get_color(index): # NH, CONUS, EA
    if index == 0:
        return 'black'
    elif 1 <= index <= 5:
        return '#0554F2'
    else:
        return '#F25244'

def get_color1(index):#EU
    if index == 0:
        return 'black'
    elif 1 <= index <= 4:
        return '#0554F2'
    else:
        return '#F25244'

#    (subplot 1-4)
for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 
    
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
        
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EU
        for i, marker in enumerate(eu_markers):
            name = eu_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    
    plt.title(title, fontsize=16, loc='left', x=0)

#   :   
regions_data = {
    'NH': {'data': NH_ds, 'title': '(e) ', 'titles': ea_data_title[1:], 'markers': ea_markers},
    'CONUS': {'data': conus_ds, 'title': '(f) ', 'titles': new_data_title[1:], 'markers': new_markers},
    'EU': {'data': eu_ds, 'title': '(g) ', 'titles': titles[1:], 'markers': markers},
    'EA': {'data': ea_ds, 'title': '(h) ', 'titles': ea_data_title[1:], 'markers': ea_markers}
}

all_corr_results = {}

amp_color = '#E74C3C'
lst_color = '#27AE60'
for region_name, region_info in regions_data.items():
    corr_data = {}
    data = region_info['data']
    region_titles = region_info['titles']
    
    i = 0
    for title in region_titles:
        i += 1
        if title == 'Obs':
            lst_column = 'lst'
            amp_column = 'amplitude'
        else:
            lst_column = f'lst_{title}'
            amp_column = f'amp_{title}'
            
        if region_name == 'NH':
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
        else:
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
            
        df_drop = merged_ds.to_dataframe().dropna()
        Had_hours = df_drop['lst'].tolist()
        Had_amp = df_drop['amplitude'].tolist()
        hours = df_drop[lst_column].tolist()
        amp = df_drop[amp_column].tolist()
        
        lst_r = circular_correlation(Had_hours, hours)
        lst_p = fisher_lee_test(Had_hours, hours)
        amp_r = stats.spearmanr(Had_amp, amp)[0]
        amp_p = stats.spearmanr(Had_amp, amp)[1]
        
        corr_data[title] = {
            'lst_corr': lst_r,
            'lst_p': lst_p,
            'amp_corr': amp_r,
            'amp_p': amp_p,
        }
    
    all_corr_results[region_name] = pd.DataFrame(corr_data)

#      (subplot 5-8)
for idx, (region_name, region_info) in enumerate(regions_data.items(), 5):  # 5 
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 5-8
    
    cor_df = all_corr_results[region_name]
    datasets = cor_df.columns
    lst_corr = cor_df.loc['lst_corr']
    amp_corr = cor_df.loc['amp_corr']
    
    angles = np.linspace(0, 2*np.pi, len(datasets), endpoint=False)
    lst_corr = np.concatenate((lst_corr, [lst_corr[0]]))
    amp_corr = np.concatenate((amp_corr, [amp_corr[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    y_min = min(min(lst_corr), min(amp_corr), 0.0)
    y_max = max(max(lst_corr), max(amp_corr))
    ax.set_ylim(y_min, y_max)
    
    ax.yaxis.grid(True, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill(np.linspace(0, 2*np.pi, 100), np.zeros(100), alpha=0.1, color='blue')
    
    #   - label   correlation subplot 
    if idx == 5:  #   correlation subplot
        ax.plot(angles, lst_corr, '-', linewidth=1.5, label='Phase',c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5, label='Amplitude',c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    else:
        ax.plot(angles, lst_corr, '-', linewidth=1.5,c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5,c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.set_yticks(np.arange(0, y_max + 0.2, 0.2))
    ax.set_title(region_info['title'], fontsize=16, loc='left', x=0, y=1.1)
    
    k = 0
    for angle, label in zip(angles[:-1], datasets):
        if angle == 0:
            ha, va = 'left', 'center'
        elif 0 < angle < np.pi/2:
            ha, va = 'left', 'bottom'
        elif angle == np.pi/2:
            ha, va = 'center', 'bottom'
        elif np.pi/2 < angle < np.pi:
            ha, va = 'right', 'bottom'
        elif angle == np.pi:
            ha, va = 'right', 'center'
        elif np.pi < angle < 3*np.pi/2:
            ha, va = 'right', 'top'
        elif angle == 3*np.pi/2:
            ha, va = 'center', 'top'
        else:
            ha, va = 'left', 'top'
            
        k += 1
        radius = 1.03 if region_name in ['NH', 'CONUS', 'EA'] else 0.83
        
        if k == 0:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='black')
        elif 1 <= k <= (4 if region_name == 'EU' else 5):
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#0554F2')
        else:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#F25244')

# #     ()
# legend1 = fig.legend(bbox_to_anchor=(1.02, 0.85), loc='upper left',
#                     labelspacing=1.2, markerscale=1, handletextpad=0.5, frameon=False)
# # title

# #    
# for i, text in enumerate(legend1.get_texts()):
#     if i == 0:
#         text.set_color('black')
#     elif 1 <= i <= 5:
#         text.set_color('#0554F2')
#     else:
#         text.set_color('#F25244')

#     ()
legend2 = fig.legend(bbox_to_anchor=(.98, 0.5), loc='center left',
                    labelspacing=1.2, markerscale=1, handletextpad=0.5,
                    frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(wspace=0., hspace=0.3)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/combined_regional_plots.png', 
#             format='png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

#  figure  (2 4)
fig = plt.figure(figsize=(20, 8))

obs_col = '#2C3A47'  #   (charcoal)
sattle_col = '#2E86AB'  #  (turquoise)
reana_col = '#E74C3C'  #  (pumpkin)

#   :    
regions = {
    'NH': (NH_ds, '(a) Global', 0.8),
    'CONUS': (conus_ds, '(b) CONUS', 0.8),
    'EU': (eu_ds, '(c) EU', 0.8),
    'EA': (ea_ds, '(d) EA', 0.8)
}


def get_color(index): # NH, CONUS, EA
    if index == 0:
        return obs_col
    elif 1 <= index <= 5:
        return sattle_col
    else:
        return reana_col

def get_color1(index):#EU
    if index == 0:
        return obs_col
    elif 1 <= index <= 4:
        return sattle_col
    else:
        return reana_col

#    (subplot 1-4)
for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 
    
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
        
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EU
        for i, marker in enumerate(eu_markers):
            name = eu_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    
    plt.title(title, fontsize=16, loc='left', x=0)

#   :   
regions_data = {
    'NH': {'data': NH_ds, 'title': '(e) ', 'titles': ea_data_title[1:], 'markers': ea_markers},
    'CONUS': {'data': conus_ds, 'title': '(f) ', 'titles': new_data_title[1:], 'markers': new_markers},
    'EU': {'data': eu_ds, 'title': '(g) ', 'titles': titles[1:], 'markers': markers},
    'EA': {'data': ea_ds, 'title': '(h) ', 'titles': ea_data_title[1:], 'markers': ea_markers}
}

all_corr_results = {}

amp_color = '#E74C3C'
lst_color = '#27AE60'
# amp_color = '#6B4C8A'   #  (plum)
# lst_color = '#9CAF5E'   #   (olive)

for region_name, region_info in regions_data.items():
    corr_data = {}
    data = region_info['data']
    region_titles = region_info['titles']
    
    i = 0
    for title in region_titles:
        i += 1
        if title == 'Obs':
            lst_column = 'lst'
            amp_column = 'amplitude'
        else:
            lst_column = f'lst_{title}'
            amp_column = f'amp_{title}'
            
        if region_name == 'NH':
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
        else:
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
            
        df_drop = merged_ds.to_dataframe().dropna()
        Had_hours = df_drop['lst'].tolist()
        Had_amp = df_drop['amplitude'].tolist()
        hours = df_drop[lst_column].tolist()
        amp = df_drop[amp_column].tolist()
        
        lst_r = circular_correlation(Had_hours, hours)
        lst_p = fisher_lee_test(Had_hours, hours)
        amp_r = stats.spearmanr(Had_amp, amp)[0]
        amp_p = stats.spearmanr(Had_amp, amp)[1]
        
        corr_data[title] = {
            'lst_corr': lst_r,
            'lst_p': lst_p,
            'amp_corr': amp_r,
            'amp_p': amp_p,
        }
    
    all_corr_results[region_name] = pd.DataFrame(corr_data)

#      (subplot 5-8)
for idx, (region_name, region_info) in enumerate(regions_data.items(), 5):  # 5 
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 5-8
    
    cor_df = all_corr_results[region_name]
    datasets = cor_df.columns
    lst_corr = cor_df.loc['lst_corr']
    amp_corr = cor_df.loc['amp_corr']
    
    angles = np.linspace(0, 2*np.pi, len(datasets), endpoint=False)
    lst_corr = np.concatenate((lst_corr, [lst_corr[0]]))
    amp_corr = np.concatenate((amp_corr, [amp_corr[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    y_min = min(min(lst_corr), min(amp_corr), 0.0)
    y_max = max(max(lst_corr), max(amp_corr))
    ax.set_ylim(y_min, y_max)
    
    ax.yaxis.grid(True, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill(np.linspace(0, 2*np.pi, 100), np.zeros(100), alpha=0.1, color='blue')
    
    #   - label   correlation subplot 
    if idx == 5:  #   correlation subplot
        ax.plot(angles, lst_corr, '-', linewidth=1.5, label='Phase',c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5, label='Amplitude',c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    else:
        ax.plot(angles, lst_corr, '-', linewidth=1.5,c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5,c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.set_yticks(np.arange(0, y_max + 0.2, 0.2))
    ax.set_title(region_info['title'], fontsize=16, loc='left', x=0, y=1.1)
    
    k = 0
    for angle, label in zip(angles[:-1], datasets):
        if angle == 0:
            ha, va = 'left', 'center'
        elif 0 < angle < np.pi/2:
            ha, va = 'left', 'bottom'
        elif angle == np.pi/2:
            ha, va = 'center', 'bottom'
        elif np.pi/2 < angle < np.pi:
            ha, va = 'right', 'bottom'
        elif angle == np.pi:
            ha, va = 'right', 'center'
        elif np.pi < angle < 3*np.pi/2:
            ha, va = 'right', 'top'
        elif angle == 3*np.pi/2:
            ha, va = 'center', 'top'
        else:
            ha, va = 'left', 'top'
            
        k += 1
        radius = 1.03 if region_name in ['NH', 'CONUS', 'EA'] else 0.83
        
        if k == 0:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='black')
        elif 1 <= k <= (4 if region_name == 'EU' else 5):
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c=sattle_col)
        else:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c=reana_col)


#     ()
legend2 = fig.legend(bbox_to_anchor=(.98, 0.5), loc='center left',
                    labelspacing=1.2, markerscale=1, handletextpad=0.5,
                    frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(wspace=0., hspace=0.3)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/combined_regional_plots.png', 
            format='png', dpi=400, bbox_inches='tight')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

#  figure  (2 4)
fig = plt.figure(figsize=(20, 8))
# domain_names = ['Global', 'CONUS', 'EU', 'EA']
domain_names = ['Globe', 'CONUS', 'Europe', 'East Asia']
#   :    
regions = {
    'NH': (NH_ds, '(a)', 0.8),
    'CONUS': (conus_ds, '(b)', 0.8),
    'EU': (eu_ds, '(c)', 0.8),
    'EA': (ea_ds, '(d)', 0.8)
}

def get_color(index): # NH, CONUS, EA
    if index == 0:
        return 'black'
    elif 1 <= index <= 5:
        return '#0554F2'
    else:
        return '#F25244'

def get_color1(index):#EU
    if index == 0:
        return 'black'
    elif 1 <= index <= 4:
        return '#0554F2'
    else:
        return '#F25244'

#    (subplot 1-4)
for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 
    
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
            ax.text((np.pi/2)*3, 1.3, 'Bias', ha='center', va='center', fontsize=18 ,rotation = 90   ) #-------------------------------------------------
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EU
        for i, marker in enumerate(eu_markers):
            name = eu_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    ax.set_title(domain_names[idx-1], fontsize=18,loc='center',y = 1.15)
    plt.title(title, fontsize=14, loc='left', x=0,y = 1.)

#   :   
regions_data = {
    'NH': {'data': NH_ds, 'title': '(e) ', 'titles': ea_data_title[1:], 'markers': ea_markers},
    'CONUS': {'data': conus_ds, 'title': '(f) ', 'titles': new_data_title[1:], 'markers': new_markers},
    'EU': {'data': eu_ds, 'title': '(g) ', 'titles': titles[1:], 'markers': markers},
    'EA': {'data': ea_ds, 'title': '(h) ', 'titles': ea_data_title[1:], 'markers': ea_markers}
}

all_corr_results = {}

amp_color = '#E74C3C'
lst_color = '#27AE60'
for region_name, region_info in regions_data.items():
    corr_data = {}
    data = region_info['data']
    region_titles = region_info['titles']
    
    i = 0
    for title in region_titles:
        i += 1
        if title == 'Obs':
            lst_column = 'lst'
            amp_column = 'amplitude'
        else:
            lst_column = f'lst_{title}'
            amp_column = f'amp_{title}'
            
        if region_name == 'NH':
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
        else:
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
            
        df_drop = merged_ds.to_dataframe().dropna()
        Had_hours = df_drop['lst'].tolist()
        Had_amp = df_drop['amplitude'].tolist()
        hours = df_drop[lst_column].tolist()
        amp = df_drop[amp_column].tolist()
        
        lst_r = circular_correlation(Had_hours, hours)
        lst_p = fisher_lee_test(Had_hours, hours)
        amp_r = stats.spearmanr(Had_amp, amp)[0]
        amp_p = stats.spearmanr(Had_amp, amp)[1]
        
        corr_data[title] = {
            'lst_corr': lst_r,
            'lst_p': lst_p,
            'amp_corr': amp_r,
            'amp_p': amp_p,
        }
    
    all_corr_results[region_name] = pd.DataFrame(corr_data)

#      (subplot 5-8)
for idx, (region_name, region_info) in enumerate(regions_data.items(), 5):  # 5 
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 5-8
    
    cor_df = all_corr_results[region_name]
    datasets = cor_df.columns
    lst_corr = cor_df.loc['lst_corr']
    amp_corr = cor_df.loc['amp_corr']
    
    angles = np.linspace(0, 2*np.pi, len(datasets), endpoint=False)
    lst_corr = np.concatenate((lst_corr, [lst_corr[0]]))
    amp_corr = np.concatenate((amp_corr, [amp_corr[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    y_min = min(min(lst_corr), min(amp_corr), 0.0)
    y_max = max(max(lst_corr), max(amp_corr))
    ax.set_ylim(y_min, y_max)
    
    ax.yaxis.grid(True, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill(np.linspace(0, 2*np.pi, 100), np.zeros(100), alpha=0.1, color='blue')
    
    #   - label   correlation subplot 
    if idx == 5:  #   correlation subplot
        ax.plot(angles, lst_corr, '-', linewidth=1.5, label='Phase',c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5, label='Amplitude',c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.text(np.pi, 1.6, 'Correlation', ha='center', va='center', fontsize=18,rotation = 90  ) #-------------------------------------------------
    else:
        ax.plot(angles, lst_corr, '-', linewidth=1.5,c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5,c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.set_yticks(np.arange(0, 1 + 0.2, 0.2))
    # ax.set_title(region_info['title'], fontsize=14, loc='left', x=0,y = 1.)
    plt.title(region_info['title'], fontsize=14, loc='left', x=0,y = 1.)
    k = 0
    for angle, label in zip(angles[:-1], datasets):
        if angle == 0:
            ha, va = 'left', 'center'
        elif 0 < angle < np.pi/2:
            ha, va = 'left', 'bottom'
        elif angle == np.pi/2:
            ha, va = 'center', 'bottom'
        elif np.pi/2 < angle < np.pi:
            ha, va = 'right', 'bottom'
        elif angle == np.pi:
            ha, va = 'right', 'center'
        elif np.pi < angle < 3*np.pi/2:
            ha, va = 'right', 'top'
        elif angle == 3*np.pi/2:
            ha, va = 'center', 'top'
        else:
            ha, va = 'left', 'top'
            
        k += 1
        radius = 1.03 if region_name in ['NH', 'CONUS', 'EA'] else 1.03
        
        if k == 0:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='black')
        elif 1 <= k <= (4 if region_name == 'EU' else 5):
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#0554F2')
        else:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#F25244')
    
legend2 = fig.legend(bbox_to_anchor=(.98, 0.5), loc='center left',
                    labelspacing=1.2, markerscale=1, handletextpad=0.5,
                    frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(wspace=0., hspace=0.3)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/combined_regional_plots.png', 
            format='png', dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle

#  figure  (2 4)
fig = plt.figure(figsize=(20, 8))
domain_names = ['NH', 'CONUS', 'EU', 'EA']

#   :    
regions = {
    'NH': (NH_ds, '(a)', 0.8),
    'CONUS': (conus_ds, '(b)', 0.8),
    'EU': (eu_ds, '(c)', 0.8),
    'EA': (ea_ds, '(d)', 0.8)
}

def get_color(index): # NH, CONUS, EA
    if index == 0:
        return 'black'
    elif 1 <= index <= 5:
        return '#0554F2'
    else:
        return '#F25244'

def get_color1(index):#EU
    if index == 0:
        return 'black'
    elif 1 <= index <= 4:
        return '#0554F2'
    else:
        return '#F25244'

#    (subplot 1-4)
for idx, (region, (data, title, max_val)) in enumerate(regions.items(), 1):
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 
    
    if idx == 1:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
            ax.text((np.pi/2)*3, 1.3, 'Bias', ha='center', va='center', fontsize=18 ,rotation = 90   ) #-------------------------------------------------
        #  scatter legend 
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            color = get_color(i)
            face_color = mcolors.to_rgba(color, alpha=0.8)
            
            #      scatter  (legend)
            ax.scatter([], [], s=220, label=name,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5)
    elif idx == 2:
        for i, marker in enumerate(new_markers):
            name = new_data_title[i]
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
        
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    elif idx == 3:#EU
        for i, marker in enumerate(eu_markers):
            name = eu_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color1(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)
    else:
        for i, marker in enumerate(ea_markers):
            name = ea_data_title[i]
            
            dataset = data[name]
            mean_hour = circular_mean_hours(dataset['lst'].values[~np.isnan(dataset['lst'].values)])
            mean_amp = dataset['amplitude'].mean(['longitude','latitude']).item()
                
            angle = mean_hour * 2 * np.pi / 24
            color = get_color(i)
            
            face_color = mcolors.to_rgba(color, alpha=0.8)
            ax.scatter(angle, mean_amp, s=220,
                      marker=MarkerStyle(marker, fillstyle='full'),
                      facecolor=face_color, edgecolors='black', linewidth=0.5,
                      zorder=4)

    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')
    ax.set_title(domain_names[idx-1], fontsize=18,loc='center',y = 1.15)
    plt.title(title, fontsize=14, loc='left', x=0,y = 1.)

#   :   
regions_data = {
    'NH': {'data': NH_ds, 'title': '(e) ', 'titles': ea_data_title[1:], 'markers': ea_markers},
    'CONUS': {'data': conus_ds, 'title': '(f) ', 'titles': new_data_title[1:], 'markers': new_markers},
    'EU': {'data': eu_ds, 'title': '(g) ', 'titles': titles[1:], 'markers': markers},
    'EA': {'data': ea_ds, 'title': '(h) ', 'titles': ea_data_title[1:], 'markers': ea_markers}
}

all_corr_results = {}

amp_color = '#E74C3C'
lst_color = '#27AE60'
for region_name, region_info in regions_data.items():
    corr_data = {}
    data = region_info['data']
    region_titles = region_info['titles']
    
    i = 0
    for title in region_titles:
        i += 1
        if title == 'Obs':
            lst_column = 'lst'
            amp_column = 'amplitude'
        else:
            lst_column = f'lst_{title}'
            amp_column = f'amp_{title}'
            
        if region_name == 'NH':
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
        else:
            merged_ds = xr.merge([data['Obs'][['lst','amplitude']],
                                data[title].rename(lst=lst_column, amplitude=amp_column)[[lst_column,amp_column]]])
            
        df_drop = merged_ds.to_dataframe().dropna()
        Had_hours = df_drop['lst'].tolist()
        Had_amp = df_drop['amplitude'].tolist()
        hours = df_drop[lst_column].tolist()
        amp = df_drop[amp_column].tolist()
        
        lst_r = circular_correlation(Had_hours, hours)
        lst_p = fisher_lee_test(Had_hours, hours)
        amp_r = stats.spearmanr(Had_amp, amp)[0]
        amp_p = stats.spearmanr(Had_amp, amp)[1]
        
        corr_data[title] = {
            'lst_corr': lst_r,
            'lst_p': lst_p,
            'amp_corr': amp_r,
            'amp_p': amp_p,
        }
    
    all_corr_results[region_name] = pd.DataFrame(corr_data)

#      (subplot 5-8)
for idx, (region_name, region_info) in enumerate(regions_data.items(), 5):  # 5 
    ax = fig.add_subplot(2, 4, idx, projection='polar')  # 2 4 5-8
    
    cor_df = all_corr_results[region_name]
    datasets = cor_df.columns
    lst_corr = cor_df.loc['lst_corr']
    amp_corr = cor_df.loc['amp_corr']
    
    angles = np.linspace(0, 2*np.pi, len(datasets), endpoint=False)
    lst_corr = np.concatenate((lst_corr, [lst_corr[0]]))
    amp_corr = np.concatenate((amp_corr, [amp_corr[0]]))
    angles = np.concatenate((angles, [angles[0]]))
    
    y_min = min(min(lst_corr), min(amp_corr), 0.0)
    y_max = max(max(lst_corr), max(amp_corr))
    ax.set_ylim(y_min, y_max)
    
    ax.yaxis.grid(True, color='gray', linestyle='--', alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill(np.linspace(0, 2*np.pi, 100), np.zeros(100), alpha=0.1, color='blue')
    
    #   - label   correlation subplot 
    if idx == 5:  #   correlation subplot
        ax.plot(angles, lst_corr, '-', linewidth=1.5, label='Phase',c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5, label='Amplitude',c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.text(np.pi, 1.6, 'Correlation', ha='center', va='center', fontsize=18,rotation = 90  ) #-------------------------------------------------
    else:
        ax.plot(angles, lst_corr, '-', linewidth=1.5,c = lst_color)
        ax.fill(angles, lst_corr, alpha=0.25,c = lst_color)
        ax.plot(angles, amp_corr, '-.', linewidth=1.5,c = amp_color)
        ax.fill(angles, amp_corr, alpha=0.25,c = amp_color)
        ax.scatter(angles[:-1], lst_corr[:-1], s=30, color=lst_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
        ax.scatter(angles[:-1], amp_corr[:-1], s=30, color=amp_color, 
                  edgecolors='white', linewidth=1.5, zorder=4, alpha=0.9)
    
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([])
    ax.set_yticks(np.arange(0, 1 + 0.2, 0.2))
    # ax.set_title(region_info['title'], fontsize=14, loc='left', x=0,y = 1.)
    plt.title(region_info['title'], fontsize=14, loc='left', x=0,y = 1.)
    k = 0
    for angle, label in zip(angles[:-1], datasets):
        if angle == 0:
            ha, va = 'left', 'center'
        elif 0 < angle < np.pi/2:
            ha, va = 'left', 'bottom'
        elif angle == np.pi/2:
            ha, va = 'center', 'bottom'
        elif np.pi/2 < angle < np.pi:
            ha, va = 'right', 'bottom'
        elif angle == np.pi:
            ha, va = 'right', 'center'
        elif np.pi < angle < 3*np.pi/2:
            ha, va = 'right', 'top'
        elif angle == 3*np.pi/2:
            ha, va = 'center', 'top'
        else:
            ha, va = 'left', 'top'
            
        k += 1
        radius = 1.03 if region_name in ['NH', 'CONUS', 'EA'] else 1.03
        
        if k == 0:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='black')
        elif 1 <= k <= (4 if region_name == 'EU' else 5):
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#0554F2')
        else:
            ax.text(angle, radius, label, ha=ha, va=va, fontsize=12, c='#F25244')
    
legend2 = fig.legend(bbox_to_anchor=(.98, 0.5), loc='center left',
                    labelspacing=1.2, markerscale=1, handletextpad=0.5,
                    frameon=False, fontsize=12)
plt.tight_layout()
plt.subplots_adjust(wspace=0., hspace=0.3)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/combined_regional_plots_NH.png', 
            format='png', dpi=300, bbox_inches='tight')
# plt.show()

In [ ]:
angles

## 6. Precipitation Types


In [ ]:
rdir = '${DATA_DIR}/'
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc').rename(lst='tp_lst')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc').rename(lst='tp_lst')

jra = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc')

In [ ]:
obs = cmorph
# obs = obs.rename
condition_mask = ((obs['both_mask']== True)|(era5['both_mask']== True)|(merra2['both_mask']== True)|(jra['both_mask']== True))

tp_mask = ((obs['tp_lst'].isnull())|(era5['tp_lst'].isnull())|(merra2['tp_lst'].isnull())|(jra['tp_lst'].isnull()))
cp_mask = ((era5['cp_lst'].isnull())|(merra2['cp_lst'].isnull())|(jra['cp_lst'].isnull()))
lp_mask = ((era5['lp_lst'].isnull())|(merra2['lp_lst'].isnull())|(jra['lp_lst'].isnull()))

final_tp_mask = condition_mask | tp_mask
final_cp_mask = condition_mask | cp_mask
final_lp_mask = condition_mask | lp_mask

print(f"Number of grids satisfying tp_lst condition: {(~final_tp_mask).sum().values}")
print(f"Number of grids satisfying cp_lst condition: {(~final_cp_mask).sum().values}")
print(f"Number of grids satisfying lp_lst condition: {(~final_lp_mask).sum().values}")

mask_ds = xr.Dataset({
    'tp_valid_region': ~final_tp_mask,
    'cp_valid_region': ~final_cp_mask,
    'lp_valid_region': ~final_lp_mask
})

In [ ]:
mask_ds['lp_valid_region'] = mask_ds['lp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>-60),0.)
mask_ds['tp_valid_region'] = mask_ds['tp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>-60),0.)
mask_ds['cp_valid_region'] = mask_ds['cp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>-60),0.)

In [ ]:
# restricted to NH..& points with observations
obs_result = xr.open_dataset('${DATA_DIR}/obs/obs_gridded.nc')
mask_ds['lp_valid_region'] = mask_ds['lp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>0)&(~obs_result['lst'].isnull()),0.)
mask_ds['tp_valid_region'] = mask_ds['tp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>0)&(~obs_result['lst'].isnull()),0.)
mask_ds['cp_valid_region'] = mask_ds['cp_valid_region'].where((mask_ds.latitude<60)&(mask_ds.latitude>0)&(~obs_result['lst'].isnull()),0.)

In [ ]:
def masking_def(ds,mask,sat):
    if sat == True:
        ds['tp_lst']  = ds['tp_lst'].where(mask['tp_valid_region'])
    else:
        ds['tp_lst']  = ds['tp_lst'].where(mask['tp_valid_region'])
        ds['cp_lst']  = ds['cp_lst'].where(mask['cp_valid_region'])
        ds['lp_lst']  = ds['lp_lst'].where(mask['lp_valid_region'])
    return ds
jra = masking_def(jra,mask_ds,False)
merra2 = masking_def(merra2,mask_ds,False)
era5 = masking_def(era5,mask_ds,False)
obs = masking_def(obs,mask_ds,True)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

def plot_precipitation_distribution(datasets, lsm, sat_mean):
    variables = ['tp_lst', 'cp_lst', 'lp_lst']
    # titles = ['Total Precipitation', 'Convective Precipitation', 'Large-scale Precipitation']
    titles = ['TP', 'CP', 'LSP']

    dataset_names = ['JRA-3Q', 'ERA5', 'MERRA2']
    y_limits = [9000, 9000, 9000, 9000]

    fig, axes = plt.subplots(4, 3, figsize=(17, 9), sharey='row')
    
    # sat_mean  
    ax_sat = axes[0, 0]
    data = cmorph['tp_lst'].sel(latitude=slice(-60, 60))
    land_data = data.where(lsm == 1)
    ocean_data = data.where(lsm == 0)
    land_counts = []
    ocean_counts = []
    for i in range(24):
        land_count = ((land_data >= i) & (land_data < i+1)).sum().values
        ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
        land_counts.append(land_count)
        ocean_counts.append(ocean_count)
    total_land = np.sum(land_counts)
    total_ocean = np.sum(ocean_counts)
    x = np.arange(24)
    width = 0.6
    ax_sat.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
    ax_sat.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
    
    land_max_idx = np.argmax(land_counts)
    ocean_max_idx = np.argmax(ocean_counts)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#F25041', 
                   alpha=0.4)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#0367A6', 
                   alpha=0.4)

    
    ax_sat.set_ylabel('Number of Grids', fontsize=10)
    # ax_sat.set_title('CMORPH', fontsize=14)
    ax_sat.grid(axis='y', linestyle='--', alpha=0.7)
    ax_sat.set_xlim(0, 24)
    ax_sat.set_ylim(0, y_limits[0])
    ax_sat.set_xticks(range(0, 25, 3))
    ax_sat.legend(loc='upper right', fontsize=11)
    ax_sat.text(0.02, 1.02, '(a) CMORPH', transform=ax_sat.transAxes, fontsize=14, va='bottom')
    ax_sat.text(-5, 9000/2, titles[0], fontsize=14, va='center',ha = 'center',rotation = 90)
    # sat_mean    
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    # axes[0, 3].axis('off')

    labels = ['(b) JRA-3Q', '(c) ERA5', '(d) MERRA-2', 
              '(e)', '(f)', '(g)', 
              '(h)', '(i)', '(j)']#, '(k)', '(l)', '(m)']
    label_index = 0
    
    for row, (var, title, y_lim) in enumerate(zip(variables, titles, y_limits[1:])):
        for col, (ds, ds_name) in enumerate(zip(datasets, dataset_names)):
            ax = axes[row+1, col]
            
            data = ds.sel(latitude=slice(-60, 60))[var]
            land_data = data.where(lsm == 1)
            ocean_data = data.where(lsm == 0)
            land_counts = []
            ocean_counts = []
            for i in range(24):
                land_count = ((land_data >= i) & (land_data < i+1)).sum().values
                ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
                land_counts.append(land_count)
                ocean_counts.append(ocean_count)
            total_land = np.sum(land_counts)
            total_ocean = np.sum(ocean_counts)
            x = np.arange(24)
            width = 0.6
            ax.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
            ax.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
            ax.axvline(x= 5 + width*0.75, color='#0367A6', linestyle='--',alpha = .6,lw= .7)
            ax.axvline(x= 18 + width*0.25, color='#F25041', linestyle='--',alpha = .6,lw= .7)
            #   shade 
            land_max_idx = np.argmax(land_counts)
            ocean_max_idx = np.argmax(ocean_counts)
            # ax.set_xticks(range(0, 25, 3))
            #     ( 2 )
            ax.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#F25041', 
                           alpha=0.4)
            
            #     ( 2 )
            ax.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#0367A6', 
                           alpha=0.4)
            ax.text(0.02, 1.02, labels[label_index], transform=ax.transAxes, fontsize=14, va='bottom')
            label_index += 1

            if row == 3:
                ax.set_xlabel('LST', fontsize=10)
            if col == 0:
                ax.set_ylabel('Number of Grids', fontsize=10)
                ax.text(-5, 9000/2, titles[row], fontsize=14, va='center',ha = 'center',rotation = 90)
            if row == 0:
                ax.set_title('')

                # ax.set_title(f'{ds_name}', fontsize=14)
            else:
                ax.set_title('')
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.set_xlim(0, 24)
            ax.set_ylim(0, y_lim)

            if row == 2:
                ax.set_xlabel('LST', fontsize=10)

                ax.set_xticks(range(0, 25, 3))
            else:
                # ax.set_xlabel('LST', fontsize=10)
                ax.set_xticks(range(0, 25, 3))
            if col != 0:
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
                
    plt.tight_layout(rect=[0.1, 0.03, 1, 0.95])
    plt.show()
    # plt.savefig('${FIG_DIR}/pre1diurnal_cycle/model_grid_finish.png',dpi=400,format='png',bbox_inches='tight')

datasets = [jra, era5, merra2]
plot_precipitation_distribution(datasets, lsm, obs)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

def plot_precipitation_distribution(datasets, lsm, sat_mean):
    variables = ['tp_lst', 'cp_lst', 'lp_lst']
    # titles = ['Total Precipitation', 'Convective Precipitation', 'Large-scale Precipitation']
    titles = ['TP', 'CP', 'LSP']

    # land_col = '#F25041'
    # ocean_col = '#0367A6'
    land_col = '#C9956B'  #   (warm beige)
    ocean_col = '#1A5F7A'  #   (cool blue)

    dataset_names = ['JRA-3Q', 'ERA5', 'MERRA2']
    y_limits = [9000, 9000, 9000, 9000]

    fig, axes = plt.subplots(4, 3, figsize=(13, 8), sharey='row')
    
    # sat_mean  
    ax_sat = axes[0, 0]
    data = cmorph['tp_lst'].sel(latitude=slice(-60, 60))
    land_data = data.where(lsm == 1)
    ocean_data = data.where(lsm == 0)
    land_counts = []
    ocean_counts = []
    for i in range(24):
        land_count = ((land_data >= i) & (land_data < i+1)).sum().values
        ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
        land_counts.append(land_count)
        ocean_counts.append(ocean_count)
    total_land = np.sum(land_counts)
    total_ocean = np.sum(ocean_counts)
    x = np.arange(24)
    width = 0.6
    ax_sat.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color=land_col, alpha=0.7)
    ax_sat.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color=ocean_col, alpha=0.7)
    
    land_max_idx = np.argmax(land_counts)
    ocean_max_idx = np.argmax(ocean_counts)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                   0, y_limits[0], 
                   color=land_col, 
                   alpha=0.4)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                   0, y_limits[0], 
                   color=ocean_col, 
                   alpha=0.4)

    
    ax_sat.set_ylabel('Number of Grids', fontsize=10)
    # ax_sat.set_title('CMORPH', fontsize=14)
    ax_sat.grid(axis='y', linestyle='--', alpha=0.7)
    ax_sat.set_xlim(0, 24)
    ax_sat.set_ylim(0, y_limits[0])
    ax_sat.set_xticks(range(0, 25, 3))
    ax_sat.tick_params(axis='x',labelsize = 11)
    ax_sat.legend(loc='upper right', fontsize='small')
    ax_sat.text(0.02, 1.02, '(a) CMORPH', transform=ax_sat.transAxes, fontsize=14, va='bottom')
    ax_sat.text(-6, 9000/2, titles[0], fontsize=14, va='center',ha = 'center',rotation = 90)
    # sat_mean    
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    # axes[0, 3].axis('off')

    labels = ['(b) JRA-3Q', '(c) ERA5', '(d) MERRA-2', 
              '(e)', '(f)', '(g)', 
              '(h)', '(i)', '(j)']#, '(k)', '(l)', '(m)']
    label_index = 0
    
    for row, (var, title, y_lim) in enumerate(zip(variables, titles, y_limits[1:])):
        for col, (ds, ds_name) in enumerate(zip(datasets, dataset_names)):
            ax = axes[row+1, col]
            
            data = ds.sel(latitude=slice(-60, 60))[var]
            land_data = data.where(lsm == 1)
            ocean_data = data.where(lsm == 0)
            land_counts = []
            ocean_counts = []
            for i in range(24):
                land_count = ((land_data >= i) & (land_data < i+1)).sum().values
                ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
                land_counts.append(land_count)
                ocean_counts.append(ocean_count)
            total_land = np.sum(land_counts)
            total_ocean = np.sum(ocean_counts)
            x = np.arange(24)
            width = 0.6
            ax.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color=land_col, alpha=0.7)
            ax.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color=ocean_col, alpha=0.7)
            ax.axvline(x= 5 + width*0.75, color=ocean_col, linestyle='--',alpha = .6,lw= .7)
            ax.axvline(x= 18 + width*0.25, color=land_col, linestyle='--',alpha = .6,lw= .7)
            #   shade 
            land_max_idx = np.argmax(land_counts)
            ocean_max_idx = np.argmax(ocean_counts)
            # ax.set_xticks(range(0, 25, 3))
            #     ( 2 )
            ax.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                           0, y_limits[0], 
                           color=land_col, 
                           alpha=0.4)
            
            #     ( 2 )
            ax.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                           0, y_limits[0], 
                           color=ocean_col, 
                           alpha=0.4)
            ax.text(0.02, 1.02, labels[label_index], transform=ax.transAxes, fontsize=14, va='bottom')
            label_index += 1

            if row == 3:
                ax.set_xlabel('LST [hour]', fontsize=12)
            if col == 0:
                ax.set_ylabel('Number of Grids', fontsize=10)
                ax.text(-6, 9000/2, titles[row], fontsize=14, va='center',ha = 'center',rotation = 90)
            if row == 0:
                ax.set_title('')

                # ax.set_title(f'{ds_name}', fontsize=14)
            else:
                ax.set_title('')
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.set_xlim(0, 24)
            ax.set_ylim(0, y_lim)

            if row == 2:
                ax.set_xlabel('LST [hour]', fontsize=12)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            else:
                # ax.set_xlabel('LST', fontsize=10)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            if col != 0:
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
                
    # plt.tight_layout(rect=[0.1, 0.03, 1, 0.95])
    # plt.subplots(constrained_layout=True)
    plt.subplots_adjust(wspace=0.05, hspace=0.43)
    # plt.show()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/model_grid_finish.png',dpi=400,format='png',bbox_inches='tight')

datasets = [jra, era5, merra2]
plot_precipitation_distribution(datasets, lsm, obs)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

def plot_precipitation_distribution(datasets, lsm, sat_mean):
    variables = ['tp_lst', 'cp_lst', 'lp_lst']
    # titles = ['Total Precipitation', 'Convective Precipitation', 'Large-scale Precipitation']
    titles = ['TP', 'CP', 'LSP']

    dataset_names = ['JRA-3Q', 'ERA5', 'MERRA2']
    y_limits = [9000, 9000, 9000, 9000]

    fig, axes = plt.subplots(4, 3, figsize=(13, 8), sharey='row')
    
    # sat_mean  
    ax_sat = axes[0, 0]
    data = cmorph['tp_lst'].sel(latitude=slice(-60, 60))
    land_data = data.where(lsm == 1)
    ocean_data = data.where(lsm == 0)
    land_counts = []
    ocean_counts = []
    for i in range(24):
        land_count = ((land_data >= i) & (land_data < i+1)).sum().values
        ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
        land_counts.append(land_count)
        ocean_counts.append(ocean_count)
    total_land = np.sum(land_counts)
    total_ocean = np.sum(ocean_counts)
    x = np.arange(24)
    width = 0.6
    ax_sat.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
    ax_sat.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
    
    land_max_idx = np.argmax(land_counts)
    ocean_max_idx = np.argmax(ocean_counts)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#F25041', 
                   alpha=0.4)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#0367A6', 
                   alpha=0.4)

    
    ax_sat.set_ylabel('Number of Grids', fontsize=10)
    # ax_sat.set_title('CMORPH', fontsize=14)
    ax_sat.grid(axis='y', linestyle='--', alpha=0.7)
    ax_sat.set_xlim(0, 24)
    ax_sat.set_ylim(0, y_limits[0])
    ax_sat.set_xticks(range(0, 25, 3))
    ax_sat.tick_params(axis='x',labelsize = 11)
    ax_sat.legend(loc='upper right', fontsize='small')
    ax_sat.text(0.02, 1.02, '(a) CMORPH', transform=ax_sat.transAxes, fontsize=14, va='bottom')
    ax_sat.text(-6, 9000/2, titles[0], fontsize=14, va='center',ha = 'center',rotation = 90)
    # sat_mean    
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    # axes[0, 3].axis('off')

    labels = ['(b) JRA-3Q', '(c) ERA5', '(d) MERRA-2', 
              '(e)', '(f)', '(g)', 
              '(h)', '(i)', '(j)']#, '(k)', '(l)', '(m)']
    label_index = 0
    
    for row, (var, title, y_lim) in enumerate(zip(variables, titles, y_limits[1:])):
        for col, (ds, ds_name) in enumerate(zip(datasets, dataset_names)):
            ax = axes[row+1, col]
            
            data = ds.sel(latitude=slice(-60, 60))[var]
            land_data = data.where(lsm == 1)
            ocean_data = data.where(lsm == 0)
            land_counts = []
            ocean_counts = []
            for i in range(24):
                land_count = ((land_data >= i) & (land_data < i+1)).sum().values
                ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
                land_counts.append(land_count)
                ocean_counts.append(ocean_count)
            total_land = np.sum(land_counts)
            total_ocean = np.sum(ocean_counts)
            x = np.arange(24)
            width = 0.6
            ax.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
            ax.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
            ax.axvline(x= 5 + width*0.75, color='#0367A6', linestyle='--',alpha = .6,lw= .7)
            ax.axvline(x= 18 + width*0.25, color='#F25041', linestyle='--',alpha = .6,lw= .7)
            #   shade 
            land_max_idx = np.argmax(land_counts)
            ocean_max_idx = np.argmax(ocean_counts)
            # ax.set_xticks(range(0, 25, 3))
            #     ( 2 )
            ax.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#F25041', 
                           alpha=0.4)
            
            #     ( 2 )
            ax.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#0367A6', 
                           alpha=0.4)
            ax.text(0.02, 1.02, labels[label_index], transform=ax.transAxes, fontsize=14, va='bottom')
            label_index += 1

            if row == 3:
                ax.set_xlabel('LST [hour]', fontsize=12)
            if col == 0:
                ax.set_ylabel('Number of Grids', fontsize=10)
                ax.text(-6, 9000/2, titles[row], fontsize=14, va='center',ha = 'center',rotation = 90)
            if row == 0:
                ax.set_title('')

                # ax.set_title(f'{ds_name}', fontsize=14)
            else:
                ax.set_title('')
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.set_xlim(0, 24)
            ax.set_ylim(0, y_lim)

            if row == 2:
                ax.set_xlabel('LST [hour]', fontsize=12)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            else:
                # ax.set_xlabel('LST', fontsize=10)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            if col != 0:
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
                
    # plt.tight_layout(rect=[0.1, 0.03, 1, 0.95])
    # plt.subplots(constrained_layout=True)
    plt.subplots_adjust(wspace=0.05, hspace=0.43)
    plt.show()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/model_grid_finish.png',dpi=400,format='png',bbox_inches='tight')

datasets = [jra, era5, merra2]
plot_precipitation_distribution(datasets, lsm, obs)

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

def plot_precipitation_distribution(datasets, lsm, sat_mean):
    variables = ['tp_lst', 'cp_lst', 'lp_lst']
    # titles = ['Total Precipitation', 'Convective Precipitation', 'Large-scale Precipitation']
    titles = ['TP', 'CP', 'LSP']

    dataset_names = ['JRA-3Q', 'ERA5', 'MERRA2']
    y_limits = [1000, 1000, 1000, 1000]

    fig, axes = plt.subplots(4, 3, figsize=(13, 8), sharey='row')
    
    # sat_mean  
    ax_sat = axes[0, 0]
    data = cmorph['tp_lst'].sel(latitude=slice(-60, 60))
    land_data = data.where(lsm == 1)
    ocean_data = data.where(lsm == 0)
    land_counts = []
    ocean_counts = []
    for i in range(24):
        land_count = ((land_data >= i) & (land_data < i+1)).sum().values
        ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
        land_counts.append(land_count)
        ocean_counts.append(ocean_count)
    total_land = np.sum(land_counts)
    total_ocean = np.sum(ocean_counts)
    x = np.arange(24)
    width = 0.6
    ax_sat.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
    ax_sat.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
    
    land_max_idx = np.argmax(land_counts)
    ocean_max_idx = np.argmax(ocean_counts)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#F25041', 
                   alpha=0.4)
    
    #     ( 2 )
    ax_sat.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                   0, y_limits[0], 
                   color='#0367A6', 
                   alpha=0.4)

    
    ax_sat.set_ylabel('Number of Grids', fontsize=10)
    # ax_sat.set_title('CMORPH', fontsize=14)
    ax_sat.grid(axis='y', linestyle='--', alpha=0.7)
    ax_sat.set_xlim(0, 24)
    ax_sat.set_ylim(0, y_limits[0])
    ax_sat.set_xticks(range(0, 25, 3))
    ax_sat.tick_params(axis='x',labelsize = 11)
    ax_sat.legend(loc='upper right', fontsize='small')
    ax_sat.text(0.02, 1.02, '(a) CMORPH', transform=ax_sat.transAxes, fontsize=14, va='bottom')
    ax_sat.text(-6, 1000/2, titles[0], fontsize=14, va='center',ha = 'center',rotation = 90)
    # sat_mean    
    axes[0, 1].axis('off')
    axes[0, 2].axis('off')
    # axes[0, 3].axis('off')

    labels = ['(b) JRA-3Q', '(c) ERA5', '(d) MERRA-2', 
              '(e)', '(f)', '(g)', 
              '(h)', '(i)', '(j)']#, '(k)', '(l)', '(m)']
    label_index = 0
    
    for row, (var, title, y_lim) in enumerate(zip(variables, titles, y_limits[1:])):
        for col, (ds, ds_name) in enumerate(zip(datasets, dataset_names)):
            ax = axes[row+1, col]
            
            data = ds.sel(latitude=slice(-60, 60))[var]
            land_data = data.where(lsm == 1)
            ocean_data = data.where(lsm == 0)
            land_counts = []
            ocean_counts = []
            for i in range(24):
                land_count = ((land_data >= i) & (land_data < i+1)).sum().values
                ocean_count = ((ocean_data >= i) & (ocean_data < i+1)).sum().values
                land_counts.append(land_count)
                ocean_counts.append(ocean_count)
            total_land = np.sum(land_counts)
            total_ocean = np.sum(ocean_counts)
            x = np.arange(24)
            width = 0.6
            ax.bar(x + width*0.25, land_counts, width*0.5, label=f'Land ({total_land})', color='#F25041', alpha=0.7)
            ax.bar(x + width*0.75, ocean_counts, width*0.5, label=f'Ocean ({total_ocean})', color='#0367A6', alpha=0.7)
            ax.axvline(x= 5 + width*0.75, color='#0367A6', linestyle='--',alpha = .6,lw= .7)
            ax.axvline(x= 18 + width*0.25, color='#F25041', linestyle='--',alpha = .6,lw= .7)
            #   shade 
            land_max_idx = np.argmax(land_counts)
            ocean_max_idx = np.argmax(ocean_counts)
            # ax.set_xticks(range(0, 25, 3))
            #     ( 2 )
            ax.fill_between([max(0, land_max_idx - .25), min(24, land_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#F25041', 
                           alpha=0.4)
            
            #     ( 2 )
            ax.fill_between([max(0, ocean_max_idx - .25), min(24, ocean_max_idx + .75)], 
                           0, y_limits[0], 
                           color='#0367A6', 
                           alpha=0.4)
            ax.text(0.02, 1.02, labels[label_index], transform=ax.transAxes, fontsize=14, va='bottom')
            label_index += 1

            if row == 3:
                ax.set_xlabel('LST [hour]', fontsize=12)
            if col == 0:
                ax.set_ylabel('Number of Grids', fontsize=10)
                ax.text(-6, 1000/2, titles[row], fontsize=14, va='center',ha = 'center',rotation = 90)
            if row == 0:
                ax.set_title('')

                # ax.set_title(f'{ds_name}', fontsize=14)
            else:
                ax.set_title('')
            ax.grid(axis='y', linestyle='--', alpha=0.7)
            ax.set_xlim(0, 24)
            ax.set_ylim(0, y_lim)

            if row == 2:
                ax.set_xlabel('LST [hour]', fontsize=12)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            else:
                # ax.set_xlabel('LST', fontsize=10)
                ax.set_xticks(range(0, 25, 3))
                ax.tick_params(axis='x',labelsize = 11)
            if col != 0:
                ax.tick_params(axis='y', which='both', left=False, labelleft=False)
                
    # plt.tight_layout(rect=[0.1, 0.03, 1, 0.95])
    # plt.subplots(constrained_layout=True)
    plt.subplots_adjust(wspace=0.05, hspace=0.43)
    plt.show()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/model_grid_finish.png',dpi=400,format='png',bbox_inches='tight')

datasets = [jra, era5, merra2]
plot_precipitation_distribution(datasets, lsm, obs)

## 7. IGRA SGP Station Analysis


### Level and Precipitation


In [ ]:
thsdir = '${CODE_DIR}/ARM/diurnal_cycle/prec/lst/not_null_tot/3hourly'
dsdir = '${CODE_DIR}/ARM/diurnal_cycle/prec/lst/not_null_tot/daily'
levsdir = '${CODE_DIR}/ARM/diurnal_cycle/lev/lst/not_null_tot'

sname_list = ['ARM_prec.nc',
              'ERA5_prec.nc',
              'NARR_prec.nc',
              'JRA3Q_prec.nc',
              'MERRA2_prec.nc']

lev_sname_list = ['ARM_lev.nc',
              'ERA5_lev.nc',
              'NARR_lev.nc',
              'JRA3Q_lev.nc',
              'MERRA2_lev.nc']

notnll_lev_ds = []
notnll_rain_ds = []
notnll_daily_ds = []

for i in range(len(sname_list)):
    notnll_lev_ds.append(xr.open_dataset(os.path.join(levsdir,lev_sname_list[i])))
    notnll_rain_ds.append(xr.open_dataset(os.path.join(thsdir,sname_list[i])))
    notnll_daily_ds.append(xr.open_dataset(os.path.join(dsdir,sname_list[i])))

In [ ]:
rain_dates = []
for i in range(len(notnll_daily_ds)):
    rain_dates.append(np.unique(notnll_daily_ds[i].time.dt.floor('D').values))

In [ ]:
from functools import reduce
not_null_rain_date = reduce(np.intersect1d, rain_dates)
len(not_null_rain_date)#990

### Surface Flux


In [ ]:
arm_flx = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/FLX/LST/ARM_FLX.nc')
era_flx = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/FLX/UTC/ERA5_FLX.nc')
narr_flx = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/FLX/UTC/NARR_FLX.nc')  #   3 .
merra_flx = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/FLX/UTC/MERRA2_FLX.nc')
jra_flx = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/FLX/UTC/JRA3Q_FLX.nc')

In [ ]:
arm_flx_test = xr.open_dataset('${CODE_DIR}/ARM/diurnal_cycle/advc/lst/ARM_advc.nc') # 3h
arm_flx_test = arm_flx_test[['h','le']]

In [ ]:
arm_flx['time'] = arm_flx['time'].values - np.timedelta64(1,'h')  # then 1-hour average
era_flx['time'] = era_flx['time'].values - np.timedelta64(7,'h')  # then 1-hour average
narr_flx['time'] = narr_flx['time'].values - np.timedelta64(6,'h')  #   3 
merra_flx['time'] = merra_flx['time'].values - np.timedelta64(6,'h')- np.timedelta64(30,'m')  # then 1-hour average
jra_flx['time'] = jra_flx['time'].values - np.timedelta64(7,'h')  # then 1-hour average

In [ ]:
era_flx = - era_flx

In [ ]:
def filter_full_day(ds,num,var):
    ddd = ds.copy(deep = True)
    # ddd['time'] = ddd['time'].values - np.timedelta64(6,'h') # to LST
    ddd = ddd.sel(time = ddd.time.dt.month.isin([6,7,8]))
    ddd = ddd.dropna('time')
    counts = ddd.resample(time='D').count()
    valid_days =counts[var]==num  #  24     True
    
    daily_ds = ddd.resample(time='D').sum('time', skipna=False)
    daily_ds = daily_ds.where(valid_days)  #     NaN    #
    daily_ds = daily_ds.dropna('time')
    
    filtered_ds = ddd.sel(time=ddd.time.dt.floor('D').isin(daily_ds.time.values))
    return(filtered_ds)

In [ ]:
era_flx['ef'] = era_flx['le']/(era_flx['le']+era_flx['h'])
narr_flx['ef'] = narr_flx['le']/(narr_flx['le']+narr_flx['h'])
merra_flx['ef'] = merra_flx['le']/(merra_flx['le']+merra_flx['h'])
jra_flx['ef'] = jra_flx['le']/(jra_flx['le']+jra_flx['h'])

In [ ]:
era_flx['ef'] = era_flx['ef'].clip(0,1)
narr_flx['ef'] = narr_flx['ef'].clip(0,1)
merra_flx['ef'] = merra_flx['ef'].clip(0,1)
jra_flx['ef'] = jra_flx['ef'].clip(0,1)

In [ ]:
arm_flx = filter_full_day(arm_flx,24,'le')
era_flx = filter_full_day(era_flx,24,'le')
narr_flx = filter_full_day(narr_flx,8,'le')
merra_flx = filter_full_day(merra_flx,24,'le')
jra_flx = filter_full_day(jra_flx,24,'le')

In [ ]:
flx_list = [arm_flx,
            era_flx,
            narr_flx,
            jra_flx,
            merra_flx]

In [ ]:
th_flx_list = [arm_flx_test, #arm_flx.resample(time = '3H').mean('time'),
            era_flx.resample(time = '3H').mean('time'),
            narr_flx,
            jra_flx.resample(time = '3H').mean('time'),
            merra_flx.resample(time = '3H').mean('time')]  #  3 

In [ ]:
th_flx_list[0] = filter_full_day(th_flx_list[0],8,'le')
th_flx_list[1] = filter_full_day(th_flx_list[1],8,'le')
th_flx_list[2] = filter_full_day(th_flx_list[2],8,'le')
th_flx_list[3] = filter_full_day(th_flx_list[3],8,'le')
th_flx_list[4] = filter_full_day(th_flx_list[4],8,'le')

In [ ]:
len(np.unique(arm_flx.time.dt.floor('D').values)) #271

In [ ]:
len(np.unique(th_flx_list[0].time.dt.floor('D').values)) #735

In [ ]:
flx_dates = []
for i in range(len(th_flx_list)):
    if i == 0:
        # flx_dates.append(np.unique(adv_list[0][['ef']].time.dt.floor('D').values))
        flx_dates.append(np.unique(th_flx_list[i].time.dt.floor('D').values))
    else: 
        flx_dates.append(np.unique(th_flx_list[i].time.dt.floor('D').values))

In [ ]:
from functools import reduce

not_null_flx_date = reduce(np.intersect1d, flx_dates)

len(not_null_flx_date) # 735

### Rainy Days


In [ ]:
test1 = reduce(np.intersect1d, [not_null_rain_date, 
                                # not_null_sm_date, 
                                not_null_flx_date,
                                #not_null_adv_date
                               ])

In [ ]:
len(test1)  # LE varanal  #454

In [ ]:
total_lev_ds = []
total_rain_ds = []
total_daily_ds = []
# total_adv_ds = []
total_flx_ds = []
# total_sm_ds = []

for i in range(len(notnll_lev_ds)):
    total_lev_ds.append(notnll_lev_ds[i].sel(time=notnll_lev_ds[i].time.dt.floor('D').isin(test1)))
    total_rain_ds.append(notnll_rain_ds[i].sel(time=notnll_rain_ds[i].time.dt.floor('D').isin(test1)))
    total_daily_ds.append(notnll_daily_ds[i].sel(time =notnll_daily_ds[i].time.dt.floor('D').isin(test1)))
    # total_adv_ds.append(adv_list[i].sel(time=adv_list[i].time.dt.floor('D').isin(test1)))
    # total_sm_ds.append(sm_list[i].sel(time=sm_list[i].time.dt.floor('D').isin(test1)))
    total_flx_ds.append(th_flx_list[i].sel(time=th_flx_list[i].time.dt.floor('D').isin(test1)))
    print(i)

In [ ]:
def return_raindate(daily_ds):
        
    filtered_daliy = daily_ds.where(daily_ds['tp'] > 0.275, np.nan)  #  prec_daily
    
    rainy_days = filtered_daliy.dropna('time')

    rainy_date = rainy_days.time.values
    return rainy_date

In [ ]:
rainy_dates = []
for i in range(len(total_daily_ds)):
    rainy_dates.append(return_raindate(total_daily_ds[i]))
    print(i)

In [ ]:
from functools import reduce
common_rainy_dates = reduce(np.intersect1d, rainy_dates)

len(common_rainy_dates) #61

In [ ]:
fig_lev_ds = []  
fig_rain_ds = []  
# fig_adv_ds = []  
fig_flx_ds = []  
# fig_sm_ds = []

# target_date =common_rainy_dates[0]

for i in range(len(total_rain_ds)):
    fig_rain_ds.append(total_rain_ds[i].sel(time = total_rain_ds[i].time.dt.floor('D').isin(common_rainy_dates)))#.groupby('time.hour').mean('time'))
    fig_lev_ds.append(total_lev_ds[i].sel(time = total_lev_ds[i].time.dt.floor('D').isin(common_rainy_dates)))#.groupby('time.hour').mean('time'))
    # fig_adv_ds.append(total_adv_ds[i].sel(time = total_adv_ds [i].time.dt.floor('D').isin(common_rainy_dates)))#.groupby('time.hour').mean('time'))
    fig_flx_ds.append(th_flx_list[i].sel(time = th_flx_list[i].time.dt.floor('D').isin(common_rainy_dates)))#.groupby('time.hour').mean('time'))
    #fig_sm_ds.append(total_sm_ds[i].sel(time = total_sm_ds [i].time.dt.floor('D').isin(common_rainy_dates)))#.groupby('time.hour').mean('time'))
    

In [ ]:
for j in range(len(fig_flx_ds)):
    fig_flx_ds[j]['pw_P'] = -fig_rain_ds[j]['tp'] # mm/3h
    fig_flx_ds[j]['pw_E'] = fig_flx_ds[j]['le']*3*3600/(2.6*10**(6)) # mm/3h

In [ ]:
fig_lev_ds[0]['pw_tend'] = fig_lev_ds[0]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h

fig_lev_ds[1]['pw_tend'] = fig_lev_ds[1]['pw'].diff(dim='time', n=1, label='lower') # mm/3h
fig_lev_ds[2]['pw_tend'] = fig_lev_ds[2]['pw'].diff(dim='time', n=1, label='lower') # mm/3h

fig_lev_ds[3]['pw_tend'] = fig_lev_ds[3]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h
fig_lev_ds[4]['pw_tend'] = fig_lev_ds[4]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h

In [ ]:
for j in range(len(fig_lev_ds)):
    fig_lev_ds[j]['pw_P'] = fig_flx_ds[j]['pw_P'].sel(time = fig_lev_ds[j]['pw_tend'].time)
    fig_lev_ds[j]['pw_E'] = fig_flx_ds[j]['pw_E'].sel(time = fig_lev_ds[j]['pw_tend'].time)

In [ ]:
for j in range(len(fig_lev_ds)):
    fig_lev_ds[j]['pw_res'] = fig_lev_ds[j]['pw_tend'] - fig_lev_ds[j]['pw_P'] - fig_lev_ds[j]['pw_E'] #mm/3h

In [ ]:
def extend_dataset_to_24h(ds):
    """
    Extend an xarray Dataset with hourly data (0, 6, 12, 18) to include hour 24
    by copying the data from hour 0.
    """
    # Create a new coordinate for hour that includes 24
    new_hour = np.append(ds.hour.values, 24)
    
    # Initialize dictionary to store extended variables
    extended_vars = {}
    
    # For each data variable in the dataset
    for var_name in ds.data_vars:
        var = ds[var_name]
        
        # Check if the variable has the 'hour' dimension
        if 'hour' in var.dims:
            # Get the variable values
            values = var.values
            
            # If the variable is 2D (e.g., temp, Td, etc. with dimensions (hour, pres))
            if var.ndim == 2:
                # Append the values from hour 0 to create hour 24
                extended_values = np.vstack([values, values[0:1]])
            # If the variable is 1D (e.g., pw, cape, cin with dimension (hour))
            elif var.ndim == 1:
                # Append the value from hour 0 to create hour 24
                extended_values = np.append(values, values[0])
            
            # Create a new DataArray with the extended values
            dims = var.dims
            coords = {dim: (ds[dim] if dim != 'hour' else new_hour) for dim in dims}
            extended_vars[var_name] = xr.DataArray(data=extended_values, dims=dims, coords=coords, 
                                                 attrs=var.attrs)
        else:
            # If the variable doesn't have the 'hour' dimension, keep it as is
            extended_vars[var_name] = var
    
    # Create a new dataset with the extended variables
    extended_ds = xr.Dataset(extended_vars, attrs=ds.attrs)
    
    # Update the hour coordinate
    extended_ds = extended_ds.assign_coords(hour=new_hour)
    
    return extended_ds

# Example usage:
# extended_ds = extend_dataset_to_24h(ds)
# Now you can plot with a complete 24-hour cycle

In [ ]:
for i in range(len(fig_lev_ds)):
    fig_rain_ds[i] = fig_rain_ds[i].groupby('time.hour').mean('time')
    fig_lev_ds[i] = fig_lev_ds[i].groupby('time.hour').mean('time')

In [ ]:
for i in range(len(fig_lev_ds)):
    fig_lev_ds[i].to_netcdf(f'${CODE_DIR}/ARM/diurnal_cycle/{i}_diurnal_ds.nc')

In [ ]:
for i in range(len(total_rain_ds)):
    fig_lev_ds[i] = extend_dataset_to_24h(fig_lev_ds[i])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

# GridSpec    2 (  SM/EF )  
#   SM/EF    3:1 
gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 0), (0, 1)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 0), (1, 1)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
cape_color = '#0072B2'
cin_color = '#D55E00'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

common_levels = np.linspace(328, 340, 25)

# PW      
six_hourly = [0, 6, 12, 18, 24]

def create_arrow_scale(ax):
    """
    Create a scale legend for arrows showing PW differences
    """
    ax.text(0.904, 0.95, 'PW diff',
            transform=ax.transAxes,
            horizontalalignment='center',
            fontsize=10)
    
    arrow_scales = [
        (1, '1 mm', 'black', 30*1.5),
        (2, '2 mm', 'black', 50*1.5),
        (3, '3 mm', 'black', 70*1.5)
    ]
    
    for i, (diff, label, color, q_len) in enumerate(arrow_scales):
        ax.quiver(0.84, 0.83, 0, q_len, 
                  color=color, width=0.01, zorder=10,
                  headwidth=3.5, headaxislength=2, headlength=2,
                  scale_units='xy', scale=1, angles='xy', transform=ax.transAxes)
        
        ax.text(0.87, 0.83 + i*0.03, label, transform=ax.transAxes, 
                verticalalignment='bottom', fontsize=10)

for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    #    y  
    if (i < 3 and main_col == 0) or (i >= 3 and main_col == 0):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 

    bar_width = 0.6
    bar_s = .7
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-400, 800)
    
    #   PW     (  () )
    if i > 0:
        obs_diur = fig_lev_ds[0]
        pw_diff = (current_diur['pw'].sel(hour=six_hourly) - obs_diur['pw'].sel(hour=six_hourly)).values
        
        for j, hour in enumerate(six_hourly):
            if hour in current_diur.hour.values:
                # CAPE  
                if 'cape' in current_diur and hour in current_diur.hour.values:
                    cape_val = float(current_diur['cape'].sel(hour=hour).values)
                else:
                    # CAPE       
                    cape_val = 300  #  y  
                
                diff = pw_diff[j]
                
                if diff > 0:
                    color = 'red'
                else:
                    color = 'blue'
                
                if abs(diff) > 3:
                    arrow_length = 70*1.5
                elif abs(diff) > 2.5:
                    arrow_length = 60*1.5
                elif abs(diff) > 2.0:
                    arrow_length = 50*1.5
                elif abs(diff) > 1.5:
                    arrow_length = 40*1.5
                elif abs(diff) > 1:
                    arrow_length = 30*1.5
                elif abs(diff) > 0.5:
                    arrow_length = 20*1.5
                else:
                    arrow_length = 10*1.5
                
                if diff > 0:
                    ax3.quiver(hour, cape_val, 0, arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
                else:
                    ax3.quiver(hour, cape_val, 0, -arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
        
        if i == 1:  # ERA5 (  )
            create_arrow_scale(ax3)
    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #    x  
    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # # EF   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = fig_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    pw_bar_width = 2
    
    for hour, value in zip(hours, values):
        #    ( )
        # bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.6, value, width=bar_width*3, 
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.6, value, width=bar_width*3,
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    # ef_line = ax_ef.plot(current_ef.hour+1.5, current_ef.ef.values, '-', 
    #                     color=ef_color, linewidth=2.5, marker='s', markersize=6, 
    #                     markeredgecolor='black', markeredgewidth=0.5, label='EF')
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Advc',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Advc', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    # ax_ef.set_yticks(np.arange(-5,5,2.5))
    if (i < 3 and sm_ef_col == 2) or (i >= 3 and sm_ef_col == 1):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    # ax_ef.tick_params(axis='y', colors=ef_color)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    Line2D([0], [0], color='red', lw=0, marker=r'$\uparrow$', markersize=15,
           label='PW(Model) > PW(Obs)'),
    Line2D([0], [0], color='blue', lw=0, marker=r'$\downarrow$', markersize=15,
           label='PW(Model) < PW(Obs)'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 0.05))

# Colorbar 
cbar_ax = fig.add_axes([0.74, 0.17, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(328, 340, 7))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']
reanalysis_datasets = datasets[1:]  #  (IGRA SGP Obs )

fig = plt.figure(figsize=(16, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.5)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.5)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]

#    (  )
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef, dataset_idx=0, x_label=False):
    current_diur = fig_lev_ds[dataset_idx]
    current_rain = fig_rain_ds[dataset_idx]
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(925, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    ax.set_ylabel('Pressure [hPa]', fontsize=14)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    ax2.set_ylim(0, 8)
    ax2.tick_params(axis='y', which='both', right=True, labelright=False)
    # ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    # ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    ax3.set_ylim(-400, 800)
    ax3.tick_params(axis='y', which='both', right=False, labelright=False)
    ax3.spines['right'].set_visible(False)

    ax.set_title('IGRA SGP Obs', fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = current_diur
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-4, 4)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

def create_diff_plot(ax, ax_sm_ef, dataset_idx, position, x_label=False):
    model_idx = dataset_idx
    dataset_name = datasets[model_idx]
    
    current_diur = fig_lev_ds[model_idx]
    current_rain = fig_rain_ds[model_idx]
    
    #    MSE  
    model_hours = current_diur.hour.values
    model_six_hour_mask = np.isin(model_hours, six_hourly)
    
    if len(model_hours) > len(six_hourly):
        mse_model = current_diur.mse.values[model_six_hour_mask, :]
        model_hours_6h = model_hours[model_six_hour_mask]
    else:
        mse_model = current_diur.mse.values
        model_hours_6h = model_hours
    
    mse_diff = mse_model - obs_mse
    mse_diff_masked = np.ma.masked_invalid(mse_diff)
    
    # MSE   
    CS0 = ax.contourf(model_hours_6h, current_diur.pres, mse_diff_masked.T, 
                     levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both')
    
    ax.set_yscale('log')
    ax.set_ylim(925, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])
    
    # y  ( )
    if position in [(0, 0), (1, 0)]:
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP  
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    
    #    ( )
    if position in [(0, 2), (1, 2)]:
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    ax2.set_ylim(0, 8)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE')
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN')
        
    # CAPE/CIN   ( )
    if position in [(0, 2), (1, 2)]:
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)
    
    ax3.set_ylim(-400, 800)
    
    ax.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW    
    ax_ef = ax_sm_ef.twinx()
    
    obs_pw = fig_lev_ds[0]
    current_pw = fig_lev_ds[model_idx]
    hours = obs_pw.hour.values
    
    # PW tendency 
    obs_values = obs_pw['pw_tend'].sel(hour=hours).values
    current_values = current_pw['pw_tend'].sel(hour=hours).values
    pw_tend_diff = current_values - obs_values
    
    for hour, value in zip(hours, pw_tend_diff):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW     
    bars_P = ax_ef.bar(hours+bar_s, (current_pw['pw_P'].sel(hour=hours)-obs_pw['pw_P']), 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, (current_pw['pw_E'].sel(hour=hours)-obs_pw['pw_E']), 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, (current_pw['pw_res'].sel(hour=hours)-obs_pw['pw_res']), 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-4, 4)
    
    if position in [(0, 2), (1, 2)]:
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    
    return CS0

#   (IGRA SGP Obs) -   ,   
ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1, dataset_idx=0)

#   (ERA5, NARR) -   ,      
ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_era5 = create_diff_plot(ax2, ax_sm_ef2, dataset_idx=1, position=(0, 1))

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_narr = create_diff_plot(ax3, ax_sm_ef3, dataset_idx=2, position=(0, 2))

#   (JRA-3Q, MERRA-2) -   ,      
ax5 = fig.add_subplot(gs_bottom[0, 1])
ax_sm_ef5 = fig.add_subplot(gs_bottom[1, 1])
CS_jra = create_diff_plot(ax5, ax_sm_ef5, dataset_idx=3, position=(1, 1), x_label=True)

ax6 = fig.add_subplot(gs_bottom[0, 2])
ax_sm_ef6 = fig.add_subplot(gs_bottom[1, 2])
CS_merra = create_diff_plot(ax6, ax_sm_ef6, dataset_idx=4, position=(1, 2), x_label=True)

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=1, frameon=True, fontsize=14, bbox_to_anchor=(0.35, 0.5))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax2.get_position()
cbar_ax_obs = fig.add_axes([(pos1.x1 + pos2.x0)/2 - 0.04, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))

#    -  
cbar_ax_diff = fig.add_axes([1.01, (1-.315)/2, 0.02, 0.315])
cbar_diff = fig.colorbar(CS_era5, cax=cbar_ax_diff)  # ERA5  
cbar_diff.set_label('ΔMSE [kJ/kg]', fontsize=14, labelpad=20)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))

labels = ['(a)', '(b)', '(c)', '(d)', '(e)']
axes = [ax1, ax2, ax3, ax5, ax6]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/vertical_fig_rain.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 1 5    
fig = plt.figure(figsize=((15/3)*5, 12/2))

# GridSpec  - 1 5 
gs = GridSpec(2, 6, figure=fig,
              top=0.90, bottom=0.15,
              width_ratios=[1,.1, 1, 1, 1, 1],  # 5   
              height_ratios=[4.5, 1],  #  :PW  = 4.5:1 
              hspace=0.0,
              wspace=0.2)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]

#    (  )
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

CS_obs = None  #   colorbar
CS_diff = None  #   colorbar
axis = []
fig_idx = [0,2,3,4,5]
for i, dataset_name in enumerate(datasets):
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs[0, fig_idx[i]])
    axis.append(ax1)
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    #   () () 
    if i == 0:  #   (IGRA SGP Obs)
        # MSE    
        mse_data = current_diur.mse.values.copy()
        mse_data_masked = np.ma.masked_invalid(mse_data)
        
        # MSE    ( )
        CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                          levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
        CS_obs = CS_current
        
    else:  #   ( )
        #    MSE  
        model_hours = current_diur.hour.values
        model_six_hour_mask = np.isin(model_hours, six_hourly)
        
        if len(model_hours) > len(six_hourly):
            mse_model = current_diur.mse.values[model_six_hour_mask, :]
            model_hours_6h = model_hours[model_six_hour_mask]
        else:
            mse_model = current_diur.mse.values
            model_hours_6h = model_hours
        
        mse_diff = mse_model - obs_mse
        mse_diff_masked = np.ma.masked_invalid(mse_diff)
        
        # MSE    ( )
        CS_current = ax1.contourf(model_hours_6h, current_diur.pres, mse_diff_masked.T, 
                         levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
        if CS_diff is None:
            CS_diff = CS_current
    
    #    ( ) -     
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (PW  )
    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #    y  
    if i == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
        
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)

    bar_width = 0.6
    bar_s = .6
    
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp   
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        # CP LP  
        if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
            bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5, zorder=3)
            
            bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5, zorder=3)
    
    #    -  (i==4) y  
    if i == 4:
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False) 

    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN   -  (i==4) y  
    if i == 4:
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
        
    ax3.set_ylim(-400, 800)    
    ax1.set_title(dataset_name, fontsize=18)
    
    # PW  
    ax_sm_ef = fig.add_subplot(gs[1, fig_idx[i]])
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #   x  
    ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   ( )
    ax_ef = ax_sm_ef.twinx()
    
    if i == 0:
        pw_info = current_diur
        hours = pw_info.hour.values
        values = pw_info['pw_tend'].values
        
        # PW tendency  
        for hour, value in zip(hours, values):
            if value > 0:
                ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                         color='blue', alpha=0.3, 
                         edgecolor='darkblue', linewidth=.8, zorder=1)
            else:
                ax_ef.bar(hour+1.5, value, width=bar_width*3,
                         color='red', alpha=0.3,
                         edgecolor='darkred', linewidth=.8, zorder=1)
        
        # PW    
        bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                     width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                     width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                     width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        # PW   (  )
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                          fontsize=10, frameon=False,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)
        
    else:  #   ( )
        #   PW  
        obs_pw = fig_lev_ds[0]
        current_pw = fig_lev_ds[i]
        hours = obs_pw.hour.values
        
        # PW tendency 
        obs_values = obs_pw['pw_tend'].sel(hour=hours).values
        current_values = current_pw['pw_tend'].sel(hour=hours).values
        pw_tend_diff = current_values - obs_values
        
        for hour, value in zip(hours, pw_tend_diff):
            if value > 0:
                ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                         color='blue', alpha=0.3, 
                         edgecolor='darkblue', linewidth=.8, zorder=1)
            else:
                ax_ef.bar(hour+1.5, value, width=bar_width*3,
                         color='red', alpha=0.3,
                         edgecolor='darkred', linewidth=.8, zorder=1)
        
        # PW     
        bars_P = ax_ef.bar(hours+bar_s, (current_pw['pw_P'].sel(hour=hours)-obs_pw['pw_P']), 
                     width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_E = ax_ef.bar(hours+bar_s+bar_width, (current_pw['pw_E'].sel(hour=hours)-obs_pw['pw_E']), 
                     width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_res = ax_ef.bar(hours+bar_s+bar_width*2, (current_pw['pw_res'].sel(hour=hours)-obs_pw['pw_res']), 
                     width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                     edgecolor='black', linewidth=0.5, zorder=3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3) 
    ax_ef.set_ylim(-4, 4)
    
    #  (i==4) PW y  
    if i == 4:
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='lower center', ncol=3, frameon=False, 
          fontsize=14, bbox_to_anchor=(1., 0.1))

#    -       
pos_first = axis[0].get_position()
cbar_ax_obs = fig.add_axes([pos_first.x1 + 0.005, pos_first.y0, 0.015, pos_first.height])

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))

pos_last = ax1.get_position()
cbar_ax_diff = fig.add_axes([pos_last.x1 + 0.07, pos_last.y0, 0.015, pos_last.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('ΔMSE [kJ/kg]', fontsize=14, labelpad=20)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))

plt.tight_layout(rect=[0, 0.1, 0.95, 0.95])
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/vertical_fig_1row5col_v2.png', dpi=400, bbox_inches='tight')

plt.show()

##### CpT


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

# GridSpec    2 (  SM/EF )  
#   SM/EF    3:1 
gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 0), (0, 1)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 0), (1, 1)]  #  GridSpec SM/EF 

mse_cmap = cmaps.WhiteYellowOrangeRed
cape_color = '#0072B2'
cin_color = '#D55E00'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

common_levels = np.linspace(260, 310, 26)

# PW      
six_hourly = [0, 6, 12, 18, 24]

def create_arrow_scale(ax):
    """
    Create a scale legend for arrows showing PW differences
    """
    ax.text(0.904, 0.95, 'PW diff',
            transform=ax.transAxes,
            horizontalalignment='center',
            fontsize=10)
    
    arrow_scales = [
        (1, '1 mm', 'black', 30*1.5),
        (2, '2 mm', 'black', 50*1.5),
        (3, '3 mm', 'black', 70*1.5)
    ]
    
    for i, (diff, label, color, q_len) in enumerate(arrow_scales):
        ax.quiver(0.84, 0.83, 0, q_len, 
                  color=color, width=0.01, zorder=10,
                  headwidth=3.5, headaxislength=2, headlength=2,
                  scale_units='xy', scale=1, angles='xy', transform=ax.transAxes)
        
        ax.text(0.87, 0.83 + i*0.03, label, transform=ax.transAxes, 
                verticalalignment='bottom', fontsize=10)

for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse_h.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    #    y  
    if (i < 3 and main_col == 0) or (i >= 3 and main_col == 0):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax_pw = ax2.twiny()
    ax_pw.set_zorder(1)  #  zorder
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 
    pw_tend = fig_lev_ds[i]['pw_tend'][:-1]
    hours = pw_tend.hour.values
    values = pw_tend.values
    
    bar_width = 2
    base_value= 8  # 
    
    for hour, value in zip(hours, values):
        #    ( )
        bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width, 
                     bottom=base_value-abs(value),
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width,
                     bottom=base_value-abs(value),
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks([])  # x  
    ax_pw.set_xticklabels([])
    bar_width = 0.6
    bar_s = .7
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-400, 800)
    
    #   PW     (  () )
    if i > 0:
        obs_diur = fig_lev_ds[0]
        pw_diff = (current_diur['pw'].sel(hour=six_hourly) - obs_diur['pw'].sel(hour=six_hourly)).values
        
        for j, hour in enumerate(six_hourly):
            if hour in current_diur.hour.values:
                # CAPE  
                if 'cape' in current_diur and hour in current_diur.hour.values:
                    cape_val = float(current_diur['cape'].sel(hour=hour).values)
                else:
                    # CAPE       
                    cape_val = 300  #  y  
                
                diff = pw_diff[j]
                
                if diff > 0:
                    color = 'red'
                else:
                    color = 'blue'
                
                if abs(diff) > 3:
                    arrow_length = 70*1.5
                elif abs(diff) > 2.5:
                    arrow_length = 60*1.5
                elif abs(diff) > 2.0:
                    arrow_length = 50*1.5
                elif abs(diff) > 1.5:
                    arrow_length = 40*1.5
                elif abs(diff) > 1:
                    arrow_length = 30*1.5
                elif abs(diff) > 0.5:
                    arrow_length = 20*1.5
                else:
                    arrow_length = 10*1.5
                
                if diff > 0:
                    ax3.quiver(hour, cape_val, 0, arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
                else:
                    ax3.quiver(hour, cape_val, 0, -arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
        
        if i == 1:  # ERA5 (  )
            create_arrow_scale(ax3)
    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    
    # SM   ( )
    # sm_line = ax_sm_ef.plot(current_sm.hour, current_sm.sm_ano.values, '-', 
    #                        color=sm_color, linewidth=2.5, marker='o', markersize=6, 
    #                        markeredgecolor='black', markeredgewidth=0.5, label='SM')
    # SM  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    # ax_sm_ef.set_ylim(0, 0.5)  # SM  (  )
    # ax_sm_ef.grid(True, ls='--', alpha=0.3)
    
    # if (i < 3 and sm_ef_col == 0) or (i >= 3 and sm_ef_col == 0):
        # ax_sm_ef.set_ylabel('SM [m³/m³]', fontsize=11, color=sm_color)
    # ax_sm_ef.tick_params(axis='y', colors=sm_color)
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #    x  
    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # # EF   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = fig_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    pw_bar_width = 2
    
    for hour, value in zip(hours, values):
        #    ( )
        # bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.6, value, width=bar_width*3, 
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.6, value, width=bar_width*3,
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    # ef_line = ax_ef.plot(current_ef.hour+1.5, current_ef.ef.values, '-', 
    #                     color=ef_color, linewidth=2.5, marker='s', markersize=6, 
    #                     markeredgecolor='black', markeredgewidth=0.5, label='EF')
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Advc',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Advc', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    # ax_ef.set_yticks(np.arange(-5,5,2.5))
    if (i < 3 and sm_ef_col == 2) or (i >= 3 and sm_ef_col == 1):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    # ax_ef.tick_params(axis='y', colors=ef_color)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    Line2D([0], [0], color='red', lw=0, marker=r'$\uparrow$', markersize=15,
           label='PW(Model) > PW(Obs)'),
    Line2D([0], [0], color='blue', lw=0, marker=r'$\downarrow$', markersize=15,
           label='PW(Model) < PW(Obs)'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 0.05))

# Colorbar 
cbar_ax = fig.add_axes([0.74, 0.17, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('$C_p \cdot T$ [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(260, 310, 11))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)

plt.show()

##### Lvq


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

# GridSpec    2 (  SM/EF )  
#   SM/EF    3:1 
gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 0), (0, 1)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 0), (1, 1)]  #  GridSpec SM/EF 

mse_cmap = cmaps.WhiteBlue
cape_color = '#0072B2'
cin_color = '#D55E00'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

common_levels = np.linspace(5,35,31)

# PW      
six_hourly = [0, 6, 12, 18, 24]

def create_arrow_scale(ax):
    """
    Create a scale legend for arrows showing PW differences
    """
    ax.text(0.904, 0.95, 'PW diff',
            transform=ax.transAxes,
            horizontalalignment='center',
            fontsize=10)
    
    arrow_scales = [
        (1, '1 mm', 'black', 30*1.5),
        (2, '2 mm', 'black', 50*1.5),
        (3, '3 mm', 'black', 70*1.5)
    ]
    
    for i, (diff, label, color, q_len) in enumerate(arrow_scales):
        ax.quiver(0.84, 0.83, 0, q_len, 
                  color=color, width=0.01, zorder=10,
                  headwidth=3.5, headaxislength=2, headlength=2,
                  scale_units='xy', scale=1, angles='xy', transform=ax.transAxes)
        
        ax.text(0.87, 0.83 + i*0.03, label, transform=ax.transAxes, 
                verticalalignment='bottom', fontsize=10)

for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse_le.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    #    y  
    if (i < 3 and main_col == 0) or (i >= 3 and main_col == 0):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax_pw = ax2.twiny()
    ax_pw.set_zorder(1)  #  zorder
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 
    pw_tend = fig_lev_ds[i]['pw_tend'][:-1]
    hours = pw_tend.hour.values
    values = pw_tend.values
    
    bar_width = 2
    base_value= 8  # 
    
    for hour, value in zip(hours, values):
        #    ( )
        bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width, 
                     bottom=base_value-abs(value),
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width,
                     bottom=base_value-abs(value),
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks([])  # x  
    ax_pw.set_xticklabels([])
    bar_width = 0.6
    bar_s = .7
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-400, 800)
    
    #   PW     (  () )
    if i > 0:
        obs_diur = fig_lev_ds[0]
        pw_diff = (current_diur['pw'].sel(hour=six_hourly) - obs_diur['pw'].sel(hour=six_hourly)).values
        
        for j, hour in enumerate(six_hourly):
            if hour in current_diur.hour.values:
                # CAPE  
                if 'cape' in current_diur and hour in current_diur.hour.values:
                    cape_val = float(current_diur['cape'].sel(hour=hour).values)
                else:
                    # CAPE       
                    cape_val = 300  #  y  
                
                diff = pw_diff[j]
                
                if diff > 0:
                    color = 'red'
                else:
                    color = 'blue'
                
                if abs(diff) > 3:
                    arrow_length = 70*1.5
                elif abs(diff) > 2.5:
                    arrow_length = 60*1.5
                elif abs(diff) > 2.0:
                    arrow_length = 50*1.5
                elif abs(diff) > 1.5:
                    arrow_length = 40*1.5
                elif abs(diff) > 1:
                    arrow_length = 30*1.5
                elif abs(diff) > 0.5:
                    arrow_length = 20*1.5
                else:
                    arrow_length = 10*1.5
                
                if diff > 0:
                    ax3.quiver(hour, cape_val, 0, arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
                else:
                    ax3.quiver(hour, cape_val, 0, -arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
        
        if i == 1:  # ERA5 (  )
            create_arrow_scale(ax3)
    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    
    # SM   ( )
    # sm_line = ax_sm_ef.plot(current_sm.hour, current_sm.sm_ano.values, '-', 
    #                        color=sm_color, linewidth=2.5, marker='o', markersize=6, 
    #                        markeredgecolor='black', markeredgewidth=0.5, label='SM')
    # SM  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    # ax_sm_ef.set_ylim(0, 0.5)  # SM  (  )
    # ax_sm_ef.grid(True, ls='--', alpha=0.3)
    
    # if (i < 3 and sm_ef_col == 0) or (i >= 3 and sm_ef_col == 0):
        # ax_sm_ef.set_ylabel('SM [m³/m³]', fontsize=11, color=sm_color)
    # ax_sm_ef.tick_params(axis='y', colors=sm_color)
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #    x  
    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # # EF   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = fig_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    pw_bar_width = 2
    
    for hour, value in zip(hours, values):
        #    ( )
        # bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.6, value, width=bar_width*3, 
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.6, value, width=bar_width*3,
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    # ef_line = ax_ef.plot(current_ef.hour+1.5, current_ef.ef.values, '-', 
    #                     color=ef_color, linewidth=2.5, marker='s', markersize=6, 
    #                     markeredgecolor='black', markeredgewidth=0.5, label='EF')
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Advc',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Advc', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    # ax_ef.set_yticks(np.arange(-5,5,2.5))
    if (i < 3 and sm_ef_col == 2) or (i >= 3 and sm_ef_col == 1):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    # ax_ef.tick_params(axis='y', colors=ef_color)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    Line2D([0], [0], color='red', lw=0, marker=r'$\uparrow$', markersize=15,
           label='PW(Model) > PW(Obs)'),
    Line2D([0], [0], color='blue', lw=0, marker=r'$\downarrow$', markersize=15,
           label='PW(Model) < PW(Obs)'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 0.05))

# Colorbar 
cbar_ax = fig.add_axes([0.74, 0.17, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('$L_v \cdot q$ [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(5,35,7))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)

plt.show()

##### RH


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

# GridSpec    2 (  SM/EF )  
#   SM/EF    3:1 
gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 0), (0, 1)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 0), (1, 1)]  #  GridSpec SM/EF 

mse_cmap = cmaps.WhiteBlue
cape_color = '#0072B2'
cin_color = '#D55E00'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

common_levels = np.linspace(40, 80, 21)

# PW      
six_hourly = [0, 6, 12, 18, 24]

def create_arrow_scale(ax):
    """
    Create a scale legend for arrows showing PW differences
    """
    ax.text(0.904, 0.95, 'PW diff',
            transform=ax.transAxes,
            horizontalalignment='center',
            fontsize=10)
    
    arrow_scales = [
        (1, '1 mm', 'black', 30*1.5),
        (2, '2 mm', 'black', 50*1.5),
        (3, '3 mm', 'black', 70*1.5)
    ]
    
    for i, (diff, label, color, q_len) in enumerate(arrow_scales):
        ax.quiver(0.84, 0.83, 0, q_len, 
                  color=color, width=0.01, zorder=10,
                  headwidth=3.5, headaxislength=2, headlength=2,
                  scale_units='xy', scale=1, angles='xy', transform=ax.transAxes)
        
        ax.text(0.87, 0.83 + i*0.03, label, transform=ax.transAxes, 
                verticalalignment='bottom', fontsize=10)

for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.rhumi.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    #    y  
    if (i < 3 and main_col == 0) or (i >= 3 and main_col == 0):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax_pw = ax2.twiny()
    ax_pw.set_zorder(1)  #  zorder
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 
    pw_tend = fig_lev_ds[i]['pw_tend'][:-1]
    hours = pw_tend.hour.values
    values = pw_tend.values
    
    bar_width = 2
    base_value= 8  # 
    
    for hour, value in zip(hours, values):
        #    ( )
        bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width, 
                     bottom=base_value-abs(value),
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width,
                     bottom=base_value-abs(value),
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks([])  # x  
    ax_pw.set_xticklabels([])
    bar_width = 0.6
    bar_s = .7
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-400, 800)
    
    #   PW     (  () )
    if i > 0:
        obs_diur = fig_lev_ds[0]
        pw_diff = (current_diur['pw'].sel(hour=six_hourly) - obs_diur['pw'].sel(hour=six_hourly)).values
        
        for j, hour in enumerate(six_hourly):
            if hour in current_diur.hour.values:
                # CAPE  
                if 'cape' in current_diur and hour in current_diur.hour.values:
                    cape_val = float(current_diur['cape'].sel(hour=hour).values)
                else:
                    # CAPE       
                    cape_val = 300  #  y  
                
                diff = pw_diff[j]
                
                if diff > 0:
                    color = 'red'
                else:
                    color = 'blue'
                
                if abs(diff) > 3:
                    arrow_length = 70*1.5
                elif abs(diff) > 2.5:
                    arrow_length = 60*1.5
                elif abs(diff) > 2.0:
                    arrow_length = 50*1.5
                elif abs(diff) > 1.5:
                    arrow_length = 40*1.5
                elif abs(diff) > 1:
                    arrow_length = 30*1.5
                elif abs(diff) > 0.5:
                    arrow_length = 20*1.5
                else:
                    arrow_length = 10*1.5
                
                if diff > 0:
                    ax3.quiver(hour, cape_val, 0, arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
                else:
                    ax3.quiver(hour, cape_val, 0, -arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
        
        if i == 1:  # ERA5 (  )
            create_arrow_scale(ax3)
    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    # SM  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    ax_ef = ax_sm_ef.twinx()
    pw_info = fig_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    pw_bar_width = 2
    
    for hour, value in zip(hours, values):
        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.6, value, width=bar_width*3, 
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.6, value, width=bar_width*3,
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Advc',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Advc', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    # ax_ef.set_yticks(np.arange(-5,5,2.5))
    if (i < 3 and sm_ef_col == 2) or (i >= 3 and sm_ef_col == 1):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    # ax_ef.tick_params(axis='y', colors=ef_color)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    Line2D([0], [0], color='red', lw=0, marker=r'$\uparrow$', markersize=15,
           label='PW(Model) > PW(Obs)'),
    Line2D([0], [0], color='blue', lw=0, marker=r'$\downarrow$', markersize=15,
           label='PW(Model) < PW(Obs)'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 0.05))

# Colorbar 
cbar_ax = fig.add_axes([0.74, 0.17, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('RH [%]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(40, 80, 9))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)

plt.show()

##### DCP


In [ ]:
def reconstruct_1st_harmonic(time_array, values):
    ff = np.fft.fft(values)
    ff_filtered = np.zeros_like(ff, dtype=complex)
    ff_filtered[1] = ff[1]  # Retain first harmonic
    # ff_filtered[-1] = ff[-1]  # Retain conjugate
    ff_filtered[-1] = np.conj(ff[1])
    reconstructed = np.fft.irfft(ff_filtered, n=len(time_array))
    
    T = len(values)
    amp1 = np.sqrt(ff[1].real**2 + ff[1].imag**2)/T  # 1st harmonic amplitude
    amp2 = np.sqrt(ff[2].real**2 + ff[2].imag**2)/T  # 2nd harmonic amplitude
    
    ratio = amp2/amp1
    return reconstructed,ratio
def find_first_harmonic_peak(time_array, values,longitude, acc_hour=1):
    # FFT        
    ff = np.fft.fft(values)
    ff_filtered = np.zeros_like(ff, dtype=complex)
    ff_filtered[1] = ff[1]
    ff_filtered[-1] = np.conj(ff_filtered[1])
    reconstructed = np.fft.irfft(ff_filtered, n=len(time_array))

    T = len(values)
    amplitude = np.sqrt(ff[1].real**2 + ff[1].imag**2)/T

    # reconstructed    
    peak_idx = np.argmax(reconstructed)
    peak_hour = time_array[peak_idx]

    utc_hour = peak_hour - (acc_hour/2)

    # 24  
    if utc_hour < 0:
        utc_hour += 24
    elif utc_hour >= 24:
        utc_hour -= 24
        
    '''calculate lst hour'''
    if (utc_hour + float(longitude/15))<0. :
        lst_hour = (utc_hour + float(longitude/15)) + 24

    elif  (utc_hour + float(longitude/15))>=24. :
        lst_hour = (utc_hour + float(longitude/15)) - 24
        
    else:
        lst_hour = (utc_hour + float(longitude/15))
    # return(lst,amp,norm_amp)

    return utc_hour,lst_hour,amplitude
def analyze_diurnal_harmonics(data):
    """
    Analyze diurnal harmonics of precipitation data
    
    Parameters:
    -----------
    data : xarray.DataArray or numpy.ndarray
        Input precipitation data with 24 hourly values
    
    Returns:
    --------
    dict
        Dictionary containing powers of 1st, 2nd, 3rd harmonics and remaining power
    """
    if isinstance(data, xr.DataArray):
        values = data.values
    else:
        values = data
    
    # Compute FFT
    ff = np.fft.fft(values)
    power_spectrum = np.abs(ff)**2
    
    # Calculate total power (excluding DC component)
    total_power = np.sum(power_spectrum[1:len(power_spectrum)//2])
    
    # Calculate power ratios for first 3 harmonics
    harmonic_powers = []
    for i in range(1, 4):
        power_ratio = power_spectrum[i] / total_power * 100
        harmonic_powers.append(power_ratio)
    
    # Calculate remaining power (higher harmonics)
    remaining_power = 100 - sum(harmonic_powers)
    harmonic_powers.append(remaining_power)
    
    return harmonic_powers

def cal_maxtime(hour,pre,longitude,acc_hour):
    '''calculate amplitude'''
    amp = (np.max(pre)-np.min(pre))/2
    '''calculate normalized amplitude'''
    norm_amp = amp/pre.mean()
    '''calculate utc hour'''
    if (np.argmax(pre)-(acc_hour/2))<0:  #   -1.5 
        utc = hour[np.argmax(pre)]-(acc_hour/2)+24

    else :
        utc=hour[np.argmax(pre)]-(acc_hour/2)
        
    '''calculate lst hour'''
    if (utc + float(longitude/15))<0. :
        lst = (utc + float(longitude/15)) + 24

    elif  (utc + float(longitude/15))>=24. :
        lst = (utc + float(longitude/15)) - 24
        
    else:
        lst = (utc + float(longitude/15))
    return(lst,amp,norm_amp)


def make_nan_array(reference_data,variavle_list): # ex) variable name = ['utc','lst','amplitude']
    import xarray as xr 
    import numpy as np
    longitude = reference_data.longitude.values
    latitude = reference_data.latitude.values
    
    dataset = xr.Dataset({'latitude': (['latitude'], latitude),'longitude': (['longitude'], longitude)})
    for i in range(len(variavle_list)):
        
        dataset[variavle_list[i]] = xr.DataArray(np.full((len(dataset['latitude']), len(dataset['longitude'])), np.nan), dims=['latitude', 'longitude'])

        
    return(dataset)

def make_1st_freq_diurnal(hour_array,diurnal_data,acc_hour,var):
    output_data = make_nan_array(diurnal_data,[var+'_lst',var+'_amp',var+'_SAD',var+'_1st',var+'_2nd',var+'_3rd',var+'_higher'])
    output_data.load()
    sd_diurnal_data = (diurnal_data - diurnal_data.mean('hour',skipna = False))/diurnal_data.std('hour',skipna = False)
    for i in range (len(diurnal_data.latitude.values)): #latitude loop
        for j in range (len(diurnal_data.longitude.values)): #longitude loop
            # diurnal_data dims Frozen({'hour': 8, 'latitude': 1440, 'longitude': 2880}) .
            if diurnal_data[:,i,j].isnull().any().item():
                output_data[var+'_lst'][i][j] = np.nan
                output_data[var+'_amp'][i][j] = np.nan
                # output_data['norm_amplitude'][i][j] = np.nan
                output_data[var+'_SAD'][i][j] = np.nan
                # output_data['RMSE'][i][j] = np.nan
                # output_data['amp_ratio'][i][j] = np.nan
                # output_data['new_harm_lst'][i][j] = np.nan 
                # output_data['new_harm_amp'][i][j] = np.nan 
                output_data[var+'_1st'][i][j] = np.nan 
                output_data[var+'_2nd'][i][j] = np.nan 
                output_data[var+'_3rd'][i][j] = np.nan 
                output_data[var+'_higher'][i][j] = np.nan 

            else:
                
                # lst,amp,norm_amp = cal_maxtime(hour_array,diurnal_data[:,i,j].values ,diurnal_data.longitude.values[j],acc_hour)
                _,harm_lst,harm_amp = find_first_harmonic_peak(hour_array,diurnal_data[:,i,j].values ,diurnal_data.longitude.values[j],acc_hour)
                first,second,third,higher = analyze_diurnal_harmonics(diurnal_data[:,i,j].values)

                
                # output_data['lst'][i][j] = lst 
                # output_data['amp'][i][j] = amp
                # output_data['norm_amplitude'][i][j] = norm_amp
                output_data[var+'_SAD'][i][j] = np.abs(reconstruct_1st_harmonic(hour_array, sd_diurnal_data[:,i,j].values)[0] - sd_diurnal_data[:,i,j].values).sum()
                # output_data['RMSE'][i][j] = np.sqrt(np.mean((reconstruct_1st_harmonic(hour_array, sd_diurnal_data[:,i,j].values)[0] - sd_diurnal_data[:,i,j].values)**2))
                # output_data['amp_ratio'][i][j] = reconstruct_1st_harmonic(hour_array, sd_diurnal_data[:,i,j].values)[1]
                output_data[var+'_lst'][i][j] = harm_lst
                output_data[var+'_amp'][i][j] = harm_amp
                output_data[var+'_1st'][i][j] = first
                output_data[var+'_2nd'][i][j] = second 
                output_data[var+'_3rd'][i][j] = third 
                output_data[var+'_higher'][i][j] = higher 
        print(f"latitude : {i}", end='\r')
    return(output_data)

In [ ]:
def cal_diurnal_static(hour_array,diurnal_data,acc_hour,var):


    sd_diurnal_data = (diurnal_data[var] - diurnal_data[var].mean('hour',skipna = False))/diurnal_data[var].std('hour',skipna = False)
    _,harm_lst,harm_amp = find_first_harmonic_peak(hour_array,diurnal_data[var].values ,0.,acc_hour)
    first,second,third,higher = analyze_diurnal_harmonics(diurnal_data[var].values)

    
    diurnal_data[var+'_SAD'] = np.abs(reconstruct_1st_harmonic(hour_array, sd_diurnal_data.values)[0] - sd_diurnal_data.values).sum()
    diurnal_data[var+'_lst'] = harm_lst
    diurnal_data[var+'_amp'] = harm_amp
    diurnal_data[var+'_1st'] = first
    diurnal_data[var+'_2nd'] = second 
    diurnal_data[var+'_3rd'] = third 
    diurnal_data[var+'_higher'] = higher 


In [ ]:
hour_list = np.arange(0,24,3)
cal_diurnal_static(hour_list,fig_rain_ds[0],-3.,'tp')

cal_diurnal_static(hour_list,fig_rain_ds[1],-3.,'tp')
cal_diurnal_static(hour_list,fig_rain_ds[1],-3.,'lp')
cal_diurnal_static(hour_list,fig_rain_ds[1],-3.,'cp')

cal_diurnal_static(hour_list,fig_rain_ds[2],-3.,'tp')
cal_diurnal_static(hour_list,fig_rain_ds[2],-3.,'lp')
cal_diurnal_static(hour_list,fig_rain_ds[2],-3.,'cp')

cal_diurnal_static(hour_list,fig_rain_ds[3],-3.,'tp')
cal_diurnal_static(hour_list,fig_rain_ds[3],-3.,'lp')
cal_diurnal_static(hour_list,fig_rain_ds[3],-3.,'cp')

cal_diurnal_static(hour_list,fig_rain_ds[4],-3.,'tp')
cal_diurnal_static(hour_list,fig_rain_ds[4],-3.,'lp')
cal_diurnal_static(hour_list,fig_rain_ds[4],-3.,'cp')


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as mcolors
from matplotlib.markers import MarkerStyle
import matplotlib.cm as cm

fig, axes = plt.subplots(1, 3, figsize=(12, 4), subplot_kw={'projection': 'polar'})
max_val = 0.6
data_title = ['Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

markers = ['o' ,'s','X','>', '*']
plot_titles = ['Total Precipitation (TP)', 'Convective Precipitation (CP)', 'Large-scale Precipitation (LP)']

variables = ['tp', 'cp', 'lp']

#     (   ,   )
def get_color_palette(n_colors):
    colors = ['black']
    
    # 'viridis', 'plasma', 'inferno', 'magma', 'cividis'  
    #  'tab10', 'tab20', 'Set1', 'Set2', 'Set3'    
    cmap = cm.get_cmap('Set2')
    for i in range(1, n_colors):
        colors.append(cmap(i-1))
    
    return colors

color_palette = get_color_palette(len(markers))

#   plot 
for var_idx, variable in enumerate(variables):
    ax = axes[var_idx]
    
    for i, marker in enumerate(markers):
        name = data_title[i]
        
        # index 0 tp ,  index tp, cp, lp  
        if i == 0 and variable != 'tp':
            #    tp  cp lp   
            continue
        
        mean_hour = fig_rain_ds[i][f'{variable}_lst'].item()
        mean_amp = fig_rain_ds[i][f'{variable}_amp'].item()
            
        angle = mean_hour * 2 * np.pi / 24
        color = color_palette[i]
        
        face_color = color if isinstance(color, str) else color  # mcolors.to_rgba    
        ax.scatter(angle, mean_amp, s=220, label=name if var_idx == 0 else "",
                  marker=MarkerStyle(marker, fillstyle='full'),
                  facecolor=face_color, edgecolors='black', linewidth=0.5,
                  zorder=4,alpha=1)
    
    ctick = 0.2
    ax.set_ylim(0, max_val)
    ax.set_yticks(np.arange(0, max_val, ctick))
    ax.set_yticklabels([f'{x:.1f}' for x in np.arange(0, max_val, ctick)])
    ax.set_xticks(np.linspace(0, 2*np.pi, 8, endpoint=False))
    ax.set_xticklabels(['0', '3', '6', '9', '12', '15', '18', '21'], fontsize=12)
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_title(plot_titles[var_idx], fontsize=14, pad=15)
    
    for line in ax.get_xgridlines() + ax.get_ygridlines():
        line.set_linestyle('--')

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, bbox_to_anchor=(.89, 0.6), loc='center',
           labelspacing=1.2, markerscale=1, handletextpad=0.5, frameon=False)

plt.tight_layout()
plt.subplots_adjust(right=0.8)
plt.show()

#### Efficiency


In [ ]:
for i in range(len(fig_lev_ds)):
    fig_lev_ds[i] = fig_lev_ds[i].mean()*8

for j in range(len(fig_lev_ds)):
    fig_lev_ds[j]['pre_eff'] = -fig_lev_ds[j]['pw_P']/(fig_lev_ds[j]['pw_E']+fig_lev_ds[j]['pw_res'])
    fig_lev_ds[j]['pw_P'] = -fig_lev_ds[j]['pw_P']

In [ ]:
test = []
for i in range(len(fig_lev_ds)):
    test.append(fig_lev_ds[i].sel(hour = fig_lev_ds[i].hour.isin([12,18])).mean('hour')*8)

for j in range(len(test)):
    test[j]['pre_eff'] = -test[j]['pw_P']/(test[j]['pw_E']+test[j]['pw_res'])
    test[j]['pw_P'] = -test[j]['pw_P']

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

data = {
    'Obs': [
        float(fig_lev_ds[0]['pw_P'].values),
        float(fig_lev_ds[0]['pw_E'].values), 
        float(fig_lev_ds[0]['pw_res'].values),
        float(fig_lev_ds[0]['pre_eff'].values)
    ],
    'ERA5': [
        float(fig_lev_ds[1]['pw_P'].values),
        float(fig_lev_ds[1]['pw_E'].values), 
        float(fig_lev_ds[1]['pw_res'].values),
        float(fig_lev_ds[1]['pre_eff'].values)
    ],
    'NARR': [
        float(fig_lev_ds[2]['pw_P'].values),
        float(fig_lev_ds[2]['pw_E'].values), 
        float(fig_lev_ds[2]['pw_res'].values),
        float(fig_lev_ds[2]['pre_eff'].values)
    ],

    'JRA-3Q': [
        float(fig_lev_ds[3]['pw_P'].values),
        float(fig_lev_ds[3]['pw_E'].values), 
        float(fig_lev_ds[3]['pw_res'].values),
        float(fig_lev_ds[3]['pre_eff'].values)
    ],
    'MERRA-2': [
        float(fig_lev_ds[4]['pw_P'].values),
        float(fig_lev_ds[4]['pw_E'].values), 
        float(fig_lev_ds[4]['pw_res'].values),
        float(fig_lev_ds[4]['pre_eff'].values)
    ]
}

categories = ['P', 'E', 'MFC','$\chi$']

# colors = ['#2E86AB', '#A23B72', '#F18F01']  #   
colors = ['#D989A6', '#022859', '#266AA6','#D98A29','#D94625']


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
test = []
for i in range(len(fig_lev_ds)):
    test.append(fig_lev_ds[i].sel(hour = fig_lev_ds[i].hour.isin([0,18])).mean('hour')*8)

for j in range(len(test)):
    test[j]['pre_eff'] = -test[j]['pw_P']/(test[j]['pw_E']+test[j]['pw_res'])
    test[j]['pw_P'] = -test[j]['pw_P']
data = {
    'Obs': [
        float(test[0]['pw_P'].values),
        float(test[0]['pw_E'].values), 
        float(test[0]['pw_res'].values),
        float(test[0]['pre_eff'].values)
    ],
    'ERA5': [
        float(test[1]['pw_P'].values),
        float(test[1]['pw_E'].values), 
        float(test[1]['pw_res'].values),
        float(test[1]['pre_eff'].values)
    ],
    'NARR': [
        float(test[2]['pw_P'].values),
        float(test[2]['pw_E'].values), 
        float(test[2]['pw_res'].values),
        float(test[2]['pre_eff'].values)
    ],

    'JRA-3Q': [
        float(test[3]['pw_P'].values),
        float(test[3]['pw_E'].values), 
        float(test[3]['pw_res'].values),
        float(test[3]['pre_eff'].values)
    ],
    'MERRA-2': [
        float(test[4]['pw_P'].values),
        float(test[4]['pw_E'].values), 
        float(test[4]['pw_res'].values),
        float(test[4]['pre_eff'].values)
    ]
}

categories = ['P', 'E', 'MFC','$\chi$']

# colors = ['#2E86AB', '#A23B72', '#F18F01']  #   
colors = ['#D989A6', '#022859', '#266AA6','#D98A29','#D94625']


In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(11, 6)) #00LST

#   (χ )
df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories[:-1]):  #  χ 
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })
df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn   (P, E, MFC)
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax1, alpha=0.8)

ax2 = ax1.twinx()

# χ    
n_datasets = len(data.keys())
bar_width = 0.8 / n_datasets
x_chi = len(categories) - 1  # χ x 

for i, (dataset, color) in enumerate(zip(data.keys(), colors)):
    chi_value = data[dataset][-1]  #   (χ)
    x_position = x_chi + (i - n_datasets/2 + 0.5) * bar_width
    
    ax2.bar(x_position, chi_value, width=bar_width, 
            color=color, alpha=0.8)
    
    # χ   
    ax2.text(x_position, chi_value + 0.02, f'{chi_value:.2f}', 
             ha='center', va='bottom', fontsize=9)

#    (  )
ax1.set_ylim(0, 30)  #   0~30
ax2.set_ylim(0, 1)  #   0~1

ax1.set_yticks(np.linspace(0,30,6)) 
ax2.set_yticks(np.linspace(0,1.,6))  

ax1.set_ylabel('[mm/day]', fontsize=12)
ax2.set_ylabel('[-]', fontsize=12, color='red')
ax2.tick_params(axis='y', colors='red')

ax1.tick_params(axis='x', labelsize=11)
ax1.tick_params(axis='y', labelsize=10)
ax1.legend(fontsize=11, title_fontsize=12, frameon=True, bbox_to_anchor=(1.3, 1))

# x   (  )
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories)

#    (P, E, MFC)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(11, 6)) #00LST

#   (χ )
df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories[:-1]):  #  χ 
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })
df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn   (P, E, MFC)
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax1, alpha=0.8)

ax2 = ax1.twinx()

# χ    
n_datasets = len(data.keys())
bar_width = 0.8 / n_datasets
x_chi = len(categories) - 1  # χ x 

for i, (dataset, color) in enumerate(zip(data.keys(), colors)):
    chi_value = data[dataset][-1]  #   (χ)
    x_position = x_chi + (i - n_datasets/2 + 0.5) * bar_width
    
    ax2.bar(x_position, chi_value, width=bar_width, 
            color=color, alpha=0.8)
    
    # χ   
    ax2.text(x_position, chi_value + 0.02, f'{chi_value:.2f}', 
             ha='center', va='bottom', fontsize=9)

#    (  )
ax1.set_ylim(0, 30)  #   0~30
ax2.set_ylim(0, 1)  #   0~1

ax1.set_yticks(np.linspace(0,30,6)) 
ax2.set_yticks(np.linspace(0,1.,6))  

ax1.set_ylabel('[mm/day]', fontsize=12)
ax2.set_ylabel('[-]', fontsize=12, color='red')
ax2.tick_params(axis='y', colors='red')

ax1.tick_params(axis='x', labelsize=11)
ax1.tick_params(axis='y', labelsize=10)
ax1.legend(fontsize=11, title_fontsize=12, frameon=True, bbox_to_anchor=(1.3, 1))

# x   (  )
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories)

#    (P, E, MFC)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(11, 6)) #daily mean

#   (χ )
df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories[:-1]):  #  χ 
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })
df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn   (P, E, MFC)
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax1, alpha=0.8)

ax2 = ax1.twinx()

# χ    
n_datasets = len(data.keys())
bar_width = 0.8 / n_datasets
x_chi = len(categories) - 1  # χ x 

for i, (dataset, color) in enumerate(zip(data.keys(), colors)):
    chi_value = data[dataset][-1]  #   (χ)
    x_position = x_chi + (i - n_datasets/2 + 0.5) * bar_width
    
    ax2.bar(x_position, chi_value, width=bar_width, 
            color=color, alpha=0.8)
    
    # χ   
    ax2.text(x_position, chi_value + 0.02, f'{chi_value:.2f}', 
             ha='center', va='bottom', fontsize=9)

#    (  )
ax1.set_ylim(0, 20)  #   0~30
ax2.set_ylim(0, 1)  #   0~1

ax1.set_yticks(np.linspace(0,20,6)) 
ax2.set_yticks(np.linspace(0,1.,6))  

ax1.set_ylabel('[mm/day]', fontsize=12)
ax2.set_ylabel('[-]', fontsize=12, color='red')
ax2.tick_params(axis='y', colors='red')

ax1.tick_params(axis='x', labelsize=11)
ax1.tick_params(axis='y', labelsize=10)
ax1.legend(fontsize=11, title_fontsize=12, frameon=True, bbox_to_anchor=(1.3, 1))

# x   (  )
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories)

#    (P, E, MFC)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
np.linspace(0,20,6)

### Non-Rainy Days


In [ ]:
from functools import reduce

def return_non_raindate(daily_ds):
    filtered_daily = daily_ds.where(daily_ds['tp'] < 0.275, np.nan)
    
    non_rainy_days = filtered_daily.dropna('time')
    non_rainy_date = non_rainy_days.time.values
    return non_rainy_date

non_rain_date = []
for i in range(len(total_daily_ds)):
    non_rain_date.append(return_non_raindate(total_daily_ds[i]))
    print(i)

common_non_rainy_date = reduce(np.intersect1d, non_rain_date)

In [ ]:
len(common_non_rainy_date) #179

In [ ]:
fig_non_lev_ds = []  
fig_non_rain_ds = []  
fig_non_flx_ds = []  

for i in range(len(total_rain_ds)):
    fig_non_rain_ds.append(total_rain_ds[i].sel(time = total_rain_ds[i].time.dt.floor('D').isin(common_non_rainy_date)))#.groupby('time.hour').mean('time'))
    fig_non_lev_ds.append(total_lev_ds[i].sel(time = total_lev_ds[i].time.dt.floor('D').isin(common_non_rainy_date)))#.groupby('time.hour').mean('time'))
    fig_non_flx_ds.append(th_flx_list[i].sel(time = th_flx_list[i].time.dt.floor('D').isin(common_non_rainy_date)))#.groupby('time.hour').mean('time'))
    

In [ ]:
for j in range(len(fig_non_flx_ds)):
    fig_non_flx_ds[j]['pw_P'] = -fig_non_rain_ds[j]['tp'] # mm/3h
    fig_non_flx_ds[j]['pw_E'] = fig_non_flx_ds[j]['le']*3*3600/(2.6*10**(6)) # mm/3h

In [ ]:
fig_non_lev_ds[0]['pw_tend'] = fig_non_lev_ds[0]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h

fig_non_lev_ds[1]['pw_tend'] = fig_non_lev_ds[1]['pw'].diff(dim='time', n=1, label='lower') # mm/3h
fig_non_lev_ds[2]['pw_tend'] = fig_non_lev_ds[2]['pw'].diff(dim='time', n=1, label='lower') # mm/3h

fig_non_lev_ds[3]['pw_tend'] = fig_non_lev_ds[3]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h
fig_non_lev_ds[4]['pw_tend'] = fig_non_lev_ds[4]['pw'].diff(dim='time', n=1, label='lower')/2 #mm/6h ++> mm/3h

In [ ]:
for j in range(len(fig_non_lev_ds)):
    fig_non_lev_ds[j]['pw_P'] = fig_non_flx_ds[j]['pw_P'].sel(time = fig_non_lev_ds[j]['pw_tend'].time)
    fig_non_lev_ds[j]['pw_E'] = fig_non_flx_ds[j]['pw_E'].sel(time = fig_non_lev_ds[j]['pw_tend'].time)

In [ ]:
for j in range(len(fig_non_lev_ds)):
    fig_non_lev_ds[j]['pw_res'] = fig_non_lev_ds[j]['pw_tend'] - fig_non_lev_ds[j]['pw_P'] - fig_non_lev_ds[j]['pw_E'] #mm/3h

In [ ]:
def extend_dataset_to_24h(ds):
    """
    Extend an xarray Dataset with hourly data (0, 6, 12, 18) to include hour 24
    by copying the data from hour 0.
    """
    # Create a new coordinate for hour that includes 24
    new_hour = np.append(ds.hour.values, 24)
    
    # Initialize dictionary to store extended variables
    extended_vars = {}
    
    # For each data variable in the dataset
    for var_name in ds.data_vars:
        var = ds[var_name]
        
        # Check if the variable has the 'hour' dimension
        if 'hour' in var.dims:
            # Get the variable values
            values = var.values
            
            # If the variable is 2D (e.g., temp, Td, etc. with dimensions (hour, pres))
            if var.ndim == 2:
                # Append the values from hour 0 to create hour 24
                extended_values = np.vstack([values, values[0:1]])
            # If the variable is 1D (e.g., pw, cape, cin with dimension (hour))
            elif var.ndim == 1:
                # Append the value from hour 0 to create hour 24
                extended_values = np.append(values, values[0])
            
            # Create a new DataArray with the extended values
            dims = var.dims
            coords = {dim: (ds[dim] if dim != 'hour' else new_hour) for dim in dims}
            extended_vars[var_name] = xr.DataArray(data=extended_values, dims=dims, coords=coords, 
                                                 attrs=var.attrs)
        else:
            # If the variable doesn't have the 'hour' dimension, keep it as is
            extended_vars[var_name] = var
    
    # Create a new dataset with the extended variables
    extended_ds = xr.Dataset(extended_vars, attrs=ds.attrs)
    
    # Update the hour coordinate
    extended_ds = extended_ds.assign_coords(hour=new_hour)
    
    return extended_ds

# Example usage:
# extended_ds = extend_dataset_to_24h(ds)
# Now you can plot with a complete 24-hour cycle

In [ ]:
for i in range(len(fig_non_lev_ds)):
    fig_non_rain_ds[i] = fig_non_rain_ds[i].groupby('time.hour').mean('time')
    fig_non_lev_ds[i] = fig_non_lev_ds[i].groupby('time.hour').mean('time')

In [ ]:
for i in range(len(fig_non_rain_ds)):
    fig_non_lev_ds[i] = extend_dataset_to_24h(fig_non_lev_ds[i])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

# GridSpec    2 (  SM/EF )  
#   SM/EF    3:1 
gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 0), (0, 1)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 0), (1, 1)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
cape_color = '#0072B2'
cin_color = '#D55E00'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

common_levels = np.linspace(328, 340, 25)

# PW      
six_hourly = [0, 6, 12, 18, 24]

def create_arrow_scale(ax):
    """
    Create a scale legend for arrows showing PW differences
    """
    ax.text(0.904, 0.95, 'PW diff',
            transform=ax.transAxes,
            horizontalalignment='center',
            fontsize=10)
    
    arrow_scales = [
        (1, '1 mm', 'black', 30*1.5),
        (2, '2 mm', 'black', 50*1.5),
        (3, '3 mm', 'black', 70*1.5)
    ]
    
    for i, (diff, label, color, q_len) in enumerate(arrow_scales):
        ax.quiver(0.84, 0.83, 0, q_len, 
                  color=color, width=0.01, zorder=10,
                  headwidth=3.5, headaxislength=2, headlength=2,
                  scale_units='xy', scale=1, angles='xy', transform=ax.transAxes)
        
        ax.text(0.87, 0.83 + i*0.03, label, transform=ax.transAxes, 
                verticalalignment='bottom', fontsize=10)

for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = fig_non_lev_ds[i]
    current_rain = fig_non_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    #    y  
    if (i < 3 and main_col == 0) or (i >= 3 and main_col == 0):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax_pw = ax2.twiny()
    ax_pw.set_zorder(1)  #  zorder
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 
    pw_tend = fig_non_lev_ds[i]['pw_tend'][:-1]
    hours = pw_tend.hour.values
    values = pw_tend.values
    
    bar_width = 2
    base_value= 8  # 
    
    for hour, value in zip(hours, values):
        #    ( )
        bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width, 
                     bottom=base_value-abs(value),
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            # : 
            ax_pw.bar(hour+1.5, bar_height, width=bar_width,
                     bottom=base_value-abs(value),
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks([])  # x  
    ax_pw.set_xticklabels([])
    bar_width = 0.6
    bar_s = .7
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i < 3 and main_col == 2) or (i >= 3 and main_col == 1):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-400, 800)
    
    #   PW     (  () )
    if i > 0:
        obs_diur = fig_non_lev_ds[0]
        pw_diff = (current_diur['pw'].sel(hour=six_hourly) - obs_diur['pw'].sel(hour=six_hourly)).values
        
        for j, hour in enumerate(six_hourly):
            if hour in current_diur.hour.values:
                # CAPE  
                if 'cape' in current_diur and hour in current_diur.hour.values:
                    cape_val = float(current_diur['cape'].sel(hour=hour).values)
                else:
                    # CAPE       
                    cape_val = 300  #  y  
                
                diff = pw_diff[j]
                
                if diff > 0:
                    color = 'red'
                else:
                    color = 'blue'
                
                if abs(diff) > 3:
                    arrow_length = 70*1.5
                elif abs(diff) > 2.5:
                    arrow_length = 60*1.5
                elif abs(diff) > 2.0:
                    arrow_length = 50*1.5
                elif abs(diff) > 1.5:
                    arrow_length = 40*1.5
                elif abs(diff) > 1:
                    arrow_length = 30*1.5
                elif abs(diff) > 0.5:
                    arrow_length = 20*1.5
                else:
                    arrow_length = 10*1.5
                
                if diff > 0:
                    ax3.quiver(hour, cape_val, 0, arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
                else:
                    ax3.quiver(hour, cape_val, 0, -arrow_length,
                           color=color, width=0.01, zorder=10,
                           headwidth=3.5, headaxislength=2, headlength=2,
                           scale_units='xy', scale=1, angles='xy')
        
        if i == 1:  # ERA5 (  )
            create_arrow_scale(ax3)
    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    
    # SM   ( )
    # sm_line = ax_sm_ef.plot(current_sm.hour, current_sm.sm_ano.values, '-', 
    #                        color=sm_color, linewidth=2.5, marker='o', markersize=6, 
    #                        markeredgecolor='black', markeredgewidth=0.5, label='SM')
    # SM  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    # ax_sm_ef.set_ylim(0, 0.5)  # SM  (  )
    # ax_sm_ef.grid(True, ls='--', alpha=0.3)
    
    # if (i < 3 and sm_ef_col == 0) or (i >= 3 and sm_ef_col == 0):
        # ax_sm_ef.set_ylabel('SM [m³/m³]', fontsize=11, color=sm_color)
    # ax_sm_ef.tick_params(axis='y', colors=sm_color)
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #    x  
    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # # EF   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = fig_non_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    pw_bar_width = 2
    
    for hour, value in zip(hours, values):
        #    ( )
        # bar_height = abs(value)  # 3mm   
        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.6, value, width=bar_width*3, 
                     color='red', alpha=0.3, 
                     edgecolor='darkred', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.6, value, width=bar_width*3,
                     color='blue', alpha=0.3,
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
    # ef_line = ax_ef.plot(current_ef.hour+1.5, current_ef.ef.values, '-', 
    #                     color=ef_color, linewidth=2.5, marker='s', markersize=6, 
    #                     markeredgecolor='black', markeredgewidth=0.5, label='EF')
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Advc',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Advc', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    # ax_ef.set_yticks(np.arange(-5,5,2.5))
    if (i < 3 and sm_ef_col == 2) or (i >= 3 and sm_ef_col == 1):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)
    # ax_ef.tick_params(axis='y', colors=ef_color)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    Line2D([0], [0], color='red', lw=0, marker=r'$\uparrow$', markersize=15,
           label='PW(Model) > PW(Obs)'),
    Line2D([0], [0], color='blue', lw=0, marker=r'$\downarrow$', markersize=15,
           label='PW(Model) < PW(Obs)'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, fontsize=14, bbox_to_anchor=(0.5, 0.05))

# Colorbar 
cbar_ax = fig.add_axes([0.74, 0.17, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(328, 340, 7))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)

plt.show()

### Rainy Days


In [ ]:
len(compare_lev_ds)

In [ ]:
sg_rain_list_daily = []
sg_dry_list_daily = []

sg_diff_list_daily = []

for j in range(len(fig_lev_ds)):
    sg_rain_list_daily.append(fig_lev_ds[j].sel(hour = 0)[['pw_P','pw_E','pw_res']].mean())
    sg_dry_list_daily.append(fig_non_lev_ds[j].sel(hour = 0)[['pw_P','pw_E','pw_res']].mean())
    
    sg_rain_list_daily[j]['pre_eff'] = -sg_rain_list_daily[j]['pw_P']/(sg_rain_list_daily[j]['pw_E']+sg_rain_list_daily[j]['pw_res'])
    sg_dry_list_daily[j]['pre_eff'] = -sg_dry_list_daily[j]['pw_P']/(sg_dry_list_daily[j]['pw_E']+sg_dry_list_daily[j]['pw_res'])
    
    sg_diff_list_daily.append(sg_rain_list_daily[j]-sg_dry_list_daily[j])


    sg_diff_list_daily[j]['effc_ef'] = sg_diff_list_daily[j]['pre_eff']*(sg_dry_list_daily[j]['pw_E']+sg_dry_list_daily[j]['pw_res'])
    
    sg_diff_list_daily[j]['surf_ef'] = sg_dry_list_daily[j]['pre_eff']*sg_diff_list_daily[j]['pw_E']
    
    sg_diff_list_daily[j]['remo_ef'] = sg_dry_list_daily[j]['pre_eff']*sg_diff_list_daily[j]['pw_res']
    
    sg_diff_list_daily[j]['resi'] = sg_diff_list_daily[j]['pre_eff']*(sg_diff_list_daily[j]['pw_E']+sg_diff_list_daily[j]['pw_res'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

data = {
    'Obs': [
        float(sg_diff_list_daily[0]['effc_ef'].values*8),
        float(sg_diff_list_daily[0]['surf_ef'].values*8), 
        float(sg_diff_list_daily[0]['remo_ef'].values*8)
    ],
    'ERA5': [
        float(sg_diff_list_daily[1]['effc_ef'].values*8),
        float(sg_diff_list_daily[1]['surf_ef'].values*8), 
        float(sg_diff_list_daily[1]['remo_ef'].values*8)
    ],
    'NARR': [
        float(sg_diff_list_daily[2]['effc_ef'].values*8),
        float(sg_diff_list_daily[2]['surf_ef'].values*8),
        float(sg_diff_list_daily[2]['remo_ef'].values*8)
    ],

    'JRA-3Q': [
        float(sg_diff_list_daily[3]['effc_ef'].values*8),
        float(sg_diff_list_daily[3]['surf_ef'].values*8),
        float(sg_diff_list_daily[3]['remo_ef'].values*8)
    ],
    'MERRA-2': [
        float(sg_diff_list_daily[4]['effc_ef'].values*8),
        float(sg_diff_list_daily[4]['surf_ef'].values*8),
        float(sg_diff_list_daily[4]['remo_ef'].values*8)
    ]
}

categories = ['Efficiency effect', 'Surface effect', 'Remote effect']

colors = ['#F25244', '#F2A20C', '#4EA67D','#856DA6','#D9415D']


In [ ]:
fig, ax2 = plt.subplots(1, 1, figsize=(6, 6))

df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories):
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })

df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn  
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax2, alpha=0.8)

# ax2.set_xlabel(none)
ax2.set_ylabel('[mm/day]', fontsize=12)
# ax2.set_title('Precipitation Effects Comparison', fontsize=14, fontweight='bold', pad=20)
ax2.tick_params(axis='x', labelsize=11)
ax2.tick_params(axis='y', labelsize=10)
ax2.legend( fontsize=11, title_fontsize=12, frameon=True)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
for j in range(len(rain_list_daily)):
    rain_list_daily[j]['pre_eff'] = -rain_list_daily[j]['pw_P']/(rain_list_daily[j]['pw_E']+rain_list_daily[j]['pw_res'])
    dry_list_daily[j]['pre_eff'] = -dry_list_daily[j]['pw_P']/(dry_list_daily[j]['pw_E']+dry_list_daily[j]['pw_res'])

In [ ]:
diff_list_daily = []
for j in range(len(rain_list_daily)):
    diff_list_daily.append(rain_list_daily[j]-dry_list_daily[j])

In [ ]:
compare_lev_ds = []
compare_rain_ds = []

for j in range(len(fig_rain_ds)):
    compare_lev_ds.append(fig_lev_ds[j]-fig_non_lev_ds[j])
    compare_rain_ds.append(fig_rain_ds[j]-fig_non_rain_ds[j])

In [ ]:
compare_lev_ds = []
compare_rain_ds = []

for j in range(len(fig_rain_ds)):
    compare_lev_ds.append(fig_lev_ds[j]-fig_non_lev_ds[j])
    compare_rain_ds.append(fig_rain_ds[j]-fig_non_rain_ds[j])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

fig = plt.figure(figsize=(15, 12))  #   SM/EF   

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                  hspace=0.0)  #   SM/EF    

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 3:1 
                     hspace=0.)  #   SM/EF    

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
# main_positions_bottom = [(0, 0), (0, 1)]          #  GridSpec    (   )
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec    (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
# sm_ef_positions_bottom = [(1, 0), (1, 1)]         #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

common_levels = np.linspace(-6, 6, 25)

# PW      
six_hourly = [0, 6, 12, 18, 24]


for i, dataset_name in enumerate(datasets):
    if i < 3:  #  3  (ARM SGP, ERA5, NARR) -  
        gs_current = gs_top
        main_idx = i
        main_pos = main_positions_top
        sm_ef_pos = sm_ef_positions_top
    else:  #  2  (JRA-3Q, MERRA-2) -  
        gs_current = gs_bottom
        main_idx = i - 3
        main_pos = main_positions_bottom
        sm_ef_pos = sm_ef_positions_bottom
        if main_idx >= len(main_positions_bottom):
            continue
    
    # MSE/CAPE/CIN   
    main_row, main_col = main_pos[main_idx]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs_current[main_row, main_col])
    
    current_diur = compare_lev_ds[i]
    current_rain = compare_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both',zorder = 0)
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (SM/EF  )
    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #    y  
    if (i == 0) or (i ==3):
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else :
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)  # PW tendency 

    bar_width = 0.6
    bar_s = .6
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        # tp, cp, lp    -     
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # cp  (tp   )
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
        
        # lp  (cp   )
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5,zorder = 3)
    
    if (i == 2 and main_col == 2) or (i ==4 and main_col == 2):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False) 


    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   -     
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE',zorder = 3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN',zorder = 3)
    
    # CAPE/CIN  
    if (i == 2 and main_col == 2) or (i ==4 and main_col == 2):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
    ax3.set_ylim(-200, 400)    
    ax1.set_title(dataset_name, fontsize=18)
    
    # SM/EF  
    sm_ef_row, sm_ef_col = sm_ef_pos[main_idx]
    ax_sm_ef = fig.add_subplot(gs_current[sm_ef_row, sm_ef_col])
    
    # SM  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #    x  
    if i >= 3:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # # EF   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = compare_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    for hour, value in zip(hours, values):        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8,zorder = 1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8,zorder = 1)

    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5,zorder = 3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5,zorder = 3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c = 'black',ls = '-',lw = .3) 
    ax_ef.set_ylim(-4, 4)  # EF  (0-1)
    
    if i == 0:  #    (IGRA SGP Obs)
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right',  ncol=3,
                          fontsize=10, frameon=False,#framealpha=0.7,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    if (i == 2 and sm_ef_col == 2) or (i == 4 and sm_ef_col == 2):
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)

#   (SM EF )
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=1, frameon=True, fontsize=14, bbox_to_anchor=(0.28, 0.5))

# Colorbar 
cbar_ax = fig.add_axes([1.01, (1-0.315)/2, 0.02, 0.315])  #    .74
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(-6, 6, 7))

plt.tight_layout(rect=[0, 0.05, 0.9, 0.95])
plt.subplots_adjust(wspace=0.3, hspace=0.3)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/vertical_fig_nonrain.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 1 5    
fig = plt.figure(figsize=((15/3)*5, 12/2))

# GridSpec  - 1 5 
gs = GridSpec(2, 6, figure=fig,
              top=0.90, bottom=0.15,
              width_ratios=[1,.1, 1, 1, 1, 1],  # 5   
              height_ratios=[4.5, 1],  #  :PW  = 4.5:1 
              hspace=0.0,
              wspace=0.2)

mse_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
labels = ['(a)', '(b)', '(c)', '(d)', '(e)']

common_levels = np.linspace(-6, 6, 25)
fig_idx = [0,2,3,4,5]
for i, dataset_name in enumerate(datasets):
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs[0, fig_idx[i]])
    
    current_diur = compare_lev_ds[i]
    current_rain = compare_rain_ds[i]
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    # mse_data = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax1.contourf(current_diur.hour, current_diur.pres, mse_data.T, 
                      levels=common_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    #    ( ) -     
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    
    # x  -   x  
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])  # x   (PW  )
    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #      ,     
    if i == 0:
        ax1.set_ylim(925, 400)

        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.set_ylim(925, 400)

        #   y    
        ax1.tick_params(axis='y', which='both', labelleft=False)
    
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)

    bar_width = 0.6
    bar_s = .6
    
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    else:
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                     width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                     width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    
    #    -  (i==4) y  
    ax2.set_ylim(0, 6)
    if i == 4:
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=False, labelright=False)
        ax2.spines['right'].set_visible(False)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    ax3.set_ylim(-100, 400)
    if i == 4:
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)
    
    ax1.set_title(dataset_name, fontsize=18)    
    # PW  
    ax_sm_ef = fig.add_subplot(gs[1, fig_idx[i]])
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False) 

    #   x  
    ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   ( )
    ax_ef = ax_sm_ef.twinx()
    pw_info = compare_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    for hour, value in zip(hours, values):        
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)

    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)

    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3) 
    ax_ef.set_ylim(-4.8, 4.8)
    
    #   (i==0) PW  
    if i == 0:
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                              edgecolor='black', linewidth=0.3)
        ]
        
        #   -   
        leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                          fontsize=10, frameon=False,
                          bbox_to_anchor=(1, -0.1), 
                          bbox_transform=ax_ef.transAxes)

    #  (i==4) PW y  
    if i == 4:
        ax_ef.set_ylabel('[mm/3h]', fontsize=11)

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=5, frameon=True, 
          fontsize=14, bbox_to_anchor=(0.5, 0.05))

pos_last = ax_last.get_position()
cbar_ax = fig.add_axes([ax1.get_position().x1 + 0.07, ax1.get_position().y0, 0.015, ax1.get_position().height])
cbar = fig.colorbar(CS0, cax=cbar_ax)
cbar.set_label('MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar.set_ticks(np.linspace(-6, 6, 7))

plt.tight_layout(rect=[0, 0.1, 0.95, 0.95])

plt.show()

In [ ]:
# fig_lev_ds 
# fig_rain_ds 

In [ ]:
import numpy as np
import xarray as xr

def interpolate_to_3hourly(ds):
    """
    6-hourly (0, 6, 12, 18) data expanded via spline interpolation to 
    3-hourly interval (0, 3, 6, 9, 12, 15, 18, 21)expanded.
    """
    # 1. New    (0, 3, 6, ..., 21)
    new_hours = np.arange(0, 24, 3)
    
    # 2. (Circular/Periodic)   24  0   
    # Diurnal cycle 0 24    .
    ds_cycle = ds.sel(hour=0)
    ds_cycle.coords['hour'] = 24
    ds_extended = xr.concat([ds, ds_cycle], dim='hour')
    
    # 3. Spline(Cubic)  
    # 'cubic' 4        (0, 6, 12, 18, 24  5 )
    ds_interpolated = ds_extended.interp(hour=new_hours, method='linear')
    
    return ds_interpolated



In [ ]:
merra_lev_diurnal_splin_interp = xr.concat([merra_lev_mean.assign_coords(hour=np.arange(-24,0,6)),
          merra_lev_mean,
          merra_lev_mean.assign_coords(hour=np.arange(24,48,6))],
          dim='hour').fillna(0).interp(hour=np.arange(0, 24, 3), method='cubic')#['cape'][-1].plot()

jra_lev_diurnal_splin_interp = xr.concat([jra_lev_mean.assign_coords(hour=np.arange(-24,0,6)),
          jra_lev_mean,
          jra_lev_mean.assign_coords(hour=np.arange(24,48,6))],
          dim='hour').fillna(0).interp(hour=np.arange(0, 24, 3), method='cubic')#['cape'][-1].plot()


In [ ]:
def return_sp_interp(source_ds):
    diurnal_splin_interp = xr.concat([source_ds.assign_coords(hour=np.arange(-24,0,6)),
              source_ds,
              source_ds.assign_coords(hour=np.arange(24,48,6))],
              dim='hour').fillna(0).interp(hour=np.arange(0, 24, 3), method='cubic')#['cape'][-1].plot()
    return (diurnal_splin_interp)

In [ ]:
fig_lev_ds[0] = return_sp_interp(fig_lev_ds[0])#['cin'].plot()
fig_lev_ds[3] = return_sp_interp(fig_lev_ds[3])#['cin'].plot()
fig_lev_ds[4] = return_sp_interp(fig_lev_ds[4])#['cin'].plot()


In [ ]:
for i in range(len(total_rain_ds)):
    fig_lev_ds[i] = extend_dataset_to_24h(fig_lev_ds[i])

In [ ]:
fig_lev_ds[0]#[4]#[3]#[0]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 2 
fig = plt.figure(figsize=((24/5)*3.5, (12/4.5)*3.5))

# GridSpec  - 2 
# 1:   1 ( )
# 2:   4
gs = GridSpec(2, 4, figure=fig,
              top=0.93, bottom=0.12,
              width_ratios=[1, 1, 1, 1],  #   , colorbar, ,  3
              height_ratios=[1, 1],
              hspace=0.25,
              wspace=0.15)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
# labels = ['(a)', '(b)', '(c)', '(d)', '(e)']
labels = ['(a)',  '(d)','(b)','(c)',  '(e)']

obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]

#    (  )
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

CS_obs = None  #   colorbar
CS_diff = None  #   colorbar
axis = []

#   :   (0, 0) 
#   :   (1, 0), (1, 1), (1, 2), (1, 3) 
positions = [
    (0, 0),  # IGRA SGP Obs -   ,   
    (1, 2),  # ERA5 -   ,   
    (1, 0),  # NARR -   ,   
    (1, 1),  # JRA-3Q -   ,   
    (1, 3),  # MERRA-2 -   ,   
]

for i, dataset_name in enumerate(datasets):
    row, col = positions[i]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs[row, col])
    axis.append(ax1)
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    #   () () 
    if i == 0:  #   (IGRA SGP Obs)
        # MSE    
        mse_data = current_diur.mse.values.copy()
        mse_data_masked = np.ma.masked_invalid(mse_data)
        
        # MSE    ( )
        CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                          levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
        CS_obs = CS_current
        
    else:  #   ( )
        #    MSE  
        model_hours = current_diur.hour.values
        model_six_hour_mask = np.isin(model_hours, six_hourly)
        
        if len(model_hours) > len(six_hourly):
            mse_model = current_diur.mse.values[model_six_hour_mask, :]
            model_hours_6h = model_hours[model_six_hour_mask]
        else:
            mse_model = current_diur.mse.values
            model_hours_6h = model_hours
        
        mse_diff = mse_model - obs_mse
        mse_diff_masked = np.ma.masked_invalid(mse_diff)
        
        # MSE    ( )
        CS_current = ax1.contourf(model_hours_6h, current_diur.pres, mse_diff_masked.T, 
                         levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
        if CS_diff is None:
            CS_diff = CS_current
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.minorticks_off()
    ax1.set_yticks([925, 850, 700, 500, 400])
    ax1.set_yticklabels([925, 850, 700, 500, 400], fontsize=12)

    
    # x 
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels(np.arange(0, 24, 3), fontsize=12)
    if row !=0:
        ax1.set_xlabel('LST [hour]', fontsize=13)

    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #          y  
    if col == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
        
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)

    bar_width = 0.6
    bar_s = .6
    
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        # CP LP  
        if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
            bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5, zorder=3)
            
            bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5, zorder=3)
    
    if row == 0 or (row == 1 and col == 3):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.tick_params(labelsize=12)

    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False) 

    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    # row, col
    if row == 0 or (row == 1 and col == 3):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.tick_params(labelsize=12)

    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
        
    ax3.set_ylim(-400, 800)    
    ax1.set_title(dataset_name, fontsize=18)
    
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='lower center', ncol=5, frameon=False, 
          fontsize=18, bbox_to_anchor=(0.5, 0.01))

#    -     
pos_first = axis[0].get_position()
cbar_ax_obs = fig.add_axes([pos_first.x1 + 0.1, pos_first.y0, 0.02, pos_first.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))
# cbar_obs.set_ticklabels(np.linspace(328, 340, 7),fontsize=12)
cbar_obs.ax.tick_params(labelsize=12)

#    -    
pos_last = axis[-1].get_position()  # JRA-3Q (  )
cbar_ax_diff = fig.add_axes([pos_last.x1 + 0.1, pos_last.y0, 0.02, pos_last.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))
cbar_diff.ax.tick_params(labelsize=12)

# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/vertical_fig_2row_layout.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 2 
fig = plt.figure(figsize=((24/5)*3.5, (12/4.5)*3.5))

# GridSpec  - 2 
# 1:   1 ( )
# 2:   4
gs = GridSpec(2, 4, figure=fig,
              top=0.93, bottom=0.12,
              width_ratios=[1, 1, 1, 1],  #   , colorbar, ,  3
              height_ratios=[1, 1],
              hspace=0.25,
              wspace=0.15)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
# labels = ['(a)', '(b)', '(c)', '(d)', '(e)']
labels = ['(a)',  '(d)','(b)','(c)',  '(e)']

obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]

#    (  )
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

CS_obs = None  #   colorbar
CS_diff = None  #   colorbar
axis = []

#   :   (0, 0) 
#   :   (1, 0), (1, 1), (1, 2), (1, 3) 
positions = [
    (0, 0),  # IGRA SGP Obs -   ,   
    (1, 2),  # ERA5 -   ,   
    (1, 0),  # NARR -   ,   
    (1, 1),  # JRA-3Q -   ,   
    (1, 3),  # MERRA-2 -   ,   
]

for i, dataset_name in enumerate(datasets):
    row, col = positions[i]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs[row, col])
    axis.append(ax1)
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    #   () () 
    if i == 0:  #   (IGRA SGP Obs)
        # MSE    
        mse_data = current_diur.mse.values.copy()
        mse_data_masked = np.ma.masked_invalid(mse_data)
        
        # MSE    ( )
        CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                          levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
        CS_obs = CS_current
        
    else:  #   ( )
        #    MSE  
        model_hours = current_diur.hour.values
        model_six_hour_mask = np.isin(model_hours, six_hourly)
        
        # if len(model_hours) > len(six_hourly):
        #     mse_model = current_diur.mse.values[model_six_hour_mask, :]
        #     model_hours_6h = model_hours[model_six_hour_mask]
        # else:
        #     mse_model = current_diur.mse.values
        #     model_hours_6h = model_hours
        mse_model = current_diur.mse.values
        
        mse_diff = mse_model - obs_mse
        mse_diff_masked = np.ma.masked_invalid(mse_diff)
        
        # MSE    ( )
        CS_current = ax1.contourf(model_hours, current_diur.pres, mse_diff_masked.T, 
                         levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
        if CS_diff is None:
            CS_diff = CS_current
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.minorticks_off()
    ax1.set_yticks([925, 850, 700, 500, 400])
    ax1.set_yticklabels([925, 850, 700, 500, 400], fontsize=12)

    
    # x 
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels(np.arange(0, 24, 3), fontsize=12)
    if row !=0:
        ax1.set_xlabel('LST [hour]', fontsize=13)

    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #          y  
    if col == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
        
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)

    bar_width = 0.6
    bar_s = .6
    
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        # CP LP  
        if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
            bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5, zorder=3)
            
            bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5, zorder=3)
    
    if row == 0 or (row == 1 and col == 3):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.tick_params(labelsize=12)

    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False) 

    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    # row, col
    if row == 0 or (row == 1 and col == 3):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.tick_params(labelsize=12)

    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
        
    ax3.set_ylim(-400, 800)    
    ax1.set_title(dataset_name, fontsize=18)
    
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='lower center', ncol=5, frameon=False, 
          fontsize=18, bbox_to_anchor=(0.5, 0.01))

#    -     
pos_first = axis[0].get_position()
cbar_ax_obs = fig.add_axes([pos_first.x1 + 0.1, pos_first.y0, 0.02, pos_first.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))
# cbar_obs.set_ticklabels(np.linspace(328, 340, 7),fontsize=12)
cbar_obs.ax.tick_params(labelsize=12)

#    -    
pos_last = axis[-1].get_position()  # JRA-3Q (  )
cbar_ax_diff = fig.add_axes([pos_last.x1 + 0.1, pos_last.y0, 0.02, pos_last.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))
cbar_diff.ax.tick_params(labelsize=12)

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/vertical_fig_2row_layout_new.png', dpi=400, bbox_inches='tight')

# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 2 
fig = plt.figure(figsize=((24/5)*3.5, (12/4.5)*3.5))

# GridSpec  - 2 
# 1:   1 ( )
# 2:   4
gs = GridSpec(2, 4, figure=fig,
              top=0.93, bottom=0.12,
              width_ratios=[1, 1, 1, 1],  #   , colorbar, ,  3
              height_ratios=[1, 1],
              hspace=0.25,
              wspace=0.15)

mse_cmap = cmaps.davos_r
diff_cmap = cmaps.balance
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)
# labels = ['(a)', '(b)', '(c)', '(d)', '(e)']
labels = ['(a)',  '(d)','(b)','(c)',  '(e)']

obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 25)

# PW      
six_hourly = [0, 6, 12, 18, 24]

#    (  )
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

CS_obs = None  #   colorbar
CS_diff = None  #   colorbar
axis = []

#   :   (0, 0) 
#   :   (1, 0), (1, 1), (1, 2), (1, 3) 
positions = [
    (0, 0),  # IGRA SGP Obs -   ,   
    (1, 2),  # ERA5 -   ,   
    (1, 0),  # NARR -   ,   
    (1, 1),  # JRA-3Q -   ,   
    (1, 3),  # MERRA-2 -   ,   
]

for i, dataset_name in enumerate(datasets):
    row, col = positions[i]
    
    # MSE/CAPE/CIN  
    ax1 = fig.add_subplot(gs[row, col])
    axis.append(ax1)
    
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    #   () () 
    if i == 0:  #   (IGRA SGP Obs)
        # MSE    
        mse_data = current_diur.mse.values.copy()
        mse_data_masked = np.ma.masked_invalid(mse_data)
        
        # MSE    ( )
        CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                          levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
        CS_obs = CS_current
        
    else:  #   ( )
        #    MSE  
        model_hours = current_diur.hour.values
        model_six_hour_mask = np.isin(model_hours, six_hourly)
        
        if len(model_hours) > len(six_hourly):
            mse_model = current_diur.mse.values[model_six_hour_mask, :]
            model_hours_6h = model_hours[model_six_hour_mask]
        else:
            mse_model = current_diur.mse.values
            model_hours_6h = model_hours
        
        mse_diff = mse_model - obs_mse
        mse_diff_masked = np.ma.masked_invalid(mse_diff)
        
        # MSE    ( )
        CS_current = ax1.contourf(model_hours_6h, current_diur.pres, mse_diff_masked.T, 
                         levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
        if CS_diff is None:
            CS_diff = CS_current
    
    #    ( )
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.minorticks_off()
    ax1.set_yticks([925, 850, 700, 500, 400])
    ax1.set_yticklabels([925, 850, 700, 500, 400], fontsize=12)

    
    # x 
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels(np.arange(0, 24, 3), fontsize=12)
    if row !=0:
        ax1.set_xlabel('LST [hour]', fontsize=13)

    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')

    #          y  
    if col == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False) 
        
    #      (   )
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)

    bar_width = 0.6
    bar_s = .6
    
    # ARM SGP ( ) tp  cp, lp 
    if i == 0:
        # tp 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
    else:
        # CP LP   ERA5, NARR, JRA-3Q, MERRA-2 
        bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                     width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                     edgecolor='black', linewidth=0.5, zorder=3)
        
        # CP LP  
        if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
            bars_cp = ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5, zorder=3)
            
            bars_lp = ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5, zorder=3)
    
    if row == 0 or (row == 1 and col == 3):
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.tick_params(labelsize=12)

    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False) 

    ax2.set_ylim(0, 8)

    # CAPE CIN     (   )
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    # row, col
    if row == 0 or (row == 1 and col == 3):
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.tick_params(labelsize=12)

    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False) 
        
    ax3.set_ylim(-400, 800)    
    ax1.set_title(dataset_name, fontsize=18)
    
var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='lower center', ncol=5, frameon=False, 
          fontsize=18, bbox_to_anchor=(0.5, 0.01))

#    -     
pos_first = axis[0].get_position()
cbar_ax_obs = fig.add_axes([pos_first.x1 + 0.1, pos_first.y0, 0.02, pos_first.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))
# cbar_obs.set_ticklabels(np.linspace(328, 340, 7),fontsize=12)
cbar_obs.ax.tick_params(labelsize=12)

#    -    
pos_last = axis[-1].get_position()  # JRA-3Q (  )
cbar_ax_diff = fig.add_axes([pos_last.x1 + 0.1, pos_last.y0, 0.02, pos_last.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))
cbar_diff.ax.tick_params(labelsize=12)

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/vertical_fig_2row_layout.png', dpi=400, bbox_inches='tight')

# plt.show()

In [ ]:
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]


In [ ]:
len( fig_rain_ds)

In [ ]:
(fig_rain_ds[0]*8)['tp'].plot()

In [ ]:
np.linspace(-10, 10, 21)

In [ ]:
for i in range(len(fig_rain_ds)):
    fig_rain_ds[i] = extend_dataset_to_24h(fig_rain_ds[i])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

levels = np.linspace(-10, 10, 21)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(200,500,25)
cape_cmap = plt.cm.jet #cmaps.cet_l_worb
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'NARR': {'color': '#2ca02c', 'data': fig_lev_ds[2]*8},
    'ERA5': {'color': '#ff7f0e', 'data': fig_lev_ds[1]*8},
    'IGRA': {'color': '#1f77b4', 'data': fig_lev_ds[0]*8}
}

precip_datasets = {
    'NARR': {'color': 'gray', 'data':  fig_rain_ds[2]*8},
    'ERA5': {'color': 'gray', 'data':  fig_rain_ds[1]*8},
    'IGRA': {'color': 'black', 'data':  fig_rain_ds[0]*8}
}
ls = ['solid','dotted','dashed'] 

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
# CAPE  (8.0),   dPW(5.5), MFC(3.0), E(0.5)
sec = [5.5, 3.0, 0.5, 8.0]  # CAPE  (8.0) 
yticks = [-1.0,-0.5,0.0,0.5,1.5,2.0,2.5,3.0,4.0,4.5,5.0,5.5,6.5,7.0,7.5,8.]
lev_xds = [fig_lev_ds[2],fig_lev_ds[1],fig_lev_ds[0] ]
# pw_xds = [fig_lev_ds[2],fig_lev_ds[1],fig_lev_ds[0] ]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 2)|(ds_idx == 2):  # JRA-3Q, MERRA2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

#    (  )
dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

# CMORPH 
# ax2.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
#         color='black', linewidth=2, ls='-',
#         label='CMORPH')

#    (CAPE  )
#  CAPE, dPW, MFC, E 
all_var_labels = ['CAPE'] + var_labels  # CAPE  
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # CAPE(8.0), dPW(5.5), MFC(3.0), E(0.5)

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)
# 12
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  (CAPE   )
ax.set_ylim(-1., 8.)
ax.set_yticks([])  # y  
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5,alpha = .7)

ax.set_yticklabels([])

ax2.set_ylim(0, 30)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=1, bbox_to_anchor=(1.5, .75),frameon=False)

#    (    )
ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-10, 10, 6))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25,ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(200,500,7))
cbar2.set_label('[J/kg]', fontsize=10)

plt.show()

In [ ]:
dataset_names_ordered[::-1]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 5    
fig, ax = plt.subplots(1, 1, figsize=(6, 7))

levels = np.linspace(-10, 10, 21)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(200,500,25)
cape_cmap = cmaps.cet_l_worb
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

#  : IGRA(0) -> ERA5(1) -> NARR(2) -> JRA-3Q(3) -> MERRA-2(4)
# dataset_names_ordered = ['IGRA', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']
dataset_names_ordered = ['MERRA-2','JRA-3Q',  'NARR','ERA5','IGRA']

#   - 5  
# datasets = {
#     'IGRA': {'color': '#1f77b4', 'data': fig_lev_ds[0]*8},
#     'ERA5': {'color': '#ff7f0e', 'data': fig_lev_ds[1]*8},
#     'NARR': {'color': '#2ca02c', 'data': fig_lev_ds[2]*8},
#     'JRA-3Q': {'color': '#d62728', 'data': fig_lev_ds[3]*8},
#     'MERRA-2': {'color': '#9467bd', 'data': fig_lev_ds[4]*8}
# }

datasets = {
    'MERRA-2': {'color': '#9467bd', 'data': fig_lev_ds[4]*8},
    'JRA-3Q': {'color': '#d62728', 'data': fig_lev_ds[3]*8},
    'NARR': {'color': '#2ca02c', 'data': fig_lev_ds[2]*8},
    'ERA5': {'color': '#ff7f0e', 'data': fig_lev_ds[1]*8},
    'IGRA': {'color': '#1f77b4', 'data': fig_lev_ds[0]*8},
}
#   - 5  
# precip_datasets = {
#     'IGRA': {'color': 'black', 'data': fig_rain_ds[0]*8},
#     'ERA5': {'color': 'gray', 'data': fig_rain_ds[1]*8},
#     'NARR': {'color': 'gray', 'data': fig_rain_ds[2]*8},
#     'JRA-3Q': {'color': 'gray', 'data': fig_rain_ds[3]*8},
#     'MERRA-2': {'color': 'gray', 'data': fig_rain_ds[4]*8}
# }
precip_datasets = {
    'MERRA-2': {'color': 'gray', 'data': fig_rain_ds[4]*8},
    'JRA-3Q': {'color': 'gray', 'data': fig_rain_ds[3]*8},
    'NARR': {'color': 'gray', 'data': fig_rain_ds[2]*8},
    'ERA5': {'color': 'gray', 'data': fig_rain_ds[1]*8},
    'IGRA': {'color': 'black', 'data': fig_rain_ds[0]*8}
}



ls = ['solid','dotted','dashed','dashdot', (0, (3, 1, 1, 1))]  # 5  

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']

# 5   y  
sec = [8.5, 5.5, 2.5, 11.5]  # CAPE  (11.5) ,  
# yticks = [-1.0, 0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5, 12.0]
# yticks = [0.0, 0.5, 1.0, 1.5, 2.0,2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 6.0, 6.5, 7.0, 7.5, 8.0, 8.5, 9.0, 9.5, 10.0, 10.5, 11.0, 11.5]
yticks = np.arange(0.,12.,.5)
lev_xds = [fig_lev_ds[4], fig_lev_ds[3],fig_lev_ds[2],fig_lev_ds[1],fig_lev_ds[0]]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # IGRA JRA-3Q     (3-hourly interval  )
    if ds_idx == 0 or ds_idx == 1 or ds_idx == 4:  # IGRA  JRA-3Q
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, dataset_name in enumerate(dataset_names_ordered):
        data = datasets[dataset_name]['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

#    (5   )
for ds_idx, dataset_name in enumerate(dataset_names_ordered):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*5 + ds_idx * .5 + .25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name in dataset_names_ordered[::-1]:
    dataset_info = precip_datasets[dataset_name]
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

#    (CAPE  )
#  CAPE, dPW, MFC, E 
all_var_labels = ['CAPE'] + var_labels  # CAPE  
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # CAPE(11.5), dPW(8.5), MFC(5.5), E(2.5)

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx] - 2.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold', rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  (5    )
ax.set_ylim(0., 11.5)
ax.set_yticks([])  # y  
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)

ax.set_yticklabels([])

ax2.set_ylim(0, 40)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10, ncol=1, bbox_to_anchor=(1.5, .75), frameon=False)

#    (    )
ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03, ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-10, 10,11))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25, ax_pos.y0, 0.03, ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(200,500,7))
cbar2.set_label('[J/kg]', fontsize=10)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 5    
fig, ax = plt.subplots(1, 1, figsize=(6, 7))

levels = np.linspace(-10, 10, 21)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(200,500,25)
cape_cmap = cmaps.cet_l_worb
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

dataset_names_ordered = ['MERRA-2','JRA-3Q',  'NARR','ERA5','IGRA']

datasets = {
    'MERRA-2': {'color': '#9467bd', 'data': fig_lev_ds[4]*8},
    'JRA-3Q': {'color': '#d62728', 'data': fig_lev_ds[3]*8},
    'NARR': {'color': '#2ca02c', 'data': fig_lev_ds[2]*8},
    'ERA5': {'color': '#ff7f0e', 'data': fig_lev_ds[1]*8},
    'IGRA': {'color': '#1f77b4', 'data': fig_lev_ds[0]*8},
}

precip_datasets = {
    'MERRA-2': {'color': '#440154', 'data': fig_rain_ds[4]*8},
    'JRA-3Q': {'color': '#31688e', 'data': fig_rain_ds[3]*8},  #  -
    'NARR': {'color': '#35b779', 'data': fig_rain_ds[2]*8},
    'ERA5': {'color': '#fde725', 'data': fig_rain_ds[1]*8},
    'IGRA': {'color': '#000000', 'data': fig_rain_ds[0]*8}
}


#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']

# 5   y  
sec = [8.5, 5.5, 2.5, 11.5]  # CAPE  (11.5) ,  
yticks = np.arange(0.,12.,.5)
lev_xds = [fig_lev_ds[4], fig_lev_ds[3],fig_lev_ds[2],fig_lev_ds[1],fig_lev_ds[0]]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # IGRA JRA-3Q     (3-hourly interval  )
    if ds_idx == 0 or ds_idx == 1 or ds_idx == 4:  # IGRA  JRA-3Q
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, dataset_name in enumerate(dataset_names_ordered):
        data = datasets[dataset_name]['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*5 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#    (5   )
for ds_idx, dataset_name in enumerate(dataset_names_ordered):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*5 + ds_idx * .5 + .25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

#     -  ,    
ax2 = ax.twinx()

#  1:     
alphas = [0.9, 0.7, 0.5, 0.3, 0.8]
for i, dataset_name in enumerate(dataset_names_ordered[::-1]):  #  IGRA 
    dataset_info = precip_datasets[dataset_name]
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2., alpha=0.8, label=dataset_name)


#    (CAPE  )
all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx] - 2.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold', rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y 
ax.set_ylim(0., 11.5)
ax.set_yticks([])
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)

ax.set_yticklabels([])

ax2.set_ylim(0, 40)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

#  -    
ax2.legend(loc='lower left', fontsize=10, ncol=1, bbox_to_anchor=(1.5, .75), frameon=False)

ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03, ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-10, 10, 11))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25, ax_pos.y0, 0.03, ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(200,500,7))
cbar2.set_label('[J/kg]', fontsize=10)

plt.show()

#### Based on Paper Figure


In [ ]:
for j in range(len(fig_rain_ds)):
    compare_lev_ds[j]['pre_eff'] = -compare_lev_ds[j]['pw_P']/(compare_lev_ds[j]['pw_E']+compare_lev_ds[j]['pw_res'])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']

#   - 2 5 
fig = plt.figure(figsize=((15/3)*5, 12))  #  2

# GridSpec  -   
#   (compare )
gs_bottom = GridSpec(2, 6, figure=fig,
                  top=0.95, bottom=0.55,
                  width_ratios=[1, 0.1, 1, 1, 1, 1],
                  height_ratios=[4.5, 1],  # :PW = 4.5:1
                  hspace=0.0,  # -PW  
                  wspace=0.2)

#   (fig )
gs_top= GridSpec(2, 6, figure=fig,
                     top=0.47, bottom=0.08,
                     width_ratios=[1, 0.1, 1, 1, 1, 1],
                     height_ratios=[4.5, 1],  # :PW = 4.5:1
                     hspace=0.0,  # -PW  
                     wspace=0.2)

mse_cmap_1 = 'RdBu_r'  #    MSE 
mse_cmap_2 = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
labels = ['(f) IGRA SGP Obs', '(g) ERA5', '(h) NARR', '(i) JRA-3Q', '(j) MERRA-2','(a) IGRA SGP Obs', '(b) ERA5', '(c) NARR', '(d) JRA-3Q', '(e) MERRA-2']

# GridSpec   
fig_idx = [0, 2, 3, 4, 5]

common_levels_1 = np.linspace(-6, 6, 25)
obs_levels = np.linspace(328, 340, 31)
diff_levels = np.linspace(-3, 3, 13)

six_hourly = [0, 6, 12, 18, 24]
obs_diur = fig_lev_ds[0]
obs_rain = fig_rain_ds[0]
obs_hours = obs_diur.hour.values
obs_mse = obs_diur.mse.values.copy()

CS0_row1 = None
CS_obs = None
CS_diff = None
axis1 = []
axis2 = []

# ===========================    (compare ) ===========================
for i, dataset_name in enumerate(datasets):
    
    #   (  )
    ax1 = fig.add_subplot(gs_top[0, fig_idx[i]])
    axis1.append(ax1)
    current_diur = compare_lev_ds[i]
    current_rain = compare_rain_ds[i]
    
    # MSE  
    mse_data = current_diur.mse.values.copy()
    CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data.T, 
                             levels=common_levels_1, cmap=mse_cmap_1, alpha=0.9, extend='both', zorder=0)
    
    if i == 0:
        CS0_row1 = CS_current
    
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])
    ax1.text(0., 395, labels[i], fontsize=16, ha='left', va='bottom')
    
    # y 
    if i == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False)
    
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    if i == 0:
        ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
               width=bar_width, align='edge', color=tp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
    else:
        ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
               width=bar_width, align='edge', color=tp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
        ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
               width=bar_width, align='edge', color=cp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
        ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
               width=bar_width, align='edge', color=lp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
    
    ax2.set_ylim(0, 6)
    if i == 4:
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE/CIN 
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
            color=cape_color, linewidth=2.5, marker='o', markersize=7, 
            markeredgecolor='black', markeredgewidth=0.5, zorder=3)
    ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
            color=cin_color, linewidth=2.5, marker='s', markersize=7, 
            markeredgecolor='black', markeredgewidth=0.5, zorder=3)
    
    ax3.set_ylim(-100, 400)
    if i == 4:
        ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)
    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)
    
    # ax1.set_title(dataset_name, fontsize=18)
    
    # PW  (  )
    ax_pw = fig.add_subplot(gs_top[1, fig_idx[i]])
    
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks(np.arange(0, 25, 3))
    ax_pw.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_pw.spines['left'].set_visible(False) 
    ax_pw.set_xlabel('LST [hour]', fontsize=14)
    
    ax_pw_right = ax_pw.twinx()
    pw_info = compare_lev_ds[i]
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency 
    for hour, value in zip(hours, values):
        if value > 0:
            ax_pw_right.bar(hour+1.5, value, width=bar_width*3, 
                           color='blue', alpha=0.3, edgecolor='darkblue', 
                           linewidth=0.8, zorder=1)
        else:
            ax_pw_right.bar(hour+1.5, value, width=bar_width*3,
                           color='red', alpha=0.3, edgecolor='darkred', 
                           linewidth=0.8, zorder=1)
    
    # PW  
    ax_pw_right.bar(hours+bar_s, pw_info['pw_P'], width=bar_width, 
                   align='edge', color='r', alpha=0.7, edgecolor='black', 
                   linewidth=0.5, zorder=3)
    ax_pw_right.bar(hours+bar_s+bar_width, pw_info['pw_E'], width=bar_width, 
                   align='edge', color='green', alpha=0.7, edgecolor='black', 
                   linewidth=0.5, zorder=3)
    ax_pw_right.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], width=bar_width, 
                   align='edge', color='blue', alpha=0.7, edgecolor='black', 
                   linewidth=0.5, zorder=3)
    
    ax_pw_right.plot(np.arange(0,25,3), [0]*9, c='black', ls='-', lw=0.3)
    ax_pw_right.set_ylim(-4.8, 4.8)
    ax_pw_right.plot(np.arange(0,25,3), [2]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_pw_right.plot(np.arange(0,25,3), [-2]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    #    PW 
        # ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        # ax2.spines['right'].set_visible(False)

    if i == 4:
        ax_pw_right.set_ylabel('[mm/3h]', fontsize=11)
    else : 
        ax_pw_right.tick_params(axis='y', which='both', right=True, labelright=False)
        ax_pw_right.spines['right'].set_visible(False)

# ===========================    (fig ) ===========================
for i, dataset_name in enumerate(datasets):
    
    #   (  )
    ax1 = fig.add_subplot(gs_bottom[0, fig_idx[i]])
    axis2.append(ax1)
    current_diur = fig_lev_ds[i]
    current_rain = fig_rain_ds[i]
    
    #  vs  
    if i == 0:
        mse_data = current_diur.mse.values.copy()
        mse_data_masked = np.ma.masked_invalid(mse_data)
        CS_current = ax1.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                                 levels=obs_levels, cmap=mse_cmap_2, alpha=0.9, extend='both', zorder=0)
        CS_obs = CS_current
    else:  #   ()
        model_hours = current_diur.hour.values
        model_six_hour_mask = np.isin(model_hours, six_hourly)
        
        if len(model_hours) > len(six_hourly):
            mse_model = current_diur.mse.values[model_six_hour_mask, :]
            model_hours_6h = model_hours[model_six_hour_mask]
        else:
            mse_model = current_diur.mse.values
            model_hours_6h = model_hours
        
        mse_diff = mse_model - obs_mse
        mse_diff_masked = np.ma.masked_invalid(mse_diff)
        CS_current = ax1.contourf(model_hours_6h, current_diur.pres, mse_diff_masked.T, 
                                 levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
        if CS_diff is None:
            CS_diff = CS_current
    
    ax1.set_yscale('log')
    ax1.set_ylim(925, 400)
    ax1.grid(True, ls='--', alpha=0.3)
    ax1.set_yticks([925, 850, 700, 500, 400])
    ax1.set_xlim(0, 24)
    ax1.set_xticks(np.arange(0, 24, 3))
    ax1.set_xticklabels([])
    ax1.text(0., 395, labels[i+5], fontsize=16, ha='left', va='bottom')
    # ax1.set_title(dataset_name, fontsize=18)
    # y 
    if i == 0:
        ax1.set_ylabel('Pressure [hPa]', fontsize=14)
    else:
        ax1.tick_params(axis='y', which='both', left=False, labelleft=False)
        ax1.spines['left'].set_visible(False)
    
    ax2 = ax1.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    if i == 0:
        ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
               width=bar_width, align='edge', color=tp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
    else:
        ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
               width=bar_width, align='edge', color=tp_color, alpha=0.7,
               edgecolor='black', linewidth=0.5, zorder=3)
        if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
            ax2.bar(current_rain.hour+bar_s + bar_width, current_rain.cp.values, 
                   width=bar_width, align='edge', color=cp_color, alpha=0.7,
                   edgecolor='black', linewidth=0.5, zorder=3)
            ax2.bar(current_rain.hour+bar_s + 2*bar_width, current_rain.lp.values, 
                   width=bar_width, align='edge', color=lp_color, alpha=0.7,
                   edgecolor='black', linewidth=0.5, zorder=3)
    
    ax2.set_ylim(0, 8)
    if i == 4:
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE/CIN 
    ax3 = ax1.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
            color=cape_color, linewidth=2.5, marker='o', markersize=7, 
            markeredgecolor='black', markeredgewidth=0.5, zorder=3)
    ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
            color=cin_color, linewidth=2.5, marker='s', markersize=7, 
            markeredgecolor='black', markeredgewidth=0.5, zorder=3)
    
    ax3.set_ylim(-200, 600)
    if i == 4:
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        # ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)

    else:
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)
    
    # PW  (  )
    ax_pw = fig.add_subplot(gs_bottom[1, fig_idx[i]])
    
    ax_pw.set_xlim(0, 24)
    ax_pw.set_xticks(np.arange(0, 25, 3))
    ax_pw.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_pw.spines['left'].set_visible(False) 
    # ax_pw.set_xlabel('LST [hour]', fontsize=14)
    
    ax_pw_right = ax_pw.twinx()
    
    if i == 0:
        pw_info = current_diur
        hours = pw_info.hour.values
        values = pw_info['pw_tend'].values
        
        for hour, value in zip(hours, values):
            if value > 0:
                ax_pw_right.bar(hour+1.5, value, width=bar_width*3, 
                               color='blue', alpha=0.3, edgecolor='darkblue', 
                               linewidth=0.8, zorder=1)
            else:
                ax_pw_right.bar(hour+1.5, value, width=bar_width*3,
                               color='red', alpha=0.3, edgecolor='darkred', 
                               linewidth=0.8, zorder=1)
        
        ax_pw_right.bar(hours+bar_s, pw_info['pw_P'], width=bar_width, 
                       align='edge', color='r', alpha=0.7, edgecolor='black', 
                       linewidth=0.5, zorder=3)
        ax_pw_right.bar(hours+bar_s+bar_width, pw_info['pw_E'], width=bar_width, 
                       align='edge', color='green', alpha=0.7, edgecolor='black', 
                       linewidth=0.5, zorder=3)
        ax_pw_right.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], width=bar_width, 
                       align='edge', color='blue', alpha=0.7, edgecolor='black', 
                       linewidth=0.5, zorder=3)
    else:  #   ()
        obs_pw = fig_lev_ds[0]
        current_pw = fig_lev_ds[i]
        # hours = obs_pw.hour.values
        hours = current_pw.hour.values
        
        # obs_values = obs_pw['pw_tend'].sel(hour=hours).values
        current_values = current_pw['pw_tend'].sel(hour=hours).values
        # pw_tend_diff = current_values - obs_values
        pw_tend_diff = current_values
        
        for hour, value in zip(hours, pw_tend_diff):
            if value > 0:
                ax_pw_right.bar(hour+1.5, value, width=bar_width*3, 
                               color='blue', alpha=0.3, edgecolor='darkblue', 
                               linewidth=0.8, zorder=1)
            else:
                ax_pw_right.bar(hour+1.5, value, width=bar_width*3,
                               color='red', alpha=0.3, edgecolor='darkred', 
                               linewidth=0.8, zorder=1)
                
        ax_pw_right.bar(hours+bar_s, current_pw['pw_P'].sel(hour=hours), 
                       width=bar_width, align='edge', color='r', alpha=0.7, 
                       edgecolor='black', linewidth=0.5, zorder=3)
        ax_pw_right.bar(hours+bar_s+bar_width, current_pw['pw_E'].sel(hour=hours), 
                       width=bar_width, align='edge', color='green', alpha=0.7, 
                       edgecolor='black', linewidth=0.5, zorder=3)
        ax_pw_right.bar(hours+bar_s+bar_width*2, current_pw['pw_res'].sel(hour=hours), 
                       width=bar_width, align='edge', color='blue', alpha=0.7, 
                       edgecolor='black', linewidth=0.5, zorder=3)
    
        # ax_pw_right.bar(hours+bar_s, (current_pw['pw_P'].sel(hour=hours)-obs_pw['pw_P']), 
        #                width=bar_width, align='edge', color='r', alpha=0.7, 
        #                edgecolor='black', linewidth=0.5, zorder=3)
        # ax_pw_right.bar(hours+bar_s+bar_width, (current_pw['pw_E'].sel(hour=hours)-obs_pw['pw_E']), 
        #                width=bar_width, align='edge', color='green', alpha=0.7, 
        #                edgecolor='black', linewidth=0.5, zorder=3)
        # ax_pw_right.bar(hours+bar_s+bar_width*2, (current_pw['pw_res'].sel(hour=hours)-obs_pw['pw_res']), 
        #                width=bar_width, align='edge', color='blue', alpha=0.7, 
        #                edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_pw_right.plot(np.arange(0,25,3), [0]*9, c='black', ls='-', lw=0.3)
    if i==0:
        ax_pw_right.plot(np.arange(0,25,3), [2.5]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
        ax_pw_right.plot(np.arange(0,25,3), [-2.5]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

        ax_pw_right.set_ylim(-4.8, 4.8)
    else : 
        ax_pw_right.plot(np.arange(0,25,3), [2]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
        ax_pw_right.plot(np.arange(0,25,3), [-2]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
        ax_pw_right.set_ylim(-4.8, 4.8)
        # ax_pw_right.set_ylim(-2.8, 2.8)
    if i == 0:
        handles = [
            mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                              edgecolor='black', linewidth=0.3),
            mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                              edgecolor='black', linewidth=0.3)
        ]
        ax_pw_right.legend(handles=handles, loc='lower right', ncol=3,
                          fontsize=10, frameon=False, bbox_to_anchor=(1, -0.1))
    elif i == 4:
        ax_pw_right.set_ylabel('[mm/3h]', fontsize=11)
    else:
        ax_pw_right.tick_params(axis='y', which='both', right=True, labelright=False)
        ax_pw_right.spines['right'].set_visible(False)


var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='lower left', ncol=1, frameon=False, 
          fontsize=14, bbox_to_anchor=(.93, 0.48))

pos_first = axis1[4].get_position()
cbar_ax1 = fig.add_axes([pos_first.x1 + 0.07, pos_first.y0, 0.015, pos_first.height])
cbar1 = fig.colorbar(CS0_row1, cax=cbar_ax1)
cbar1.set_label('$\Delta$MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar1.set_ticks(np.linspace(-6, 6, 7))

pos_obs = axis2[0].get_position()
cbar_ax_obs = fig.add_axes([pos_obs.x1 + 0.005, pos_obs.y0, 0.015, pos_obs.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(328, 340, 7))

pos_diff = axis2[4].get_position()
cbar_ax_diff = fig.add_axes([pos_diff.x1 + 0.07, pos_diff.y0, 0.015, pos_diff.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$MSE [kJ/kg]', fontsize=14, labelpad=20)
cbar_diff.set_ticks(np.linspace(-3, 3, 7))

plt.tight_layout(rect=[0, 0.05, 0.95, 0.97])
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/vertical_fig_obs.png', dpi=400, bbox_inches='tight')
# plt.show()

### skew_t


In [ ]:
fig_lev_ds[0]

In [ ]:
#fig_lev_ds[i]
#datasets = ['IGRA SGP Obs', 'ERA5', 'NARR', 'JRA-3Q', 'MERRA-2']


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_lev_ds[0].isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
fig.suptitle('IGRA',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
fig_lev_ds[1]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 2, 4, 6]):  # : 4  
    ax = axes[idx]
    now_lev = fig_lev_ds[1].isel(hour=hour)
    hour = fig_lev_ds[1].isel(hour=hour)['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('ERA5',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 2, 4, 6]):  # : 4  
    ax = axes[idx]
    now_lev = fig_lev_ds[2].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('NARR',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_lev_ds[3].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('JRA-3Q',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_lev_ds[4].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('MERRA-2',fontsize = 20) 
plt.tight_layout()
plt.show()

#### Dry Day


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_non_lev_ds[0].isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
fig.suptitle('IGRA',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 2, 4, 6]):  # : 4  
    ax = axes[idx]
    now_lev = fig_non_lev_ds[1].isel(hour=hour)
    hour = fig_non_lev_ds[1].isel(hour=hour)['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('ERA5',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 2, 4, 6]):  # : 4  
    ax = axes[idx]
    now_lev = fig_non_lev_ds[2].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('NARR',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_non_lev_ds[3].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('JRA-3Q',fontsize = 20) 
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = fig_non_lev_ds[4].isel(hour=hour)
    hour = now_lev['hour'].values
    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(925, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
    ax.set_title(f'Hour = {hour:02d} LST', fontsize=10)
fig.suptitle('MERRA-2',fontsize = 20) 
plt.tight_layout()
plt.show()

## 8. Vertical Profile - Reanalysis


### Updated Figure


In [ ]:
def extend_dataset_to_24h(ds):
    """
    Extend an xarray Dataset with hourly data (0, 6, 12, 18) to include hour 24
    by copying the data from hour 0.
    """
    # Create a new coordinate for hour that includes 24
    new_hour = np.append(ds.hour.values, 24)
    
    # Initialize dictionary to store extended variables
    extended_vars = {}
    
    # For each data variable in the dataset
    for var_name in ds.data_vars:
        var = ds[var_name]
        
        # Check if the variable has the 'hour' dimension
        if 'hour' in var.dims:
            # Get the variable values
            values = var.values
            
            # If the variable is 2D (e.g., temp, Td, etc. with dimensions (hour, pres))
            if var.ndim == 2:
                # Append the values from hour 0 to create hour 24
                extended_values = np.vstack([values, values[0:1]])
            # If the variable is 1D (e.g., pw, cape, cin with dimension (hour))
            elif var.ndim == 1:
                # Append the value from hour 0 to create hour 24
                extended_values = np.append(values, values[0])
            
            # Create a new DataArray with the extended values
            dims = var.dims
            coords = {dim: (ds[dim] if dim != 'hour' else new_hour) for dim in dims}
            extended_vars[var_name] = xr.DataArray(data=extended_values, dims=dims, coords=coords, 
                                                 attrs=var.attrs)
        else:
            # If the variable doesn't have the 'hour' dimension, keep it as is
            extended_vars[var_name] = var
    
    # Create a new dataset with the extended variables
    extended_ds = xr.Dataset(extended_vars, attrs=ds.attrs)
    
    # Update the hour coordinate
    extended_ds = extended_ds.assign_coords(hour=new_hour)
    
    return extended_ds

# Example usage:
# extended_ds = extend_dataset_to_24h(ds)
# Now you can plot with a complete 24-hour cycle

In [ ]:
jra_lev_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/use_lev/jra_lev_mean.nc')
jra_prec_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/use_lev/jra_prec_mean.nc')
jra_pw_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/use_lev/jra_pw_mean.nc')
jra_pw_E_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/use_lev/jra_pw_E_mean.nc')

era_lev_mean=xr.open_dataset('${DATA_DIR}/ERA5/use_lev/era_lev_mean.nc')
era_prec_mean=xr.open_dataset('${DATA_DIR}/ERA5/use_lev/era_prec_mean.nc')
era_pw_mean=xr.open_dataset('${DATA_DIR}/ERA5/use_lev/era_pw_mean.nc')

merra_lev_mean=xr.open_dataset('${DATA_DIR}/MERRA2/use_lev/merra_lev_mean.nc')
merra_prec_mean=xr.open_dataset('${DATA_DIR}/MERRA2/use_lev/merra_prec_mean.nc')
merra_pw_mean=xr.open_dataset('${DATA_DIR}/MERRA2/use_lev/merra_pw_mean.nc')

In [ ]:
cmo_prec_mean = xr.open_dataset('${DATA_DIR}/CMORPH/for_lev_fig/rainy/prec_mean_new1.nc')
cmo_prec_mean = extend_dataset_to_24h(cmo_prec_mean)

In [ ]:
jra_prec_mean = extend_dataset_to_24h(jra_prec_mean)
merra_prec_mean = extend_dataset_to_24h(merra_prec_mean)
era_prec_mean = extend_dataset_to_24h(era_prec_mean)

In [ ]:
def fill_with_previous_hour(ds):
    """fill missing hours with previous hour values
    03h -> use 00h value
    09h -> use 06h value  
    15h -> use 12h value
    21h -> use 18h value
    """
    
    result_data = {}
    
    for var_name in ds.data_vars:
        values = ds[var_name].values
        
        # New 8    
        new_values = np.zeros(8)
        
        #    (0,6,12,18 ->  0,2,4,6)
        new_values[0] = values[0]  # 0
        new_values[2] = values[1]  # 6
        new_values[4] = values[2]  # 12
        new_values[6] = values[3]  # 18
        
        new_values[1] = values[0]  # 3 = 0 
        new_values[3] = values[1]  # 9 = 6 
        new_values[5] = values[2]  # 15 = 12 
        new_values[7] = values[3]  # 21 = 18 
        
        result_data[var_name] = (('hour',), new_values)
    
    # New dataset 
    full_hours = np.arange(0, 24, 3)
    ds_filled = xr.Dataset(result_data, coords={'hour': full_hours})
    
    return ds_filled

In [ ]:
jra_pw_mean = fill_with_previous_hour(jra_pw_mean)

In [ ]:
jra_pw_mean['pw_E'] = jra_pw_E_mean['pw_E']

In [ ]:
era_prec_mean = era_prec_mean.rename({'lsp':'lp'})

In [ ]:
jra_pw_mean['pw_tend'] = jra_pw_mean['pw_E']+jra_pw_mean['pw_P']+jra_pw_mean['pw_res']

In [ ]:
jra_pw_mean['pw_tend'].plot()

In [ ]:
import numpy as np
import xarray as xr

def interpolate_to_3hourly(ds):
    """
    6-hourly (0, 6, 12, 18) data expanded via spline interpolation to 
    3-hourly interval (0, 3, 6, 9, 12, 15, 18, 21)expanded.
    """
    # 1. New    (0, 3, 6, ..., 21)
    new_hours = np.arange(0, 24, 3)
    
    # 2. (Circular/Periodic)   24  0   
    # Diurnal cycle 0 24    .
    ds_cycle = ds.sel(hour=0)
    ds_cycle.coords['hour'] = 24
    ds_extended = xr.concat([ds, ds_cycle], dim='hour')
    
    # 3. Spline(Cubic)  
    # 'cubic' 4        (0, 6, 12, 18, 24  5 )
    ds_interpolated = ds_extended.interp(hour=new_hours, method='linear')
    
    return ds_interpolated



In [ ]:
merra_lev_diurnal_splin_interp = xr.concat([merra_lev_mean.assign_coords(hour=np.arange(-24,0,6)),
          merra_lev_mean,
          merra_lev_mean.assign_coords(hour=np.arange(24,48,6))],
          dim='hour').fillna(0).interp(hour=np.arange(0, 24, 3), method='cubic')#['cape'][-1].plot()

jra_lev_diurnal_splin_interp = xr.concat([jra_lev_mean.assign_coords(hour=np.arange(-24,0,6)),
          jra_lev_mean,
          jra_lev_mean.assign_coords(hour=np.arange(24,48,6))],
          dim='hour').fillna(0).interp(hour=np.arange(0, 24, 3), method='cubic')#['cape'][-1].plot()


In [ ]:
merra_lev_diurnal_splin_interp['cin'].plot()
merra_lev_mean['cin'].plot()

In [ ]:
jra_lev_diurnal_splin_interp['cin'].plot()
jra_lev_mean['cin'].plot()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [2, 6]},
                                        sharex=True)

# levels = np.linspace(-6, 6, 25) np.linspace(-5.5, 5.5, 21)
levels =np.arange(-5.5, 5.51, .5)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels  = np.arange(150,351,5) #np.linspace(150,350,31) #np.linspace(50,150,21)
# cape_cmap = cmaps.WhiteYellowOrangeRed#NCV_jet
cape_cmap = cmaps.redy3
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
# ls = ['solid','dotted','dashed'] 
ls = ['solid','solid','solid'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(3.5, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9,frameon=False, ncol=2)
# ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.2))

ax_precip.set_xlim(0, 24)  # 25.5  
# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  # CAPE  (8.0) 
yticks = np.arange(-1.0,7.5,.5)

lev_xds = [merra_lev_diurnal_splin_interp, era_lev_mean, jra_lev_diurnal_splin_interp]
# lev_xds =  [merra_lev_mean, era_lev_mean, jra_lev_mean]
# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 2):  # JRA-3Q, MERRA-2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, 0.41])  #   x ,  y , , 
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
# cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))
cbar1.set_ticks([-5,-2.5,0,2.5,5])

cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0+0.41, 0.03,0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(150,350,3))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/paper_fig/fig8_new.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [2, 6]},
                                        sharex=True)

#   (MFC, E ) ########################################################
levels =np.arange(-5.5, 5.51, .5)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE   ################################################################
cape_levels = np.arange(50.,251,10) #np.linspace(0.,200,31) #np.linspace(50,150,21)
cape_cmap = cmaps.WhiteYellowOrangeRed
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='max')

# CIN   (dPW  ) ###############################################
cin_levels = np.linspace(-60, -20, 31)  # CIN    
cin_cmap = cmaps.WhiteBlue_r  #  colormap  
cin_norm = BoundaryNorm(cin_levels, cin_cmap.N, extend='max')

# datasets = {
#     'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0, 24, 3)) * 8},
#     'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0, 24, 3)) * 8},
#     'NARR': {'color': '#1f77b4', 'data': narr_pw_mean * 8}
# }

# #  
# precip_datasets = {
#     'NARR': {'color': '#1f77b4', 'data': narr_prec_mean * 8},
#     'ERA5': {'color': '#2ca02c', 'data': era_prec_mean * 8},
#     'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean * 8}
# }
datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}



ls = ['solid', 'solid', 'solid']

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended * 8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'):
        bars_cp = ax_precip.bar(precip_data.hour + bar_s + bar_width * i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        print(i)
        bars_lp = ax_precip.bar(precip_data.hour + bar_s + bar_width * i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)
    i += 1

#    ################################################################
ax_precip.set_ylim(3.5, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9, frameon=False, ncol=2)
ax_precip.set_xlim(0, 24)
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()
ax_precip.yaxis.set_label_position("right")

# =============================================================
#  : 
# =============================================================

# MFC, E RdBu cmap 
variables = ['pw_res', 'pw_E']
var_labels = ['MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  #  : [dPW→CIN, MFC, E, CAPE]
yticks = np.arange(-1.0, 7.5, .5)

lev_xds = [merra_lev_diurnal_splin_interp, era_lev_mean, jra_lev_diurnal_splin_interp]

# ----- CAPE ( ,   ) -----
cape_var_idx = 3
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data['cape'].values
    
    if (ds_idx == 0) | (ds_idx == 2):
        for hour in np.arange(3, 24, 6):
            rect_bg = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                    facecolor='white', alpha=.7,
                                    edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
    
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            rect = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                 facecolor=color_intensity, alpha=1.,
                                 edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

# ----- CIN ( dPW , sec[0]=4.5) -----
cin_var_idx = 0
y_base_cin = sec[cin_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cin_values = lev_data['cin'].values
    
    if (ds_idx == 0) | (ds_idx == 2):
        for hour in np.arange(3, 24, 6):
            rect_bg = plt.Rectangle((hour, y_base_cin - .5 * 3 + ds_idx * .5), 3, .5,
                                    facecolor='white', alpha=.7,
                                    edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
    
    for i, (hour, cin_value) in enumerate(zip(hours, cin_values)):
        if not np.isnan(cin_value):
            color_intensity = cin_cmap(cin_norm(cin_value))
            rect = plt.Rectangle((hour, y_base_cin - .5 * 3 + ds_idx * .5), 3, .5,
                                 facecolor=color_intensity, alpha=1.,
                                 edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

# -----   (MFC, E) - RdBu cmap -----
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx + 1]  # sec[1]=MFC(2.5), sec[2]=E(0.5)
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                rect = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                     facecolor=color_intensity, alpha=1.,
                                     edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 2):  # CAPE + CIN + MFC + E = 4
        y_pos = sec[var_idx] - .5 * 3 + ds_idx * .5 + .25
        ax_heat.text(0 - .5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

#    ( : CAPE, CIN, MFC, E)
all_var_labels = ['CAPE', 'CIN', 'MFC', 'E']
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # 6.5, 4.5, 2.5, 0.5

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx] - 1.5 / 2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

ax_heat_pos = ax_heat.get_position()

#    (MFC, E  - RdBu)
cbar_ax1 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0, 0.03, 0.25])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))
cbar1.set_ticks([-5,-2.5,0,2.5,5])
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0 + 0.41, 0.03, 0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks([50,100,150,200,250])
cbar2.set_label('[J/kg]', fontsize=10)

#    (CIN) - CAPE MFC  
cbar_ax3 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0 + 0.41-0.16, 0.03, 0.16])
mappable3 = plt.cm.ScalarMappable(cmap=cin_cmap, norm=cin_norm)
mappable3.set_array([])
cbar3 = plt.colorbar(mappable3, cax=cbar_ax3, orientation='vertical', extend='both')
# cbar3.set_ticks(np.linspace(-60, 0, 5))
cbar3.set_ticks([-55,-40,-25])
cbar3.set_label('[J/kg]', fontsize=10)

plt.subplots_adjust(hspace=0)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/paper_fig/fig12_new.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [2, 6]},
                                        sharex=True)

#   (MFC, E ) ########################################################
levels =np.arange(-5.5, 5.51, .5)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE   ################################################################
cape_levels = np.arange(100.,301,10) #np.linspace(0.,200,31) #np.linspace(50,150,21)
cape_cmap = cmaps.WhiteYellowOrangeRed
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='max')

# CIN   (dPW  ) ###############################################
cin_levels = np.linspace(-60, -20, 31)  # CIN    
cin_cmap = cmaps.WhiteBlue_r  #  colormap  
cin_norm = BoundaryNorm(cin_levels, cin_cmap.N, extend='max')

# datasets = {
#     'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0, 24, 3)) * 8},
#     'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0, 24, 3)) * 8},
#     'NARR': {'color': '#1f77b4', 'data': narr_pw_mean * 8}
# }

# #  
# precip_datasets = {
#     'NARR': {'color': '#1f77b4', 'data': narr_prec_mean * 8},
#     'ERA5': {'color': '#2ca02c', 'data': era_prec_mean * 8},
#     'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean * 8}
# }
datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}



ls = ['solid', 'solid', 'solid']

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended * 8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'):
        bars_cp = ax_precip.bar(precip_data.hour + bar_s + bar_width * i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        # print(i)
        # bars_lp = ax_precip.bar(precip_data.hour + bar_s + bar_width * i, precip_data.lp.values, 
        #                  width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
        #                  edgecolor='black', linewidth=0.5)
    i += 1

#    ################################################################
ax_precip.set_ylim(0, 10)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9, frameon=False, ncol=2)
ax_precip.set_xlim(0, 24)
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.set_yticks(np.arange(1,11,3))
ax_precip.yaxis.tick_right()
ax_precip.yaxis.set_label_position("right")

# =============================================================
#  : 
# =============================================================

# MFC, E RdBu cmap 
variables = ['pw_res', 'pw_E']
var_labels = ['MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  #  : [dPW→CIN, MFC, E, CAPE]
yticks = np.arange(-1.0, 7.5, .5)

lev_xds = [merra_lev_diurnal_splin_interp, era_lev_mean, jra_lev_diurnal_splin_interp]

# ----- CAPE ( ,   ) -----
cape_var_idx = 3
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data['cape'].values
    
    if (ds_idx == 0) | (ds_idx == 2):
        for hour in np.arange(3, 24, 6):
            rect_bg = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                    facecolor='white', alpha=.7,
                                    edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
    
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            rect = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                 facecolor=color_intensity, alpha=1.,
                                 edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

# ----- CIN ( dPW , sec[0]=4.5) -----
cin_var_idx = 0
y_base_cin = sec[cin_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cin_values = lev_data['cin'].values
    
    if (ds_idx == 0) | (ds_idx == 2):
        for hour in np.arange(3, 24, 6):
            rect_bg = plt.Rectangle((hour, y_base_cin - .5 * 3 + ds_idx * .5), 3, .5,
                                    facecolor='white', alpha=.7,
                                    edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
    
    for i, (hour, cin_value) in enumerate(zip(hours, cin_values)):
        if not np.isnan(cin_value):
            color_intensity = cin_cmap(cin_norm(cin_value))
            rect = plt.Rectangle((hour, y_base_cin - .5 * 3 + ds_idx * .5), 3, .5,
                                 facecolor=color_intensity, alpha=1.,
                                 edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

# -----   (MFC, E) - RdBu cmap -----
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx + 1]  # sec[1]=MFC(2.5), sec[2]=E(0.5)
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                rect = plt.Rectangle((hour, y_base - .5 * 3 + ds_idx * .5), 3, .5,
                                     facecolor=color_intensity, alpha=1.,
                                     edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 2):  # CAPE + CIN + MFC + E = 4
        y_pos = sec[var_idx] - .5 * 3 + ds_idx * .5 + .25
        ax_heat.text(0 - .5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

#    ( : CAPE, CIN, MFC, E)
all_var_labels = ['CAPE', 'CIN', 'MFC', 'E']
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # 6.5, 4.5, 2.5, 0.5

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx] - 1.5 / 2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

ax_heat_pos = ax_heat.get_position()

#    (MFC, E  - RdBu)
cbar_ax1 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0, 0.03, 0.25])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))
cbar1.set_ticks([-5,-2.5,0,2.5,5])
cbar1.set_label('[mm/day]', fontsize=10)
cbar1.ax.minorticks_off()

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0 + 0.41, 0.03, 0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks([100,150,200,250,300])
cbar2.set_label('[J/kg]', fontsize=10)
cbar2.ax.minorticks_off()

#    (CIN) - CAPE MFC  
cbar_ax3 = fig.add_axes([ax_heat_pos.x1 + .02, ax_heat_pos.y0 + 0.41-0.16, 0.03, 0.16])
mappable3 = plt.cm.ScalarMappable(cmap=cin_cmap, norm=cin_norm)
mappable3.set_array([])
cbar3 = plt.colorbar(mappable3, cax=cbar_ax3, orientation='vertical', extend='both')
# cbar3.set_ticks(np.linspace(-60, 0, 5))
cbar3.set_ticks([-55,-40,-25])
cbar3.set_label('[J/kg]', fontsize=10)
cbar3.ax.minorticks_off()
plt.subplots_adjust(hspace=0)
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/fig8_new.png', dpi=400, bbox_inches='tight')
# plt.show()

In [ ]:
np.linspace(0.,250,6)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [2, 6]},
                                        sharex=True)

# levels = np.linspace(-6, 6, 25) np.linspace(-5.5, 5.5, 21)
levels =np.arange(-5.5, 5.51, .5)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.arange(150,351,10) #np.linspace(150,350,31) #np.linspace(50,150,21)
# cape_cmap = cmaps.WhiteYellowOrangeRed#NCV_jet
cape_cmap = cmaps.redy3
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
# ls = ['solid','dotted','dashed'] 
ls = ['solid','solid','solid'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(3.5, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9,frameon=False, ncol=2)
# ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.2))

ax_precip.set_xlim(0, 24)  # 25.5  
# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  # CAPE  (8.0) 
yticks = np.arange(-1.0,7.5,.5)

lev_xds = [merra_lev_diurnal_splin_interp, era_lev_mean, jra_lev_diurnal_splin_interp]
# lev_xds =  [merra_lev_mean, era_lev_mean, jra_lev_mean]
# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 2):  # JRA-3Q, MERRA-2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, 0.41])  #   x ,  y , , 
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
# cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))
cbar1.set_ticks([-5,-2.5,0,2.5,5])

cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0+0.41, 0.03,0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(150,350,3))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/paper_fig/fig8_new.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [2, 6]},
                                        sharex=True)

# levels = np.linspace(-6, 6, 25) np.linspace(-5.5, 5.5, 21)
levels =np.arange(-5.5, 5.51, .5)
cmap = cmaps.vik_r

norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.arange(150,351,10) #np.linspace(150,350,31) #np.linspace(50,150,21)
# cape_cmap = cmaps.WhiteYellowOrangeRed#NCV_jet
cape_cmap = cmaps.matter
# cmaps.vik
# cape_cmap = cmaps.redy3
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
# ls = ['solid','dotted','dashed'] 
ls = ['solid','solid','solid'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(3.5, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9,frameon=False, ncol=2)
# ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.2))

ax_precip.set_xlim(0, 24)  # 25.5  
# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  # CAPE  (8.0) 
yticks = np.arange(-1.0,7.5,.5)

lev_xds = [merra_lev_mean, era_lev_mean, jra_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 2):  # JRA-3Q, MERRA-2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, 0.41])  #   x ,  y , , 
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
# cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))
cbar1.set_ticks([-5,-2.5,0,2.5,5])

cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0+0.41, 0.03,0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(150,350,3))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/pknu_paper/fig8.png', dpi=400, bbox_inches='tight')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [1, 6]},
                                        sharex=True)

levels = np.linspace(-6, 6, 25)
# levels = np.linspace(-5, 5, 21)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(100,250,31) #np.linspace(50,150,21)
cape_cmap = cmaps.NCV_jet
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_mean.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_mean*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
hour_extended = np.concatenate([[-1.5], cmo_prec_mean.hour + 1.5])
precip_extended = np.concatenate([[cmo_prec_mean.tp.values[-2]], cmo_prec_mean.tp.values])

ax_precip.plot(hour_extended, precip_extended*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    hour_extended = np.concatenate([[-1.5], precip_data.hour + 1.5])
    precip_extended = np.concatenate([[precip_data.tp.values[-2]], precip_data.tp.values])
    
    ax_precip.plot(hour_extended, precip_extended, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(2.5, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.2))
ax_precip.set_xlim(0, 24)  # 25.5  
# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  # CAPE  (8.0) 
yticks = np.arange(-1.0,7.,.5)

lev_xds = [merra_lev_mean, era_lev_mean, jra_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 2):  # JRA-3Q, MERRA-2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, 0.49])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
# cbar1.set_ticks(np.linspace(-5.5, 5.5, 5))

cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.02, 0.6, 0.03,0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(100,250,4))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new_paper_reanalysis_v3.png', dpi=400, bbox_inches='tight')
plt.show()

### Rainy Days


In [ ]:
jra_lev_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
jra_prec_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
jra_pw_tend=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

era_lev_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
era_prec_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
era_pw_tend=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

merra_lev_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/lev_mean_new.nc')
merra_prec_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
# merra_pw_tend=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean_new.nc')#6hourly
merra_pw_tend=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean_new_3hourly.nc')#3hourly

In [ ]:
merra_pw_tend_new=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean_new_3hourly.nc')#3hourly
merra_pw_tend_old=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')#6hourly

In [ ]:
merra_pw_tend.close()
del(merra_pw_tend)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

fig = plt.figure(figsize=(15, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.3)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.3)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(314, 326, 25)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]


#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef,lev_ds,prec_ds,pw_ds,title,position, dataset_idx=0, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.set_ylim(0, 2)
    else:
        ax2.set_ylim(0, 2)
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-1, 1)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1,jra_lev_mean,jra_prec_mean,jra_pw_tend,'JRA3Q', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_obs = create_obs_plot(ax2, ax_sm_ef2,merra_lev_mean,merra_prec_mean,merra_pw_tend.sel(hour = np.arange(0,24,3)),'MERRA2', 'mid')

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_era = create_obs_plot(ax3, ax_sm_ef3,era_lev_mean.sel(hour = np.arange(0,25,3)),era_prec_mean,era_pw_tend.sel(hour = np.arange(0,24,3)),'ERA5', 'right')

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='|CIN|'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=3, frameon=False, fontsize=14, bbox_to_anchor=(1.05, 0.63))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos2.x1+.1, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(314, 326, 7))


labels = ['(a)','(b)','(c)']
axes = [ax1,ax2,ax3]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

fig = plt.figure(figsize=(15, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.3)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.3)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(314, 326, 25)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]


#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef,lev_ds,prec_ds,pw_ds,title,position, dataset_idx=0, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.set_ylim(0, 2)
    else:
        ax2.set_ylim(0, 2)
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-1, 1)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1,jra_lev_mean,jra_prec_mean,jra_pw_tend,'JRA3Q', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_obs = create_obs_plot(ax2, ax_sm_ef2,merra_lev_mean,merra_prec_mean,merra_pw_tend.sel(hour = np.arange(0,24,3)),'MERRA2', 'mid')

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_era = create_obs_plot(ax3, ax_sm_ef3,era_lev_mean.sel(hour = np.arange(0,25,3)),era_prec_mean,era_pw_tend.sel(hour = np.arange(0,24,3)),'ERA5', 'right')

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='|CIN|'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=3, frameon=False, fontsize=14, bbox_to_anchor=(1.05, 0.63))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos2.x1+.1, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(314, 326, 7))


labels = ['(a)','(b)','(c)']
axes = [ax1,ax2,ax3]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
plt.show()

In [ ]:
cmo_prec_mean = xr.open_dataset('${DATA_DIR}/CMORPH/for_lev_fig/rainy/prec_mean_new.nc')
cmo_prec_mean = extend_dataset_to_24h(cmo_prec_mean)

In [ ]:
jra_prec_mean = extend_dataset_to_24h(jra_prec_mean)
merra_prec_mean = extend_dataset_to_24h(merra_prec_mean)
era_prec_mean = extend_dataset_to_24h(era_prec_mean)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm

fig, ax = plt.subplots(1, 1, figsize=(6, 4))

levels = np.linspace(-6, 6, 13)  # -5 5 11 
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8},
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8}
}

precip_datasets = {
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [6.5,4.,1.5]

for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        # JRA-3Q    
        if dataset_name == 'JRA-3Q':
            for hour in np.arange(3,24,6):
                rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                      facecolor='white', alpha=.7,
                                      edgecolor='gray', hatch='///', linewidth=0.5)
                ax.add_patch(rect_bg)
                
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

for ds_idx, dataset_name in enumerate(datasets.keys()):
    for var_idx in range(len(variables)):
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

#    (    )
for var_idx, var_label in enumerate(var_labels):
    y_pos = sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  ( )
ax.set_ylim(0., 6.5)
ax.set_yticks([])  # y  
# ax.set_yticks([0,1.5, 4.0,4.0-1.5, 6.5,6.5-1.5])  #  y
ax.set_yticks([0,.5,1.,1.5, 2.5,3.,3.5,4.,5.,5.5,6.,6.5])  #  y

ax.grid(True, alpha=0.5, axis='y')  # y 
ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=3, bbox_to_anchor=(0., .64),frameon=False)

ax_pos = ax.get_position()
cbar_ax = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03, ax_pos.y1-ax_pos.y0])

#   (BoundaryNorm )
mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable.set_array([])  #  !
cbar = plt.colorbar(mappable, cax=cbar_ax, orientation='vertical', extend='both')
cbar.set_label('[mm/day]', fontsize=12)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm

fig, ax = plt.subplots(1, 1, figsize=(6, 4))

levels = np.linspace(-6, 6, 13)  # -5 5 11 
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8},
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8}
}

precip_datasets = {
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [6.5,4.,1.5]

for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        # JRA-3Q    
        if dataset_name == 'JRA-3Q':
            for hour in np.arange(3,24,6):
                rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                      facecolor='white', alpha=.7,
                                      edgecolor='gray', hatch='///', linewidth=0.5)
                ax.add_patch(rect_bg)
                
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

for ds_idx, dataset_name in enumerate(datasets.keys()):
    for var_idx in range(len(variables)):
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

#    (    )
for var_idx, var_label in enumerate(var_labels):
    y_pos = sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  ( )
ax.set_ylim(0., 6.5)
ax.set_yticks([])  # y  
# ax.set_yticks([0,1.5, 4.0,4.0-1.5, 6.5,6.5-1.5])  #  y
ax.set_yticks([0,.5,1.,1.5, 2.5,3.,3.5,4.,5.,5.5,6.,6.5])  #  y

ax.grid(True, alpha=0.5, axis='y')  # y 
ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=3, bbox_to_anchor=(0., .64),frameon=False)

ax_pos = ax.get_position()
cbar_ax = fig.add_axes([ax_pos.x0, -0.04, ax_pos.x1-ax_pos.x0, 0.03])

#   (BoundaryNorm )
mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable.set_array([])  #  !
cbar = plt.colorbar(mappable, cax=cbar_ax, orientation='horizontal', extend='both')
cbar.set_label('[mm/day]', fontsize=12)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm

fig, ax = plt.subplots(1, 1, figsize=(6, 4))

levels = np.linspace(-6, 6, 13)  # -5 5 11 
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8},
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8}
}

precip_datasets = {
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [6.5,4.,1.5]
lev_xds = [jra_lev_mean,merra_lev_mean,era_lev_mean]
pw_xds = [jra_pw_tend,merra_pw_tend,era_pw_tend]

for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        # JRA-3Q    
        if dataset_name == 'JRA-3Q':
            for hour in np.arange(3,24,6):
                rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                      facecolor='white', alpha=.7,
                                      edgecolor='gray', hatch='///', linewidth=0.5)
                ax.add_patch(rect_bg)
                
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

for ds_idx, dataset_name in enumerate(datasets.keys()):
    for var_idx in range(len(variables)):
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

ax3 = ax.twinx()
ax3.spines['right'].set_position(('outward', 50))
cape_color = '#F21B1B'

# CAPE CIN  
for i in range(3):
    
    cape_line = ax3.plot(lev_xds[i].hour, lev_xds[i].cape.values,
                        color=cape_color, linewidth=2.5, marker='o', markersize=4, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3,ls = ls[i],alpha = .5)
    # cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
    #                    color=cin_color, linewidth=2.5, marker='s', markersize=7, 
    #                    markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)

# CAPE/CIN  
# if position == 'right':
ax3.set_ylabel('CAPE [J/kg]', fontsize=12,c = cape_color)
ax3.set_ylim(0, 250)



ax4 = ax.twinx()
ax4.spines['right'].set_position(('outward', 100))
chi_color = 'green'

# CAPE CIN  
for i in range(3):
    
    chi_line = ax4.plot(pw_xds[i].hour, pw_xds[i].chi.values,
                        color=chi_color, linewidth=2.5, marker='o', markersize=3, 
                        markeredgecolor='black', markeredgewidth=0.5, label='$\chi$', zorder=3,ls = ls[i],alpha = .5)
    # cin_line = ax4.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
    #                    color=cin_color, linewidth=2.5, marker='s', markersize=7, 
    #                    markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)

# CAPE/CIN  
# if position == 'right':
ax4.set_ylabel('$\chi$ [-]', fontsize=12,c = chi_color)
ax4.set_ylim(0, .2)

# else :
    # ax3.set_ylim(0, 250)
    # ax3.tick_params(axis='y', which='both', right=False, labelright=False)
    # ax3.spines['right'].set_visible(False)



#    (    )
for var_idx, var_label in enumerate(var_labels):
    y_pos = sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  ( )
ax.set_ylim(0., 6.5)
ax.set_yticks([])  # y  
# ax.set_yticks([0,1.5, 4.0,4.0-1.5, 6.5,6.5-1.5])  #  y
ax.set_yticks([0,.5,1.,1.5, 2.5,3.,3.5,4.,5.,5.5,6.,6.5])  #  y

ax.grid(True, alpha=0.5, axis='y')  # y 
ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=3, bbox_to_anchor=(0., .64),frameon=False)

ax_pos = ax.get_position()
cbar_ax = fig.add_axes([ax_pos.x0, -0.04, ax_pos.x1-ax_pos.x0, 0.03])

#   (BoundaryNorm )
mappable = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable.set_array([])  #  !
cbar = plt.colorbar(mappable, cax=cbar_ax, orientation='horizontal', extend='both')
cbar.set_label('[mm/day]', fontsize=12)

plt.show()

In [ ]:
def fill_with_previous_hour(ds):
    """fill missing hours with previous hour values
    03h -> use 00h value
    09h -> use 06h value  
    15h -> use 12h value
    21h -> use 18h value
    """
    
    result_data = {}
    
    for var_name in ds.data_vars:
        values = ds[var_name].values
        
        # New 8    
        new_values = np.zeros(8)
        
        #    (0,6,12,18 ->  0,2,4,6)
        new_values[0] = values[0]  # 0
        new_values[2] = values[1]  # 6
        new_values[4] = values[2]  # 12
        new_values[6] = values[3]  # 18
        
        new_values[1] = values[0]  # 3 = 0 
        new_values[3] = values[1]  # 9 = 6 
        new_values[5] = values[2]  # 15 = 12 
        new_values[7] = values[3]  # 21 = 18 
        
        result_data[var_name] = (('hour',), new_values)
    
    # New dataset 
    full_hours = np.arange(0, 24, 3)
    ds_filled = xr.Dataset(result_data, coords={'hour': full_hours})
    
    return ds_filled

In [ ]:
jra_pw_tend = fill_with_previous_hour(jra_pw_tend)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps
fig, ax = plt.subplots(1, 1, figsize=(6, 5))

levels = np.linspace(-6, 6, 25)  # -6 6 13 
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE   ( )
cape_levels = np.linspace(60,200,29)  # 0 250
cape_cmap = cmaps.cet_l_worb  # CAPE 
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8},
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8}
}

precip_datasets = {
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

#   (CAPE )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [8.0, 5.5, 3.0, 0.5]  # CAPE    ( )
yticks = [-1.0,-0.5,0.0,0.5,1.5,2.0,2.5,3.0,4.0,4.5,5.0,5.5,6.5,7.0,7.5,8.]
lev_xds = [jra_lev_mean, merra_lev_mean, era_lev_mean]
pw_xds = [jra_pw_tend, merra_pw_tend, era_pw_tend]

for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y   (CAPE   )
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

# CAPE  New row 
cape_var_idx = 3  #    ( 3)
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 1):  # JRA-3Q
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#    (CAPE )
dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1
# cmo_prec_mean
ax2.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
        color='black', linewidth=2, ls='-',
        label='CMORPH')

#    (CAPE )
all_var_labels = var_labels + ['CAPE']  # CAPE  
for var_idx, var_label in enumerate(all_var_labels):
    y_pos = sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)

ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  (CAPE   )
ax.set_ylim(-1., 8.)
ax.set_yticks([])  # y  
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5,alpha = .7)

# ax.grid(True, alpha=0.5, axis='y')  # y 
ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=12)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=1, bbox_to_anchor=(1.5, .75),frameon=False)

#    (    )
ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25,ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(60,200,8))
cbar2.set_label('[J/kg]', fontsize=10)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

levels = np.linspace(-6, 6, 25)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(60,200,29)
cape_cmap = cmaps.cet_l_worb
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8}
}

precip_datasets = {
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8},
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
# CAPE  (8.0),   dPW(5.5), MFC(3.0), E(0.5)
sec = [5.5, 3.0, 0.5, 8.0]  # CAPE  (8.0) 
yticks = [-1.0,-0.5,0.0,0.5,1.5,2.0,2.5,3.0,4.0,4.5,5.0,5.5,6.5,7.0,7.5,8.]
lev_xds = [merra_lev_mean, jra_lev_mean,  era_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 1):  # JRA-3Q, MERRA2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

#    (  )
dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

# CMORPH 
ax2.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
        color='black', linewidth=2, ls='-',
        label='CMORPH')

#    (CAPE  )
#  CAPE, dPW, MFC, E 
all_var_labels = ['CAPE'] + var_labels  # CAPE  
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # CAPE(8.0), dPW(5.5), MFC(3.0), E(0.5)

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right',rotation=90)
    # ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  (CAPE   )
ax.set_ylim(-1., 8.)
ax.set_yticks([])  # y  
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5,alpha = .7)

ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=10)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=1, bbox_to_anchor=(1.5, .75),frameon=False)

#    (    )
ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25,ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(60,200,8))
cbar2.set_label('[J/kg]', fontsize=10)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new_paper_reanalysis.png', dpi=400, bbox_inches='tight')

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

fig, ax = plt.subplots(1, 1, figsize=(6, 5))

levels = np.linspace(-6, 6, 25)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(50,150,21)
cape_cmap = cmaps.NCV_jet
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8}

}

precip_datasets = {
    'JRA-3Q': {'color': 'gray', 'data': jra_prec_mean*8},
    'ERA5': {'color': 'gray', 'data': era_prec_mean*8},
    'MERRA2': {'color': 'gray', 'data': merra_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
# CAPE  (8.0),   dPW(5.5), MFC(3.0), E(0.5)
sec = [5.5, 3.0, 0.5, 8.0]  # CAPE  (8.0) 
yticks = [-1.0,-0.5,0.0,0.5,1.5,2.0,2.5,3.0,4.0,4.5,5.0,5.5,6.5,7.0,7.5,8.]
lev_xds = [merra_lev_mean, jra_lev_mean,  era_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    
    # CAPE 
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 1):  # JRA-3Q, MERRA2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            # CAPE  norm 
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    # y  
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        
        hours = data.hour.values
        
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                # BoundaryNorm   
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax.add_patch(rect)

#    (  )
dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

ax2 = ax.twinx()
ax2.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
        color='black', linewidth=2, ls='-',
        label='CMORPH')

i=0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax2.plot(precip_data.hour, precip_data.tp.values, 
            color=color, linewidth=2, ls=ls[i],
            label=dataset_name)
    i=i+1

#    (CAPE  )
#  CAPE, dPW, MFC, E 
all_var_labels = ['CAPE'] + var_labels  # CAPE  
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]  # CAPE(8.0), dPW(5.5), MFC(3.0), E(0.5)

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right',rotation=90)
    # ax.text(-4, y_pos, var_label, fontsize=12, va='center', ha='right', weight='bold',rotation=90)
ax.set_xlim(0, 24)
ax.set_xticks(np.arange(0, 25, 3))
ax.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])

# y  (CAPE   )
ax.set_ylim(-1., 8.)
ax.set_yticks([])  # y  
ax.set_yticks(yticks)
for i in range(len(yticks)):
    ax.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5,alpha = .7)

ax.set_yticklabels([])

ax2.set_ylim(0, 12)
ax2.set_ylabel('precipitation [mm/day]', fontsize=10)

ax.set_xlabel('[LST]', fontsize=12)

ax2.legend(loc='lower left', fontsize=10,ncol=1, bbox_to_anchor=(1.5, .75),frameon=False)

#    (    )
ax_pos = ax.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_pos.x1+.1, ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_pos.x1+.25,ax_pos.y0, 0.03,ax_pos.y1-ax_pos.y0])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(50,150,5))
cbar2.set_label('[J/kg]', fontsize=10)
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new_paper_reanalysis_v1.png', dpi=400, bbox_inches='tight')

# plt.show()

In [ ]:
jra_prec_mean

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [1, 6]},
                                        sharex=True)

levels = np.linspace(-6, 6, 25)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(50,150,21)
cape_cmap = cmaps.NCV_jet
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
ax_precip.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax_precip.plot(precip_data.hour, precip_data.tp.values, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        # bars_lp = ax2.bar(precip_data.hour+bar_s+2*bar_width, precip_data.lp.values, 
        #                  width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
        #                  edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(2, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.1))

# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [5.5, 3.0, 0.5, 8.0]  # CAPE  (8.0) 
yticks = [-1.0,-0.5,0.0,0.5,1.5,2.0,2.5,3.0,4.0,4.5,5.0,5.5,6.5,7.0,7.5,8.]
lev_xds = [merra_lev_mean, jra_lev_mean, era_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 1):  # JRA-3Q, MERRA2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 8.)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, ax_heat_pos.height])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.1, ax_heat_pos.y0, 0.03, ax_heat_pos.height])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(50,150,5))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 

# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new_paper_reanalysis_v2.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as colors
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm
import colormaps as cmaps

#   - 2  (: , : )
fig, (ax_precip, ax_heat) = plt.subplots(2, 1, figsize=(6, 6), 
                                        gridspec_kw={'height_ratios': [1, 6]},
                                        sharex=True)

levels = np.linspace(-6, 6, 25)
cmap = plt.cm.RdBu
norm = BoundaryNorm(levels, cmap.N, extend='both')

# CAPE  
cape_levels = np.linspace(50,150,21)
cape_cmap = cmaps.NCV_jet
cape_norm = BoundaryNorm(cape_levels, cape_cmap.N, extend='both')

datasets = {
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'ERA5': {'color': '#2ca02c', 'data': era_pw_tend.sel(hour=np.arange(0,24,3))*8},
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_pw_tend*8}
}

precip_datasets = {
    'JRA-3Q': {'color': '#1f77b4', 'data': jra_prec_mean*8},
    'ERA5': {'color': '#2ca02c', 'data': era_prec_mean*8},
    'MERRA-2': {'color': '#ff7f0e', 'data': merra_prec_mean*8}
}
ls = ['solid','dotted','dashed'] 

# =============================================================
#  :  
# =============================================================

# CMORPH  ()
ax_precip.plot(cmo_prec_mean.hour, cmo_prec_mean.tp.values*8, 
               color='black', linewidth=2, ls='-', label='CMORPH')

i = 0
for dataset_name, dataset_info in precip_datasets.items():
    precip_data = dataset_info['data']
    color = dataset_info['color']
    
    ax_precip.plot(precip_data.hour, precip_data.tp.values, 
                   color=color, linewidth=2, ls=ls[i], label=dataset_name)
    bar_width = 0.6
    bar_s = 0.6
    
    if hasattr(precip_data, 'cp'): #and hasattr(precip_data, 'lp'):
        bars_cp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.cp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.7, 
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax_precip.bar(precip_data.hour+bar_s+bar_width*i, precip_data.lp.values, 
                         width=bar_width, align='edge', color=color, alpha=0.3, hatch='////',
                         edgecolor='black', linewidth=0.5)

    i += 1

ax_precip.set_ylim(2, 8)
ax_precip.set_ylabel('[mm/day]', fontsize=10)
ax_precip.grid(True, alpha=0.3)
ax_precip.legend(fontsize=9, ncol=1, frameon=False,bbox_to_anchor=(1.4, 1.2))

# x   (  )
ax_precip.set_xticks(np.arange(0, 25, 3))
ax_precip.set_xticklabels([])
ax_precip.yaxis.tick_right()  # y  
ax_precip.yaxis.set_label_position("right")  # y  

# =============================================================
#  :  
# =============================================================

#   (CAPE  )
variables = ['pw_tend', 'pw_res', 'pw_E']
var_labels = ['dPW', 'MFC', 'E']
sec = [4.5, 2.5, 0.5, 6.5]  # CAPE  (8.0) 
yticks = np.arange(-1.0,7.,.5)

lev_xds = [merra_lev_mean, era_lev_mean, jra_lev_mean]

# CAPE  New row  ( )
cape_var_idx = 3  #    ( )
y_base = sec[cape_var_idx]

for ds_idx, lev_data in enumerate(lev_xds):
    hours = lev_data.hour.values
    cape_values = lev_data.cape.values
    
    # JRA-3Q    
    if (ds_idx == 0)|(ds_idx == 2):  # JRA-3Q, MERRA-2
        for hour in np.arange(3,24,6):
            rect_bg = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                  facecolor='white', alpha=.7,
                                  edgecolor='gray', hatch='///', linewidth=0.5)
            ax_heat.add_patch(rect_bg)
            
    #   CAPE     
    for i, (hour, cape_value) in enumerate(zip(hours, cape_values)):
        if not np.isnan(cape_value):
            color_intensity = cape_cmap(cape_norm(cape_value))
            
            rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                               facecolor=color_intensity, alpha=1.,
                               edgecolor='none', linewidth=0.5)
            ax_heat.add_patch(rect)

#   (dPW, MFC, E)
for var_idx, (var, var_label) in enumerate(zip(variables, var_labels)):
    y_base = sec[var_idx]
    
    for ds_idx, (dataset_name, dataset_info) in enumerate(datasets.items()):
        data = dataset_info['data']
        hours = data.hour.values
        values = data[var].values
        
        for i, (hour, value) in enumerate(zip(hours, values)):
            if not np.isnan(value):
                color_intensity = cmap(norm(value))
                
                rect = plt.Rectangle((hour, y_base - .5*3 + ds_idx * .5), 3, .5,
                                   facecolor=color_intensity, alpha=1.,
                                   edgecolor='none', linewidth=0.5)
                ax_heat.add_patch(rect)

dataset_names = list(datasets.keys())
for ds_idx, dataset_name in enumerate(dataset_names):
    for var_idx in range(len(variables) + 1):  # CAPE  +1
        y_pos = sec[var_idx] - .5*3 + ds_idx * .5+.25
        ax_heat.text(0-.5, y_pos, dataset_name, fontsize=10, va='center', ha='right')

all_var_labels = ['CAPE'] + var_labels
reordered_sec = [sec[3], sec[0], sec[1], sec[2]]

for var_idx, var_label in enumerate(all_var_labels):
    y_pos = reordered_sec[var_idx]-1.5/2
    ax_heat.text(-4, y_pos, var_label, fontsize=14, va='center', ha='right', rotation=90)

ax_heat.set_xlim(0, 24)
ax_heat.set_xticks(np.arange(0, 25, 3))
ax_heat.set_xticklabels([f'{h:02d}' for h in np.arange(0, 25, 3)])
ax_heat.set_xlabel('[LST]', fontsize=12)

ax_heat.set_ylim(-1., 6.5)
ax_heat.set_yticks(yticks)
for i in range(len(yticks)):
    ax_heat.axhline(yticks[i], color='gray', linestyle='-', linewidth=0.5, alpha=.7)
ax_heat.set_yticklabels([])

fig_pos = fig.get_window_extent()
ax_heat_pos = ax_heat.get_position()

#    ( )
cbar_ax1 = fig.add_axes([ax_heat_pos.x1+.02, ax_heat_pos.y0, 0.03, 0.49])
mappable1 = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
mappable1.set_array([])
cbar1 = plt.colorbar(mappable1, cax=cbar_ax1, orientation='vertical', extend='both')
cbar1.set_ticks(np.linspace(-6, 6, 7))
cbar1.set_label('[mm/day]', fontsize=10)

#    (CAPE)
cbar_ax2 = fig.add_axes([ax_heat_pos.x1+.02, 0.6, 0.03,0.16])
mappable2 = plt.cm.ScalarMappable(cmap=cape_cmap, norm=cape_norm)
mappable2.set_array([])
cbar2 = plt.colorbar(mappable2, cax=cbar_ax2, orientation='vertical', extend='both')
cbar2.set_ticks(np.linspace(50,150,5))
cbar2.set_label('[J/kg]', fontsize=10)

# plt.tight_layout()
plt.subplots_adjust(hspace=0)  #  0 

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new_paper_reanalysis_v2.png', dpi=400, bbox_inches='tight')
# plt.show()

In [ ]:
ax_heat_pos.height

In [ ]:
import numpy as np # new
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

fig = plt.figure(figsize=(15, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.3)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.3)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(314, 326, 25)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]


#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef,lev_ds,prec_ds,pw_ds,title,position, dataset_idx=0, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.set_ylim(0, 2)
    else:
        ax2.set_ylim(0, 2)
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-1, 1)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1,jra_lev_mean,jra_prec_mean,jra_pw_tend,'JRA3Q', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_obs = create_obs_plot(ax2, ax_sm_ef2,merra_lev_mean,merra_prec_mean,merra_pw_tend,'MERRA2', 'mid')

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_era = create_obs_plot(ax3, ax_sm_ef3,era_lev_mean.sel(hour = np.arange(0,25,3)),era_prec_mean,era_pw_tend.sel(hour = np.arange(0,24,3)),'ERA5', 'right')

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='|CIN|'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=3, frameon=False, fontsize=14, bbox_to_anchor=(1.05, 0.63))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos2.x1+.1, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(314, 326, 7))


labels = ['(a)','(b)','(c)']
axes = [ax1,ax2,ax3]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
plt.show()

#### Efficiency


In [ ]:
jra_lev_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
jra_prec_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
jra_pw_tend=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

era_lev_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
era_prec_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
era_pw_tend=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

merra_lev_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
merra_prec_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
# merra_pw_tend=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')
merra_pw_tend=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean_new_3hourly.nc')#3hourly

In [ ]:
jra_pw_tend['pw'] = jra_lev_mean['pw']
era_pw_tend['pw'] = era_lev_mean['pw']
# merra  tend  

jra_pw_tend['chi'] = -jra_pw_tend['pw_P']/jra_pw_tend['pw']
era_pw_tend['chi'] = -era_pw_tend['pw_P']/era_pw_tend['pw']
merra_pw_tend['chi'] = -merra_pw_tend['pw_P']/merra_pw_tend['pw']

In [ ]:
jra_pw_tend['chi'].plot()

In [ ]:
era_pw_tend['chi'].plot()

In [ ]:
merra_pw_tend['chi'].plot()

In [ ]:
jra_pw_tend = jra_pw_tend.mean()*8
era_pw_tend = era_pw_tend.mean()*8
merra_pw_tend = merra_pw_tend.mean()*8

In [ ]:
rain_list =  [era_pw_tend,jra_pw_tend,merra_pw_tend]

test3 = []
for i in range(len(rain_list)):
    test3.append(rain_list[i].sel(hour = rain_list[i].hour.isin([12])).mean('hour')*8)

for j in range(len(test3)):
    test3[j]['pre_eff'] = -test3[j]['pw_P']/(test3[j]['pw_E']+test3[j]['pw_res'])
    test3[j]['pw_P'] = -test3[j]['pw_P']

In [ ]:
rain_list =  [era_pw_tend,jra_pw_tend,merra_pw_tend]

for j in range(len(rain_list)):
    rain_list[j]['pre_eff'] = -rain_list[j]['pw_P']/(rain_list[j]['pw_E']+rain_list[j]['pw_res'])
    rain_list[j]['pw_P'] = -rain_list[j]['pw_P']

In [ ]:
rain_list[0]

In [ ]:
rain_list[1]

In [ ]:
rain_list[2]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

data = {
    'ERA5': [
        float(test3[0]['pw_P'].values),
        float(test3[0]['pw_E'].values), 
        float(test3[0]['pw_res'].values),
        float(test3[0]['pre_eff'].values)
    ],
    'JRA-3Q': [
        float(test3[1]['pw_P'].values),
        float(test3[1]['pw_E'].values), 
        float(test3[1]['pw_res'].values),
        float(test3[1]['pre_eff'].values)
    ],
    'MERRA-2': [
        float(test3[2]['pw_P'].values),
        float(test3[2]['pw_E'].values), 
        float(test3[2]['pw_res'].values),
        float(test3[2]['pre_eff'].values)
    ]
}

categories = ['P', 'E', 'MFC','$\chi$']

colors = ['#2E86AB', '#A23B72', '#F18F01']


In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(11, 6)) #00LST

#   (χ )
df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories[:-1]):  #  χ 
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })
df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn   (P, E, MFC)
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax1, alpha=0.8)

ax2 = ax1.twinx()

# χ    
n_datasets = len(data.keys())
bar_width = 0.8 / n_datasets
x_chi = len(categories) - 1  # χ x 

for i, (dataset, color) in enumerate(zip(data.keys(), colors)):
    chi_value = data[dataset][-1]  #   (χ)
    x_position = x_chi + (i - n_datasets/2 + 0.5) * bar_width
    
    ax2.bar(x_position, chi_value, width=bar_width, 
            color=color, alpha=0.8)
    
    # χ   
    ax2.text(x_position, chi_value + 0.02, f'{chi_value:.2f}', 
             ha='center', va='bottom', fontsize=9)

#    (  )
ax1.set_ylim(0, 12)  #   0~30
ax2.set_ylim(0, 1)  #   0~1

ax1.set_yticks(np.linspace(0,12,6)) 
ax2.set_yticks(np.linspace(0,2.,6))  

ax1.set_ylabel('[mm/day]', fontsize=12)
ax2.set_ylabel('[-]', fontsize=12, color='red')
ax2.tick_params(axis='y', colors='red')

ax1.tick_params(axis='x', labelsize=11)
ax1.tick_params(axis='y', labelsize=10)
ax1.legend(fontsize=11, title_fontsize=12, frameon=True, bbox_to_anchor=(1.3, 1))

# x   (  )
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories)

#    (P, E, MFC)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots(1, 1, figsize=(11, 6)) #00LST

#   (χ )
df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories[:-1]):  #  χ 
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })
df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn   (P, E, MFC)
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax1, alpha=0.8)

ax2 = ax1.twinx()

# χ    
n_datasets = len(data.keys())
bar_width = 0.8 / n_datasets
x_chi = len(categories) - 1  # χ x 

for i, (dataset, color) in enumerate(zip(data.keys(), colors)):
    chi_value = data[dataset][-1]  #   (χ)
    x_position = x_chi + (i - n_datasets/2 + 0.5) * bar_width
    
    ax2.bar(x_position, chi_value, width=bar_width, 
            color=color, alpha=0.8)
    
    # χ   
    ax2.text(x_position, chi_value + 0.02, f'{chi_value:.2f}', 
             ha='center', va='bottom', fontsize=9)

#    (  )
ax1.set_ylim(0, 12)  #   0~30
ax2.set_ylim(0, 1)  #   0~1

ax1.set_yticks(np.linspace(0,12,6)) 
ax2.set_yticks(np.linspace(0,2.,6))  

ax1.set_ylabel('[mm/day]', fontsize=12)
ax2.set_ylabel('[-]', fontsize=12, color='red')
ax2.tick_params(axis='y', colors='red')

ax1.tick_params(axis='x', labelsize=11)
ax1.tick_params(axis='y', labelsize=10)
ax1.legend(fontsize=11, title_fontsize=12, frameon=True, bbox_to_anchor=(1.3, 1))

# x   (  )
ax1.set_xticks(range(len(categories)))
ax1.set_xticklabels(categories)

#    (P, E, MFC)
for container in ax1.containers:
    ax1.bar_label(container, fmt='%.2f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax2 = plt.subplots(1, 1, figsize=(6, 6))

df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories):
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })

df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn  
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax2, alpha=0.8)

# ax2.set_xlabel(none)
ax2.set_ylabel('[mm/day]', fontsize=12)
# ax2.set_title('Precipitation Effects Comparison', fontsize=14, fontweight='bold', pad=20)
ax2.tick_params(axis='x', labelsize=11)
ax2.tick_params(axis='y', labelsize=10)
ax2.legend( fontsize=11, title_fontsize=12, frameon=True)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax2 = plt.subplots(1, 1, figsize=(6, 6))

df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories):
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })

df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn  
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax2, alpha=0.8)

# ax2.set_xlabel(none)
ax2.set_ylabel('[mm/day]', fontsize=12)
# ax2.set_title('Precipitation Effects Comparison', fontsize=14, fontweight='bold', pad=20)
ax2.tick_params(axis='x', labelsize=11)
ax2.tick_params(axis='y', labelsize=10)
ax2.legend( fontsize=11, title_fontsize=12, frameon=True)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

### Non-Rainy Days


In [ ]:
jra_dry_pw_tend

In [ ]:
jra_dry_lev_mean = xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/dry/lev_mean.nc')
jra_dry_prec_mean = xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/dry/prec_mean.nc')
jra_dry_pw_tend = xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/dry/pw_tend_mean.nc')

era_dry_lev_mean = xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/dry/lev_mean.nc')
era_dry_prec_mean = xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/dry/prec_mean.nc')
era_dry_pw_tend = xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/dry/pw_tend_mean.nc')

merra_dry_lev_mean = xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/dry/lev_mean.nc')
merra_dry_prec_mean = xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/dry/prec_mean.nc')
merra_dry_pw_tend = xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/dry/pw_tend_mean.nc')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

datasets = ['JRA-3Q']
reanalysis_datasets = datasets[1:]  #  (IGRA SGP Obs )

fig = plt.figure(figsize=(15, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.3)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.3)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(314, 326, 25)
diff_levels = np.linspace(-3, 3, 13)

# PW      
six_hourly = [0, 6, 12, 18, 24]


#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef,lev_ds,prec_ds,pw_ds,title,position, dataset_idx=0, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.set_ylim(0, 2)
    else:
        ax2.set_ylim(0, 2)
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-1, 1)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

#   (IGRA SGP Obs) -   ,   
ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1,jra_dry_lev_mean,jra_dry_prec_mean,jra_dry_pw_tend,'JRA3Q', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_obs = create_obs_plot(ax2, ax_sm_ef2,merra_dry_lev_mean,merra_dry_prec_mean,merra_dry_pw_tend,'MERRA2', 'mid')

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_era = create_obs_plot(ax3, ax_sm_ef3,era_dry_lev_mean.sel(hour = np.arange(0,25,6)),era_dry_prec_mean,era_dry_pw_tend.sel(hour = np.arange(0,24,6)),'ERA5', 'right')

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='|CIN|'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=3, frameon=False, fontsize=14, bbox_to_anchor=(1.05, 0.63))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos2.x1+.1, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(314, 326, 7))

labels = ['(a)','(b)','(c)']
axes = [ax1,ax2,ax3]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
plt.show()

### Rainy Days minus Non-Rainy Days


In [ ]:
for j in range(len(fig_rain_ds)):
    compare_lev_ds[j]['pre_eff'] = -compare_lev_ds[j]['pw_P']/(compare_lev_ds[j]['pw_E']+compare_lev_ds[j]['pw_res'])

In [ ]:
rain_list =  [era_pw_tend,jra_pw_tend,merra_pw_tend]
dry_list = [era_dry_pw_tend,jra_dry_pw_tend,merra_dry_pw_tend]

for j in range(len(rain_list)):
    rain_list[j]['pre_eff'] = -rain_list[j]['pw_P']/(rain_list[j]['pw_E']+rain_list[j]['pw_res'])
    dry_list[j]['pre_eff'] = -dry_list[j]['pw_P']/(dry_list[j]['pw_E']+dry_list[j]['pw_res'])

In [ ]:
rain_list_daily =  [era_pw_tend.mean(),jra_pw_tend.mean(),merra_pw_tend.mean()]
dry_list_daily = [era_dry_pw_tend.mean(),jra_dry_pw_tend.mean(),merra_dry_pw_tend.mean()]


In [ ]:
for j in range(len(rain_list_daily)):
    rain_list_daily[j]['pre_eff'] = -rain_list_daily[j]['pw_P']/(rain_list_daily[j]['pw_E']+rain_list_daily[j]['pw_res'])
    dry_list_daily[j]['pre_eff'] = -dry_list_daily[j]['pw_P']/(dry_list_daily[j]['pw_E']+dry_list_daily[j]['pw_res'])

In [ ]:
diff_list_daily = []
for j in range(len(rain_list_daily)):
    diff_list_daily.append(rain_list_daily[j]-dry_list_daily[j])

In [ ]:
for j in range(len(diff_list_daily)):

    diff_list_daily[j]['effc_ef'] = diff_list_daily[j]['pre_eff']*(dry_list_daily[j]['pw_E']+dry_list_daily[j]['pw_res'])
    
    diff_list_daily[j]['surf_ef'] = dry_list_daily[j]['pre_eff']*diff_list_daily[j]['pw_E']
    
    diff_list_daily[j]['remo_ef'] = dry_list_daily[j]['pre_eff']*diff_list_daily[j]['pw_res']
    
    diff_list_daily[j]['resi'] = diff_list_daily[j]['pre_eff']*(diff_list_daily[j]['pw_E']+diff_list_daily[j]['pw_res'])

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

data = {
    'ERA5': [
        float(diff_list_daily[0]['effc_ef'].values*8),
        float(diff_list_daily[0]['surf_ef'].values*8), 
        float(diff_list_daily[0]['remo_ef'].values*8)
    ],
    'JRA-3Q': [
        float(diff_list_daily[1]['effc_ef'].values*8),
        float(diff_list_daily[1]['surf_ef'].values*8),
        float(diff_list_daily[1]['remo_ef'].values*8)
    ],
    'MERRA-2': [
        float(diff_list_daily[2]['effc_ef'].values*8),
        float(diff_list_daily[2]['surf_ef'].values*8),
        float(diff_list_daily[2]['remo_ef'].values*8)
    ]
}

categories = ['Efficiency effect', 'Surface effect', 'Remote effect']

colors = ['#2E86AB', '#A23B72', '#F18F01']


In [ ]:
fig, ax2 = plt.subplots(1, 1, figsize=(6, 6))

df_data = []
for dataset, values in data.items():
    for i, category in enumerate(categories):
        df_data.append({
            'Dataset': dataset,
            'Effect': category,
            'Value': values[i]
        })

df = pd.DataFrame(df_data)

# Seaborn  
sns.set_style("whitegrid")
sns.set_palette(colors)

# Seaborn  
sns.barplot(data=df, x='Effect', y='Value', hue='Dataset', ax=ax2, alpha=0.8)

# ax2.set_xlabel(none)
ax2.set_ylabel('[mm/day]', fontsize=12)
# ax2.set_title('Precipitation Effects Comparison', fontsize=14, fontweight='bold', pad=20)
ax2.tick_params(axis='x', labelsize=11)
ax2.tick_params(axis='y', labelsize=10)
ax2.legend( fontsize=11, title_fontsize=12, frameon=True)

for container in ax2.containers:
    ax2.bar_label(container, fmt='%.3f', fontsize=9, padding=3)

plt.tight_layout()
plt.show()

In [ ]:
diff_list_daily[0]

In [ ]:
diff_list_daily[1]

In [ ]:
diff_list_daily[2]

In [ ]:
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend
era_pw_diff = era_pw_tend - era_dry_pw_tend

In [ ]:
diff_list = [era_pw_diff,jra_pw_diff, merra_pw_diff]

In [ ]:
for j in range(len(rain_list)):

    diff_list[j]['effc_ef'] = diff_list[j]['pre_eff']*(dry_list[j]['pw_E']+dry_list[j]['pw_res'])
    
    diff_list[j]['surf_ef'] = dry_list[j]['pre_eff']*diff_list[j]['pw_E']
    
    diff_list[j]['remo_ef'] = dry_list[j]['pre_eff']*diff_list[j]['pw_res']
    
    diff_list[j]['resi'] = diff_list[j]['pre_eff']*(diff_list[j]['pw_E']+diff_list[j]['pw_res'])

In [ ]:
jra_pw_diff

In [ ]:
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend
era_pw_diff = era_pw_tend - era_dry_pw_tend


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

jra_lev_diff = jra_lev_mean - jra_dry_lev_mean
jra_prec_diff = jra_prec_mean - jra_dry_prec_mean
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend

# MERRA-2 
merra_lev_diff = merra_lev_mean - merra_dry_lev_mean
merra_prec_diff = merra_prec_mean - merra_dry_prec_mean
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend

# ERA5  ( )
era_lev_diff = era_lev_mean.sel(hour=np.arange(0,25,6)) - era_dry_lev_mean.sel(hour=np.arange(0,25,6))
era_prec_diff = era_prec_mean - era_dry_prec_mean
era_pw_diff = era_pw_tend.sel(hour=np.arange(0,24,6)) - era_dry_pw_tend.sel(hour=np.arange(0,24,6))

compare_lev_ds = [jra_lev_diff, merra_lev_diff, era_lev_diff]
compare_rain_ds = [jra_prec_diff, merra_prec_diff, era_prec_diff]
compare_pw_ds = [jra_pw_diff, merra_pw_diff, era_pw_diff]


fig = plt.figure(figsize=(15, 12))

total_height = 0.85  #     (   )
top_margin = 0.95
bottom_margin = 0.1  #   ( )

group_gap = 0.08

available_height = total_height - group_gap
top_height = available_height * 0.5  #    (  )
bottom_height = available_height * 0.5  #    (  )

top_bottom = top_margin - top_height
bottom_top = top_bottom - group_gap  #     (  )
bottom_bottom = bottom_top - bottom_height

gs_top = GridSpec(2, 3, figure=fig,
                  top=top_margin, bottom=top_bottom,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                  hspace=0.0,
                  wspace=0.3)  #    (default )

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=bottom_top, bottom=bottom_bottom,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],  #  :SM/EF  = 4.5:1 
                     hspace=0.0,
                     wspace=0.3)  #    (default )

main_positions_top = [(0, 0), (0, 1), (0, 2)]  #  GridSpec  
main_positions_bottom = [(0, 1), (0, 2)]  #  GridSpec   (   )

# SM/EF   
sm_ef_positions_top = [(1, 0), (1, 1), (1, 2)]  #  GridSpec SM/EF 
sm_ef_positions_bottom = [(1, 1), (1, 2)]  #  GridSpec SM/EF 

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'  # /
cp_color = '#56B4E9'
lp_color = '#E69F00'  # /
sm_color = '#009E73'  #  (SM)
ef_color = '#CC0000'  #  (EF)

obs_levels = np.linspace(-6, 6, 24)
diff_levels = np.linspace(-5, 5, 21)

# PW      
six_hourly = [0, 6, 12, 18, 24]


#    (IGRA SGP)
def create_obs_plot(ax, ax_sm_ef,lev_ds,prec_ds,pw_ds,title,position, dataset_idx=0, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    ax2 = ax.twinx()
    ax2.set_zorder(2)  #  zorder
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    #   tp 
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   ( )
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
        ax2.set_ylim(0, 2)
    else:
        ax2.set_ylim(0, 2)
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, abs(current_diur.cin.values), '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # SM/EF  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            # : 
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.set_ylim(-1.5, 1.5)
    
    # PW  
    handles = [
        mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                          edgecolor='black', linewidth=0.3),
        mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                          edgecolor='black', linewidth=0.3)
    ]
    leg = ax_ef.legend(handles=handles, loc='lower right', ncol=3,
                      fontsize=10, frameon=False,
                      bbox_to_anchor=(1, -0.1), 
                      bbox_transform=ax_ef.transAxes)
    
    return CS0

ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1,compare_lev_ds[0],compare_rain_ds[0],compare_pw_ds[0],'JRA3Q', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
CS_obs = create_obs_plot(ax2, ax_sm_ef2,compare_lev_ds[1],compare_rain_ds[1],compare_pw_ds[1],'MERRA2', 'mid')

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
CS_era = create_obs_plot(ax3, ax_sm_ef3,compare_lev_ds[2],compare_rain_ds[2],compare_pw_ds[2],'ERA5', 'right')

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='$\Delta$ CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='$\Delta$ CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

fig.legend(handles=var_handles, loc='upper center', ncol=3, frameon=False, fontsize=14, bbox_to_anchor=(1.05, 0.63))

#    - IGRA ERA5  
pos1 = ax1.get_position()
pos2 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos2.x1+.1, pos1.y0, 0.02, pos1.height])  # IGRA ERA5  

cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('$\Delta$ MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(-6, 6, 7))


labels = ['(d)','(e)','(f)']
axes = [ax1,ax2,ax3]

for i, (ax, label) in enumerate(zip(axes, labels)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout(rect=[0, 0.05, 0.95, 0.95])
plt.show()

### Final


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

jra_lev_diff = jra_lev_mean - jra_dry_lev_mean
jra_prec_diff = jra_prec_mean - jra_dry_prec_mean
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend

merra_lev_diff = merra_lev_mean - merra_dry_lev_mean
merra_prec_diff = merra_prec_mean - merra_dry_prec_mean
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend

era_lev_diff = era_lev_mean.sel(hour=np.arange(0,25,6)) - era_dry_lev_mean.sel(hour=np.arange(0,25,6))
era_prec_diff = era_prec_mean - era_dry_prec_mean
era_pw_diff = era_pw_tend.sel(hour=np.arange(0,24,6)) - era_dry_pw_tend.sel(hour=np.arange(0,24,6))

compare_lev_ds = [jra_lev_diff, merra_lev_diff, era_lev_diff]
compare_rain_ds = [jra_prec_diff, merra_prec_diff, era_prec_diff]
compare_pw_ds = [jra_pw_diff, merra_pw_diff, era_pw_diff]

fig = plt.figure(figsize=(15,12))  #    2 

# GridSpec  - 2 
gs_top = GridSpec(2, 3, figure=fig,
                  top=0.95, bottom=0.52,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],
                  hspace=0.0,
                  wspace=0.2)

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=0.45, bottom=0.05,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],
                     hspace=0.0,
                     wspace=0.2)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'
cp_color = '#56B4E9'
lp_color = '#E69F00'
sm_color = '#009E73'
ef_color = '#CC0000'

obs_levels = np.linspace(314, 326, 25)
diff_levels = np.linspace(-6, 6, 24)

#     (  )
def create_obs_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP  
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP  
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / |CIN| [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)
    
    return CS0

#     (  )
def create_diff_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP   
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(0, 250)
    else :
        ax3.set_ylim(0, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW    
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency   
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW     
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)

    if position == 'right':
        ax_ef.set_ylabel('[mm/3hr]', fontsize=11,labelpad=10)
    else :
        ax_ef.tick_params(axis='y', which='both', right=False, labelright=False)
        ax_ef.spines['right'].set_visible(False)


    return CS0

#   - Rainy Days  (  )
ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1, era_lev_mean.sel(hour=np.arange(0,25,6)), era_prec_mean, era_pw_tend.sel(hour=np.arange(0,24,6)), 'ERA5', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
create_obs_plot(ax2, ax_sm_ef2, jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q', 'mid')
# jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q'

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
create_obs_plot(ax3, ax_sm_ef3, merra_lev_mean, merra_prec_mean, merra_pw_tend, 'MERRA2', 'right')

#   -   (  )
ax4 = fig.add_subplot(gs_bottom[0, 0])
ax_sm_ef4 = fig.add_subplot(gs_bottom[1, 0])
CS_diff = create_diff_plot(ax4, ax_sm_ef4, compare_lev_ds[2], compare_rain_ds[2], compare_pw_ds[2], 'ERA5', 'left', x_label=True)

ax5 = fig.add_subplot(gs_bottom[0, 1])
ax_sm_ef5 = fig.add_subplot(gs_bottom[1, 1])
create_diff_plot(ax5, ax_sm_ef5, compare_lev_ds[0], compare_rain_ds[0], compare_pw_ds[0], 'JRA3Q', 'mid', x_label=True)

ax6 = fig.add_subplot(gs_bottom[0, 2])
ax_sm_ef6 = fig.add_subplot(gs_bottom[1, 2])
create_diff_plot(ax6, ax_sm_ef6, compare_lev_ds[1], compare_rain_ds[1], compare_pw_ds[1], 'MERRA2', 'right', x_label=True)

#   (Rainy Days )
pos1 = ax1.get_position()
pos3 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos3.x1+0.1, pos1.y0, 0.02, pos1.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_obs.set_ticks(np.linspace(314, 326, 7))

#   ( )
pos4 = ax4.get_position()
pos6 = ax6.get_position()
cbar_ax_diff = fig.add_axes([pos6.x1+0.1, pos4.y0, 0.02, pos4.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$ MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-6, 6, 7))

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

# PW  (  )
handles = [
    mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                      edgecolor='black', linewidth=0.3)
]
ax_sm_ef1.legend(handles=handles, loc='lower right', ncol=3,
              fontsize=10, frameon=False,
              bbox_to_anchor=(1, -0.1), 
              bbox_transform=ax_sm_ef1.transAxes)

fig.legend(handles=var_handles, loc='center', ncol=1, frameon=False, fontsize=14, bbox_to_anchor=(1, 0.52))

labels_top = ['(a)','(b)','(c)']
labels_bottom = ['(d)','(e)','(f)']
axes_top = [ax1, ax2, ax3]
axes_bottom = [ax4, ax5, ax6]

for i, (ax, label) in enumerate(zip(axes_top, labels_top)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

for i, (ax, label) in enumerate(zip(axes_bottom, labels_bottom)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/certical_fig_ranalysis.png', dpi=400, bbox_inches='tight')
# plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

jra_lev_diff = jra_lev_mean - jra_dry_lev_mean
jra_prec_diff = jra_prec_mean - jra_dry_prec_mean
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend

merra_lev_diff = merra_lev_mean - merra_dry_lev_mean
merra_prec_diff = merra_prec_mean - merra_dry_prec_mean
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend

era_lev_diff = era_lev_mean.sel(hour=np.arange(0,25,3)) - era_dry_lev_mean.sel(hour=np.arange(0,25,3))
era_prec_diff = era_prec_mean - era_dry_prec_mean
era_pw_diff = era_pw_tend.sel(hour=np.arange(0,24,3)) - era_dry_pw_tend.sel(hour=np.arange(0,24,3))

compare_lev_ds = [jra_lev_diff, merra_lev_diff, era_lev_diff]
compare_rain_ds = [jra_prec_diff, merra_prec_diff, era_prec_diff]
compare_pw_ds = [jra_pw_diff, merra_pw_diff, era_pw_diff]

fig = plt.figure(figsize=(15,12))  #    2 

# GridSpec  - 2 
gs_top = GridSpec(2, 3, figure=fig,
                  top=0.95, bottom=0.52,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],
                  hspace=0.0,
                  wspace=0.2)

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=0.45, bottom=0.05,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],
                     hspace=0.0,
                     wspace=0.2)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'
cp_color = '#56B4E9'
lp_color = '#E69F00'
sm_color = '#009E73'
ef_color = '#CC0000'

# obs_levels = np.linspace(314, 326, 25)  #   
obs_levels = np.linspace(316, 326, 21)

# diff_levels = np.linspace(-6, 6, 24)    #   
diff_levels = np.linspace(-4, 4, 17)

#     (  )
def create_obs_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP  
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP  
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(-50, 250)
    else :
        ax3.set_ylim(-50, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    # ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)
    
    return CS0

#     (  )
def create_diff_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP   
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(-50, 250)
    else :
        ax3.set_ylim(-50, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    # ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW    
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency   
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW     
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)

    if position == 'right':
        ax_ef.set_ylabel('[mm/3hr]', fontsize=11,labelpad=10)
    else :
        ax_ef.tick_params(axis='y', which='both', right=False, labelright=False)
        ax_ef.spines['right'].set_visible(False)


    return CS0

#   - Rainy Days  (  )
ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1, era_lev_mean.sel(hour=np.arange(0,25,6)), era_prec_mean, era_pw_tend.sel(hour=np.arange(0,24,6)), 'ERA5', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
create_obs_plot(ax2, ax_sm_ef2, jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q', 'mid')
# jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q'

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
create_obs_plot(ax3, ax_sm_ef3, merra_lev_mean, merra_prec_mean, merra_pw_tend, 'MERRA2', 'right')

#   -   (  )
ax4 = fig.add_subplot(gs_bottom[0, 0])
ax_sm_ef4 = fig.add_subplot(gs_bottom[1, 0])
CS_diff = create_diff_plot(ax4, ax_sm_ef4, compare_lev_ds[2], compare_rain_ds[2], compare_pw_ds[2], 'ERA5', 'left', x_label=True)

ax5 = fig.add_subplot(gs_bottom[0, 1])
ax_sm_ef5 = fig.add_subplot(gs_bottom[1, 1])
create_diff_plot(ax5, ax_sm_ef5, compare_lev_ds[0], compare_rain_ds[0], compare_pw_ds[0], 'JRA3Q', 'mid', x_label=True)

ax6 = fig.add_subplot(gs_bottom[0, 2])
ax_sm_ef6 = fig.add_subplot(gs_bottom[1, 2])
create_diff_plot(ax6, ax_sm_ef6, compare_lev_ds[1], compare_rain_ds[1], compare_pw_ds[1], 'MERRA2', 'right', x_label=True)

#   (Rainy Days )
pos1 = ax1.get_position()
pos3 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos3.x1+0.108, pos1.y0, 0.02, pos1.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
# cbar_obs.set_ticks(np.linspace(314, 326, 7))
cbar_obs.set_ticks(np.linspace(316, 326, 6))

#   ( )
pos4 = ax4.get_position()
pos6 = ax6.get_position()
cbar_ax_diff = fig.add_axes([pos6.x1+0.108, pos4.y0, 0.02, pos4.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$ MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-4, 4, 5))

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

# PW  (  )
handles = [
    mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                      edgecolor='black', linewidth=0.3)
]
ax_sm_ef1.legend(handles=handles, loc='lower right', ncol=3,
              fontsize=10, frameon=False,
              bbox_to_anchor=(1, -0.1), 
              bbox_transform=ax_sm_ef1.transAxes)

fig.legend(handles=var_handles, loc='center', ncol=1, frameon=False, fontsize=14, bbox_to_anchor=(1, 0.52))

labels_top = ['(a) ERA5','(b) JRA-3Q','(c) MERRA2']
labels_bottom = ['(d) ERA5','(e) JRA-3Q','(f) MERRA2']
axes_top = [ax1, ax2, ax3]
axes_bottom = [ax4, ax5, ax6]

for i, (ax, label) in enumerate(zip(axes_top, labels_top)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

for i, (ax, label) in enumerate(zip(axes_bottom, labels_bottom)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/certical_fig_ranalysis.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
import matplotlib.colors as colors
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import colormaps as cmaps
from matplotlib.offsetbox import AnnotationBbox, DrawingArea

jra_lev_diff = jra_lev_mean - jra_dry_lev_mean
jra_prec_diff = jra_prec_mean - jra_dry_prec_mean
jra_pw_diff = jra_pw_tend - jra_dry_pw_tend

merra_lev_diff = merra_lev_mean - merra_dry_lev_mean
merra_prec_diff = merra_prec_mean - merra_dry_prec_mean
merra_pw_diff = merra_pw_tend - merra_dry_pw_tend

era_lev_diff = era_lev_mean.sel(hour=np.arange(0,25,3)) - era_dry_lev_mean.sel(hour=np.arange(0,25,3))
era_prec_diff = era_prec_mean - era_dry_prec_mean
era_pw_diff = era_pw_tend.sel(hour=np.arange(0,24,3)) - era_dry_pw_tend.sel(hour=np.arange(0,24,3))

compare_lev_ds = [jra_lev_diff, merra_lev_diff, era_lev_diff]
compare_rain_ds = [jra_prec_diff, merra_prec_diff, era_prec_diff]
compare_pw_ds = [jra_pw_diff, merra_pw_diff, era_pw_diff]

fig = plt.figure(figsize=(15,12))  #    2 

# GridSpec  - 2 
gs_top = GridSpec(2, 3, figure=fig,
                  top=0.95, bottom=0.52,
                  width_ratios=[1, 1, 1],
                  height_ratios=[4.5, 1],
                  hspace=0.0,
                  wspace=0.2)

gs_bottom = GridSpec(2, 3, figure=fig,
                     top=0.45, bottom=0.05,
                     width_ratios=[1, 1, 1],
                     height_ratios=[4.5, 1],
                     hspace=0.0,
                     wspace=0.2)

mse_cmap = cmaps.deep
diff_cmap = 'RdBu_r'
cape_color = '#F21B1B'
cin_color = '#2714A6'
tp_color = '#CC79A7'
cp_color = '#56B4E9'
lp_color = '#E69F00'
sm_color = '#009E73'
ef_color = '#CC0000'

# obs_levels = np.linspace(314, 326, 25)  #   
obs_levels = np.linspace(316, 326, 21)

# diff_levels = np.linspace(-6, 6, 24)    #   
diff_levels = np.linspace(-4, 4, 17)

#     (  )
def create_obs_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=obs_levels, cmap=mse_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP  
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP  
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN  
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(-50, 250)
    else :
        ax3.set_ylim(-50, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    # ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW   
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency  
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW    
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)
    
    return CS0

#     (  )
def create_diff_plot(ax, ax_sm_ef, lev_ds, prec_ds, pw_ds, title, position, x_label=False):
    current_diur = lev_ds
    current_rain = prec_ds
    
    # MSE    
    mse_data = current_diur.mse.values.copy()
    mse_data_masked = np.ma.masked_invalid(mse_data)
    
    # MSE   
    CS0 = ax.contourf(current_diur.hour, current_diur.pres, mse_data_masked.T, 
                      levels=diff_levels, cmap=diff_cmap, alpha=0.9, extend='both', zorder=0)
    
    ax.set_yscale('log')
    ax.set_ylim(1000, 400)
    ax.grid(True, ls='--', alpha=0.3)
    ax.set_yticks([1000,925, 850, 700, 500, 400])
    
    # x 
    ax.set_xlim(0, 24)
    ax.set_xticks(np.arange(0, 24, 3))
    ax.set_xticklabels([])  # x  
    
    # y  
    if position == 'left':
        ax.set_ylabel('Pressure [hPa]', fontsize=14)
    else : 
        ax.tick_params(axis='y', which='both', right=False, labelleft=False)
        ax.spines['left'].set_visible(False)
    
    ax2 = ax.twinx()
    ax2.set_zorder(2)
    ax2.patch.set_visible(False)
    
    bar_width = 0.6
    bar_s = 0.6
    
    # TP   
    bars_tp = ax2.bar(current_rain.hour+bar_s, current_rain.tp.values, 
                 width=bar_width, align='edge', color=tp_color, alpha=0.7, label='TP',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    # CP LP   
    if hasattr(current_rain, 'cp') and hasattr(current_rain, 'lp'):
        bars_cp = ax2.bar(current_rain.hour+bar_s+bar_width, current_rain.cp.values, 
                         width=bar_width, align='edge', color=cp_color, alpha=0.7, label='CP',
                         edgecolor='black', linewidth=0.5)
        
        bars_lp = ax2.bar(current_rain.hour+bar_s+2*bar_width, current_rain.lp.values, 
                         width=bar_width, align='edge', color=lp_color, alpha=0.7, label='LP',
                         edgecolor='black', linewidth=0.5)
    ax2.set_ylim(0, 2)
    ax2.set_yticks(np.arange(0,2.1,.5))
    if position == 'right':
        ax2.set_ylabel('Precipitation [mm/3hr]', fontsize=14)
    else:
        ax2.tick_params(axis='y', which='both', right=True, labelright=False)
        ax2.spines['right'].set_visible(False)
    
    # CAPE CIN    
    ax3 = ax.twinx()
    ax3.spines['right'].set_position(('outward', 60))
    
    # CAPE CIN   
    cape_line = ax3.plot(current_diur.hour, current_diur.cape.values, '-', 
                        color=cape_color, linewidth=2.5, marker='o', markersize=7, 
                        markeredgecolor='black', markeredgewidth=0.5, label='CAPE', zorder=3)
    cin_line = ax3.plot(current_diur.hour, current_diur.cin.values, '-', 
                       color=cin_color, linewidth=2.5, marker='s', markersize=7, 
                       markeredgecolor='black', markeredgewidth=0.5, label='CIN', zorder=3)
    
    # CAPE/CIN  
    if position == 'right':
        ax3.set_ylabel('$\Delta$ CAPE / CIN [J/kg]', fontsize=14)
        ax3.set_ylim(-50, 250)
    else :
        ax3.set_ylim(-50, 250)
        ax3.tick_params(axis='y', which='both', right=False, labelright=False)
        ax3.spines['right'].set_visible(False)

    # ax.set_title(title, fontsize=18)
    
    # PW  
    ax_sm_ef.set_xlim(0, 24)
    ax_sm_ef.set_xticks(np.arange(0, 25, 3))
    ax_sm_ef.tick_params(axis='y', which='both', left=False, labelleft=False)
    ax_sm_ef.spines['left'].set_visible(False)
    
    if x_label:
        ax_sm_ef.set_xlabel('LST [hour]', fontsize=14)
    
    # PW    
    ax_ef = ax_sm_ef.twinx()
    pw_info = pw_ds 
    hours = pw_info.hour.values
    values = pw_info['pw_tend'].values
    
    # PW tendency   
    for hour, value in zip(hours, values):
        if value > 0:
            ax_ef.bar(hour+1.5, value, width=bar_width*3, 
                     color='blue', alpha=0.3, 
                     edgecolor='darkblue', linewidth=.8, zorder=1)
        else:
            ax_ef.bar(hour+1.5, value, width=bar_width*3,
                     color='red', alpha=0.3,
                     edgecolor='darkred', linewidth=.8, zorder=1)
    
    # PW     
    bars_P = ax_ef.bar(hours+bar_s, pw_info['pw_P'], 
                 width=bar_width, align='edge', color='r', alpha=0.7, label='-P',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_E = ax_ef.bar(hours+bar_s+bar_width, pw_info['pw_E'], 
                 width=bar_width, align='edge', color='green', alpha=0.7, label='E',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    bars_res = ax_ef.bar(hours+bar_s+bar_width*2, pw_info['pw_res'], 
                 width=bar_width, align='edge', color='blue', alpha=0.7, label='Moist conv',
                 edgecolor='black', linewidth=0.5, zorder=3)
    
    ax_ef.plot(np.arange(0,25,3), [0 for nn in range(9)], c='black', ls='-', lw=.3)
    ax_ef.plot(np.arange(0,25,3), [1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )
    ax_ef.plot(np.arange(0,25,3), [-1]*9, c='gray', ls='-', lw=0.2,alpha =.8 )

    ax_ef.set_ylim(-1.5, 1.5)

    if position == 'right':
        ax_ef.set_ylabel('[mm/3hr]', fontsize=11,labelpad=10)
    else :
        ax_ef.tick_params(axis='y', which='both', right=False, labelright=False)
        ax_ef.spines['right'].set_visible(False)


    return CS0

#   - Rainy Days  (  )
ax1 = fig.add_subplot(gs_top[0, 0])
ax_sm_ef1 = fig.add_subplot(gs_top[1, 0])
CS_obs = create_obs_plot(ax1, ax_sm_ef1, era_lev_mean.sel(hour=np.arange(0,25,3)), era_prec_mean, era_pw_tend.sel(hour=np.arange(0,24,3)), 'ERA5', 'left')

ax2 = fig.add_subplot(gs_top[0, 1])
ax_sm_ef2 = fig.add_subplot(gs_top[1, 1])
create_obs_plot(ax2, ax_sm_ef2, jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q', 'mid')
# jra_lev_mean, jra_prec_mean, jra_pw_tend, 'JRA3Q'

ax3 = fig.add_subplot(gs_top[0, 2])
ax_sm_ef3 = fig.add_subplot(gs_top[1, 2])
create_obs_plot(ax3, ax_sm_ef3, merra_lev_mean, merra_prec_mean, merra_pw_tend, 'MERRA2', 'right')

#   -   (  )
ax4 = fig.add_subplot(gs_bottom[0, 0])
ax_sm_ef4 = fig.add_subplot(gs_bottom[1, 0])
CS_diff = create_diff_plot(ax4, ax_sm_ef4, compare_lev_ds[2], compare_rain_ds[2], compare_pw_ds[2], 'ERA5', 'left', x_label=True)

ax5 = fig.add_subplot(gs_bottom[0, 1])
ax_sm_ef5 = fig.add_subplot(gs_bottom[1, 1])
create_diff_plot(ax5, ax_sm_ef5, compare_lev_ds[0], compare_rain_ds[0], compare_pw_ds[0], 'JRA3Q', 'mid', x_label=True)

ax6 = fig.add_subplot(gs_bottom[0, 2])
ax_sm_ef6 = fig.add_subplot(gs_bottom[1, 2])
create_diff_plot(ax6, ax_sm_ef6, compare_lev_ds[1], compare_rain_ds[1], compare_pw_ds[1], 'MERRA2', 'right', x_label=True)

#   (Rainy Days )
pos1 = ax1.get_position()
pos3 = ax3.get_position()
cbar_ax_obs = fig.add_axes([pos3.x1+0.108, pos1.y0, 0.02, pos1.height])
cbar_obs = fig.colorbar(CS_obs, cax=cbar_ax_obs)
cbar_obs.set_label('MSE [kJ/kg]', fontsize=14, labelpad=10)
# cbar_obs.set_ticks(np.linspace(314, 326, 7))
cbar_obs.set_ticks(np.linspace(316, 326, 6))

#   ( )
pos4 = ax4.get_position()
pos6 = ax6.get_position()
cbar_ax_diff = fig.add_axes([pos6.x1+0.108, pos4.y0, 0.02, pos4.height])
cbar_diff = fig.colorbar(CS_diff, cax=cbar_ax_diff)
cbar_diff.set_label('$\Delta$ MSE [kJ/kg]', fontsize=14, labelpad=10)
cbar_diff.set_ticks(np.linspace(-4, 4, 5))

var_handles = [
    Line2D([0], [0], color=cape_color, lw=2.5, marker='o', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CAPE'),
    Line2D([0], [0], color=cin_color, lw=2.5, marker='s', markersize=7, 
          markeredgecolor='black', markeredgewidth=0.5, label='CIN'),
    mpatches.Rectangle((0, 0), 1, 1, fc=tp_color, alpha=0.7, label='TP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=cp_color, alpha=0.7, label='CP', 
                      edgecolor='black', linewidth=0.5),
    mpatches.Rectangle((0, 0), 1, 1, fc=lp_color, alpha=0.7, label='LP', 
                      edgecolor='black', linewidth=0.5)
]

# PW  (  )
handles = [
    mpatches.Rectangle((0, 0), 1, 1, fc='r', alpha=0.7, label='-P', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='green', alpha=0.7, label='E', 
                      edgecolor='black', linewidth=0.3),
    mpatches.Rectangle((0, 0), 1, 1, fc='blue', alpha=0.7, label='Moist conv', 
                      edgecolor='black', linewidth=0.3)
]
ax_sm_ef1.legend(handles=handles, loc='lower right', ncol=3,
              fontsize=10, frameon=False,
              bbox_to_anchor=(1, -0.1), 
              bbox_transform=ax_sm_ef1.transAxes)

fig.legend(handles=var_handles, loc='center', ncol=1, frameon=False, fontsize=14, bbox_to_anchor=(1, 0.52))

labels_top = ['(a) ERA5','(b) JRA-3Q','(c) MERRA2']
labels_bottom = ['(d) ERA5','(e) JRA-3Q','(f) MERRA2']
axes_top = [ax1, ax2, ax3]
axes_bottom = [ax4, ax5, ax6]

for i, (ax, label) in enumerate(zip(axes_top, labels_top)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

for i, (ax, label) in enumerate(zip(axes_bottom, labels_bottom)):
    ax.text(0., 395, label, fontsize=16, ha='left', va='bottom')

plt.tight_layout()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/certical_fig_ranalysis_comp.png', dpi=400, bbox_inches='tight')
# plt.show()

### skew_t


In [ ]:
jra_lev_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
jra_prec_mean=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
jra_pw_tend=xr.open_dataset('${DATA_DIR}/JRA-3Q/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

era_lev_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
era_prec_mean=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
era_pw_tend=xr.open_dataset('${DATA_DIR}/ERA5/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

merra_lev_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/lev_mean.nc')
merra_prec_mean=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/prec_mean.nc')
merra_pw_tend=xr.open_dataset('${DATA_DIR}/MERRA-2/diurnalcycle/for_lev_fig/rainy/pw_tend_mean.nc')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = jra_lev_mean.isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(1000, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    # Hodograph 
    
fig.suptitle('ERA5',fontsize = 20) 

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = jra_lev_mean.isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(1000, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    # Hodograph 
fig.suptitle('JRA-3Q',fontsize = 20) 

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 2, 4, 6]):  # : 4  
    ax = axes[idx]
    now_lev = era_lev_mean.isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(1000, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
fig.suptitle('ERA5',fontsize = 20) 

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import metpy
import metpy.calc as mpcalc
from metpy.cbook import get_test_data
from metpy.plots import add_metpy_logo, Hodograph, SkewT
from metpy.units import units
from metpy.calc import el, lcl

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

fig, axes = plt.subplots(2, 2, figsize=(10, 10), subplot_kw=dict(projection='skewx'))
axes = axes.flatten()

for idx, hour in enumerate([0, 1, 2, 3]):  # : 4  
    ax = axes[idx]
    now_lev = merra_lev_mean.isel(hour=hour)

    p = now_lev['pres'].values * units.hPa
    z = now_lev['gph'].values * units.m
    T = (now_lev['temp'].values - 273.15) * units.degC
    Td = (now_lev['Td'].values - 273.15) * units.degC
    # wind_speed = now_lev['winds'].values * units('m/s')
    # wind_dir = now_lev['windd'].values * units.degrees
    # u, v = mpcalc.wind_components(wind_speed, wind_dir)

    # SkewT 
    # skew = SkewT(fig, rotation=45, rect=ax.get_position().bounds)
    skew = SkewT(fig, rotation=45, subplot=ax)
    parcel_prof = mpcalc.parcel_profile(p, T[0], Td[0]).to('degC')
    cape, cin = mpcalc.cape_cin(p, T, Td, parcel_prof)

    skew.plot(p, T, 'r')
    skew.plot(p, Td, 'g')
    skew.plot(p, parcel_prof, 'k', linestyle='--')
    # skew.plot_barbs(p, u, v)

    skew.ax.set_ylim(1000, 300)
    skew.ax.set_xlim(-10, 30)
    skew.plot_dry_adiabats(lw=0.5)
    skew.plot_moist_adiabats(lw=0.5)
    skew.plot_mixing_lines(lw=0.5)
    skew.shade_cape(p, T, parcel_prof)
    skew.shade_cin(p, T, parcel_prof)
    skew.ax.text(0.01, 0.99, f'CAPE: {cape.to("J/kg").m:.1f} J/kg\nCIN: {cin.to("J/kg").m:.1f} J/kg',
                 transform=skew.ax.transAxes, fontsize=10, color='black', ha='left', va='top',
                 bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))
    # LCL & LFC
    p_sfc, T_sfc, Td_sfc = p[0], T[0], Td[0]
    lcl_p, _ = mpcalc.lcl(p_sfc, T_sfc, Td_sfc)
    lfc_p, _ = mpcalc.lfc(p, T, Td, parcel_prof)
    lcl_p = lcl_p.to('hPa')
    if lfc_p is not None:
        lfc_p = lfc_p.to('hPa')

    skew.ax.axhline(lcl_p.m, color='brown', linestyle='--', lw=1)
    skew.ax.text(0.98, lcl_p.m, f'LCL: {lcl_p.m:.0f} hPa',
                 transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                 color='brown', fontsize=7)

    if lfc_p is not None and np.isfinite(lfc_p.m):
        skew.ax.axhline(lfc_p.m, color='purple', linestyle='--', lw=1)
        skew.ax.text(0.98, lfc_p.m, f'LFC: {lfc_p.m:.0f} hPa',
                     transform=skew.ax.get_yaxis_transform(), ha='right', va='bottom',
                     color='purple', fontsize=7)
    skew.ax.set_xlabel(f'Temperature [{T.units:~P}]')
    skew.ax.set_ylabel(f'Pressure [{p.units:~P}]')
    ax.set_title(f'Hour = {hour*6:02d} LST', fontsize=10)
fig.suptitle('MERRA-2',fontsize = 20) 

plt.tight_layout()
plt.show()

## Additional Figure 2


### Sub-Daily Variance Ratio


In [ ]:
obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr_26h_30.nc')#.rename({'hfr':'hfr_tp'})

IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

In [ ]:
sat_mean_hfr = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr],'data').mean('data')
rean_mean_hfr = xr.concat([ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data')

In [ ]:
total_mask = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr,ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data',skipna = False)['hfr_tp'].isnull()

In [ ]:
IMERG_hfr = IMERG_hfr.where(~total_mask)
TRMM_hfr = TRMM_hfr.where(~total_mask)
CMORPH_hfr = CMORPH_hfr.where(~total_mask)
GSMaP_hfr = GSMaP_hfr.where(~total_mask)
MSWEP_hfr = MSWEP_hfr.where(~total_mask)

#reanalysis
ERA5_hfr = ERA5_hfr.where(~total_mask)
JRA3Q_hfr = JRA3Q_hfr.where(~total_mask)
# NARRa_hfr = NARRa_hfr.where(~total_mask)
# NARRf_hfr = NARRf_hfr.where(~total_mask)
MERRA2_hfr = MERRA2_hfr.where(~total_mask)

In [ ]:
sat_mean_hfr = sat_mean_hfr.clip(0,100)
rean_mean_hfr = rean_mean_hfr.clip(0,100)

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

### 1st Harmonic Ratio


In [ ]:
rdir = '${DATA_DIR}/'
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly.nc')
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_nonmswep.nc') #CMORPH GSMAP IMERG 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_gs_ms.nc') #cmorph_gsmap_mswep 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_cmo_im_ms.nc') # cmorph, imerg , msewp 
# sat_mean = xr.open_dataset(rdir+'Multi-sat/MS_diurnal_result_3hourly_diurmean.nc') # diurnal  mean  (   ?)
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
#obs = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
sat_mean_rat1 = xr.concat([imerg,trmm,cmorph, gsmap,mswep],'data').mean('data')#,skipna=False)
rean_mean_rat1 = xr.concat([era5,jra55,merra2],'data').mean('data')#,skipna=False)

In [ ]:
(rean_mean_rat1['rat_1st']-result_ds_list[1]['1st']).plot()

In [ ]:
(sat_mean_rat1['rat_1st']-result_ds_list[0]['1st']).plot()

In [ ]:
(rean_mean_rat1['rat_1st']-sat_mean_rat1['rat_1st']).plot()

In [ ]:
((sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100).plot()

In [ ]:
(((rean_mean_rat1['var_subD']/rean_mean_rat1['var_total'])*100)-((sat_mean_rat1['var_subD']/sat_mean_rat1['var_total'])*100)).plot()

In [ ]:
(((rean_mean_rat1['var_1st']/rean_mean_rat1['var_subD'])*100)-((sat_mean_rat1['var_1st']/sat_mean_rat1['var_subD'])*100)).plot()

In [ ]:
(((rean_mean_rat1['var_1st']/rean_mean_rat1['var_total'])*100)-((sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100)).plot()

In [ ]:
rean_mean_rat1['rat_subD'].plot()

In [ ]:
((rean_mean_rat1['var_isd']/rean_mean_rat1['var_subD'])*100).plot()

In [ ]:
((rean_mean_rat1['var_mdc']/rean_mean_rat1['var_subD'])*100).plot()

In [ ]:
IMERG_var['tp_1st_var'].plot()

In [ ]:
((IMERG_var['tp_1st_var']/IMERG_hfr['total_var'])*100).plot()

In [ ]:
((imerg['var_1st']/imerg['var_total'])*100).plot()

In [ ]:
((IMERG_var['tp_1st_var']/IMERG_hfr['total_var'])*100).plot()

In [ ]:
(((rean_mean_rat1['var_isd']/rean_mean_rat1['var_subD'])*100)+((rean_mean_rat1['var_mdc']/rean_mean_rat1['var_subD'])*100)).plot()

In [ ]:
rean_mean_rat1['rat_mdc'].plot()

In [ ]:
IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})


In [ ]:
sat_mean_rat1 = sat_mean_rat1.clip(0,100)
rean_mean_rat1 = rean_mean_rat1.clip(0,100)

In [ ]:
((sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100).plot()

In [ ]:
import numpy as np # var(1st harm)/var(total)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
from cmap import Colormap
def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(20, 60, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,30))
    ax.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')

result_ds_list = [(((sat_mean_rat1['1st']/100)*(sat_mean_hfr['hfr_tp']/100))*100),
                  (((rean_mean_rat1['1st']/100)*(rean_mean_hfr['hfr_tp']/100))*100)]

def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    # cmap = cmaps.WhiteBlueGreenYellowRed
    levels = 40  # 30  
    cmap = Colormap('colorcet:CET_L17').to_mpl()
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name = 'rat'), 'rat', lsm, extent,True)
    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name = 'rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    ax2_hist.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=11)
    ax2_hist.tick_params(labelsize=9)
    
    # cbar_ticks = np.linspace(0, 100, 11)  # 70, 75, 80, 85, 90, 95, 100
    cbar_ticks = np.linspace(20, 60, 5)  # 70, 75, 80, 85, 90, 95, 100

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    # cax = fig.add_axes([pos1.x1 + 0.11,  (pos1.y1-pos2.y0)/3, 0.02, (pos1.y1-pos2.y0)*(2/3)])  #  
    cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0)-.06,0.03])
    # cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0),0.02])  #  
    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='both', ticks=cbar_ticks)
    cbar.set_label(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import copy

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1,rean_mean_rat1['rat']]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 40
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(b) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)

    ax1_hist.set_xlim(10, 70)  #  x  
    ax1_hist.set_xticks([20,40,60])
    ax1_hist.set_xlabel(None)

    
    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(c) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)    
    ax2_hist.set_xlim(10, 70)  #  x  
    ax2_hist.set_xticks([20,40,60])
    ax2_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -30, 30, 'diff')
    ax3.set_title('(d) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent)    
    ax3_hist.set_xlim(-20, 20)  #  x  
    ax3_hist.set_xticks([-10,0,10])
    ax3_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(20, 60, 5)  # 70, 75, 80, 85, 90, 95, 100
    
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='both', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)
    # cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    # cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
result_ds_list[0].plot()#(vmin = 20, vmax = 60)

In [ ]:
result_ds_list[1].plot()#(vmin = 20, vmax = 60)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import copy
def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(0, 15, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)
    return pc

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [(sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100,(rean_mean_rat1['var_1st']/rean_mean_rat1['var_total'])*100]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 40
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(b) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)

    ax1_hist.set_xlim(0, 10)  #  x  
    ax1_hist.set_xticks([0,5,10])
    ax1_hist.set_xlabel(None)

    
    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(c) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)    
    ax2_hist.set_xlim(0, 10)  #  x  
    ax2_hist.set_xticks([0,5,10])
    ax2_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -10, 10, 'diff')
    ax3.set_title('(d) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent)    
    ax3_hist.set_xlim(-10, 10)  #  x  
    ax3_hist.set_xticks([-5,0,5])
    ax3_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 15, 6)  # 70, 75, 80, 85, 90, 95, 100
    
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)
    # cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    # cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
np.linspace(0, 15, 6)

In [ ]:
result_ds_list = [(sat_mean_hfr['total_var']*((sat_mean_rat1['1st']/100)*(sat_mean_hfr['hfr_tp']/100)))/9,
                  (rean_mean_hfr['total_var']*((rean_mean_rat1['1st']/100)*(rean_mean_hfr['hfr_tp']/100)))/9]
# var(1st harm)

In [ ]:
(rean_mean_rat1['var_1st']/9).plot(vmin = 0 ,vmax = 0.2)

In [ ]:
(sat_mean_rat1['var_1st']/9).plot(vmin = 0 ,vmax = 0.2)

In [ ]:
cmap_main = Colormap('colorcet:CET_L17').to_mpl()
result_ds_list[0].plot(vmax = 60,vmin = 0,cmap =  cmap_main )

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, 60, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  

    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1['var_1st']*64,
                  rean_mean_rat1['var_1st']*64]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 30
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    ax1_hist.set_xlim(0, 60)
    ax1_hist.set_xlabel(None, fontsize=11)
    # ax1_hist.set_xticks([0,10,20,30,40,50,60])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 60)
    # ax2_hist.set_xticks([0,10,20,30,40,50,60])
    # ax2_hist.set_xlabel(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=11)
    ax2_hist.set_xlabel(None, fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -20, 20, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)
    ax3_hist.set_xlim(-20, 20)
    ax3_hist.set_xticks([-10,0,10])
    ax3_hist.set_xlabel(None, fontsize=11)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 60, 7)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/d^2$]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/d^2$]', fontsize=12)
    # cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    # cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, .2, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  

    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1['var_1st']/9,
                  rean_mean_rat1['var_1st']/9]
# var(1st harm)
sat_mean_rat1 = sat_mean_rat1.clip(0,100)
rean_mean_rat1 = rean_mean_rat1.clip(0,100)
def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 30
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    ax1_hist.set_xlim(0, 0.2)
    ax1_hist.set_xlabel(None, fontsize=11)
    ax1_hist.set_xticks([0,0.1,0.2])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 0.2)
    ax2_hist.set_xticks([0,0.1,0.2])
    # ax2_hist.set_xlabel(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=11)
    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -1, 1, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)
    ax3_hist.set_xlim(-.1, .1)
    # ax3_hist.set_xlabel(None, fontsize=11)
    ax3_hist.set_xticks([-.1,0.,.1])
    ax3_hist.set_xlabel(None, fontsize=11)
    # 1.     (0 to .2)
    cbar_ticks = np.linspace(0, .2, 7)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    # cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    # cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
(result_ds_list[1]-result_ds_list[0]).plot(vmin = -3, vmax = 3,cmap = 'RdBu')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, 1.5, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / SFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,30))
    ax.set_xlabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=12)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=12, frameon=False, facecolor='none',bbox_to_anchor= (.0,.96),loc = 'lower left')
def main():
    # extent = [-150, 150, 0, 60]
    extent = [-180., 180., -60.001, 60.001]
    fig = plt.figure(figsize=(11, 8.5))
    # cmap = 'RdBu_r'
    levels = 40  # 30  
    cmap = Colormap('colorcet:CET_L17').to_mpl()
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().y1-ax1.get_position().y0])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name = 'rat'), 'rat', lsm, extent,True)
    ax1_hist.set_xlim(0, 1)  #  x  
    ax1_hist.set_xticks([0.5])
    ax1_hist.set_xlabel(None)
    ax1_hist.tick_params(labelsize=9)

    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name = 'rat'), extent, cmap, levels, 'rat', obs_hfr)
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12,  ax2.get_position().y1-ax2.get_position().y0])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name = 'rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 1)  #  x  
    ax2_hist.set_xticks([0.5])
    ax2_hist.set_xlabel(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=11)
    ax2_hist.tick_params(labelsize=9)
    
    cbar_ticks = np.linspace(0, 1.5, 7)  

    pos1 = ax1.get_position()
    pos2 = ax2.get_position()
    
    cax = fig.add_axes([pos2.x0+.03, pos2.y0 - 0.06,(pos2.x1-pos2.x0)-.06,0.03])
    cbar = fig.colorbar(scatter2, cax=cax, orientation='horizontal', 
                       extend='max', ticks=cbar_ticks)
    cbar.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    cbar.ax.tick_params(labelsize=11)
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_map_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()
if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, 1.5, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  

    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 30
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    ax1_hist.set_xlim(0, 1)
    ax1_hist.set_xlabel(None, fontsize=11)
    ax1_hist.set_xticks([0.5])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 1)
    ax2_hist.set_xticks([0.5])
    ax2_hist.set_xlabel(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=11)
    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -1, 1, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)
    ax3_hist.set_xlim(-1, 1)
    ax3_hist.set_xlabel(None, fontsize=11)
    ax3_hist.set_xticks([-.5,0.,.5])
    ax3_hist.set_xlabel(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=11)
    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 1.5, 7)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    # cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    # cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import copy

result_ds_list = [sat_mean_hfr['high_var']/9,rean_mean_hfr['high_var']/9]

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 40
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(c) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    ax1_hist.set_xlim(0, 1)
    ax1_hist.set_xlabel(None, fontsize=11)
    ax1_hist.set_xticks([0.5])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(d) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 1)
    ax2_hist.set_xticks([0.5])
    ax2_hist.set_xlabel(r'$\sigma_{subD}^2$ [$mm^2/h^2$]', fontsize=11)
    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -1, 1, 'diff')
    ax3.set_title('(e) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 1.5, 7)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{subD}^2$ [$mm^2/h^2$]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    # cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    # cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    # cbar2.set_label(r'$\sigma_{subD}^2$ [$mm^2/h^2$]', fontsize=12)
    cax2 = fig.add_axes([pos3.x1+.01, pos3.y0,0.02,(pos3.y1-pos3.y0)])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='vertical', extend='both')
    cbar2.set_label(r'$\sigma_{subD}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

### Bar Plot


In [ ]:
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
averaged_data

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
condition = condition&(~obs_result['left_mask'])  #    test

In [ ]:
obs_hfr = averaged_data
IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc')
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_26h_30.nc')
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_26h_30.nc')
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_26h_30.nc')
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_26h_30.nc')

In [ ]:
obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr_26h_30.nc')#.rename(hfr='hfr_tp')

IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc')
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_26h_30.nc')
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_26h_30.nc')
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_26h_30.nc')
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_26h_30.nc')

ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr_26h_30.nc')
JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr_26h_30.nc')
MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr_26h_30.nc')

In [ ]:
obs_var= xr.open_dataset('${DATA_DIR}/obs/obs_var.nc')
#satellite
IMERG_var= xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var.nc')
TRMM_var= xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var.nc')
CMORPH_var= xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var.nc')
GSMaP_var= xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var.nc')
MSWEP_var= xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var.nc')
#reanalysis
ERA5_var= xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var.nc')
JRA3Q_var= xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var.nc')
MERRA2_var= xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var.nc')

In [ ]:
rdir = '${DATA_DIR}/'
obs_result = xr.open_dataset(rdir+'obs/obs_gridded.nc')
imerg = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')
cmorph = xr.open_dataset(rdir+'CMORPH/CMORPH_diurnal_result_3hourly.nc')
mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_new_1.nc')
# mswep = xr.open_dataset(rdir + 'MSWEP/MSWEP_diurnal_result_3hourly_old.nc')
gsmap = xr.open_dataset(rdir + 'GSMaP/GSMaP_diurnal_result_3hourly.nc')
trmm = xr.open_dataset(rdir + 'TRMM/TRMM_diurnal_result_3hourly.nc')
narr = xr.open_dataset('${DATA_DIR}/NARR/ana/NARR_diurnal_result_3hourly_saperate.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
# jra55 = xr.open_dataset(rdir + 'JRA-55/JRA55_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
merra2 = xr.open_dataset(rdir + 'MERRA2/MERRA2_diurnal_result_3hourly.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')
era5 = xr.open_dataset(rdir + 'ERA5/ERA5_diurnal_result_3hourly_new.nc').rename(tp_lst='lst',tp_amp = 'amplitude',tp_SAD='SAD',tp_1st='1st',tp_2nd='2nd')

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'Obs': obs_hfr.where(condition)  # 'obs_hfr_fil' DataArray  
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'Obs': obs_result.where(condition)  # 'obs_hfr_fil' DataArray  
}


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','JRA-3Q','ERA5', 'Obs']

dataframes_list = []

#  (plot_order) 
for name in plot_order:
    print(f"Processing {name}...")
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    # 1D 
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    # 1. (Value) 2. (Dataset)  DataFrame 
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

print("\n--- Data points per group ---")
print(df_long.groupby('Dataset').size())

fig, axes = plt.subplots(1, 1, figsize=(5, 5))

sns.barplot(
    x='Dataset',  # X
    y='1st Var',  # Y
    data=df_long,
    order=plot_order,  # X 
    ax=axes
)
axes.set_title('Bar Plot (Mean & 95% CI) of All Grid Points')
axes.set_ylabel('Mean Total Var Value')
axes.set_xlabel('Data Source')
plt.tight_layout()
plt.show()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP', 'Obs']

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'Obs': obs_hfr_fil
}

dataframes_list = []

for name in plot_order:
    print(f"Processing {name}...")
    da = datasets_to_plot[name]
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total,
        'High Var': all_values_valid_high,
        'Dataset': name
    })
    
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var']].mean()
df_summary = df_summary.reindex(plot_order)  # plot_order  

fig, ax = plt.subplots(figsize=(8, 6))

# Stacked bar plot 
x = np.arange(len(plot_order))
width = 0.6

# Total Var 
bars1 = ax.bar(x, df_summary['Total Var'], width, label='Total Var', color='steelblue')

# High Var Total Var  
bars2 = ax.bar(x, df_summary['High Var'], width, 
                label='High Var', color='coral')

ax.set_xlabel('Data Source', fontsize=12)
ax.set_ylabel('Mean Value', fontsize=12)
ax.set_title('Stacked Bar Plot: Total Var vs High Var', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(plot_order)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
(7/8)*9

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var'] - df_summary['High Var'], width, 
               label=r'$\sigma_{\text{Low freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    percentage = (high_val / total_val) * 100
    
    text_high = low_val+ high_val/ 2
    ax.text(i, text_high, f'{high_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    text_low = low_val/ 2
    ax.text(i, text_low, f'{low_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
legend_elements = [
    Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
]
ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_new_eseo.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))


datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot_rat1[name]
    
    all_values_total = da['var_total'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['var_subD'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var'] - df_summary['High Var'], width, 
               label=r'$\sigma_{\text{Low freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    percentage = (high_val / total_val) * 100
    
    text_high = low_val+ high_val/ 2
    ax.text(i, text_high, f'{high_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    text_low = low_val/ 2
    ax.text(i, text_low, f'{low_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
legend_elements = [
    Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
]
ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_new_eseo.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
obs_var= xr.open_dataset('${DATA_DIR}/obs/obs_var.nc')  #     .
#satellite
IMERG_var= xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var.nc')
TRMM_var= xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var.nc')
CMORPH_var= xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var.nc')
GSMaP_var= xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var.nc')
MSWEP_var= xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var.nc')
#reanalysis
ERA5_var= xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var.nc')
JRA3Q_var= xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var.nc')
MERRA2_var= xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var.nc')

In [ ]:
obs_var.where(condition)['tp_1st_var'].plot(vmin = 0,vmax= 1.5,cmap = Colormap('colorcet:CET_L17').to_mpl())

In [ ]:
obs_var.where(condition)['tp_1st_var'].mean()

In [ ]:
IMERG_var.where(condition)['tp_1st_var'].mean()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}
datasets_to_plot_var = {
    'IMERG': IMERG_var.where(condition),
    'TRMM': TRMM_var.where(condition),
    'CMORPH': CMORPH_var.where(condition),
    'GSMaP': GSMaP_var.where(condition),
    'MSWEP': MSWEP_var.where(condition),
    'JRA-3Q': JRA3Q_var.where(condition),
    'ERA5': ERA5_var.where(condition),
    'MERRA-2': MERRA2_var.where(condition),
    'Obs': obs_var.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['total_var'].values.flatten()
    all_values_high = da['high_var'].values.flatten()
    all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    
    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st
                                                                         )
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_high = all_values_high[~alllc]
    all_values_valid_1st = all_values_1st[~alllc]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st,
        # '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var','1st Var']].mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
df_summary

In [ ]:
# 2. compute ratios (%) (: %)
# subD = High Var, 1st = 1st Var, total = Total Var
analysis_df = pd.DataFrame(index=[
    r'$\sigma^2_{subD} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{subD}$ (%)'
])

for col in df_summary.index:
    total = df_summary.loc[col, 'Total Var']
    subD = df_summary.loc[col, 'High Var']
    first = df_summary.loc[col, '1st Var']
    
    analysis_df[col] = [
        (subD / total) * 100,
        (first / total) * 100,
        (first / subD) * 100
    ]

# 3. (Table) 
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = analysis_df.round(2).values
columns = analysis_df.columns
rows = analysis_df.index

table = ax.table(cellText=table_data,
                 rowLabels=rows,
                 colLabels=columns,
                 cellLoc='center',
                 loc='center',
                 colColours=['#f2f2f2']*len(columns),
                 rowColours=['#f2f2f2']*len(rows))

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.5)  #    (, )

# plt.title('Ratio Analysis of Precipitation Variance Components', fontsize=14, pad=20)
plt.show()

In [ ]:
# 2. compute ratios (%) (: %)
# subD = High Var, 1st = 1st Var, total = Total Var
analysis_df = pd.DataFrame(index=[
    r'$\sigma^2_{subD} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{subD}$ (%)'
])

for col in df_summary.index:
    total = df_summary.loc[col, 'Total Var']
    subD = df_summary.loc[col, 'High Var']
    first = df_summary.loc[col, '1st Var']
    
    analysis_df[col] = [
        (subD / total) * 100,
        (first / total) * 100,
        (first / subD) * 100
    ]

# 3. (Table) 
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = analysis_df.round(2).values
columns = analysis_df.columns
rows = analysis_df.index

table = ax.table(cellText=table_data,
                 rowLabels=rows,
                 colLabels=columns,
                 cellLoc='center',
                 loc='center',
                 colColours=['#f2f2f2']*len(columns),
                 rowColours=['#f2f2f2']*len(rows))

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.5)  #    (, )

# plt.title('Ratio Analysis of Precipitation Variance Components', fontsize=14, pad=20)
plt.show()

In [ ]:
# 2. compute ratios (%) (: %) #exclude obs where rat1 < 70
# subD = High Var, 1st = 1st Var, total = Total Var
analysis_df = pd.DataFrame(index=[
    r'$\sigma^2_{subD} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{total}$ (%)',
    r'$\sigma^2_{1st} / \sigma^2_{subD}$ (%)'
])

for col in df_summary.index:
    total = df_summary.loc[col, 'Total Var']
    subD = df_summary.loc[col, 'High Var']
    first = df_summary.loc[col, '1st Var']
    
    analysis_df[col] = [
        (subD / total) * 100,
        (first / total) * 100,
        (first / subD) * 100
    ]

# 3. (Table) 
fig, ax = plt.subplots(figsize=(8, 4))
ax.axis('off')

table_data = analysis_df.round(2).values
columns = analysis_df.columns
rows = analysis_df.index

table = ax.table(cellText=table_data,
                 rowLabels=rows,
                 colLabels=columns,
                 cellLoc='center',
                 loc='center',
                 colColours=['#f2f2f2']*len(columns),
                 rowColours=['#f2f2f2']*len(rows))

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.5)  #    (, )

# plt.title('Ratio Analysis of Precipitation Variance Components', fontsize=14, pad=20)
plt.show()

In [ ]:
import xarray as xr  # # variance computed from actual 1st harmonic
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }

# datasets_to_plot_rat1 = {
#     'IMERG': imerg.where(condition),
#     'TRMM': trmm.where(condition),
#     'CMORPH': cmorph.where(condition),
#     'GSMaP': gsmap.where(condition),
#     'MSWEP': mswep.where(condition),
#     'JRA-3Q': jra55.where(condition),
#     'ERA5': era5.where(condition),
#     'MERRA-2': merra2.where(condition),
#     'Obs': obs_result.where(condition)  
# }


# total_color = '#F2BF5E'
# high_color = '#F28379'

# dataframes_list = []
# for name in plot_order:
#     da = datasets_to_plot[name]
#     var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
#     all_values_total = da['total_var'].values.flatten()
#     all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
#     all_values_high = da['high_var'].values.flatten()
#     all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
#     all_values_1st = var_1st.values.flatten()
#     all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

#     # temp_df = pd.DataFrame({
#     #     'Total Var': all_values_valid_total/9.,
#     #     'High Var': all_values_valid_high/9.,
#     #     'Dataset': name
#     # })
#     temp_df = pd.DataFrame({
#         'Total Var': all_values_valid_total/9.,
#         'High Var': all_values_valid_high/9.,
#         '1st Var' : all_values_valid_1st/9.,
# 'Dataset': name  # 'Dataset'     
#     })
#     dataframes_list.append(temp_df)

# df_long = pd.concat(dataframes_list)

# def calculate_ci(data, confidence=0.95):
# """95%  """
#     n = len(data)
#     mean = np.mean(data)
# se = stats.sem(data)  #  
#     ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
#     return mean, ci

# df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
# df_summary = df_summary.reindex(plot_order)

# # CI 
# ci_total = []
# ci_high = []

# for dataset in plot_order:
#     data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
#     data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
#     _, ci_t = calculate_ci(data_total)
#     _, ci_h = calculate_ci(data_high)
    
#     ci_total.append(ci_t)
#     ci_high.append(ci_h)

# ci_total = np.array(ci_total)
# ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var'] - df_summary['High Var'], width, 
               label=r'$\sigma_{\text{Low freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=0)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    percentage = (high_val / total_val) * 100
    
    text_high = low_val+ high_val/ 2
    ax.text(i, text_high, f'{high_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    text_low = low_val/ 2
    ax.text(i, text_low, f'{low_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
legend_elements = [
    Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
            dpi=400, bbox_inches='tight')

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'
fst_color = '#FCF42F'
dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7/0.5, 5/0.5))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
# bars1 = ax.bar(x, df_summary['Total Var'], width, 
#                label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
#                alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       color=fst_color, label=r'$\sigma_{\text{1st}}$',
       # edgecolor='black', 
       # hatch='///', 
       alpha=0.5, 
       error_kw={'elinewidth': 2})

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    fst_val = df_summary.loc[dataset, '1st Var']

    low_val = total_val - high_val
    
    percentage = (fst_val / high_val) * 100
    
    # text_high = fst_val+(high_val-fst_val)/ 2
    ax.text(i, high_val, f'{high_val:.1f}',  
           ha='center', va='bottom', fontsize=20, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    text_low = fst_val/ 2
    ax.text(i, text_low, f'{fst_val:.1f}',  
           ha='center', va='center', fontsize=20, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, high_val+.05, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=25, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.2,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=30)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=18)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.2)
ax.set_yticklabels([0.,0.2,0.4,0.6,0.8,1.0,1.2],fontsize=25)
ax.set_xticklabels(plot_order, rotation=45, fontsize=25)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=.6, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=fst_color, alpha=1., label=r'$\sigma^2_{\text{1st}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=25, loc='upper right', frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    fst_val = df_summary.loc[dataset, '1st Var']

    low_val = total_val - high_val
    
    percentage = (fst_val / high_val) * 100

    text_high = low_val+ fst_val/ 2
    ax.text(i, text_high, f'{high_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])

    ax.text(i, total_val, f'{total_val:.2f}',  
           ha='center', va='bottom', fontsize=11, color='gray', 
           path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    fst_val = df_summary.loc[dataset, '1st Var']
    
    percentage = (fst_val / high_val) * 100
    ########## 1st text
    text_high = low_val+ fst_val/ 2
    ax.text(i, text_high, f'{fst_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ########## low text
    ax.text(i, low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{high_val:.2f}',  
           ha='center', va='bottom', fontsize=11, color='gray', 
           path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    fst_val = df_summary.loc[dataset, '1st Var']
    
    percentage = (fst_val / high_val) * 100
    ########## 1st text
    text_high = low_val+ fst_val/ 2
    ax.text(i, text_high, f'{fst_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ########## low text
    ax.text(i, low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{high_val:.2f}',  
           ha='center', va='bottom', fontsize=11, color='gray', 
           path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
# import xarray as xr
# import numpy as np
# import matplotlib.pyplot as plt
# import pandas as pd
# from scipy import stats
# from matplotlib.patches import Patch
# import matplotlib.patheffects as pe
# plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
# positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }

# datasets_to_plot_rat1 = {
#     'IMERG': imerg.where(condition),
#     'TRMM': trmm.where(condition),
#     'CMORPH': cmorph.where(condition),
#     'GSMaP': gsmap.where(condition),
#     'MSWEP': mswep.where(condition),
#     'JRA-3Q': jra55.where(condition),
#     'ERA5': era5.where(condition),
#     'MERRA-2': merra2.where(condition),
#     'Obs': obs_result.where(condition)  
# }


# total_color = '#F2BF5E'
# high_color = '#F28379'

# dataframes_list = []
# for name in plot_order:
#     da = datasets_to_plot[name]
#     var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
#     all_values_total = da['total_var'].values.flatten()
#     all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
#     all_values_high = da['high_var'].values.flatten()
#     all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
#     all_values_1st = var_1st.values.flatten()
#     all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

#     # temp_df = pd.DataFrame({
#     #     'Total Var': all_values_valid_total/9.,
#     #     'High Var': all_values_valid_high/9.,
#     #     'Dataset': name
#     # })
#     temp_df = pd.DataFrame({
#         'Total Var': all_values_valid_total/9.,
#         'High Var': all_values_valid_high/9.,
#         '1st Var' : all_values_valid_1st/9.,
# 'Dataset': name  # 'Dataset'     
#     })
#     dataframes_list.append(temp_df)

# df_long = pd.concat(dataframes_list)

# def calculate_ci(data, confidence=0.95):
# """95%  """
#     n = len(data)
#     mean = np.mean(data)
# se = stats.sem(data)  #  
#     ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
#     return mean, ci

# df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
# df_summary = df_summary.reindex(plot_order)

# # CI 
# ci_total = []
# ci_high = []

# for dataset in plot_order:
#     data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
#     data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
#     _, ci_t = calculate_ci(data_total)
#     _, ci_h = calculate_ci(data_high)
    
#     ci_total.append(ci_t)
#     ci_high.append(ci_h)

# ci_total = np.array(ci_total)
# ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    fst_val = df_summary.loc[dataset, '1st Var']
    
    percentage = (fst_val / high_val) * 100
    ########## 1st text
    text_high = low_val+ fst_val/ 2
    ax.text(i, text_high, f'{fst_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ########## low text
    ax.text(i, low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{high_val:.2f}',  
           ha='center', va='bottom', fontsize=11, color='gray', 
           path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
imerg['var_total'].plot()

In [ ]:
imerg

In [ ]:
(imerg['var_1st']-IMERG_var['tp_1st_var']).max()

In [ ]:
(imerg['var_1st']-IMERG_var['tp_1st_var']).min()

In [ ]:
IMERG_var['tp_1st_var'].plot()

In [ ]:
IMERG_var= xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var.nc')
IMERG_var

In [ ]:
(imerg['var_subD']-imerg['var_mdc']-imerg['var_isd']).plot()

In [ ]:
df_summary

In [ ]:
datasets_to_plot_rat1['IMERG']['var_1st'].plot()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}
datasets_to_plot_var = {
    'IMERG': IMERG_var.where(condition),
    'TRMM': TRMM_var.where(condition),
    'CMORPH': CMORPH_var.where(condition),
    'GSMaP': GSMaP_var.where(condition),
    'MSWEP': MSWEP_var.where(condition),
    'JRA-3Q': JRA3Q_var.where(condition),
    'ERA5': ERA5_var.where(condition),
    'MERRA-2': MERRA2_var.where(condition),
    'Obs': obs_var.where(condition)
}

total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    
    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st
                                                                         )
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_high = all_values_high[~alllc]
    all_values_valid_1st = all_values_1st[~alllc]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st,
        # '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var','1st Var']].mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
df_summary

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}
datasets_to_plot_var = {
    'IMERG': IMERG_var.where(condition),
    'TRMM': TRMM_var.where(condition),
    'CMORPH': CMORPH_var.where(condition),
    'GSMaP': GSMaP_var.where(condition),
    'MSWEP': MSWEP_var.where(condition),
    'JRA-3Q': JRA3Q_var.where(condition),
    'ERA5': ERA5_var.where(condition),
    'MERRA-2': MERRA2_var.where(condition),
    'Obs': obs_var.where(condition)
}

total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_1st = datasets_to_plot[name]['var_mdc'].values.flatten()
    
    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st
                                                                         )
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_high = all_values_high[~alllc]
    all_values_valid_1st = all_values_1st[~alllc]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9,
        # '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var','1st Var']].mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
0.654475

In [ ]:
df_summary

In [ ]:
df_summary

In [ ]:
(imerg['var_1st']+imerg['var_2nd']+imerg['var_3rd']+imerg['var_mdc_res']).plot()

In [ ]:
(imerg['var_mdc']-(imerg['var_1st']+imerg['var_2nd']+imerg['var_3rd']+imerg['var_mdc_res'])).plot()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))


# datasets_to_plot_rat1 = {
#     'IMERG': imerg.where(condition),
#     'TRMM': trmm.where(condition),
#     'CMORPH': cmorph.where(condition),
#     'GSMaP': gsmap.where(condition),
#     'MSWEP': mswep.where(condition),
#     'JRA-3Q': jra55.where(condition),
#     'ERA5': era5.where(condition),
#     'MERRA-2': merra2.where(condition),
#     'Obs': obs_result.where(condition)  
# }

# datasets_to_plot_var = {
#     'IMERG': IMERG_var.where(condition),
#     'TRMM': TRMM_var.where(condition),
#     'CMORPH': CMORPH_var.where(condition),
#     'GSMaP': GSMaP_var.where(condition),
#     'MSWEP': MSWEP_var.where(condition),
#     'JRA-3Q': JRA3Q_var.where(condition),
#     'ERA5': ERA5_var.where(condition),
#     'MERRA-2': MERRA2_var.where(condition),
#     'Obs': obs_var.where(condition)
# }

# total_color = '#F2BF5E'
# high_color = '#F28379'

# dataframes_list = []
# for name in plot_order:
#     da = datasets_to_plot_rat1[name]
    
#     all_values_total = da['var_total'].values.flatten()
#     all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
#     all_values_high = da['var_subD'].values.flatten()
#     all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
#     all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
#     all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

#     # temp_df = pd.DataFrame({
#     #     'Total Var': all_values_valid_total/9.,
#     #     'High Var': all_values_valid_high/9.,
#     #     'Dataset': name
#     # })
#     temp_df = pd.DataFrame({
#         'Total Var': all_values_valid_total/9.,
#         'High Var': all_values_valid_high/9.,
#         '1st Var' : all_values_valid_1st/9.,
# 'Dataset': name  # 'Dataset'     
#     })
#     dataframes_list.append(temp_df)

# df_long = pd.concat(dataframes_list)

# def calculate_ci(data, confidence=0.95):
# """95%  """
#     n = len(data)
#     mean = np.mean(data)
# se = stats.sem(data)  #  
#     ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
#     return mean, ci

# df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
# df_summary = df_summary.reindex(plot_order)

# # CI 
# ci_total = []
# ci_high = []

# for dataset in plot_order:
#     data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
#     data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
#     _, ci_t = calculate_ci(data_total)
#     _, ci_h = calculate_ci(data_high)
    
#     ci_total.append(ci_t)
#     ci_high.append(ci_h)

# ci_total = np.array(ci_total)
# ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    fst_val = df_summary.loc[dataset, '1st Var']
    
    percentage = (fst_val / high_val) * 100
    ########## 1st text
    text_high = low_val+ fst_val/ 2
    ax.text(i, text_high, f'{fst_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ########## low text
    ax.text(i, low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{high_val:.2f}',  
           ha='center', va='bottom', fontsize=11, color='gray', 
           path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

### Updated Version


### Figure 2


In [ ]:
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')

In [ ]:
datasets_to_plot_rat1['IMERG']['rat_1st'].mean()

In [ ]:
imerg_test = xr.open_dataset(rdir+'IMERG/IMREG_diurnal_result_3hourly_new.nc')

In [ ]:
imerg_test['1st'].where(condition).mean()

In [ ]:
(imerg_test['1st']-datasets_to_plot_rat1['IMERG']['rat_1st']).plot(vmax= 5,vmin = -5)

In [ ]:
datasets_to_plot_rat1['IMERG']['rat_1st'].mean()

In [ ]:
obs_test = xr.open_dataset('${DATA_DIR}/obs/obs_gridded.nc')

In [ ]:
obs_test['1st'].plot()

In [ ]:
(datasets_to_plot_rat1['Obs']['rat_1st']-obs_test['1st']).plot(vmax= 5,vmin = -5)

In [ ]:
obs_result['rat_1st'].where(condition).plot()

In [ ]:
obs_result['rat_1st'].where(condition).mean()

In [ ]:
datasets_to_plot_rat1['Obs']['rat_1st'].mean()

In [ ]:
datasets_to_plot_rat1['IMERG']['rat_2nd'].mean()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }
# datasets_to_plot_var = {
#     'IMERG': IMERG_var.where(condition),
#     'TRMM': TRMM_var.where(condition),
#     'CMORPH': CMORPH_var.where(condition),
#     'GSMaP': GSMaP_var.where(condition),
#     'MSWEP': MSWEP_var.where(condition),
#     'JRA-3Q': JRA3Q_var.where(condition),
#     'ERA5': ERA5_var.where(condition),
#     'MERRA-2': MERRA2_var.where(condition),
#     'Obs': obs_var.where(condition)
# }

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_low = da['var_Low'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_irreg = da['var_isd'].values.flatten()
    all_values_reg = da['var_mdc'].values.flatten()


    all_values_1st = da['var_1st'].values.flatten()
    all_values_2nd = da['var_2nd'].values.flatten()
    all_values_3rd = da['var_3rd'].values.flatten()
    all_values_res = da['var_mdc_res'].values.flatten()

    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st)
    
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_low = all_values_low[~alllc]
    all_values_valid_high = all_values_high[~alllc]

    all_values_valid_irreg = all_values_irreg[~alllc]
    all_values_valid_reg = all_values_reg[~alllc]

    all_values_valid_1st = all_values_1st[~alllc]
    all_values_valid_2nd = all_values_2nd[~alllc]
    all_values_valid_3rd = all_values_3rd[~alllc]
    all_values_valid_res = all_values_res[~alllc]


    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'Low Var': all_values_valid_low/9.,
        'High Var': all_values_valid_high/9.,

        'reg Var': all_values_valid_reg/9.,
        'irreg Var': all_values_valid_irreg/9.,

        '1st Var' : all_values_valid_1st/9,
        '2nd Var' : all_values_valid_2nd/9,
        '3rd Var' : all_values_valid_3rd/9,
        'res Var' : all_values_valid_res/9,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset').mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mpltern

# ──   ─────────────────────────────────────────
# df_summary   
datasets = ['Obs', 'IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP', 'ERA5', 'JRA-3Q', 'MERRA-2']

reg_var = np.array([
    df_summary.loc['Obs',     'reg Var'],
    df_summary.loc['IMERG',   'reg Var'],
    df_summary.loc['TRMM',    'reg Var'],
    df_summary.loc['CMORPH',  'reg Var'],
    df_summary.loc['GSMaP',   'reg Var'],
    df_summary.loc['MSWEP',   'reg Var'],
    df_summary.loc['ERA5',    'reg Var'],
    df_summary.loc['JRA-3Q',  'reg Var'],
    df_summary.loc['MERRA-2', 'reg Var'],
])
var_1st = np.array([df_summary.loc[d, '1st Var'] for d in datasets])
var_2nd = np.array([df_summary.loc[d, '2nd Var'] for d in datasets])
var_3rd = np.array([df_summary.loc[d, '3rd Var'] for d in datasets])

#   (%,  100)
r_1st    = var_1st / reg_var * 100  # right axis ()
r_2nd    = var_2nd / reg_var * 100  # top axis   ()
r_higher = 100 - r_1st - r_2nd  # left axis  ()

# ──   ─────────────────────────────────────────
colors_hex = {
    'top':   '#F20505',   # 2nd  (red)
    'left':  '#0BD904',   # higher (green)
    'right': '#1B3BF2'    # 1st  (blue)
}

def rgb_colormap(t, l, r, tmin, lmin, rmin, tmax, lmax, rmax, colors_hex):
    def hex2rgb(h):
        h = h.lstrip('#')
        return np.array([int(h[i:i+2], 16)/255 for i in (0, 2, 4)])
    ct = hex2rgb(colors_hex['top'])
    cl = hex2rgb(colors_hex['left'])
    cr = hex2rgb(colors_hex['right'])
    tn = np.clip((t - tmin) / (tmax - tmin), 0, 1)
    ln = np.clip((l - lmin) / (lmax - lmin), 0, 1)
    rn = np.clip((r - rmin) / (rmax - rmin), 0, 1)
    total = tn + ln + rn
    total = np.where(total == 0, 1, total)
    wt, wl, wr = tn/total, ln/total, rn/total
    colors = wt[:,None]*ct + wl[:,None]*cl + wr[:,None]*cr
    return np.clip(colors, 0, 1)

# ──  Dirichlet scatter ──────────────────────────────
np.random.seed(42)
from numpy.random import dirichlet
samples = dirichlet([10, 10, 10], size=3000) * 100  # (N, 3): t, l, r

tmin, tmax = 10, 35
lmin, lmax =  0, 25
rmin, rmax = 65, 90

mask = ((samples[:,0] >= tmin) & (samples[:,0] <= tmax) &
        (samples[:,1] >= lmin) & (samples[:,1] <= lmax) &
        (samples[:,2] >= rmin) & (samples[:,2] <= rmax))
t_bg, l_bg, r_bg = samples[mask,0], samples[mask,1], samples[mask,2]
bg_colors = rgb_colormap(t_bg, l_bg, r_bg, tmin, lmin, rmin, tmax, lmax, rmax, colors_hex)

# ── /  ──────────────────────────────────────
marker_style = {
    'Obs':     ('*', 'black',   250),
    'IMERG':   ('s', '#C2E5F2', 180),
    'TRMM':    ('D', '#C2E5F2', 150),
    'CMORPH':  ('^', '#C2E5F2', 180),
    'GSMaP':   ('X', '#C2E5F2', 180),
    'MSWEP':   ('P', '#C2E5F2', 180),
    'ERA5':    ('o', '#F24405', 180),
    'JRA-3Q':  ('v', '#F24405', 180),
    'MERRA-2': ('h', '#F24405', 180),
}

# ──  ────────────────────────────────────────────────
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(projection='ternary', ternary_sum=100.0)

ax.scatter(t_bg, l_bg, r_bg, c=bg_colors, s=25, marker='D', zorder=0, alpha=0.6)

#  scatter
for i, name in enumerate(datasets):
    mk, fc, sz = marker_style[name]
    ax.scatter(r_2nd[i], r_higher[i], r_1st[i],
               marker=mk, color=fc, edgecolor='black',
               s=sz, zorder=5, lw=0.8, alpha=0.9, label=name)

ax.set_ternary_lim(tmin, tmax, lmin, lmax, rmin, rmax)

ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

tick_interval = 5
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval),
                   labels=np.arange(tmin, tmax+1, tick_interval), fontsize=11)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval),
                   labels=np.arange(lmin, lmax+1, tick_interval), fontsize=11)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval),
                   labels=np.arange(rmin, rmax+1, tick_interval), fontsize=11)

ax.set_tlabel('2nd harmonic amplitude [%]',    fontsize=12, color=colors_hex['top'])
ax.set_llabel('Higher-order harmonic amplitude [%]', fontsize=12, color=colors_hex['left'])
ax.set_rlabel('1st harmonic amplitude [%]',    fontsize=12, color=colors_hex['right'])

ax.grid(True, alpha=0.4)
ax.legend(loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=12, frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
df_summary

In [ ]:
print(df_summary)

In [ ]:
def get_harmonic_ratios(df_summary):
    """
    Compute 1st, 2nd, higher-order harmonic ratios from df_summary
    
    Returns
    -------
    df_ratio : DataFrame
        columns: '1st [%]', '2nd [%]', 'higher [%]'
        Denominator: reg Var
    """
    reg = df_summary['reg Var']
    
    r1st    = df_summary['1st Var'] / reg * 100
    r2nd    = df_summary['2nd Var'] / reg * 100
    r3rd    = df_summary['3rd Var'] / reg * 100
    rhigher = 100 - r1st - r2nd  # 3rd   

    #  3rd   :
    # rhigher = r3rd + df_summary['res Var'] / reg * 100

    df_ratio = pd.DataFrame({
        '1st [%]':    r1st,
        '2nd [%]':    r2nd,
        'higher [%]': rhigher,
    })
    
    assert np.allclose(df_ratio.sum(axis=1), 100), "sum is not 100%"
    
    return df_ratio

df_ratio = get_harmonic_ratios(df_summary)

In [ ]:
df_ratio

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
# bars1 = ax.bar(x, df_summary['Total Var'], width, 
#                label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
#                alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})

bars3 = ax.bar(x, 
       df_summary['reg Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['reg Var'],                 
       color='#5C9EAD', 
       alpha=0.6, 
       linewidth=.3, label=r'$\sigma_{\text{reg}}$')

bars4 = ax.bar(x, 
       df_summary['irreg Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color=total_color, 
       alpha=0.6, 
       linewidth=.3, label=r'$\sigma_{\text{irreg}}$')


for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    reg_val = df_summary.loc[dataset, 'reg Var']
    irreg_val = df_summary.loc[dataset, 'irreg Var']
    low_val = total_val - high_val
    # fst_val = df_summary.loc[dataset, '1st Var']
    
    # percentage = (fst_val / high_val) * 100
    ########## irreg text
    ax.text(i, low_val+irreg_val/2, f'{irreg_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    ########## reg text
    # ax.text(i, total_val, f'{reg_val:.3f}',  
    #        ha='center', va='bottom', fontsize=11, color='gray', 
    #        path_effects=[pe.withStroke(linewidth=1., foreground='black', alpha=0.6)])
    ax.text(i, total_val, f'{reg_val:.3f}',  
           ha='center', va='bottom', fontsize=9, color='lightgray', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    ########## low text
    ax.text(i, low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    # Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    # Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor='#5C9EAD', alpha=0.6, label=r'$\sigma^2_{\text{reg}}$'),
    Patch(facecolor=total_color, alpha=0.6, label=r'$\sigma^2_{\text{irreg}}$'),
    Patch(facecolor=high_color, alpha=0.6, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
# plt.show()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
            dpi=400, bbox_inches='tight')

### Figure 4


In [ ]:
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')

In [ ]:
imerg

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }
# datasets_to_plot_var = {
#     'IMERG': IMERG_var.where(condition),
#     'TRMM': TRMM_var.where(condition),
#     'CMORPH': CMORPH_var.where(condition),
#     'GSMaP': GSMaP_var.where(condition),
#     'MSWEP': MSWEP_var.where(condition),
#     'JRA-3Q': JRA3Q_var.where(condition),
#     'ERA5': ERA5_var.where(condition),
#     'MERRA-2': MERRA2_var.where(condition),
#     'Obs': obs_var.where(condition)
# }

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_low = da['var_Low'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_irreg = da['var_isd'].values.flatten()
    all_values_reg = da['var_mdc'].values.flatten()


    all_values_1st = da['var_1st'].values.flatten()
    all_values_2nd = da['var_2nd'].values.flatten()
    all_values_3rd = da['var_3rd'].values.flatten()
    all_values_res = da['var_mdc_res'].values.flatten()

    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st)
    
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_low = all_values_low[~alllc]
    all_values_valid_high = all_values_high[~alllc]

    all_values_valid_irreg = all_values_irreg[~alllc]
    all_values_valid_reg = all_values_reg[~alllc]

    all_values_valid_1st = all_values_1st[~alllc]
    all_values_valid_2nd = all_values_2nd[~alllc]
    all_values_valid_3rd = all_values_3rd[~alllc]
    all_values_valid_res = all_values_res[~alllc]


    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'Low Var': all_values_valid_low/9.,
        'High Var': all_values_valid_high/9.,

        'reg Var': all_values_valid_reg/9.,
        'irreg Var': all_values_valid_irreg/9.,

        '1st Var' : all_values_valid_1st/9,
        '2nd Var' : all_values_valid_2nd/9,
        '3rd Var' : all_values_valid_3rd/9,
        'res Var' : all_values_valid_res/9,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset').mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
df_summary

In [ ]:
from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6
ax.yaxis.set_major_formatter(formatter)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
offset_text = ax.yaxis.get_offset_text()

offset_text.set_horizontalalignment('right')
# offset_text.set_x(-0.15) #     .
# # High Var Total Var   ( )
# bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
#                label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
#                alpha=0.6, error_kw={'elinewidth': 2})

# bars3 = ax.bar(x, 
#        df_summary['reg Var'], 
#        width,
#        bottom=df_summary['Total Var'] - df_summary['reg Var'],                 
#        color='#5C9EAD', 
#        alpha=0.6, 
#        linewidth=.3, label=r'$\sigma_{\text{reg}}$')

bars4 = ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=0,                 
       color='#F28379', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{1st}}$')

bars5 = ax.bar(x, 
       df_summary['2nd Var'], 
       width,
       bottom=df_summary['1st Var'] ,                 
       color='#F2BF5E', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{2nd}}$')
#F28379
bars6 = ax.bar(x, 
       df_summary['3rd Var'], 
       width,
       bottom=df_summary['1st Var']+ df_summary['2nd Var'],                 
       color='#5C9EAD', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{3rd}}$')


for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    reg_val = df_summary.loc[dataset, 'reg Var']
    irreg_val = df_summary.loc[dataset, 'irreg Var']
    fst_val = df_summary.loc[dataset, '1st Var']
    nd_val = df_summary.loc[dataset, '2nd Var']
    rd_val = df_summary.loc[dataset, '3rd Var']

    # percentage = (fst_val / high_val) * 100
    ########## irreg text
    ax.text(i, fst_val/2, f'{fst_val*100:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])

    ax.text(i, fst_val+nd_val/2, f'{nd_val*100:.2f}',  
           ha='center', va='center', fontsize=9, color='lightgray', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    ########## low text
    ax.text(i, fst_val+nd_val+rd_val, f'{rd_val*100:.2f}',  
           ha='center', va='bottom', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    
    # ax.text(i, total_val+.1, f'{percentage:.1f}%',  
    #    ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

# ax.text(-0.5, 1.7,  '(a)',ha='left', va='bottom', fontsize=18)
ax.text(0.001, 1., '(a)', transform=ax.transAxes, 
        fontsize=18, va='bottom', ha='left')
ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)

ax.set_xlim(-0.5, len(plot_order) - 0.5)
# ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    # Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    # Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor='#5C9EAD', alpha=0.6, label=r'$\sigma^2_{\text{3rd}}$'),
    Patch(facecolor='#F2BF5E', alpha=0.6, label=r'$\sigma^2_{\text{2nd}}$'),
    Patch(facecolor='#F28379', alpha=0.6, label=r'$\sigma^2_{\text{1st}}$')
]#F28379
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)
# ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0), useMathText=True)

# offset text(×10⁻²) y  
# ax.yaxis.get_offset_text().set_visible(False)  #   

# y    
# offset = ax.yaxis.get_major_formatter().get_offset()
# ax.text(-0.5, 0.01, offset, transform=ax.transAxes, fontsize=11, ha='left')


ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

In [ ]:
#obs = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
sat_mean_rat1 = xr.concat([imerg,trmm,cmorph, gsmap,mswep],'data').mean('data')#,skipna=False)
rean_mean_rat1 = xr.concat([era5,jra55,merra2],'data').mean('data')#,skipna=False)

In [ ]:
sat_mean_rat1 = sat_mean_rat1.clip(0,100)
rean_mean_rat1 = rean_mean_rat1.clip(0,100)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import copy
def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(0, 15, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)
    return pc

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [(sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100,
                  (rean_mean_rat1['var_1st']/rean_mean_rat1['var_total'])*100]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 40
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(b) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)

    ax1_hist.set_xlim(0, 10)  #  x  
    ax1_hist.set_xticks([0,5,10])
    ax1_hist.set_xlabel(None)

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(c) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)    
    ax2_hist.set_xlim(0, 10)  #  x  
    ax2_hist.set_xticks([0,5,10])
    # ax2_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    ax2_hist.set_xlabel(None)
    # ---   : Difference (MM - MS) ---
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -10, 10, 'diff')
    ax3.set_title('(d) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent)    
    ax3_hist.set_xlim(-8, 8)  #  x  
    ax3_hist.set_xticks([-5,0,5])
    # ax3_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    ax3_hist.set_xlabel(None)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 15, 6)  # 70, 75, 80, 85, 90, 95, 100
    
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0), useMathText=True)

    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, 60, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

# (: Colormap lsm, result_ds_list      )

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  

    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1['var_1st']*64,
                  rean_mean_rat1['var_1st']*64]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 30
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    ax1_hist.set_xlim(0, 60)
    ax1_hist.set_xlabel(None, fontsize=11)
    # ax1_hist.set_xticks([0,10,20,30,40,50,60])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.set_xlim(0, 60)
    ax2_hist.set_xlabel(None, fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -20, 20, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)
    ax3_hist.set_xlim(-20, 20)
    ax3_hist.set_xticks([-10,0,10])
    ax3_hist.set_xlabel(None, fontsize=11)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 60, 7)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
from matplotlib.ticker import ScalarFormatter

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0), useMathText=True)

    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, .4, 8*4 + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=-.2, vmax=.2)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  
    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1['var_1st'],
                  rean_mean_rat1['var_1st']]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 28
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))
    ax1_hist.xaxis.set_major_formatter(formatter_hist)
    
    ax1_hist.set_xlim(0, .4)
    ax1_hist.set_xlabel(None, fontsize=11)
    # ax1_hist.set_xticks([0,10,20,30,40,50,60])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))

    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.xaxis.set_major_formatter(formatter_hist)

    ax2_hist.set_xlim(0, .4)
    ax2_hist.set_xlabel(None, fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -20, 20, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)

    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))
    ax3_hist.xaxis.set_major_formatter(formatter_hist)
    
    ax3_hist.set_xlim(-.2, .2)
    # ax3_hist.set_xticks([-10,0,10])
    ax3_hist.set_xlabel(None, fontsize=11)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, .4, 9)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    cbar1.ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    cbar1.ax.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    cbar2.ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    cbar2.ax.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)

    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': IMERG_hfr.where(condition),
    'TRMM': TRMM_hfr.where(condition),
    'CMORPH': CMORPH_hfr.where(condition),
    'GSMaP': GSMaP_hfr.where(condition),
    'MSWEP': MSWEP_hfr.where(condition),
    'JRA-3Q': JRA3Q_hfr.where(condition),
    'ERA5': ERA5_hfr.where(condition),
    'MERRA-2': MERRA2_hfr.where(condition),
    'Obs': obs_hfr.where(condition)
}

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    all_values_total = da['total_var'].values.flatten()
    all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    all_values_high = da['high_var'].values.flatten()
    all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    all_values_1st = var_1st.values.flatten()
    all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    # temp_df = pd.DataFrame({
    #     'Total Var': all_values_valid_total/9.,
    #     'High Var': all_values_valid_high/9.,
    #     'Dataset': name
    # })
    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'High Var': all_values_valid_high/9.,
        '1st Var' : all_values_valid_1st/9.,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset')[['Total Var', 'High Var', '1st Var']].mean()
df_summary = df_summary.reindex(plot_order)

# CI 
ci_total = []
ci_high = []

for dataset in plot_order:
    data_total = df_long[df_long['Dataset'] == dataset]['Total Var'].values
    data_high = df_long[df_long['Dataset'] == dataset]['High Var'].values
    
    _, ci_t = calculate_ci(data_total)
    _, ci_h = calculate_ci(data_high)
    
    ci_total.append(ci_t)
    ci_high.append(ci_h)

ci_total = np.array(ci_total)
ci_high = np.array(ci_high)

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6

# Total Var  ( )
bars1 = ax.bar(x, df_summary['Total Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=total_color, #color='#F2BF5E', 
               alpha=0.6, error_kw={'elinewidth': 2})

# High Var Total Var   ( )
bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
               label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})
ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=df_summary['Total Var'] - df_summary['High Var'],                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3)

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    low_val = total_val - high_val
    percentage = (high_val / total_val) * 100
    
    text_high = low_val+ high_val/ 2
    ax.text(i, text_high, f'{high_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    text_low = low_val/ 2
    ax.text(i, text_low, f'{low_val:.1f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    ax.text(i, total_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=high_color, alpha=0.7, label=r'$\sigma^2_{\text{Low}}$')
]
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_1st.png', 
#             dpi=400, bbox_inches='tight')

### Violin Plot


In [ ]:
obs_hfr = xr.open_dataset('${DATA_DIR}/obs/obs_hfr_26h_30.nc')#.rename({'hfr':'hfr_tp'})

IMERG_hfr = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
TRMM_hfr = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
CMORPH_hfr = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
GSMaP_hfr = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MSWEP_hfr = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

ERA5_hfr = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
JRA3Q_hfr = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})
MERRA2_hfr = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_hfr_26h_30.nc').rename({'hfr':'hfr_tp'})

In [ ]:
sat_mean = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr],'data').mean('data')
rean_mean = xr.concat([ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data')

In [ ]:
total_mask = xr.concat([IMERG_hfr,TRMM_hfr,CMORPH_hfr, GSMaP_hfr,MSWEP_hfr,ERA5_hfr,JRA3Q_hfr,MERRA2_hfr],'data').mean('data',skipna = False)['hfr_tp'].isnull()

In [ ]:
IMERG_hfr = IMERG_hfr.where(~total_mask)
TRMM_hfr = TRMM_hfr.where(~total_mask)
CMORPH_hfr = CMORPH_hfr.where(~total_mask)
GSMaP_hfr = GSMaP_hfr.where(~total_mask)
MSWEP_hfr = MSWEP_hfr.where(~total_mask)

#reanalysis
ERA5_hfr = ERA5_hfr.where(~total_mask)
JRA3Q_hfr = JRA3Q_hfr.where(~total_mask)
# NARRa_hfr = NARRa_hfr.where(~total_mask)
# NARRf_hfr = NARRf_hfr.where(~total_mask)
MERRA2_hfr = MERRA2_hfr.where(~total_mask)

In [ ]:
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
sat_mean

In [ ]:
TRMM_hfr = TRMM_hfr.clip(0,100)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

def create_grouped_hfr_boxplots():
    """// HFR boxplot """
    
    satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    
    reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
    reanalysis_names = ['ERA5', 'JRA3Q', 'MERRA2']
    
    all_datasets = satellite_datasets + reanalysis_datasets
    all_names = satellite_names + reanalysis_names
    
    plot_data = []
    
    for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
        global_data = ds['hfr_tp'].values.flatten()
        global_data = global_data[~np.isnan(global_data)]
        
        land_data = ds.where(lsm)['hfr_tp'].values.flatten()
        land_data = land_data[~np.isnan(land_data)]
        
        ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        data_type = 'Satellite' if i < 5 else 'Reanalysis'
        
        for val in global_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Global', 
                            'Type': data_type, 'Position': i})
        
        for val in land_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Land', 
                            'Type': data_type, 'Position': i})
        
        for val in ocean_data:
            plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Ocean', 
                            'Type': data_type, 'Position': i})
    
    df = pd.DataFrame(plot_data)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    
    surface_colors = {
        'Land': '#D9665B',
        'Ocean': '#03738C',
        'Global': '#54728C'
    }
    
    sns.boxplot(data=df, x='Dataset', y='HFR', hue='Surface',
                order=all_names, hue_order=['Global', 'Land', 'Ocean'],
                palette=surface_colors, ax=ax, showfliers=False,
                showmeans=True, meanprops={"marker":"o","markerfacecolor":"white", 
                                          "markeredgecolor":"black","markersize":5})
    
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1)
    
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_xlabel('')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', labelsize=11, rotation=45)
    ax.tick_params(axis='y', labelsize=10)
    
    ax.set_title('High Frequency Ratio Distribution by Surface Type', 
                fontsize=14, pad=15)
    
    ax.legend( fontsize=10, 
             loc='lower left', frameon=False)
    
    # ax.text(2, -12, 'Satellite Products', ha='center', va='top', 
    #        fontsize=12, fontweight='bold', transform=ax.transData)
    # ax.text(6, -12, 'Reanalysis Products', ha='center', va='top', 
    #        fontsize=12, fontweight='bold', transform=ax.transData)
    
    sns.despine()
    
    plt.tight_layout()
    plt.show()

def create_grouped_hfr_boxplots_matplotlib():
    """matplotlib     """
    
    satellite_datasets = [IMERG_hfr, TRMM_hfr, CMORPH_hfr, GSMaP_hfr, MSWEP_hfr]
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    
    reanalysis_datasets = [ERA5_hfr, JRA3Q_hfr, MERRA2_hfr]
    reanalysis_names = ['ERA5', 'JRA3Q', 'MERRA2']
    
    all_datasets = satellite_datasets + reanalysis_datasets
    all_names = satellite_names + reanalysis_names
    
    land_data_all = []
    ocean_data_all = []
    global_data_all = []
    
    for ds in all_datasets:
        global_data = ds['hfr_tp'].values.flatten()
        global_data = global_data[~np.isnan(global_data)]
        global_data_all.append(global_data)
        
        land_data = ds.where(lsm)['hfr_tp'].values.flatten()
        land_data = land_data[~np.isnan(land_data)]
        land_data_all.append(land_data)
        
        ocean_data = ds.where(~lsm)['hfr_tp'].values.flatten()
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        ocean_data_all.append(ocean_data)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    
    land_color = '#D9665B'
    ocean_color = '#03738C'
    global_color = '#54728C'
    
    width = 0.25
    positions = np.arange(len(all_names))
    
    for i, pos in enumerate(positions):
        bp1 = ax.boxplot([global_data_all[i]], positions=[pos - width], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=global_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
        
        bp2 = ax.boxplot([land_data_all[i]], positions=[pos], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=land_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
        
        bp3 = ax.boxplot([ocean_data_all[i]], positions=[pos + width], widths=width,
                        patch_artist=True, showfliers=False,  showmeans=True,
                        boxprops=dict(facecolor=ocean_color, alpha=0.8),
                        meanprops={"marker":"o","markerfacecolor":"white", 
                                  "markeredgecolor":"black","markersize":5},medianprops={"color":"black","linewidth":1})
    
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=1)
    
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=11)
    ax.set_ylabel('HFR [%]', fontsize=13)
    ax.set_ylim(0, 100)
    
    ax.set_title('High Frequency Ratio Distribution by Surface Type', 
                fontsize=14, pad=15)
    
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=global_color, alpha=0.8, label='Global'),
                      Patch(facecolor=land_color, alpha=0.8, label='Land'),
                      Patch(facecolor=ocean_color, alpha=0.8, label='Ocean')]
    ax.legend(handles=legend_elements, fontsize=10, loc='lower left', frameon=False)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.set_xlim(-.5,7.5)
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/HFR_boxplot.png', dpi=400, bbox_inches='tight')
    # plt.show()
# print("Seaborn :")
# create_grouped_hfr_boxplots()

# print("\nMatplotlib :")
create_grouped_hfr_boxplots_matplotlib()

In [ ]:
obs_hfr = obs_hfr.rename({'hfr':'hfr_tp'})

In [ ]:
violin_plot_ds = {
    'IMERG': IMERG_hfr.where(~total_mask),
    'TRMM': TRMM_hfr.where(~total_mask),
    'CMORPH': CMORPH_hfr.where(~total_mask),
    'GSMaP': GSMaP_hfr.where(~total_mask),
    'MSWEP': MSWEP_hfr.where(~total_mask),
    'ERA5': ERA5_hfr.where(~total_mask),
    'JRA-3Q': JRA3Q_hfr.where(~total_mask),
    'MERRA-2': MERRA2_hfr.where(~total_mask),
}

violin_plot_ds_rat1 = {
    'IMERG': imerg.where(~total_mask),
    'TRMM': trmm.where(~total_mask),
    'CMORPH': cmorph.where(~total_mask),
    'GSMaP': gsmap.where(~total_mask),
    'MSWEP': mswep.where(~total_mask),
    'ERA5': era5.where(~total_mask), 
    'JRA-3Q': jra55.where(~total_mask),
    'MERRA-2': merra2.where(~total_mask),
}

In [ ]:
satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
all_names = satellite_names + reanalysis_names


In [ ]:
ds_var_list = []
for name in all_names:
    # ds_var_list.append(((violin_plot_ds[name]['total_var']*((violin_plot_ds_rat1[name]['1st']/100)*(violin_plot_ds[name]['hfr']/100)))/9).to_dataset(name='var_1'))
    ds_var_list.append((((violin_plot_ds_rat1[name]['1st']/100)*(violin_plot_ds[name]['hfr']/100))*100).to_dataset(name='var_1'))

In [ ]:
all_datasets = ds_var_list

In [ ]:
plot_data = []
for i, (ds, name) in enumerate(zip(all_datasets, all_names)):
    global_data = ds['var_1'].values.flatten()
    global_data = global_data[~np.isnan(global_data)]
    
    land_data = ds.where(lsm)['var_1'].values.flatten()
    land_data = land_data[~np.isnan(land_data)]
    
    ocean_data = ds.where(~lsm)['var_1'].values.flatten()
    ocean_data = ocean_data[~np.isnan(ocean_data)]
    
    data_type = 'Satellite' if i < 5 else 'Reanalysis'
    
    for val in global_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Global', 
                        'Type': data_type, 'Position': i})
    
    for val in land_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Land', 
                        'Type': data_type, 'Position': i})
    
    for val in ocean_data:
        plot_data.append({'Dataset': name, 'HFR': val, 'Surface': 'Ocean', 
                        'Type': data_type, 'Position': i})
df = pd.DataFrame(plot_data)

In [ ]:
df[df['Dataset']=='ERA5']['HFR'].values

In [ ]:
df[df['Dataset']=='ERA5'][df['Surface'] == 'Global']['HFR'].values

In [ ]:
df[df['Dataset']=='IMERG'][df['Surface'] == 'Global']['HFR'].values

In [ ]:
df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    fig, ax = plt.subplots(figsize=(7, 5))
    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel(r'$rat_{1st} $[%]', fontsize=13)
    # ax.set_yscale('log')
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(C)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30.png', dpi=400, bbox_inches='tight')
    plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    # fig, ax = plt.subplots(figsize=(7, 5))
    fig, ax = plt.subplots(figsize=(7, 4.25))

    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('$rat_{subD}$ [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(b)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    # plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    # fig, ax = plt.subplots(figsize=(7, 5))
    fig, ax = plt.subplots(figsize=(7, 4.25))

    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel('$rat_{subD}$ [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(b)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    # plt.show()

create_split_violin_hfr()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch
from scipy import stats

def get_violin_width_at_y(data, y_value, position, violin_width=0.8):
    """ y   """
    if len(data) == 0:
        return 0
    
    # KDE  density 
    kde = stats.gaussian_kde(data)
    density = kde(y_value)[0] if np.isscalar(y_value) else kde(y_value)
    
    #    density 
    y_range = np.linspace(data.min(), data.max(), 100)
    max_density = kde(y_range).max()
    
    #   (0~violin_width/2)
    normalized_width = (density / max_density) * (violin_width / 2)
    
    return normalized_width

def create_split_violin_hfr():
    """   HFR  
    Adaptive quantile lines fitted to violin width
    """
    satellite_names = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP']
    reanalysis_names = ['ERA5', 'JRA-3Q', 'MERRA-2']
    all_names = satellite_names + reanalysis_names

    # fig, ax = plt.subplots(figsize=(7, 5))
    fig, ax = plt.subplots(figsize=(7, 4.25))

    ymax = 80
    ymin = 0

    global_color = '#54728C'
    land_color = '#D9665B'
    ocean_color = "#03438C"
    
    positions = np.arange(len(all_names))
    
    for i, name in enumerate(all_names):
        dataset_data = df[df['Dataset'] == name]
        
        global_vals = dataset_data[dataset_data['Surface'] == 'Global']['HFR'].values
        land_vals = dataset_data[dataset_data['Surface'] == 'Land']['HFR'].values
        ocean_vals = dataset_data[dataset_data['Surface'] == 'Ocean']['HFR'].values
        
        # Global  ( )
        if len(global_vals) > 0:
            parts_global = ax.violinplot([global_vals], positions=[i], widths=0.8, 
                                       showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_global['bodies']:
                pc.set_facecolor(global_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('black')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], -np.inf, i)
                pc.get_paths()[0].vertices = verts
            
            # Global  
            global_q1 = np.percentile(global_vals, 25)
            global_median = np.median(global_vals)
            global_q3 = np.percentile(global_vals, 75)
            global_mean = np.mean(global_vals)
            
            q1_width = get_violin_width_at_y(global_vals, global_q1, i, 0.8)
            median_width = get_violin_width_at_y(global_vals, global_median, i, 0.8)
            q3_width = get_violin_width_at_y(global_vals, global_q3, i, 0.8)
            
            # Global   (  )
            # Q1 (25%)
            ax.plot([i-q1_width*0.8, i], [global_q1, global_q1], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i-.4, i], [global_median, global_median], 
                   color=global_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i-q3_width*0.8, i], [global_q3, global_q3], 
                   color=global_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Global mean 
            # mean_width = get_violin_width_at_y(global_vals, global_mean, i, 0.8)
            ax.scatter([i-.2], [global_mean], color='white', s=50, 
                      edgecolor=global_color, linewidth=1.5, zorder=10)
                
        # Ocean  ( , Land )
        if len(ocean_vals) > 0:
            parts_ocean = ax.violinplot([ocean_vals], positions=[i], widths=0.8, 
                                      showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_ocean['bodies']:
                pc.set_facecolor(ocean_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor("#03438C")
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Ocean  
            ocean_q1 = np.percentile(ocean_vals, 25)
            ocean_median = np.median(ocean_vals)
            ocean_q3 = np.percentile(ocean_vals, 75)
            ocean_mean = np.mean(ocean_vals)
            
            q1_width = get_violin_width_at_y(ocean_vals, ocean_q1, i, 0.8)
            median_width = get_violin_width_at_y(ocean_vals, ocean_median, i, 0.8)
            q3_width = get_violin_width_at_y(ocean_vals, ocean_q3, i, 0.8)
            
            # Ocean   (  ,  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [ocean_q1, ocean_q1], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':') 
            # Median (50%)
            ax.plot([i, i+.4], [ocean_median, ocean_median], 
                   color=ocean_color, linewidth=2, alpha=0.8, linestyle='-')
            # Q3 (75%)
            ax.plot([i, i+q3_width], [ocean_q3, ocean_q3], 
                   color=ocean_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Ocean mean 
            # mean_width = get_violin_width_at_y(ocean_vals, ocean_mean, i, 0.8)
            ax.scatter([i+.2], [ocean_mean], color='white', s=40, # marker='s',
                       edgecolor=ocean_color, linewidth=1.5, zorder=10)
        # Land  ( )
        if len(land_vals) > 0:
            parts_land = ax.violinplot([land_vals], positions=[i], widths=0.8, 
                                     showmeans=False, showmedians=False, showextrema=False)
            
            for pc in parts_land['bodies']:
                pc.set_facecolor(land_color)
                pc.set_alpha(0.4)
                pc.set_edgecolor('#D9665B')
                pc.set_linewidth(1.2)
                
                verts = pc.get_paths()[0].vertices
                verts[:, 0] = np.clip(verts[:, 0], i, np.inf)
                pc.get_paths()[0].vertices = verts
            
            # Land  
            land_q1 = np.percentile(land_vals, 25)
            land_median = np.median(land_vals)
            land_q3 = np.percentile(land_vals, 75)
            land_mean = np.mean(land_vals)
            
            q1_width = get_violin_width_at_y(land_vals, land_q1, i, 0.8)
            median_width = get_violin_width_at_y(land_vals, land_median, i, 0.8)
            q3_width = get_violin_width_at_y(land_vals, land_q3, i, 0.8)
            
            # Land   (  )
            # Q1 (25%)
            ax.plot([i, i+q1_width], [land_q1, land_q1], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            # Median (50%)
            ax.plot([i, i+.4], [land_median, land_median], 
                   color=land_color, linewidth=2.5, alpha=0.8)
            # Q3 (75%)
            ax.plot([i, i+q3_width], [land_q3, land_q3], 
                   color=land_color, linewidth=1.5, alpha=0.7, linestyle=':')
            
            # Land mean 
            # mean_width = get_violin_width_at_y(land_vals, land_mean, i, 0.8)
            ax.scatter([i+.2], [land_mean], color='white', s=50, 
                      edgecolor=land_color, linewidth=1.5, zorder=10)
    for i in range(len(all_names) - 1):
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)
    
    # Satellite/Reanalysis 
    ax.axvline(x=4.5, color='black', linestyle='-', linewidth=2)
            # ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    ax.set_xticks(positions)
    ax.set_xticklabels(all_names, rotation=45, fontsize=12)
    ax.set_ylabel(r'$rat_1 \times rat_{subD}$ [%]', fontsize=13)
    ax.set_ylim(ymin, ymax)
    ax.set_xlim(-0.5, len(all_names) - 0.5)
    
    # ax.set_title('(C)', 
    # # ax.set_title('(C) High Frequency Ratio Distribution', 
    #             fontsize=14, pad=15)
    ax.text(-0.5, ymax,  '(b)',#f'PSS:{pss_score:.3f}', 
           ha='left', va='bottom', fontsize=18)

    
    legend_elements = [
        Patch(facecolor=global_color, alpha=0.8, label='Global'),
        Patch(facecolor=land_color, alpha=0.7, label='Land'),
        Patch(facecolor=ocean_color, alpha=0.5, label='Ocean'),
        plt.Line2D([0], [0], color='black', linewidth=3, label='Median'),
        plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='w', 
                  markeredgecolor='black', markersize=8, label='Mean', linewidth=0)
    ]
    ax.legend(handles=legend_elements, fontsize=11, loc='lower left', frameon=False, ncol=2)
    
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/HFR_violinplot_26h_30_eseo.png', dpi=400, bbox_inches='tight')
    plt.show()

create_split_violin_hfr()

### Paper Figure


#### Figure 2
'''A
irreg, reg, Low
subD    ,  hatched   irreg  reg  ,
Low irreg  , bar
'''


In [ ]:
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
# plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
# plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')

In [ ]:
# must be executed
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }
# datasets_to_plot_var = {
#     'IMERG': IMERG_var.where(condition),
#     'TRMM': TRMM_var.where(condition),
#     'CMORPH': CMORPH_var.where(condition),
#     'GSMaP': GSMaP_var.where(condition),
#     'MSWEP': MSWEP_var.where(condition),
#     'JRA-3Q': JRA3Q_var.where(condition),
#     'ERA5': ERA5_var.where(condition),
#     'MERRA-2': MERRA2_var.where(condition),
#     'Obs': obs_var.where(condition)
# }

datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_low = da['var_Low'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_irreg = da['var_isd'].values.flatten()
    all_values_reg = da['var_mdc'].values.flatten()


    all_values_1st = da['var_1st'].values.flatten()
    all_values_2nd = da['var_2nd'].values.flatten()
    all_values_3rd = da['var_3rd'].values.flatten()
    all_values_res = da['var_mdc_res'].values.flatten()

    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st)
    
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_low = all_values_low[~alllc]
    all_values_valid_high = all_values_high[~alllc]

    all_values_valid_irreg = all_values_irreg[~alllc]
    all_values_valid_reg = all_values_reg[~alllc]

    all_values_valid_1st = all_values_1st[~alllc]
    all_values_valid_2nd = all_values_2nd[~alllc]
    all_values_valid_3rd = all_values_3rd[~alllc]
    all_values_valid_res = all_values_res[~alllc]


    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'Low Var': all_values_valid_low/9.,
        'High Var': all_values_valid_high/9.,

        'reg Var': all_values_valid_reg/9.,
        'irreg Var': all_values_valid_irreg/9.,

        '1st Var' : all_values_valid_1st/9,
        '2nd Var' : all_values_valid_2nd/9,
        '3rd Var' : all_values_valid_3rd/9,
        'res Var' : all_values_valid_res/9,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)

def calculate_ci(data, confidence=0.95):
    """95%  """
    n = len(data)
    mean = np.mean(data)
    se = stats.sem(data)
    ci = se * stats.t.ppf((1 + confidence) / 2., n - 1)
    return mean, ci

df_summary = df_long.groupby('Dataset').mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
def get_harmonic_ratios(df_summary):
    """
    Compute 1st, 2nd, higher-order harmonic ratios from df_summary
    
    Returns
    -------
    df_ratio : DataFrame
        columns: '1st [%]', '2nd [%]', 'higher [%]'
        Denominator: reg Var
    """
    reg = df_summary['reg Var']
    
    r1st    = df_summary['1st Var'] / reg * 100
    r2nd    = df_summary['2nd Var'] / reg * 100
    r3rd    = df_summary['3rd Var'] / reg * 100
    rhigher = 100 - r1st - r2nd  # 3rd   

    #  3rd   :
    # rhigher = r3rd + df_summary['res Var'] / reg * 100

    df_ratio = pd.DataFrame({
        '1st [%]':    r1st,
        '2nd [%]':    r2nd,
        'higher [%]': rhigher,
    })
    
    assert np.allclose(df_ratio.sum(axis=1), 100), "sum is not 100%"
    
    return df_ratio

df_ratio = get_harmonic_ratios(df_summary)

In [ ]:
df_ratio

In [ ]:
df_summary

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6


bars2 = ax.bar(x, df_summary['High Var'], width, 
               label=r'$\sigma_{\text{subD}}$', color=high_color, #color='#F28379', 
               alpha=0.6, error_kw={'elinewidth': 2})

bars4 = ax.bar(x, 
       df_summary['irreg Var'], 
       width,
       bottom=0.,                 
       color='none', 
       edgecolor='black', 
       hatch='///', 
       alpha=0.2, 
       linewidth=.3,
       label=r'$\sigma_{\text{irreg}}$')

bars3 = ax.bar(x, 
       df_summary['Low Var'], 
       width,
       bottom=df_summary['High Var'],                 
       color=total_color, 
       alpha=0.6, 
       linewidth=.3, label=r'$\sigma_{\text{Low}}$')

for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    reg_val = df_summary.loc[dataset, 'reg Var']
    irreg_val = df_summary.loc[dataset, 'irreg Var']
    low_val = total_val - high_val
    # fst_val = df_summary.loc[dataset, '1st Var']
    
    percentage = (high_val / total_val) * 100
    ########## irreg text
    ax.text(i, irreg_val/2, f'{irreg_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    # ax.text(i, total_val, f'{reg_val:.3f}',  
    #        ha='center', va='bottom', fontsize=9, color='lightgray', 
    #        path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    ########## low text
    ax.text(i,  high_val+low_val/2, f'{low_val:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    
    ax.text(i, total_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

ax.text(-0.5, 1.7,  '(a)',#f'PSS:{pss_score:.3f}', 
       ha='left', va='bottom', fontsize=18)

ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)
ax.set_xlim(-0.5, len(plot_order) - 0.5)
ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    # Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    # Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor=total_color, alpha=0.6, label=r'$\sigma^2_{\text{Low}}$'),
    Patch(facecolor=high_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{irreg}}$'),
    Patch(facecolor=high_color, alpha=0.6, label=r'$\sigma^2_{\text{subD}}$')
]
    # Patch(facecolor=high_color, alpha=0.6, label=r'$\sigma^2_{\text{Low}}$')
    # Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{irreg}}$'),
    # Patch(facecolor='#5C9EAD', alpha=0.6, label=r'$\sigma^2_{\text{reg}}$'),



# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)

ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
# plt.show()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_subD.png', 
            dpi=400, bbox_inches='tight')

#### Figure 4
1.	  1st,2nd,higher
2.	1st  2nd
3.	     b,c,  , ㄷ ,   MS,MM,MM-MS
4.	         , histogram
5.
6.	   0~13


##### Bar


In [ ]:
imerg

In [ ]:
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
# plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')

In [ ]:
condition.plot()

In [ ]:
# test # condition = condition.where((condition.latitude < 50) & (condition.latitude > -50)).fillna(0.)#.plot()

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs']
positions = np.arange(len(plot_order))

# datasets_to_plot = {
#     'IMERG': IMERG_hfr.where(condition),
#     'TRMM': TRMM_hfr.where(condition),
#     'CMORPH': CMORPH_hfr.where(condition),
#     'GSMaP': GSMaP_hfr.where(condition),
#     'MSWEP': MSWEP_hfr.where(condition),
#     'JRA-3Q': JRA3Q_hfr.where(condition),
#     'ERA5': ERA5_hfr.where(condition),
#     'MERRA-2': MERRA2_hfr.where(condition),
#     'Obs': obs_hfr.where(condition)
# }
# datasets_to_plot_var = {
#     'IMERG': IMERG_var.where(condition),
#     'TRMM': TRMM_var.where(condition),
#     'CMORPH': CMORPH_var.where(condition),
#     'GSMaP': GSMaP_var.where(condition),
#     'MSWEP': MSWEP_var.where(condition),
#     'JRA-3Q': JRA3Q_var.where(condition),
#     'ERA5': ERA5_var.where(condition),
#     'MERRA-2': MERRA2_var.where(condition),
#     'Obs': obs_var.where(condition)
# }
# .sel(latitude = slice(-50,50))
datasets_to_plot_rat1 = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition)  
}
# datasets_to_plot_rat1 = {
#     'IMERG': imerg.where(condition).sel(latitude = slice(-50,50)),
#     'TRMM': trmm.where(condition).sel(latitude = slice(-50,50)),
#     'CMORPH': cmorph.where(condition).sel(latitude = slice(-50,50)),
#     'GSMaP': gsmap.where(condition).sel(latitude = slice(-50,50)),
#     'MSWEP': mswep.where(condition).sel(latitude = slice(-50,50)),
#     'JRA-3Q': jra55.where(condition).sel(latitude = slice(-50,50)),
#     'ERA5': era5.where(condition).sel(latitude = slice(-50,50)),
#     'MERRA-2': merra2.where(condition).sel(latitude = slice(-50,50)),
#     'Obs': obs_result.where(condition).sel(latitude = slice(-50,50))  
# }


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_low = da['var_Low'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_irreg = da['var_isd'].values.flatten()
    all_values_reg = da['var_mdc'].values.flatten()


    all_values_1st = da['var_1st'].values.flatten()
    all_values_2nd = da['var_2nd'].values.flatten()
    all_values_3rd = da['var_3rd'].values.flatten()
    all_values_res = da['var_mdc_res'].values.flatten()

    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st)
    
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_low = all_values_low[~alllc]
    all_values_valid_high = all_values_high[~alllc]

    all_values_valid_irreg = all_values_irreg[~alllc]
    all_values_valid_reg = all_values_reg[~alllc]

    all_values_valid_1st = all_values_1st[~alllc]
    all_values_valid_2nd = all_values_2nd[~alllc]
    all_values_valid_3rd = all_values_3rd[~alllc]
    all_values_valid_res = all_values_res[~alllc]


    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'Low Var': all_values_valid_low/9.,
        'High Var': all_values_valid_high/9.,

        'reg Var': all_values_valid_reg/9.,
        'irreg Var': all_values_valid_irreg/9.,

        '1st Var' : all_values_valid_1st/9,
        '2nd Var' : all_values_valid_2nd/9,
        '3rd Var' : all_values_valid_3rd/9,
        'res Var' : all_values_valid_res/9,
        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)
df_summary = df_long.groupby('Dataset').mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
df_summary

In [ ]:
df_summary

In [ ]:
from matplotlib.ticker import ScalarFormatter

formatter = ScalarFormatter(useMathText=True)
formatter.set_powerlimits((0, 0))

fig, ax = plt.subplots(figsize=(7, 4.25))
# plt.figure(figsize=(11, 8.5))
x = np.arange(len(plot_order))
width = 0.6
ax.yaxis.set_major_formatter(formatter)
ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0))
offset_text = ax.yaxis.get_offset_text()

offset_text.set_horizontalalignment('right')
# offset_text.set_x(-0.15) #     .
# # High Var Total Var   ( )
# bars2 = ax.bar(x, df_summary['Total Var']-df_summary['High Var'], width, 
#                label=r'$\sigma_{\text{High freq}}$', color=high_color, #color='#F28379', 
#                alpha=0.6, error_kw={'elinewidth': 2})

# bars3 = ax.bar(x, 
#        df_summary['reg Var'], 
#        width,
#        bottom=df_summary['Total Var'] - df_summary['reg Var'],                 
#        color='#5C9EAD', 
#        alpha=0.6, 
#        linewidth=.3, label=r'$\sigma_{\text{reg}}$')

bars4 = ax.bar(x, 
       df_summary['1st Var'], 
       width,
       bottom=0,                 
       color='#F28379', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{1st}}$')

bars5 = ax.bar(x, 
       df_summary['2nd Var'], 
       width,
       bottom=df_summary['1st Var'] ,                 
       color='#F2BF5E', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{2nd}}$')
#F28379
bars6 = ax.bar(x, 
       df_summary['3rd Var'], 
       width,
       bottom=df_summary['1st Var']+ df_summary['2nd Var'],                 
       color='#5C9EAD', 
       alpha=0.7, 
       linewidth=.3, label=r'$\sigma_{\text{3rd}}$')


for i, dataset in enumerate(plot_order):
    total_val = df_summary.loc[dataset, 'Total Var']
    high_val = df_summary.loc[dataset, 'High Var']
    reg_val = df_summary.loc[dataset, 'reg Var']
    irreg_val = df_summary.loc[dataset, 'irreg Var']
    fst_val = df_summary.loc[dataset, '1st Var']
    nd_val = df_summary.loc[dataset, '2nd Var']
    rd_val = df_summary.loc[dataset, '3rd Var']

    percentage = (fst_val / reg_val) * 100
    ########## irreg text
    ax.text(i, fst_val/2, f'{fst_val*100:.2f}',  
           ha='center', va='center', fontsize=9, color='white', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])

    ax.text(i, fst_val+nd_val/2, f'{nd_val*100:.2f}',  
           ha='center', va='center', fontsize=9, color='lightgray', 
           path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    ########## low text
    # ax.text(i, fst_val+nd_val+rd_val, f'{rd_val*100:.2f}',  
    #        ha='center', va='bottom', fontsize=9, color='white', 
    #        path_effects=[pe.withStroke(linewidth=1.5, foreground='black', alpha=0.6)])
    
    
    ax.text(i, reg_val, f'{percentage:.1f}%',  
       ha='center', va='bottom', fontsize=11, color='gray',weight = 'bold') 

for i in range(len(plot_order) - 1):
    if i == 4:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    elif i == 7:
        ax.axvline(x=i + 0.5, color='black', linestyle='-', linewidth=2)
    else:
        ax.axvline(x=i + 0.5, color='gray', linestyle='--', linewidth=0.8, alpha=0.6)

# ax.text(-0.5, 1.7,  '(a)',ha='left', va='bottom', fontsize=18)
ax.text(0.001, 1., '(a)', transform=ax.transAxes, 
        fontsize=18, va='bottom', ha='left')
ax.set_xticks(positions)
ax.set_ylabel('$\sigma^2$ [$mm^2/h^2$]', fontsize=13)

ax.set_xlim(-0.5, len(plot_order) - 0.5)
# ax.set_ylim(0, 1.7)
ax.set_xticklabels(plot_order, rotation=45, fontsize=12)
# ax.legend(fontsize=11)
# legend_elements = [
#     Patch(facecolor=total_color, alpha=1, label=r'$\sigma^2_{\text{subD}}$'),
#     Patch(facecolor=high_color, alpha=1, label=r'$\sigma^2_{\text{Low}}$')
# ]

legend_elements = [
    # Patch(facecolor=total_color, alpha=0.5, label=r'$\sigma^2_{\text{subD}}$'),
    # Patch(facecolor=total_color, alpha=0.2, hatch='//////', edgecolor='black', label=r'$\sigma^2_{\text{1st}}$'),
    Patch(facecolor='#5C9EAD', alpha=0.6, label=r'$\sigma^2_{\text{higher}}$'),
    Patch(facecolor='#F2BF5E', alpha=0.6, label=r'$\sigma^2_{\text{2nd}}$'),
    Patch(facecolor='#F28379', alpha=0.6, label=r'$\sigma^2_{\text{1st}}$')
]#F28379
# ax.legend(handles=legend_elements, fontsize=11, loc='upper right', 
#           bbox_to_anchor=(1, 1), frameon=True, edgecolor='inherit')

ax.legend(handles=legend_elements, fontsize=13, loc='upper right',bbox_to_anchor = (1.,1.1), frameon=True)
# ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0), useMathText=True)

# offset text(×10⁻²) y  
# ax.yaxis.get_offset_text().set_visible(False)  #   

# y    
# offset = ax.yaxis.get_major_formatter().get_offset()
# ax.text(-0.5, 0.01, offset, transform=ax.transAxes, fontsize=11, ha='left')


ax.grid(axis='y', alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
# plt.show()
plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/Variance_26h_30_reg.png', 
            dpi=400, bbox_inches='tight')

In [ ]:
obs_result  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
averaged_data  = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
import seaborn as sns
da = averaged_data['var_total']

# 2. convert 2D data to 1D array (Flatten)
# .values  numpy    flatten()
all_values = da.values.flatten()

# 3. NaN   ( !)
#    NaN()      .
all_values_valid = all_values[~np.isnan(all_values)]

# --- 3.5. IQR    ---
Q1 = np.percentile(all_values_valid, 25)
Q3 = np.percentile(all_values_valid, 75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# (lower/upper_bound)   ""  
all_values_filtered = all_values_valid[
    (all_values_valid >= lower_bound) & (all_values_valid <= upper_bound)
]

print(f"Original data points: {len(all_values_valid)}")
print(f"Filtered data points: {len(all_values_filtered)}")


# 4.  (!!!  'all_values_filtered'  !!!)
fig, axes = plt.subplots(1, 2, figsize=(5, 3))

# --- Plot 1: Boxplot (  ) ---
sns.boxplot(y=all_values_filtered, ax=axes[0])
axes[0].set_title('Boxplot (Outliers Removed)')
axes[0].set_ylabel('Total Var Value')
axes[0].set_xlabel('Filtered Data')

# --- Plot 2: Bar plot (  ) ---
#    (  )
sns.barplot(y=all_values_filtered, ax=axes[1])
axes[1].set_title('Bar Plot (Outliers Removed)')
axes[1].set_ylabel('Mean Total Var Value')
axes[1].set_xlabel('Filtered Data')

plt.suptitle('Comparison (Outliers removed from data)')
plt.tight_layout()
plt.show()

In [ ]:
condition = (da >= lower_bound) & (da <= upper_bound)

# 2. .where()   (True)  .
outlier_da = da.where(condition)

# 3.  
# - outlier_da   ,  NaN.
# - .plot()    (grid cell)      .
plt.figure(figsize=(5, 3))
outlier_da.plot()
# plt.title('Map of Outlier Locations')
# plt.show()

# outlier_da    NaN     .
# print(outlier_da)

In [ ]:
sat_mean = xr.concat([imerg ,trmm,cmorph,gsmap,mswep],'data').mean('data')#,skipna = False)

In [ ]:
rean_mean = xr.concat([jra55,era5,merra2],'data').mean('data',skipna = False)

In [ ]:
new_markers = ['Obs','Multi-Satellite (MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
               'Multi-Model (MM)','JRA-3Q','ERA5','MERRA-2']

new_data_title = ['o','$S$','^' ,'s', 'H','X', 'D',
          '$R$','<', 'p', '*']

In [ ]:
sat_mean['var_total'].plot()

In [ ]:
rean_mean['var_total'].plot(vmax = 70)

In [ ]:
(sat_mean['var_total']-rean_mean['var_total']).plot(vmin = -20, vmax = 20,cmap = 'RdBu')

condition = condition.where((condition.latitude < 50) & (condition.latitude > -50)).fillna(0.)#.plot()


In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
from matplotlib.patches import Patch
import matplotlib.patheffects as pe
plot_order = ['IMERG', 'TRMM', 'CMORPH', 'GSMaP', 'MSWEP','ERA5','JRA-3Q','MERRA-2', 'Obs','Multi-Satellite (MS)','Multi-Model (MM)']
positions = np.arange(len(plot_order))

datasets_to_plot = {
    'IMERG': imerg.where(condition),
    'TRMM': trmm.where(condition),
    'CMORPH': cmorph.where(condition),
    'GSMaP': gsmap.where(condition),
    'MSWEP': mswep.where(condition),
    'JRA-3Q': jra55.where(condition),
    'ERA5': era5.where(condition),
    'MERRA-2': merra2.where(condition),
    'Obs': obs_result.where(condition),
    'Multi-Satellite (MS)': sat_mean.where(condition),
    'Multi-Model (MM)': rean_mean.where(condition)
}


total_color = '#F2BF5E'
high_color = '#F28379'

dataframes_list = []
for name in plot_order:
    da = datasets_to_plot[name]
    # var_1st = (datasets_to_plot[name]['total_var']*((datasets_to_plot_rat1[name]['1st']/100)*(datasets_to_plot[name]['hfr']/100)))
    
    # all_values_total = da['total_var'].values.flatten()
    # all_values_valid_total = all_values_total[~np.isnan(all_values_total)]
    
    # all_values_high = da['high_var'].values.flatten()
    # all_values_valid_high = all_values_high[~np.isnan(all_values_high)]
    
    # # all_values_1st = var_1st.values.flatten()
    # all_values_1st = datasets_to_plot_var[name]['tp_1st_var'].values.flatten()
    # all_values_valid_1st = all_values_1st[~np.isnan(all_values_1st)]

    
    all_values_total = da['var_total'].values.flatten()
    all_values_low = da['var_Low'].values.flatten()
    all_values_high = da['var_subD'].values.flatten()
    all_values_irreg = da['var_isd'].values.flatten()
    all_values_reg = da['var_mdc'].values.flatten()


    all_values_1st = da['var_1st'].values.flatten()
    all_values_2nd = da['var_2nd'].values.flatten()
    all_values_3rd = da['var_3rd'].values.flatten()
    all_values_res = da['var_mdc_res'].values.flatten()
    
    all_values_rat1 = da['rat_1st'].values.flatten()
    all_values_rat2 = da['rat_2nd'].values.flatten()
    # all_values_rat1 = da['rat_1st'].values.flatten()

    alllc = np.isnan(all_values_total)&np.isnan(all_values_high)&np.isnan(all_values_1st)
    
    all_values_valid_total = all_values_total[~alllc]
    all_values_valid_low = all_values_low[~alllc]
    all_values_valid_high = all_values_high[~alllc]

    all_values_valid_irreg = all_values_irreg[~alllc]
    all_values_valid_reg = all_values_reg[~alllc]

    all_values_valid_1st = all_values_1st[~alllc]
    all_values_valid_2nd = all_values_2nd[~alllc]
    all_values_valid_3rd = all_values_3rd[~alllc]
    all_values_valid_res = all_values_res[~alllc]
    
    all_values_valid_rat1 = all_values_rat1[~alllc]
    all_values_valid_rat2 = all_values_rat2[~alllc]


    temp_df = pd.DataFrame({
        'Total Var': all_values_valid_total/9.,
        'Low Var': all_values_valid_low/9.,
        'High Var': all_values_valid_high/9.,

        'reg Var': all_values_valid_reg/9.,
        'irreg Var': all_values_valid_irreg/9.,

        '1st Var' : all_values_valid_1st/9,
        '2nd Var' : all_values_valid_2nd/9,
        '3rd Var' : all_values_valid_3rd/9,
        'res Var' : all_values_valid_res/9,
        'rat_1st' : all_values_valid_rat1,
        'rat_2nd' : all_values_valid_rat2,

        'Dataset': name  # 'Dataset'     
    })
    dataframes_list.append(temp_df)

df_long = pd.concat(dataframes_list)
df_summary = df_long.groupby('Dataset').mean()
df_summary = df_summary.reindex(plot_order)


In [ ]:
df_summary #-50~50

In [ ]:
df_summary

In [ ]:
df_summary

In [ ]:
df_summary

In [ ]:
new_data_title = ['Obs','Multi-Satellite (MS)','CMORPH','IMERG','TRMM','GSMaP','MSWEP',
               'Multi-Model (MM)','JRA-3Q','ERA5','MERRA-2']

new_markers = ['o','$MS$','^' ,'s', 'H','X', 'D',
          '$MM$','<', 'p', '*']

In [ ]:
for i in range(len(new_data_title)):
    print(new_data_title[i])
    print((df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
    print((df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)

In [ ]:
# no filtering applied
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']

# print((df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
# print((df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
for i in range(len(new_data_title)):
    # if i == 0 :
    #     ttt = obs_result.where(condition)['2nd'].mean().item()
    #     rrr = obs_result.where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr
    # else:
    #     ttt = result_ds_list[i-1].where(condition)['2nd'].mean().item()
    #     rrr = result_ds_list[i-1].where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr   
    ttt = (df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    rrr = (df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    lll = 100-ttt-rrr
    
    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# -50~50 
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 30 #25
rmax = 95 #90
tmin = 5 #10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']

# print((df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
# print((df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
for i in range(len(new_data_title)):
    # if i == 0 :
    #     ttt = obs_result.where(condition)['2nd'].mean().item()
    #     rrr = obs_result.where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr
    # else:
    #     ttt = result_ds_list[i-1].where(condition)['2nd'].mean().item()
    #     rrr = result_ds_list[i-1].where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr   
    ttt = (df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    rrr = (df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    lll = 100-ttt-rrr
    
    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=200,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=200, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# no filtering applied
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 30 #25
rmax = 95 #90
tmin = 5 #10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']

# print((df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
# print((df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
for i in range(len(new_data_title)):
    # if i == 0 :
    #     ttt = obs_result.where(condition)['2nd'].mean().item()
    #     rrr = obs_result.where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr
    # else:
    #     ttt = result_ds_list[i-1].where(condition)['2nd'].mean().item()
    #     rrr = result_ds_list[i-1].where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr   
    ttt = (df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    rrr = (df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100
    print(new_data_title[i])
    print(f'1st/total={rrr}')
    print(f'2nd/total={ttt}')

    lll = 100-ttt-rrr
    if i == 1 or i==7:
        markersize = 400
    else : 
        markersize = 200
        
    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=markersize,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=markersize, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
# no filtering applied, rat1 
ax = plt.subplot(projection='ternary',ternary_sum=100.0)
arrowstyle = ArrowStyle('simple', head_length=10, head_width=5)
kwargs_arrow = {
    'transform': ax.transAxes,  # Used with ``ax.transAxesProjection``
    'arrowstyle': arrowstyle,
    'linewidth': 3,
    'clip_on': False,  # To plot arrows outside triangle
    'zorder': -10,  # Very low value not to hide e.g. tick labels.
}
ta = np.array([ 0.0, -13,  113])
la = np.array([ 113,  0.0, -13])
ra = np.array([-13,  113,  0.0])

# End of arrows in barycentric coordinates.
tb = np.array([ 100, -13,  13])
lb = np.array([ 13,  100, -13])
rb = np.array([-13,  13,  100])
f = ax.transAxesProjection.transform

tarrow = FancyArrowPatch(f(ta), f(tb), ec=colors_hex['top'], fc=colors_hex['top'], **kwargs_arrow)
larrow = FancyArrowPatch(f(la), f(lb), ec=colors_hex['left'], fc=colors_hex['left'], **kwargs_arrow)
rarrow = FancyArrowPatch(f(ra), f(rb), ec=colors_hex['right'], fc=colors_hex['right'], **kwargs_arrow)
ax.add_patch(tarrow)
ax.add_patch(larrow)
ax.add_patch(rarrow)

kwargs_label = {'fontsize':13,
    'transform': ax.transTernaryAxes,
    'backgroundcolor': 'w',
    'ha': 'center',
    'va': 'center',
    'rotation_mode': 'anchor',
    'zorder': -9,  # A bit higher on arrows, but still lower than others.
}

tpos = (ta + tb) * 0.5
lpos = (la + lb) * 0.5
rpos = (ra + rb) * 0.5

ax.text(*tpos, '2nd harmonic amplitude [%]'  , color='black', rotation=-60, **kwargs_label)
ax.text(*lpos, 'Higher-order harmonic amplitude [%]' , color='black', rotation= 60, **kwargs_label)
ax.text(*rpos, '1st harmonic amplitude [%]', color='black', rotation=  0, **kwargs_label)

t, l, r, v = get_dirichlet_pdfs(n=400, alpha=(10, 10, 10))

# Define HEX colors for the vertices
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
tmax = 35
lmax = 25
rmax = 90
tmin = 10
lmin = 0 
rmin = 65
colors = rgb_colormap(t*100, l*100, r*100,tmin,lmin,rmin,tmax,lmax,rmax, colors_hex)

# Plot using the custom colors
ax.scatter(t, l, r, c=colors, s=30,marker='D')  #    .
# ax.scatter(t, l, r, c=colors, s=20,marker='v')

ax.set_ternary_lim(
    tmin, tmax,  # tmin, tmax
    lmin, lmax,  # lmin, lmax
    rmin, rmax,  # rmin, rmax
)
# avoid confusion.
ax.taxis.set_label_position("tick1")
ax.laxis.set_label_position("tick1")
ax.raxis.set_label_position("tick1")

# Set ticks for each axis
tick_interval = 5  # Set the desired tick interval
ax.taxis.set_ticks(np.arange(tmin, tmax+1, tick_interval), labels=np.arange(tmin, tmax+1, tick_interval), fontsize=12)
ax.laxis.set_ticks(np.arange(lmin, lmax+1, tick_interval), labels=np.arange(lmin, lmax+1, tick_interval), fontsize=12)
ax.raxis.set_ticks(np.arange(rmin, rmax+1, tick_interval), labels=np.arange(rmin, rmax+1, tick_interval), fontsize=12)
sec_legend = []
data_colors = ['black',
              '#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2','#C2E5F2',
              '#F24405','#F24405','#F24405','#F24405','#F24405']

# print((df_summary.loc[new_data_title[i]]['1st Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
# print((df_summary.loc[new_data_title[i]]['2nd Var']/df_summary.loc[new_data_title[i]]['reg Var'])*100)
for i in range(len(new_data_title)):
    # if i == 0 :
    #     ttt = obs_result.where(condition)['2nd'].mean().item()
    #     rrr = obs_result.where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr
    # else:
    #     ttt = result_ds_list[i-1].where(condition)['2nd'].mean().item()
    #     rrr = result_ds_list[i-1].where(condition)['1st'].mean().item()
    #     lll = 100-ttt-rrr   
    ttt = df_summary.loc[new_data_title[i]]['rat_2nd']
    rrr = df_summary.loc[new_data_title[i]]['rat_1st']
    lll = 100-ttt-rrr
    if i == 1 or i==7:
        markersize = 400
    else : 
        markersize = 200
        
    ax.scatter(ttt,lll,rrr, marker=new_markers[i],edgecolor='black', color=data_colors[i], s=markersize,zorder = 2,lw = .7,alpha = .7)
    sec_legend.append(ax.scatter([],[],[], color=data_colors[i], s=markersize, marker=new_markers[i],zorder = 2,lw = .7,alpha = 1,label = new_data_title [i],edgecolor='black'))

legend2 = ax.legend(handles=sec_legend, loc='upper left', bbox_to_anchor=(1.05, 1.1), fontsize=15,frameon=False)

ax.grid()
# plt.savefig('${FIG_DIR}/pre_diurnal_cycle/ternary_obs_finish.png',format='png', dpi=300,bbox_inches='tight')
plt.show()

In [ ]:
lsm = xr.open_dataarray('${DATA_DIR}/MERRA-2/MERRA2_lsm.nc')

In [ ]:
sat_mean = xr.concat([imerg ,trmm,cmorph,gsmap,mswep],'data').mean('data')###,skipna = False)
rean_mean = xr.concat([jra55,era5,merra2],'data').mean('data',skipna = False)

In [ ]:
sat_mean = xr.concat([imerg ,trmm,cmorph,gsmap,mswep],'data').mean('data',skipna = False)
rean_mean = xr.concat([jra55,era5,merra2],'data').mean('data')#,skipna = False)

In [ ]:
sat_mean = sat_mean.clip(0,100)
rean_mean = rean_mean.clip(0,100)

In [ ]:
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D
import mpltern
from matplotlib.patches import ArrowStyle, FancyArrowPatch
from mpltern.datasets import get_dirichlet_pdfs
from matplotlib.colors import to_rgb

from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from matplotlib.lines import Line2D

def rgb_colormap(t, l, r, tmin, lmin, rmin,tmax, lmax, rmax, colors):
    """
    Custom colormap function that interpolates between three HEX-defined colors.
    """    
    # Create masks for out-of-bounds regions
    t_out = t > tmax
    l_out = l > lmax
    r_out = r < rmin
    
    # Normalize the coordinates
    total = t + l + r
    # t_norm = t / tmax
    # l_norm = l / lmax 
    t_norm = (t - tmin) / (tmax - tmin)
    l_norm = (l - lmin) / (lmax - lmin)
    r_norm = (r - rmin) / (rmax - rmin)  # r 30-100  
    # r_norm = r /rmax 
    t_norm_total = t / total
    l_norm_total = l / total
    r_norm_total = r / total
    
    # Interpolate between the three corner colors
    color_top = np.array(to_rgb(colors['top']))     # HEX to RGB #2nd
    color_left = np.array(to_rgb(colors['left']))   # HEX to RGB
    color_right = np.array(to_rgb(colors['right'])) # HEX to RGB
    
    # Weighted average of the corner colors
    red = t_norm * color_top[0] + l_norm * color_left[0] + r_norm * color_right[0]
    green = t_norm * color_top[1] + l_norm * color_left[1] + r_norm * color_right[1]
    blue = t_norm * color_top[2] + l_norm * color_left[2] + r_norm * color_right[2]
    
    colors_rgba = np.column_stack([red, green, blue, np.ones_like(red)])
    colors_rgba[t_out] = [*color_top, 1]
    colors_rgba[l_out] = [*color_left, 1]
    colors_rgba[r_out] = [*color_right, 1]
    
    # Clip values
    colors_rgba = np.clip(colors_rgba, 0, 1)
    
    # Calculate alpha based on distance from center
    center_dist = np.sqrt((t_norm_total - 1/3)**2 + (l_norm_total - 1/3)**2 + (r_norm_total - 1/3)**2)
    max_dist = np.sqrt(1/2)
    normalized_dist = center_dist / max_dist
    alpha = normalized_dist * 0.9 + 0.0
    alpha = np.clip(alpha, 0, 1)
    
    # Apply alpha
    colors_rgba[:, 3] = alpha
    
    return colors_rgba

def rgb_array(first, second, tmin, lmin, rmin, tmax, lmax, rmax, colors_hex):
    """
    Create an RGB DataArray using rgb_colormap for xarray DataArrays.
    """
    # Create an empty DataArray
    tfs_afs_rgb = xr.DataArray(
        data=np.empty([len(first.latitude), len(first.longitude), 4]) * np.nan,
        dims=["lat", "lon", "RGBs"],
        coords=dict(
            lat=(["lat"], first.latitude.values),
            lon=(["lon"], first.longitude.values),
            RGBs=(["RGBs"], [x for x in range(4)]),
        ),
    )
    
    # Flatten the DataArrays
    r_flat = first.values.flatten()  # 1st (, Right Vertex)
    t_flat = second.values.flatten()  # 2nd (, Top Vertex)
    l_flat = (100 - t_flat - r_flat)  # higher order (, Left Vertex)
    
    # Filter out NaN values
    valid_mask = ~np.isnan(t_flat) & ~np.isnan(l_flat) & ~np.isnan(r_flat)
    t_valid = t_flat[valid_mask]
    l_valid = l_flat[valid_mask]
    r_valid = r_flat[valid_mask]

    t_valid = np.clip(t_valid, tmin, tmax)
    l_valid = np.clip(l_valid, lmin, lmax)
    r_valid = np.clip(r_valid, rmin, rmax)
    
    # Compute colors using rgb_colormap
    rgba_colors = rgb_colormap(t_valid, l_valid, r_valid,tmin, lmin, rmin, tmax, lmax, rmax, colors_hex)
    
    color_top = np.array([*to_rgb(colors_hex['top']), 1])
    color_left = np.array([*to_rgb(colors_hex['left']), 1])
    color_right = np.array([*to_rgb(colors_hex['right']), 1])
    
    t_out = t_valid > tmax
    l_out = l_valid > lmax
    r_out = r_valid < rmin
    
    rgba_colors[t_out] = color_top
    rgba_colors[l_out] = color_left
    rgba_colors[r_out] = color_right
    
    # Assign computed colors back to the DataArray
    tfs_afs_rgb.values.reshape(-1, 4)[valid_mask] = rgba_colors
    
    return tfs_afs_rgb

In [ ]:
result_ds_list = [sat_mean,rean_mean]

In [ ]:
titles = ['(a) Multi-Satellite (MS)','(b) Multi-Model (MM)']

In [ ]:
rgb_list = []
colors_hex = {
    'top': '#F20505',  # OrangeRed (Top vertex) #, 2nd, t
    'left': '#0BD904',  # LimeGreen (Left vertex) #, higher, l
    'right': '#1B3BF2'  # RoyalBlue (Right vertex) #, 1st, r
}
# tmax = 35
# lmax = 25
# rmax = 90
# tmin = 10
# lmin = 0 
# rmin = 65

tmax = 35
lmax = 30 #25
rmax = 95 #90
tmin = 5 #10
lmin = 0 
rmin = 65

for i in range(len(result_ds_list)):
    # rgb_list.append(rgb_array(result_ds_list[i]['1st'], result_ds_list[i]['2nd'], tmin, lmin, rmin, tmax, lmax, rmax, colors_hex))
    rgb_list.append(rgb_array(result_ds_list[i]['rat_1st'], result_ds_list[i]['rat_2nd'], tmin, lmin, rmin, tmax, lmax, rmax, colors_hex))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy

def creat_ampratio(ax,ddd,raw_ds,extent):
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120, 180]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)

    data = ddd
    data_T = data[::-1,:,:]
    cs=ax.imshow(data_T, extent=[data.lon.values.min(), data.lon.values.max(), data.lat.values.min(), data.lat.values.max()])
    # ax.coastlines()
    ax.contourf(
        ddd.lon, ddd.lat, ddd[:,:,0].isnull(),
        transform=ccrs.PlateCarree(),
        colors='gray',
        levels=[0.5,1.5]
    ) 
    return cs

def main():
    # extent = [-150, 150, -50, 50]
    extent=[-180, 180, -60, 60]
    fig = plt.figure(figsize=(11, 8.5))
    
    #   : Multi satellite
    ax1 = fig.add_subplot(2, 1,1, projection=ccrs.PlateCarree())  # 2 
    scatter1 = creat_ampratio(ax1,rgb_list[0],result_ds_list[0],extent)
    ax1.set_title(titles[0], fontsize=18)
    # ax1.text(-150, 51, '(a)', fontsize=14, ha='left', va='bottom')
    
    #   : Multi reanalysis
    ax2 = fig.add_subplot(2, 1, 2, projection=ccrs.PlateCarree())  # 2 
    scatter2 = creat_ampratio(ax2,rgb_list[1],result_ds_list[1],extent)
    ax2.set_title(titles[1], fontsize=18)
    # ax2.text(-150, 51, '(b)', fontsize=14, ha='left', va='bottom')
    
    plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/amp_ratio_finish.png', dpi=400, bbox_inches='tight')
    # plt.show()

if __name__ == "__main__":
    main()

In [ ]:
df_summary

##### Map


In [ ]:
#obs = xr.open_dataset('${DATA_DIR}/obs/obs_var_new.nc')
imerg = xr.open_dataset('${DATA_DIR}/IMERG/IMERG_var_new.nc')
trmm = xr.open_dataset('${DATA_DIR}/TRMM/TRMM_var_new.nc')
cmorph = xr.open_dataset('${DATA_DIR}/CMORPH/CMORPH_var_new.nc')
gsmap = xr.open_dataset('${DATA_DIR}/GSMaP/GSMaP_var_new.nc')
mswep = xr.open_dataset('${DATA_DIR}/MSWEP/MSWEP_var_new.nc')

jra55 = xr.open_dataset('${DATA_DIR}/JRA-3Q/JRA3Q_var_new.nc')
era5 = xr.open_dataset('${DATA_DIR}/ERA5/ERA5_var_new.nc')
merra2 = xr.open_dataset('${DATA_DIR}/MERRA2/MERRA2_var_new1.nc')

In [ ]:
sat_mean_rat1 = xr.concat([imerg,trmm,cmorph, gsmap,mswep],'data').mean('data')#,skipna=False)
rean_mean_rat1 = xr.concat([era5,jra55,merra2],'data').mean('data')#,skipna=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
import copy
def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    #  gridlines labels 
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    #  31  
    # bounds = np.linspace(0, 100, levels + 1)  # +1  
    bounds = np.linspace(0, 15, levels + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)
    return pc

def create_latitude_histogram(ax, ds, var, lsm, extent,legends = False):
    """ / HFR   """
    
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)  # 2 
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    
    land_means = []
    ocean_means = []
    land_stds = []
    ocean_stds = []
    
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i + 1])
        ds_lat = ds.where(lat_mask)
        
        land_data = ds_lat.where(lsm)[var].values.flatten()
        ocean_data = ds_lat.where(~lsm)[var].values.flatten()
        
        # NaN 
        land_data = land_data[~np.isnan(land_data)]
        ocean_data = ocean_data[~np.isnan(ocean_data)]
        
        if len(land_data) > 0:
            land_means.append(np.mean(land_data))
            land_stds.append(np.std(land_data))
        else:
            land_means.append(np.nan)
            land_stds.append(np.nan)
            
        if len(ocean_data) > 0:
            ocean_means.append(np.mean(ocean_data))
            ocean_stds.append(np.std(ocean_data))
        else:
            ocean_means.append(np.nan)
            ocean_stds.append(np.nan)
    
    land_means = np.array(land_means)
    ocean_means = np.array(ocean_means)
    land_stds = np.array(land_stds)
    ocean_stds = np.array(ocean_stds)
    
    #   ()
    valid_land = ~np.isnan(land_means)
    ax.plot(land_means[valid_land], lat_centers[valid_land], 
            color='brown', linewidth=1.5, label='Land')
    ax.fill_betweenx(lat_centers[valid_land], 
                     land_means[valid_land] - land_stds[valid_land],
                     land_means[valid_land] + land_stds[valid_land],
                     color='brown', alpha=0.3)
    
    #   ()
    valid_ocean = ~np.isnan(ocean_means)
    ax.plot(ocean_means[valid_ocean], lat_centers[valid_ocean], 
            color='blue', linewidth=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[valid_ocean], 
                     ocean_means[valid_ocean] - ocean_stds[valid_ocean],
                     ocean_means[valid_ocean] + ocean_stds[valid_ocean],
                     color='blue', alpha=0.3)
    
    ax.set_xlim(45, 95)
    
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max,20))
    ax.set_xlabel('HFR [%]', fontsize=10)
    ax.tick_params(axis='y', labelleft=False)  # y  
    ax.tick_params(labelsize=9)
    ax.grid(True, alpha=0.3)
    if legends :
        ax.legend(fontsize=9, frameon=False, facecolor='none')


def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=vmin, vmax=vmax)
    
    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [(sat_mean_rat1['var_1st']/sat_mean_rat1['var_total'])*100,
                  (rean_mean_rat1['var_1st']/rean_mean_rat1['var_total'])*100]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 40
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(b) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)

    ax1_hist.set_xlim(0, 10)  #  x  
    ax1_hist.set_xticks([0,5,10])
    ax1_hist.set_xlabel(None)

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(c) Multi-Model (MM)', fontsize=18)
    
    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)    
    ax2_hist.set_xlim(0, 10)  #  x  
    ax2_hist.set_xticks([0,5,10])
    # ax2_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    ax2_hist.set_xlabel(None)
    # ---   : Difference (MM - MS) ---
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -10, 10, 'diff')
    ax3.set_title('(d) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent)    
    ax3_hist.set_xlim(-8, 8)  #  x  
    ax3_hist.set_xticks([-5,0,5])
    # ax3_hist.set_xlabel(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=11)
    ax3_hist.set_xlabel(None)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, 15, 6)  # 70, 75, 80, 85, 90, 95, 100
    
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)

    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]', fontsize=12)
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import BoundaryNorm
import cartopy.crs as ccrs
import colormaps as cmaps
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import copy
from matplotlib.ticker import ScalarFormatter

def create_local_solar_time_plot(ax, ds, extent, cmap, levels, var, ref_data=None):
    crs = ccrs.PlateCarree()
    ax.ticklabel_format(axis='y', style='sci', scilimits=(0,0), useMathText=True)

    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    bounds = np.linspace(0, .4, 8*4 + 1)  # +1  

    norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   
    
    # NaN        
    cmap_with_gray = copy.copy(cmap)
    cmap_with_gray.set_bad(color='gray')
    
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap_with_gray,
                      norm=norm)

    return pc

def create_diff_plot(ax, ds_diff, extent, cmap, vmin, vmax, var):
    """ (Difference Map)  """
    crs = ccrs.PlateCarree()
    
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    
    x_ticks = [-180, -120, -60, 0, 60, 120]
    y_ticks = [-60, -30, 0, 30, 60]
    
    ax.set_xticks(x_ticks, crs=crs)
    ax.set_yticks(y_ticks, crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    
    # -3 3   
    norm = plt.Normalize(vmin=-.2, vmax=.2)
    # bounds = np.linspace(vmin, vmax, 40 + 1)  # +1  
    # norm = BoundaryNorm(bounds, cmap.N)  # cmap.N   

    pc = ax.pcolormesh(ds_diff.longitude, ds_diff.latitude, ds_diff[var],
                      transform=ccrs.PlateCarree(),
                      cmap=cmap,
                      norm=norm)
    return pc
result_ds_list = [sat_mean_rat1['var_1st'],
                  rean_mean_rat1['var_1st']]

def main():
    extent = [-180., 180., -60.001, 60.001]
    #  3  figsize   (8.5 -> 12)
    fig = plt.figure(figsize=(11, 12))  
    
    levels = 28
    cmap_main = Colormap('colorcet:CET_L17').to_mpl()
    
    # ---   : Multi satellite (MS) ---
    ax1 = fig.add_subplot(3, 1, 1, projection=ccrs.PlateCarree())
    create_local_solar_time_plot(ax1, result_ds_list[0].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax1.set_title('(e) Multi-Satellite (MS)', fontsize=18)
    
    ax1_hist = fig.add_axes([ax1.get_position().x1, ax1.get_position().y0, 0.12, ax1.get_position().height])
    create_latitude_histogram(ax1_hist, result_ds_list[0].to_dataset(name='rat'), 'rat', lsm, extent, True)
    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))
    ax1_hist.xaxis.set_major_formatter(formatter_hist)
    
    ax1_hist.set_xlim(0, .4)
    ax1_hist.set_xlabel(None, fontsize=11)
    # ax1_hist.set_xticks([0,10,20,30,40,50,60])

    # ---   : Multi reanalysis (MM) ---
    ax2 = fig.add_subplot(3, 1, 2, projection=ccrs.PlateCarree())
    pc_main = create_local_solar_time_plot(ax2, result_ds_list[1].to_dataset(name='rat'), extent, cmap_main, levels, 'rat')
    ax2.set_title('(f) Multi-Model (MM)', fontsize=18)
    
    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))

    ax2_hist = fig.add_axes([ax2.get_position().x1, ax2.get_position().y0, 0.12, ax2.get_position().height])
    create_latitude_histogram(ax2_hist, result_ds_list[1].to_dataset(name='rat'), 'rat', lsm, extent)
    ax2_hist.xaxis.set_major_formatter(formatter_hist)

    ax2_hist.set_xlim(0, .4)
    ax2_hist.set_xlabel(None, fontsize=11)

    # ---   : Difference (MM - MS) ---
    #        add_subplot  position  
    #   x0, width  .
    ax3 = fig.add_subplot(3, 1, 3, projection=ccrs.PlateCarree())
    diff_ds = (result_ds_list[1] - result_ds_list[0]).to_dataset(name='diff')
    pc_diff = create_diff_plot(ax3, diff_ds, extent, 'RdBu_r', -20, 20, 'diff')
    ax3.set_title('(g) MM - MS', fontsize=18)
    
    #      ax3    (  )
    pos_ref = ax2.get_position()
    ax3.set_position([pos_ref.x0, pos_ref.y0 - 0.36, pos_ref.width, pos_ref.height])
    
    ax3_hist = fig.add_axes([ax3.get_position().x1, ax3.get_position().y0, 0.12, ax3.get_position().height])
    create_latitude_histogram(ax3_hist, diff_ds, 'diff', lsm, extent, True)

    formatter_hist = ScalarFormatter(useMathText=True)
    formatter_hist.set_powerlimits((0, 0))
    ax3_hist.xaxis.set_major_formatter(formatter_hist)
    
    ax3_hist.set_xlim(-.2, .2)
    # ax3_hist.set_xticks([-10,0,10])
    ax3_hist.set_xlabel(None, fontsize=11)

    # 1.     (0 to 1.5)
    cbar_ticks = np.linspace(0, .4, 9)
    cax1 = fig.add_axes([pos_ref.x0+.03, pos_ref.y0 - 0.05,(pos_ref.x1-pos_ref.x0)-.06,0.02])
    cbar1 = fig.colorbar(pc_main, cax=cax1, orientation='horizontal', extend='max', ticks=cbar_ticks)
    cbar1.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    cbar1.ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    cbar1.ax.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
    # 2.     (-3 to 3)
    pos3 = ax3.get_position()
    cax2 = fig.add_axes([pos3.x0+.03, pos3.y0 - 0.05,(pos3.x1-pos3.x0)-.06,0.02])
    cbar2 = fig.colorbar(pc_diff, cax=cax2, orientation='horizontal', extend='both')
    cbar2.set_label(r'$\sigma_{1st}^2$ [$mm^2/h^2$]', fontsize=12)
    cbar2.ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    cbar2.ax.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)

    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import ScalarFormatter
import copy

# ──   ─────────────────────────────────────────
#  figure: 2(left=b,c,d / right=e,f,g) × 3
#   : () + ()
# gridspec_kw    

fig = plt.figure(figsize=(22, 12))

#  GridSpec: 2 (left col, right col)
outer = gridspec.GridSpec(
    1, 2,
    figure=fig,
    wspace=0.01,
    left=0.04, right=0.96,
    top=0.95, bottom=0.08
)

#   : 3 × 2 ( + )
# : = 5:1 
def make_inner_gs(outer_cell):
    return gridspec.GridSpecFromSubplotSpec(
        3, 2,
        subplot_spec=outer_cell,
        hspace=0.08,
        wspace=0.0,  #     0
        width_ratios=[5, 1]
    )

gs_left  = make_inner_gs(outer[0, 0])   # b, c, d
gs_right = make_inner_gs(outer[0, 1])   # e, f, g

# ──   ───────────────────────────────────────────────
#  : σ²_1st/σ²_total [%]
ax_b  = fig.add_subplot(gs_left[0, 0], projection=ccrs.PlateCarree())
ax_bh = fig.add_subplot(gs_left[0, 1])
ax_c  = fig.add_subplot(gs_left[1, 0], projection=ccrs.PlateCarree())
ax_ch = fig.add_subplot(gs_left[1, 1])
ax_d  = fig.add_subplot(gs_left[2, 0], projection=ccrs.PlateCarree())
ax_dh = fig.add_subplot(gs_left[2, 1])

#  : σ²_1st [mm² h⁻²]
ax_e  = fig.add_subplot(gs_right[0, 0], projection=ccrs.PlateCarree())
ax_eh = fig.add_subplot(gs_right[0, 1])
ax_f  = fig.add_subplot(gs_right[1, 0], projection=ccrs.PlateCarree())
ax_fh = fig.add_subplot(gs_right[1, 1])
ax_g  = fig.add_subplot(gs_right[2, 0], projection=ccrs.PlateCarree())
ax_gh = fig.add_subplot(gs_right[2, 1])

# ──   ───────────────────────────────────────────
extent = [-180., 180., -60.001, 60.001]

# : σ²_1st/σ²_total * 100 [%]
left_data = [
    (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100).to_dataset(name='rat'),
    (rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100).to_dataset(name='rat'),
]
left_diff = ((rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100) -
             (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100)).to_dataset(name='diff')

# : σ²_1st 
right_data = [
    sat_mean_rat1['var_1st'].to_dataset(name='rat'),
    rean_mean_rat1['var_1st'].to_dataset(name='rat'),
]
right_diff = (rean_mean_rat1['var_1st'] - sat_mean_rat1['var_1st']).to_dataset(name='diff')

# ──   ───────────────────────────────────────────
cmap_main = Colormap('colorcet:CET_L17').to_mpl()
cmap_main_copy = copy.copy(cmap_main)
cmap_main_copy.set_bad(color='gray')

# ──    ────────────────────────────────────────
def plot_map(ax, ds, var, cmap, norm,
             show_xlabel=False, show_ylabel=False, add_zeroline=False):
    crs = ccrs.PlateCarree()
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    ax.set_xticks([-180,-120,-60,0,60,120], crs=crs)
    ax.set_yticks([-60,-30,0,30,60], crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(labelsize=8)
    #  :   (show_xlabel=True) 
    if not show_xlabel:
        ax.set_xticklabels([])
    #  :  (show_ylabel=True) 
    if not show_ylabel:
        ax.set_yticklabels([])
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                       transform=crs, cmap=cmap, norm=norm)
    # diff : 0  
    return pc

def plot_hist(ax, ds, var, xlim, xticks=None, show_legend=False):
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    land_means, ocean_means = [], []
    land_stds, ocean_stds = [], []
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i+1])
        ds_lat = ds.where(lat_mask)
        ld = ds_lat.where(lsm)[var].values.flatten()
        od = ds_lat.where(~lsm)[var].values.flatten()
        ld = ld[~np.isnan(ld)]; od = od[~np.isnan(od)]
        land_means.append(np.mean(ld) if len(ld) > 0 else np.nan)
        land_stds.append(np.std(ld)  if len(ld) > 0 else np.nan)
        ocean_means.append(np.mean(od) if len(od) > 0 else np.nan)
        ocean_stds.append(np.std(od)  if len(od) > 0 else np.nan)
    lm = np.array(land_means); ls = np.array(land_stds)
    om = np.array(ocean_means); os = np.array(ocean_stds)
    vl = ~np.isnan(lm); vo = ~np.isnan(om)
    ax.plot(lm[vl], lat_centers[vl], color='brown', lw=1.5, label='Land')
    ax.fill_betweenx(lat_centers[vl], lm[vl]-ls[vl], lm[vl]+ls[vl], color='brown', alpha=0.3)
    ax.plot(om[vo], lat_centers[vo], color='blue',  lw=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[vo], om[vo]-os[vo], om[vo]+os[vo], color='blue',  alpha=0.3)
    ax.set_xlim(xlim)
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max, 20))
    ax.tick_params(axis='y', labelleft=False)
    ax.tick_params(labelsize=8)
    if xticks: ax.set_xticks(xticks)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=8, frameon=False)

# ──   (b, c, d): σ²_1st/σ²_total ────────────────
norm_left = BoundaryNorm(np.linspace(0, 15, 41), cmap_main.N)
norm_diff_left = plt.Normalize(vmin=-10, vmax=10)
cmap_diff = plt.get_cmap('RdBu_r')
cmap_diff_copy = copy.copy(cmap_diff)
cmap_diff_copy.set_bad(color='gray')

# b, c: O X / d: O O + 0
pc_left_main = plot_map(ax_b, left_data[0], 'rat', cmap_main_copy, norm_left,
                        show_xlabel=False, show_ylabel=True)
ax_b.set_title('(b) Multi-Satellite (MS)', fontsize=13)
plot_hist(ax_bh, left_data[0], 'rat', (0, 10), [0,5,10], show_legend=True)

pc_left_main2 = plot_map(ax_c, left_data[1], 'rat', cmap_main_copy, norm_left,
                         show_xlabel=False, show_ylabel=True)
ax_c.set_title('(c) Multi-Model (MM)', fontsize=13)
plot_hist(ax_ch, left_data[1], 'rat', (0, 10), [0,5,10])

pc_left_diff = plot_map(ax_d, left_diff, 'diff', cmap_diff_copy, norm_diff_left,
                        show_xlabel=True, show_ylabel=True, add_zeroline=True)
ax_d.set_title('(d) MM \u2212 MS', fontsize=13)
plot_hist(ax_dh, left_diff, 'diff', (-8, 8), [-5,0,5])

# ──   (e, f, g): σ²_1st  ────────────────
norm_right = BoundaryNorm(np.linspace(0, 0.4, 33), cmap_main.N)
norm_diff_right = plt.Normalize(vmin=-0.2, vmax=0.2)

# e, f: X X / g: X O + 0
pc_right_main = plot_map(ax_e, right_data[0], 'rat', cmap_main_copy, norm_right,
                         show_xlabel=False, show_ylabel=False)
ax_e.set_title('(e) Multi-Satellite (MS)', fontsize=13)
plot_hist(ax_eh, right_data[0], 'rat', (0, 0.4), show_legend=True)
ax_eh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_eh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)

pc_right_main2 = plot_map(ax_f, right_data[1], 'rat', cmap_main_copy, norm_right,
                          show_xlabel=False, show_ylabel=False)
ax_f.set_title('(f) Multi-Model (MM)', fontsize=13)
plot_hist(ax_fh, right_data[1], 'rat', (0, 0.4))
ax_fh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_fh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)

pc_right_diff = plot_map(ax_g, right_diff, 'diff', cmap_diff_copy, norm_diff_right,
                         show_xlabel=True, show_ylabel=False, add_zeroline=True)
ax_g.set_title('(g) MM \u2212 MS', fontsize=13)
plot_hist(ax_gh, right_diff, 'diff', (-0.2, 0.2))
ax_gh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_gh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)

# ──  ────────────────────────────────────────────────
def add_colorbar(fig, pc, cax, ticks, label, extend='neither', sci=False):
    cb = fig.colorbar(pc, cax=cax, orientation='horizontal',
                      extend=extend, ticks=ticks)
    cb.set_label('')
    cb.ax.text(1.07, 0.5, label,
               transform=cb.ax.transAxes,
               ha='left', va='center', fontsize=11)
    # if sci:
    #     cb.ax.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    #     cb.ax.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
    return cb

#    (b, c )
pos_c = ax_c.get_position()
cax_left_main = fig.add_axes([pos_c.x0+0.02, pos_c.y0-0.04, pos_c.width*0.80, 0.015])
cb1 = add_colorbar(fig, pc_left_main, cax_left_main,
                   ticks=np.linspace(0,15,6),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='max')

#  diff 
pos_d = ax_d.get_position()
cax_left_diff = fig.add_axes([pos_d.x0+0.02, pos_d.y0-0.04, pos_d.width*0.80, 0.015])
cb2 = add_colorbar(fig, pc_left_diff, cax_left_diff,
                   ticks=np.linspace(-10,10,5),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='both')

#    (e, f )
pos_f = ax_f.get_position()
cax_right_main = fig.add_axes([pos_f.x0+0.02, pos_f.y0-0.04, pos_f.width*0.80, 0.015])
cb3 = add_colorbar(fig, pc_right_main, cax_right_main,
                   ticks=np.linspace(0,0.4,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='max', sci=True)

#  diff 
pos_g = ax_g.get_position()
cax_right_diff = fig.add_axes([pos_g.x0+0.02, pos_g.y0-0.04, pos_g.width*0.80, 0.015])
cb4 = add_colorbar(fig, pc_right_diff, cax_right_diff,
                   ticks=np.linspace(-0.2,0.2,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='both', sci=True)

# ──      ──────────────────
# cartopy      GridSpec   
# draw()        
fig.canvas.draw()

hist_width = 0.055
row1_shift = -0.02
row3_shift = -0.03

for name, ax_map in [('b', ax_b), ('e', ax_e)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row1_shift, pos.width, pos.height])

for name, ax_map in [('d', ax_d), ('g', ax_g)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row3_shift, pos.width, pos.height])

# ──    ──────────────────────────────
fig.canvas.draw()

for ax_map, ax_hist in [
    (ax_b, ax_bh), (ax_c, ax_ch), (ax_d, ax_dh),
    (ax_e, ax_eh), (ax_f, ax_fh), (ax_g, ax_gh),
]:
    pos = ax_map.get_position()
    ax_hist.set_position([pos.x1, pos.y0, hist_width, pos.height])

# ──     position   ──
cbar_gap = 0.04

pos_c = ax_c.get_position()
cax_left_main.set_position([pos_c.x0+0.02, pos_c.y0-cbar_gap, pos_c.width*0.80, 0.015])

pos_d = ax_d.get_position()
cax_left_diff.set_position([pos_d.x0+0.02, pos_d.y0-cbar_gap, pos_d.width*0.80, 0.015])

pos_f = ax_f.get_position()
cax_right_main.set_position([pos_f.x0+0.02, pos_f.y0-cbar_gap, pos_f.width*0.80, 0.015])

pos_g = ax_g.get_position()
cax_right_diff.set_position([pos_g.x0+0.02, pos_g.y0-cbar_gap, pos_g.width*0.80, 0.015])

# plt.savefig('fig4_combined.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import ScalarFormatter
import copy

# ──   ─────────────────────────────────────────
#  figure: 2(left=b,c,d / right=e,f,g) × 3
#   : () + ()
# gridspec_kw    

fig = plt.figure(figsize=(22/1.5, 12/1.5))

#  GridSpec: 2 (left col, right col)
outer = gridspec.GridSpec(
    1, 2,
    figure=fig,
    wspace=0.06,
    left=0.04, right=0.96,
    top=0.95, bottom=0.08
)

#   : 3 × 2 ( + )
# : = 5:1 
def make_inner_gs(outer_cell):
    return gridspec.GridSpecFromSubplotSpec(
        3, 2,
        subplot_spec=outer_cell,
        hspace=0.08,
        wspace=0.0,  #     0
        width_ratios=[5, 1]
    )

gs_left  = make_inner_gs(outer[0, 0])   # b, c, d
gs_right = make_inner_gs(outer[0, 1])   # e, f, g

# ──   ───────────────────────────────────────────────
#  : σ²_1st/σ²_total [%]
ax_b  = fig.add_subplot(gs_left[0, 0], projection=ccrs.PlateCarree())
ax_bh = fig.add_subplot(gs_left[0, 1])
ax_c  = fig.add_subplot(gs_left[1, 0], projection=ccrs.PlateCarree())
ax_ch = fig.add_subplot(gs_left[1, 1])
ax_d  = fig.add_subplot(gs_left[2, 0], projection=ccrs.PlateCarree())
ax_dh = fig.add_subplot(gs_left[2, 1])

#  : σ²_1st [mm² h⁻²]
ax_e  = fig.add_subplot(gs_right[0, 0], projection=ccrs.PlateCarree())
ax_eh = fig.add_subplot(gs_right[0, 1])
ax_f  = fig.add_subplot(gs_right[1, 0], projection=ccrs.PlateCarree())
ax_fh = fig.add_subplot(gs_right[1, 1])
ax_g  = fig.add_subplot(gs_right[2, 0], projection=ccrs.PlateCarree())
ax_gh = fig.add_subplot(gs_right[2, 1])

# ──   ───────────────────────────────────────────
extent = [-180., 180., -60.001, 60.001]

# : σ²_1st/σ²_total * 100 [%]
left_data = [
    (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100).to_dataset(name='rat'),
    (rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100).to_dataset(name='rat'),
]
left_diff = ((rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100) -
             (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100)).to_dataset(name='diff')

# : σ²_1st 
right_data = [
    sat_mean_rat1['var_1st'].to_dataset(name='rat'),
    rean_mean_rat1['var_1st'].to_dataset(name='rat'),
]
right_diff = (rean_mean_rat1['var_1st'] - sat_mean_rat1['var_1st']).to_dataset(name='diff')

# ──   ───────────────────────────────────────────
cmap_main = Colormap('colorcet:CET_L17').to_mpl()
cmap_main_copy = copy.copy(cmap_main)
cmap_main_copy.set_bad(color='gray')

# ──    ────────────────────────────────────────
def plot_map(ax, ds, var, cmap, norm,
             show_xlabel=False, show_ylabel=False, add_zeroline=False):
    crs = ccrs.PlateCarree()
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    ax.set_xticks([-180,-120,-60,0,60,120], crs=crs)
    ax.set_yticks([-60,-30,0,30,60], crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(labelsize=8)
    #  :   (show_xlabel=True) 
    if not show_xlabel:
        ax.set_xticklabels([])
    #  :  (show_ylabel=True) 
    if not show_ylabel:
        ax.set_yticklabels([])
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                       transform=crs, cmap=cmap, norm=norm)
    return pc

def plot_hist(ax, ds, var, xlim, xticks=None, show_legend=False):
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    land_means, ocean_means = [], []
    land_stds, ocean_stds = [], []
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i+1])
        ds_lat = ds.where(lat_mask)
        ld = ds_lat.where(lsm)[var].values.flatten()
        od = ds_lat.where(~lsm)[var].values.flatten()
        ld = ld[~np.isnan(ld)]; od = od[~np.isnan(od)]
        land_means.append(np.mean(ld) if len(ld) > 0 else np.nan)
        land_stds.append(np.std(ld)  if len(ld) > 0 else np.nan)
        ocean_means.append(np.mean(od) if len(od) > 0 else np.nan)
        ocean_stds.append(np.std(od)  if len(od) > 0 else np.nan)
    lm = np.array(land_means); ls = np.array(land_stds)
    om = np.array(ocean_means); os = np.array(ocean_stds)
    vl = ~np.isnan(lm); vo = ~np.isnan(om)
    ax.plot(lm[vl], lat_centers[vl], color='brown', lw=1.5, label='Land')
    ax.fill_betweenx(lat_centers[vl], lm[vl]-ls[vl], lm[vl]+ls[vl], color='brown', alpha=0.3)
    ax.plot(om[vo], lat_centers[vo], color='blue',  lw=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[vo], om[vo]-os[vo], om[vo]+os[vo], color='blue',  alpha=0.3)
    ax.set_xlim(xlim)
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max, 20))
    ax.tick_params(axis='y', labelleft=False)
    ax.tick_params(labelsize=8)
    ax.xaxis.get_offset_text().set_fontsize(8)  # ← 
    ax.yaxis.get_offset_text().set_fontsize(8)

    if xticks: ax.set_xticks(xticks)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=8, frameon=False)

# ──   (b, c, d): σ²_1st/σ²_total ────────────────
norm_left = BoundaryNorm(np.linspace(0, 12, 12*3+5), cmap_main.N)
norm_diff_left = plt.Normalize(vmin=-10, vmax=10)
cmap_diff = plt.get_cmap('RdBu_r')
cmap_diff_copy = copy.copy(cmap_diff)
cmap_diff_copy.set_bad(color='gray')
cmap_right = cmaps.WhiteYellowOrangeRed
# b, c: O X / d: O O
pc_left_main = plot_map(ax_b, left_data[0], 'rat', cmaps.WhiteYellowOrangeRed, norm_left,
                        show_xlabel=False, show_ylabel=True)
plot_hist(ax_bh, left_data[0], 'rat', (0, 10), [0,5,10], show_legend=True)

pc_left_main2 = plot_map(ax_c, left_data[1], 'rat', cmaps.WhiteYellowOrangeRed, norm_left,
                         show_xlabel=False, show_ylabel=True)
plot_hist(ax_ch, left_data[1], 'rat', (0, 10), [0,5,10])

pc_left_diff = plot_map(ax_d, left_diff, 'diff', cmap_diff_copy, norm_diff_left,
                        show_xlabel=True, show_ylabel=True, add_zeroline=True)
plot_hist(ax_dh, left_diff, 'diff', (-8, 8), [-5,0,5])

# ──   (e, f, g): σ²_1st  ────────────────
norm_right = BoundaryNorm(np.linspace(0, 0.4, 33), cmap_main.N)
norm_diff_right = plt.Normalize(vmin=-0.2, vmax=0.2)

# e, f: X X / g: X O
pc_right_main = plot_map(ax_e, right_data[0], 'rat', cmap_main_copy, norm_right,
                         show_xlabel=False, show_ylabel=False)
plot_hist(ax_eh, right_data[0], 'rat', (0, 0.4), show_legend=True)
ax_eh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax_eh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_eh.tick_params(labelsize=8)

pc_right_main2 = plot_map(ax_f, right_data[1], 'rat', cmap_main_copy, norm_right,
                          show_xlabel=False, show_ylabel=False)
plot_hist(ax_fh, right_data[1], 'rat', (0, 0.4))
ax_fh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_fh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_fh.tick_params(labelsize=8)

pc_right_diff = plot_map(ax_g, right_diff, 'diff', cmaps.fusion_r, norm_diff_right,
                         show_xlabel=True, show_ylabel=False, add_zeroline=True)
plot_hist(ax_gh, right_diff, 'diff', (-0.2, 0.2))
ax_gh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_gh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_gh.tick_params(labelsize=8)
# ──   (    ) ────────────────
label_kwargs = dict(transform=ccrs.PlateCarree(),
                    fontsize=12,
                    ha='left', va='bottom', zorder=10,
                    bbox=dict(facecolor='white', edgecolor='none',
                              alpha=0.7, pad=1.5))
for ax, letter in [(ax_b, 'b'), (ax_c, 'c'), (ax_d, 'd'),
                   (ax_e, 'e'), (ax_f, 'f'), (ax_g, 'g')]:
    ax.text(-178, -57, f'({letter})', **label_kwargs)

# ──   (    ,  ) ──────────
# axes  : x=0  , y=0.5 
row_label_kwargs = dict(fontsize=14, fontweight='bold',
                        ha='center', va='center',
                        rotation=90, rotation_mode='anchor',
                        transform=ax_b.transAxes,
                        clip_on=False)
row_labels = [
    (ax_b, 'MS'),
    (ax_c, 'MM'),
    (ax_d, 'MM \u2212 MS'),
]
for ax, label in row_labels:
    ax.text(-0.1, 0.5, label,
            fontsize=15,
            ha='center', va='center',
            rotation=90, rotation_mode='anchor',
            transform=ax.transAxes,
            clip_on=False)

# ──  ────────────────────────────────────────────────
def add_colorbar(fig, pc, cax, ticks, label, extend='neither'):
    cb = fig.colorbar(pc, cax=cax, orientation='horizontal',
                      extend=extend, ticks=ticks)
    cb.set_label('')
    cb.ax.tick_params(labelsize=8,which='minor', length=0)
    cb.ax.text(1.07, 0.5, label,
               transform=cb.ax.transAxes,
               ha='left', va='center', fontsize=10)
    return cb

#    (b, c )
pos_c = ax_c.get_position()
cax_left_main = fig.add_axes([pos_c.x0+0.02, pos_c.y0-0.04, pos_c.width*0.80, 0.015])
cb1 = add_colorbar(fig, pc_left_main, cax_left_main,
                   ticks=np.linspace(0,15,6),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='max')

#  diff 
pos_d = ax_d.get_position()
cax_left_diff = fig.add_axes([pos_d.x0+0.02, pos_d.y0-0.04, pos_d.width*0.80, 0.015])
cb2 = add_colorbar(fig, pc_left_diff, cax_left_diff,
                   ticks=np.linspace(-10,10,5),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='both')

#    (e, f )
pos_f = ax_f.get_position()
cax_right_main = fig.add_axes([pos_f.x0+0.02, pos_f.y0-0.04, pos_f.width*0.80, 0.015])
cb3 = add_colorbar(fig, pc_right_main, cax_right_main,
                   ticks=np.linspace(0,0.4,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='max')

#  diff 
pos_g = ax_g.get_position()
cax_right_diff = fig.add_axes([pos_g.x0+0.02, pos_g.y0-0.04, pos_g.width*0.80, 0.015])
cb4 = add_colorbar(fig, pc_right_diff, cax_right_diff,
                   ticks=np.linspace(-0.2,0.2,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='both')

# ──      ──────────────────
# cartopy      GridSpec   
# draw()        
fig.canvas.draw()

hist_width = 0.055
row1_shift = -0.04  # 1  (c  )
row3_shift = -0.02  # 3  (c  )

# ──  position  ──
positions = {}
for name, ax_map in [('b', ax_b), ('c', ax_c), ('d', ax_d),
                     ('e', ax_e), ('f', ax_f), ('g', ax_g)]:
    positions[name] = ax_map.get_position()

# ── 1, 3  ──
for name, ax_map in [('b', ax_b), ('e', ax_e)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row1_shift, pos.width, pos.height])

for name, ax_map in [('d', ax_d), ('g', ax_g)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row3_shift, pos.width, pos.height])
# ──    ──────────────────────────────
fig.canvas.draw()

for ax_map, ax_hist in [
    (ax_b, ax_bh), (ax_c, ax_ch), (ax_d, ax_dh),
    (ax_e, ax_eh), (ax_f, ax_fh), (ax_g, ax_gh),
]:
    pos = ax_map.get_position()
    ax_hist.set_position([pos.x1, pos.y0, hist_width, pos.height])

# ──     position   ──
cbar_gap = 0.047

pos_c = ax_c.get_position()
cax_left_main.set_position([pos_c.x0+0.02, pos_c.y0-cbar_gap, pos_c.width*0.80, 0.018])

pos_d = ax_d.get_position()
cax_left_diff.set_position([pos_d.x0+0.02, pos_d.y0-cbar_gap, pos_d.width*0.80, 0.018])

pos_f = ax_f.get_position()
cax_right_main.set_position([pos_f.x0+0.02, pos_f.y0-cbar_gap, pos_f.width*0.80, 0.018])

pos_g = ax_g.get_position()
cax_right_diff.set_position([pos_g.x0+0.02, pos_g.y0-cbar_gap, pos_g.width*0.80, 0.018])

# plt.savefig('fig4_combined.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
from scipy.ndimage import gaussian_filter

def smooth_dataset(ds, var, sigma=3):
    """
    Apply Gaussian smoothing while handling NaN
    sigma: larger value applies more smoothing (grid units)
    """
    data = ds[var].values.copy()
    
    # NaN : NaN 0    
    nan_mask = np.isnan(data)
    data_filled = np.where(nan_mask, 0, data)
    mask_filled = np.where(nan_mask, 0, 1).astype(float)
    
    # /   →  weighted mean 
    smoothed_data = gaussian_filter(data_filled, sigma=sigma)
    smoothed_mask = gaussian_filter(mask_filled, sigma=sigma)
    
    # NaN  
    with np.errstate(invalid='ignore'):
        result = smoothed_data / smoothed_mask
    result[nan_mask] = np.nan
    
    # xarray Dataset Returns
    ds_smooth = ds.copy()
    ds_smooth[var].values[:] = result
    return ds_smooth

right_diff_smooth = smooth_dataset(right_diff, 'diff', sigma=1.5)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
from matplotlib.colors import BoundaryNorm
from matplotlib.ticker import ScalarFormatter
import copy

# ──   ─────────────────────────────────────────
#  figure: 2(left=b,c,d / right=e,f,g) × 3
#   : () + ()
# gridspec_kw    

fig = plt.figure(figsize=(22/1.5, 12/1.5))

#  GridSpec: 2 (left col, right col)
outer = gridspec.GridSpec(
    1, 2,
    figure=fig,
    wspace=0.01,
    left=0.04, right=0.96,
    top=0.95, bottom=0.08
)

#   : 3 × 2 ( + )
# : = 5:1 
def make_inner_gs(outer_cell):
    return gridspec.GridSpecFromSubplotSpec(
        3, 2,
        subplot_spec=outer_cell,
        hspace=0.08,
        wspace=0.0,  #     0
        width_ratios=[5, 1]
    )

gs_left  = make_inner_gs(outer[0, 0])   # b, c, d
gs_right = make_inner_gs(outer[0, 1])   # e, f, g

# ──   ───────────────────────────────────────────────
#  : σ²_1st/σ²_total [%]
ax_b  = fig.add_subplot(gs_left[0, 0], projection=ccrs.PlateCarree())
ax_bh = fig.add_subplot(gs_left[0, 1])
ax_c  = fig.add_subplot(gs_left[1, 0], projection=ccrs.PlateCarree())
ax_ch = fig.add_subplot(gs_left[1, 1])
ax_d  = fig.add_subplot(gs_left[2, 0], projection=ccrs.PlateCarree())
ax_dh = fig.add_subplot(gs_left[2, 1])

#  : σ²_1st [mm² h⁻²]
ax_e  = fig.add_subplot(gs_right[0, 0], projection=ccrs.PlateCarree())
ax_eh = fig.add_subplot(gs_right[0, 1])
ax_f  = fig.add_subplot(gs_right[1, 0], projection=ccrs.PlateCarree())
ax_fh = fig.add_subplot(gs_right[1, 1])
ax_g  = fig.add_subplot(gs_right[2, 0], projection=ccrs.PlateCarree())
ax_gh = fig.add_subplot(gs_right[2, 1])

# ──   ───────────────────────────────────────────
extent = [-180., 180., -60.001, 60.001]

# : σ²_1st/σ²_total * 100 [%]
left_data = [
    (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100).to_dataset(name='rat'),
    (rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100).to_dataset(name='rat'),
]
left_diff = ((rean_mean_rat1['var_1st'] / rean_mean_rat1['var_total'] * 100) -
             (sat_mean_rat1['var_1st']  / sat_mean_rat1['var_total']  * 100)).to_dataset(name='diff')

# : σ²_1st 
right_data = [
    sat_mean_rat1['var_1st'].to_dataset(name='rat'),
    rean_mean_rat1['var_1st'].to_dataset(name='rat'),
]
right_diff = (rean_mean_rat1['var_1st'] - sat_mean_rat1['var_1st']).to_dataset(name='diff')

# ──   ───────────────────────────────────────────
cmap_main = Colormap('colorcet:CET_L17').to_mpl()
cmap_main_copy = copy.copy(cmap_main)
cmap_main_copy.set_bad(color='gray')

# ──    ────────────────────────────────────────
def plot_map(ax, ds, var, cmap, norm,
             show_xlabel=False, show_ylabel=False, add_zeroline=False):
    crs = ccrs.PlateCarree()
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.9)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), edgecolor='none', facecolor='lightgray')
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.3)
    ax.set_extent(extent, crs=crs)
    ax.set_xticks([-180,-120,-60,0,60,120], crs=crs)
    ax.set_yticks([-60,-30,0,30,60], crs=crs)
    ax.xaxis.set_major_formatter(LongitudeFormatter())
    ax.yaxis.set_major_formatter(LatitudeFormatter())
    ax.tick_params(labelsize=8)
    #  :   (show_xlabel=True) 
    if not show_xlabel:
        ax.set_xticklabels([])
    #  :  (show_ylabel=True) 
    if not show_ylabel:
        ax.set_yticklabels([])
    ax.grid(True, linestyle='-.', alpha=0.2, color='gray', linewidth=1)
    pc = ax.pcolormesh(ds.longitude, ds.latitude, ds[var],
                       transform=crs, cmap=cmap, norm=norm)
    return pc

def plot_hist(ax, ds, var, xlim, xticks=None, show_legend=False):
    lat_min, lat_max = extent[2], extent[3]
    lat_bins = np.arange(lat_min, lat_max + 2, 2)
    lat_centers = (lat_bins[:-1] + lat_bins[1:]) / 2
    land_means, ocean_means = [], []
    land_stds, ocean_stds = [], []
    for i in range(len(lat_bins) - 1):
        lat_mask = (ds.latitude >= lat_bins[i]) & (ds.latitude < lat_bins[i+1])
        ds_lat = ds.where(lat_mask)
        ld = ds_lat.where(lsm)[var].values.flatten()
        od = ds_lat.where(~lsm)[var].values.flatten()
        ld = ld[~np.isnan(ld)]; od = od[~np.isnan(od)]
        land_means.append(np.mean(ld) if len(ld) > 0 else np.nan)
        land_stds.append(np.std(ld)  if len(ld) > 0 else np.nan)
        ocean_means.append(np.mean(od) if len(od) > 0 else np.nan)
        ocean_stds.append(np.std(od)  if len(od) > 0 else np.nan)
    lm = np.array(land_means); ls = np.array(land_stds)
    om = np.array(ocean_means); os = np.array(ocean_stds)
    vl = ~np.isnan(lm); vo = ~np.isnan(om)
    ax.plot(lm[vl], lat_centers[vl], color='brown', lw=1.5, label='Land')
    ax.fill_betweenx(lat_centers[vl], lm[vl]-ls[vl], lm[vl]+ls[vl], color='brown', alpha=0.3)
    ax.plot(om[vo], lat_centers[vo], color='blue',  lw=1.5, label='Ocean')
    ax.fill_betweenx(lat_centers[vo], om[vo]-os[vo], om[vo]+os[vo], color='blue',  alpha=0.3)
    ax.set_xlim(xlim)
    ax.set_ylim(lat_min, lat_max)
    ax.set_yticks(np.arange(lat_min, lat_max, 20))
    ax.tick_params(axis='y', labelleft=False)
    ax.tick_params(labelsize=8)
    ax.xaxis.get_offset_text().set_fontsize(8)  # ← 
    ax.yaxis.get_offset_text().set_fontsize(8)

    if xticks: ax.set_xticks(xticks)
    ax.grid(True, alpha=0.3)
    if show_legend:
        ax.legend(fontsize=8, frameon=False)

# ──   (b, c, d): σ²_1st/σ²_total ────────────────
norm_left = BoundaryNorm(np.linspace(0, 12, 12*3+5), cmap_main.N)
norm_diff_left = plt.Normalize(vmin=-10, vmax=10)
cmap_diff = plt.get_cmap('RdBu_r')
cmap_diff_copy = copy.copy(cmap_diff)
cmap_diff_copy.set_bad(color='gray')
cmap_right = cmaps.WhiteYellowOrangeRed
# b, c: O X / d: O O
pc_left_main = plot_map(ax_b, left_data[0], 'rat', cmaps.WhiteYellowOrangeRed, norm_left,
                        show_xlabel=False, show_ylabel=True)
plot_hist(ax_bh, left_data[0], 'rat', (0, 10), [0,5,10], show_legend=True)

pc_left_main2 = plot_map(ax_c, left_data[1], 'rat', cmaps.WhiteYellowOrangeRed, norm_left,
                         show_xlabel=False, show_ylabel=True)
plot_hist(ax_ch, left_data[1], 'rat', (0, 10), [0,5,10])

pc_left_diff = plot_map(ax_d, left_diff, 'diff', cmap_diff_copy, norm_diff_left,
                        show_xlabel=True, show_ylabel=True, add_zeroline=True)
plot_hist(ax_dh, left_diff, 'diff', (-8, 8), [-5,0,5])

# ──   (e, f, g): σ²_1st  ────────────────
norm_right = BoundaryNorm(np.linspace(0, 0.4, 33), cmap_main.N)
norm_diff_right = plt.Normalize(vmin=-0.2, vmax=0.2)

# e, f: X X / g: X O
pc_right_main = plot_map(ax_e, right_data[0], 'rat', cmap_main_copy, norm_right,
                         show_xlabel=False, show_ylabel=False)
plot_hist(ax_eh, right_data[0], 'rat', (0, 0.4), show_legend=True)
ax_eh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
# ax_eh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_eh.tick_params(labelsize=8)

pc_right_main2 = plot_map(ax_f, right_data[1], 'rat', cmap_main_copy, norm_right,
                          show_xlabel=False, show_ylabel=False)
plot_hist(ax_fh, right_data[1], 'rat', (0, 0.4))
ax_fh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_fh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_fh.tick_params(labelsize=8)

pc_right_diff = plot_map(ax_g, right_diff_smooth, 'diff', cmaps.fusion_r, norm_diff_right,
                         show_xlabel=True, show_ylabel=False, add_zeroline=True)
plot_hist(ax_gh, right_diff_smooth, 'diff', (-0.2, 0.2))
ax_gh.xaxis.set_major_formatter(ScalarFormatter(useMathText=True))
ax_gh.ticklabel_format(style='sci', scilimits=(0,0), useMathText=True)
ax_gh.tick_params(labelsize=8)
# ──   (    ) ────────────────
label_kwargs = dict(transform=ccrs.PlateCarree(),
                    fontsize=12,
                    ha='left', va='bottom', zorder=10,
                    bbox=dict(facecolor='white', edgecolor='none',
                              alpha=0.7, pad=1.5))
for ax, letter in [(ax_b, 'b'), (ax_c, 'c'), (ax_d, 'd'),
                   (ax_e, 'e'), (ax_f, 'f'), (ax_g, 'g')]:
    ax.text(-178, -57, f'({letter})', **label_kwargs)

# ──   (    ,  ) ──────────
# axes  : x=0  , y=0.5 
row_label_kwargs = dict(fontsize=14, fontweight='bold',
                        ha='center', va='center',
                        rotation=90, rotation_mode='anchor',
                        transform=ax_b.transAxes,
                        clip_on=False)
row_labels = [
    (ax_b, 'MS'),
    (ax_c, 'MM'),
    (ax_d, 'MM \u2212 MS'),
]
for ax, label in row_labels:
    ax.text(-0.1, 0.5, label,
            fontsize=15,
            ha='center', va='center',
            rotation=90, rotation_mode='anchor',
            transform=ax.transAxes,
            clip_on=False)

# ──  ────────────────────────────────────────────────
def add_colorbar(fig, pc, cax, ticks, label, extend='neither'):
    cb = fig.colorbar(pc, cax=cax, orientation='horizontal',
                      extend=extend, ticks=ticks)
    cb.set_label('')
    cb.ax.tick_params(labelsize=8,which='minor', length=0)
    cb.ax.text(1.07, 0.5, label,
               transform=cb.ax.transAxes,
               ha='left', va='center', fontsize=10)
    return cb

#    (b, c )
pos_c = ax_c.get_position()
cax_left_main = fig.add_axes([pos_c.x0+0.02, pos_c.y0-0.04, pos_c.width*0.80, 0.015])
cb1 = add_colorbar(fig, pc_left_main, cax_left_main,
                   ticks=np.linspace(0,15,6),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='max')

#  diff 
pos_d = ax_d.get_position()
cax_left_diff = fig.add_axes([pos_d.x0+0.02, pos_d.y0-0.04, pos_d.width*0.80, 0.015])
cb2 = add_colorbar(fig, pc_left_diff, cax_left_diff,
                   ticks=np.linspace(-10,10,5),
                   label=r'$\sigma_{1st}^2/\sigma_{total}^2$ [%]',
                   extend='both')

#    (e, f )
pos_f = ax_f.get_position()
cax_right_main = fig.add_axes([pos_f.x0+0.02, pos_f.y0-0.04, pos_f.width*0.80, 0.015])
cb3 = add_colorbar(fig, pc_right_main, cax_right_main,
                   ticks=np.linspace(0,0.4,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='max')

#  diff 
pos_g = ax_g.get_position()
cax_right_diff = fig.add_axes([pos_g.x0+0.02, pos_g.y0-0.04, pos_g.width*0.80, 0.015])
cb4 = add_colorbar(fig, pc_right_diff, cax_right_diff,
                   ticks=np.linspace(-0.2,0.2,5),
                   label=r'$\sigma_{1st}^2$ [mm$^2$ h$^{-2}$]',
                   extend='both')

# ──      ──────────────────
# cartopy      GridSpec   
# draw()        
fig.canvas.draw()

hist_width = 0.055
row1_shift = -0.03  # 1  (c  )
row3_shift = -0.02  # 3  (c  )

# ──  position  ──
positions = {}
for name, ax_map in [('b', ax_b), ('c', ax_c), ('d', ax_d),
                     ('e', ax_e), ('f', ax_f), ('g', ax_g)]:
    positions[name] = ax_map.get_position()

# ── 1, 3  ──
for name, ax_map in [('b', ax_b), ('e', ax_e)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row1_shift, pos.width, pos.height])

for name, ax_map in [('d', ax_d), ('g', ax_g)]:
    pos = positions[name]
    ax_map.set_position([pos.x0, pos.y0 + row3_shift, pos.width, pos.height])
# ──    ──────────────────────────────
fig.canvas.draw()

for ax_map, ax_hist in [
    (ax_b, ax_bh), (ax_c, ax_ch), (ax_d, ax_dh),
    (ax_e, ax_eh), (ax_f, ax_fh), (ax_g, ax_gh),
]:
    pos = ax_map.get_position()
    ax_hist.set_position([pos.x1, pos.y0, hist_width, pos.height])

# ──     position   ──
cbar_gap = 0.047

pos_c = ax_c.get_position()
cax_left_main.set_position([pos_c.x0+0.02, pos_c.y0-cbar_gap, pos_c.width*0.80, 0.018])

pos_d = ax_d.get_position()
cax_left_diff.set_position([pos_d.x0+0.02, pos_d.y0-cbar_gap, pos_d.width*0.80, 0.018])

pos_f = ax_f.get_position()
cax_right_main.set_position([pos_f.x0+0.02, pos_f.y0-cbar_gap, pos_f.width*0.80, 0.018])

pos_g = ax_g.get_position()
cax_right_diff.set_position([pos_g.x0+0.02, pos_g.y0-cbar_gap, pos_g.width*0.80, 0.018])

plt.savefig('${FIG_DIR}/pre_diurnal_cycle/new/paper_fig/fig4_maps.png', 
            dpi=400, bbox_inches='tight')
# plt.show()

# ARM map


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.transforms import offset_copy
import cartopy.crs as ccrs
import cartopy.io.img_tiles as cimgt
import cartopy.feature as cfeature
# proj = ccrs.Geodetic()

proj =  ccrs.PlateCarree()
def main():
    # ARM SGP C1 Lamont coordinates
    lamont_lat = 36.607322
    lamont_lon = -97.487643
    
    # Try different high-resolution tile services
    tile_options = [
        # Option 1: Google Satellite with higher zoom
        cimgt.GoogleTiles(style="satellite"),
        # Option 2: ESRI World Imagery (often higher quality)
        cimgt.GoogleTiles(url='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}.jpg'),
        # Option 3: Bing Aerial
        cimgt.GoogleTiles(url='https://ecn.t3.tiles.virtualearth.net/tiles/a{q}.jpeg?g=1'),
    ]
    
    fig = plt.figure(figsize=(4, 3), dpi=300)
    
    # Try to use the best available tile service
    for i, tiles in enumerate(tile_options):
        try:
            # Create a GeoAxes in the tile's projection
            ax = fig.add_subplot(1, 1, 1, projection=tiles.crs)
            
            # Set extent for CONUS
            ax.set_extent([-125, -66.5, 25, 48], crs=proj)
            
            # Use higher zoom level for better resolution
            # Zoom levels: 4=low, 6=medium, 8=high, 10=very high
            zoom_level = 8  # Much higher resolution
            ax.add_image(tiles, zoom_level)
            
            print(f"Successfully loaded tile service {i+1} with zoom level {zoom_level}")
            break
            
        except Exception as e:
            print(f"Tile service {i+1} failed: {e}")
            if i == len(tile_options) - 1:
                # Fallback to standard Google with maximum zoom
                ax = fig.add_subplot(1, 1, 1, projection=cimgt.GoogleTiles().crs)
                ax.set_extent([-125, -66.5, 25,45], crs=proj)
                ax.add_image(cimgt.GoogleTiles(), 10)  # Maximum zoom
            continue
    
    # # Add minimal overlays to preserve high-resolution imagery
    # ax.add_feature(cfeature.COASTLINE, color='white', linewidth=0.5, alpha=0.9)
    # ax.add_feature(cfeature.BORDERS, color='white', linewidth=0.8, alpha=0.95)
    
    # Add state boundaries with very thin lines
    states = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_1_states_provinces_lines',
        scale='10m',  # Highest resolution available
        facecolor='none',
        edgecolor='white',
        linewidth=0.2,
        alpha=0.8
    )
    ax.add_feature(states)
    
    # Add marker for ARM SGP C1 Lamont with enhanced visibility
    ax.plot(lamont_lon, lamont_lat, marker='o', color='red', markersize=5,
            alpha=1.0, markeredgecolor='white', markeredgewidth=1,
            transform=ccrs.Geodetic(), zorder=10)
        
    # Enhanced text positioning for high-resolution display
    geodetic_transform = proj._as_mpl_transform(ax)
    text_transform = offset_copy(geodetic_transform, units='dots', x=30, y=15)
    
    # Add main label
    ax.text(lamont_lon, lamont_lat, 'ARM SGP C1\nLamont, Oklahoma',
            verticalalignment='bottom', horizontalalignment='left',
            transform=text_transform, fontsize=8,
            bbox=dict(facecolor='white', alpha=0.95, boxstyle='round,pad=0.4',
                     edgecolor='black', linewidth=.7))
    
    # Add detailed coordinate information
    coord_transform = offset_copy(geodetic_transform, units='dots', x=-35, y=-25)
    coord_text = f'{lamont_lat:.4f}°N\n{abs(lamont_lon):.4f}°W'
    ax.text(lamont_lon, lamont_lat, coord_text,
            verticalalignment='top', horizontalalignment='right',
            transform=coord_transform, fontsize=5,
            bbox=dict(facecolor='lightcyan', alpha=0.9, boxstyle='round,pad=0.3',
                     edgecolor='navy', linewidth=.7))
        
    # High-quality gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=False, 
                      linewidth=0., color='white', alpha=0.8, linestyle=':')
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 6, 'color': 'black',
                       'bbox': dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.9)}
    gl.ylabel_style = {'size': 6, 'color': 'black',
                       'bbox': dict(boxstyle='round,pad=0.1', facecolor='white', alpha=0.9)}
    
    plt.tight_layout()
    plt.show()
    # plt.savefig('${FIG_DIR}/pre_diurnal_cycle/armsite.png', dpi=400, bbox_inches='tight')

if __name__ == '__main__':
    main()

#


In [ ]:
ds['DOMAINMASK'].values = new_mask

In [ ]:
ds['DOMAINMASK'].dims

In [ ]:
ds['DOMAINMASK'].plot()

In [ ]:
ds['LANDMASK'].plot()

In [ ]:
with ProgressBar():
    noahmp_test['SoilMoist_tavg'][0][0].plot()

In [ ]:
noahmp_test = xr.open_mfdataset(sorted(glob('${HOME_DIR}/LISF-7.5.2-public/nldas_smap/OL_OUTPUT/SURFACEMODEL/201703/LIS_HIST_*.nc')))

In [ ]:
noahmp_test.close()
del(noahmp_test)

In [ ]:
import numpy as np

target_lat = 36.8
target_lon = -97.5
target_lat1 = 40.0
target_lon1 = -105.5
# time   0 ( ) 
ilat = np.argmin(np.abs(noahmp_test['lat'][0, :].values - target_lat))
ilon = np.argmin(np.abs(noahmp_test['lon'][0, :].values - target_lon))

ilat1 = np.argmin(np.abs(noahmp_test['lat'][0, :].values - target_lat1))
ilon1 = np.argmin(np.abs(noahmp_test['lon'][0, :].values - target_lon1))

In [ ]:
sm_point = noahmp_test['SoilMoist_tavg'][:,0, ilat, ilon]
sm_point1 = noahmp_test['SoilMoist_tavg'][:,0, ilat1, ilon1]

In [ ]:
sm_point = noahmp_test['LAI_inst'][:, ilat, ilon]
sm_point1 = noahmp_test['LAI_inst'][:, ilat1, ilon1]

In [ ]:
sm_point.plot(c = 'r')
sm_point1.plot(c = 'blue')

In [ ]:
sm_point.plot(c = 'r')
sm_point1.plot(c = 'blue')

In [ ]:
noahmp_test['Swnet_tavg'][0].plot()

In [ ]:
noahmp_test['LAI_inst']